# Wikidata Multihop Benchmark Builder

This notebook builds a Russian multi-criteria multihop QA benchmark for testing LLMs. It queries Wikidata, constructs domain-specific questions, collects gold answer QIDs and Russian labels, and appends validated examples to the main JSONL dataset.

The notebook is organized as a reusable pipeline: shared Wikidata utilities first, domain generators in the middle, and the final append-only dataset assembly at the end. The final generation block keeps resume mode enabled so existing records are not overwritten.

## Contents

1. [Environment setup](#environment-setup) — Install the small set of Python dependencies used by the notebook.
2. [Imports](#imports) — Core Python imports and third-party libraries.
3. [Global configuration](#global-configuration) — Central paths, WDQS settings, retry limits, cache locations, and output dataset paths.
4. [Wikidata client and cache](#wikidata-client-and-cache) — Disk cache, WDQS request wrapper, retry logic, and basic result handling.
5. [Entity parsing helpers](#entity-parsing-helpers) — Helpers for QIDs, URI parsing, Russian labels, aliases, and sitelinks.
6. [Dataset schema](#dataset-schema) — BenchmarkExample schema and append-only JSONL writing utilities.
7. [Pool persistence helpers](#pool-persistence-helpers) — Pool loading, saving, cache reuse, and safe numeric conversion helpers.
8. [Core pool builders](#core-pool-builders) — Shared SPARQL builders, robust WDQS helpers, and reusable generation primitives.
9. [Cinema domain](#cinema-domain) — Film and series templates with genre, country, year, rating, franchise, actor-bridge, and label filters.
10. [Russian geography domain](#russian-geography-domain) — Stable local templates for Russian geographic entities.
11. [International geography domain](#international-geography-domain) — Global geographic templates for oceans, seas, lakes, waterfalls, deserts, mountains, rivers, and islands.
12. [Books domain](#books-domain) — Book templates with genre, language, time, setting, adaptation, and author-related constraints.
13. [Video games domain](#video-games-domain) — Cached video-game pools and bounded templates over platform, genre, developer, publisher, and release year.
14. [Music albums domain](#music-albums-domain) — Album templates with artist, genre, language or country, and release-period constraints.
15. [Software domain](#software-domain) — Software templates using class, developer, platform, license, programming language, and release period.
16. [People domain](#people-domain) — Fast people templates based on occupation, citizenship, and birth-year anchors.
17. [Scientists and mathematicians domains](#scientists-and-mathematicians-domains) — Specialized people templates with occupation chains and award-style constraints.
18. [Paintings and museums domains](#paintings-and-museums-domains) — Artwork and museum templates with collection, author, location, and stable multihop refinements.
19. [Spacecraft domain](#spacecraft-domain) — Spacecraft templates with manufacturer, operator, program, launch period, and hard L5 bridge patterns.
20. [Countries domain](#countries-domain) — Country templates over geography, region, currency, language, population, and historical/current state filters.
21. [Dishes domain](#dishes-domain) — Dish templates with cuisine, ingredient inclusion/exclusion, country, and category constraints.
22. [Smartphones domain](#smartphones-domain) — Smartphone templates with manufacturer, operating system, release year, screen, and CPU-style constraints.
23. [Universities, airports, and cars domains](#universities-airports-and-cars-domains) — Domain-specific templates for education, airport, and car-model entities.
24. [Cinema Kinopoisk extension](#cinema-kinopoisk-extension) — Additional L5 templates focused on Kinopoisk-related constraints.
25. [Final dataset assembly](#final-dataset-assembly) — Append-only generation loop, strict Russian-label postprocessing, deduplication, progress reporting, and JSONL writing.
26. [Summary](#summary) — Short description of the produced dataset and the intended next validation step.


<a id="environment-setup"></a>

## 1. Environment setup

Install the small set of Python dependencies used by the notebook.

In [1]:
!pip -q install requests pandas tqdm rapidfuzz pymorphy2 pymorphy2-dicts-ru



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


<a id="imports"></a>

## 2. Imports

Core Python imports and third-party libraries.

In [2]:
import os
import re
import json
import time
import random
import hashlib
import datetime as dt
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional, Tuple

import requests
import pandas as pd
from tqdm.auto import tqdm


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<a id="global-configuration"></a>

## 3. Global configuration

Central paths, WDQS settings, retry limits, cache locations, and output dataset paths.

In [3]:
WDQS_ENDPOINT = "https://query.wikidata.org/sparql"
WIKI_API = "https://www.wikidata.org/w/api.php"

USER_AGENT = "YandexGPT-reversal-curse-benchmark (contact: salam121asd@gmail.com)"

MIN_SECONDS_BETWEEN_REQUESTS = 1.1
MAX_RETRIES = 4
TIMEOUT_SECONDS = 30

MAX_BACKOFF_SECONDS = 20
POOL_FAIL_FAST = True
POOL_MAX_RETRIES = 4
POOL_TIMEOUT_SECONDS = 30
POOL_MAX_BACKOFF_SECONDS = 20

POOL_BUILD_DELAY_S = 4.0 
DOMAIN_PAUSE_S = 2.0     

OUT_DIR = "out_wikidata_benchmark"
CACHE_DIR = os.path.join(OUT_DIR, "cache")
POOLS_DIR = os.path.join(OUT_DIR, "pools")
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(POOLS_DIR, exist_ok=True)

DATASET_PATH = os.path.join(OUT_DIR, "dataset_ru_multicriteria.jsonl")

# === OUTPUT DATASET PATHS ===
DATASET_PATH_MAIN = os.path.join(OUT_DIR, "dataset_main.jsonl")
DATASET_PATH_ZERO = os.path.join(OUT_DIR, "dataset_zero.jsonl")

ZERO_TARGET_PER_BUCKET = {
    "default": {"L1": 0, "L2": 0, "L3": 0, "L4": 0, "L5": 0},
    "cinema":  {"L1": 0, "L2": 0, "L3": 0, "L4": 0, "L5": 0},
}

# === Debug ===
DEBUG_GENERATOR_ERRORS = False


<a id="wikidata-client-and-cache"></a>

## 4. Wikidata client and cache

Disk cache, WDQS request wrapper, retry logic, and basic result handling.

In [4]:
class DiskCache:
    def __init__(self, cache_dir: str):
        self.cache_dir = cache_dir

    def _path(self, key: str) -> str:
        return os.path.join(self.cache_dir, f"{key}.json")

    def get(self, key: str) -> Optional[dict]:
        path = self._path(key)
        if os.path.exists(path):
            try:
                with open(path, "r", encoding="utf-8") as f:
                    return json.load(f)
            except json.JSONDecodeError:
                try:
                    os.rename(path, path + ".bad")
                except Exception:
                    pass
        return None

    def set(self, key: str, value: dict) -> None:
        path = self._path(key)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(value, f, ensure_ascii=False)


def _sha1(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8")).hexdigest()


class WikidataClient:
    def __init__(
        self,
        endpoint: str = WDQS_ENDPOINT,
        user_agent: str = USER_AGENT,
        min_delay: float = MIN_SECONDS_BETWEEN_REQUESTS,
        timeout: int = TIMEOUT_SECONDS,
        max_retries: int = MAX_RETRIES,
        cache: Optional[DiskCache] = None,
    ):
        self.endpoint = endpoint
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": user_agent,
            "Accept": "application/sparql-results+json"
        })
        self.min_delay = min_delay
        self.timeout = timeout
        self.max_retries = max_retries
        self.cache = cache or DiskCache(CACHE_DIR)
        self._last_request_ts = 0.0

    def _sleep_if_needed(self):
        now = time.time()
        delta = now - self._last_request_ts
        if delta < self.min_delay:
            time.sleep(self.min_delay - delta)
    def _sleep_backoff(self, attempt: int, resp: Optional[requests.Response] = None) -> None:
        """Экспоненциальный backoff с учётом Retry-After (если есть)."""
        ra = None
        if resp is not None:
            ra = resp.headers.get("Retry-After")
        if ra:
            try:
                sec = float(str(ra).strip())
                time.sleep(sec + random.random())
                return
            except Exception:
                pass
        base = min(MAX_BACKOFF_SECONDS, (2 ** attempt))
        time.sleep(base + random.random())


    def sparql_select(self, query: str, use_cache: bool = True) -> dict:
        q = query.strip()
        key = _sha1("SELECT:" + q)
        if use_cache:
            cached = self.cache.get(key)
            if cached is not None:
                return cached

        last_err: Optional[Exception] = None

        for attempt in range(self.max_retries):
            self._sleep_if_needed()
            resp = None
            try:
                resp = self.session.post(
                    self.endpoint,
                    params={"format": "json"},
                    data={"query": q},
                    headers={"Accept": "application/sparql-results+json"},
                    timeout=self.timeout,
                )
                self._last_request_ts = time.time()

                if resp.status_code in (429, 500, 502, 503, 504):
                    last_err = RuntimeError(f"WDQS transient HTTP {resp.status_code}")
                    self._sleep_backoff(attempt, resp)
                    continue

                resp.raise_for_status()

                text = resp.text or ""
                t = text.lstrip()

                if not t.startswith("{"):
                    bad_dir = os.path.join(OUT_DIR, "bad_responses")
                    os.makedirs(bad_dir, exist_ok=True)
                    with open(os.path.join(bad_dir, f"{key}.nonjson.txt"), "w", encoding="utf-8", errors="replace") as bf:
                        bf.write(text[:250_000])
                    last_err = ValueError("Non-JSON WDQS response")
                    self._sleep_backoff(attempt, resp)
                    continue

                if not text.rstrip().endswith("}"):
                    bad_dir = os.path.join(OUT_DIR, "bad_responses")
                    os.makedirs(bad_dir, exist_ok=True)
                    with open(os.path.join(bad_dir, f"{key}.truncated.txt"), "w", encoding="utf-8", errors="replace") as bf:
                        bf.write(text[:250_000])
                    last_err = ValueError("Truncated WDQS JSON")
                    self._sleep_backoff(attempt, resp)
                    continue

                try:
                    data = json.loads(text)
                except json.JSONDecodeError:
                    data = json.loads(text, strict=False)

                if not isinstance(data, dict) or "results" not in data:
                    bad_dir = os.path.join(OUT_DIR, "bad_responses")
                    os.makedirs(bad_dir, exist_ok=True)
                    with open(os.path.join(bad_dir, f"{key}.weird.json"), "w", encoding="utf-8", errors="replace") as bf:
                        bf.write(text[:250_000])
                    last_err = ValueError("Unexpected WDQS JSON format")
                    self._sleep_backoff(attempt, resp)
                    continue

                if use_cache:
                    self.cache.set(key, data)
                return data

            except (requests.RequestException, ValueError, json.JSONDecodeError) as e:
                last_err = e
                if attempt == self.max_retries - 1:
                    raise RuntimeError(f"WDQS failed after {self.max_retries} retries: {e}") from e
                self._sleep_backoff(attempt, resp)

        raise RuntimeError(f"WDQS failed: {last_err}")


    def api_search_entity(self, text: str, language: str = "ru", limit: int = 10) -> dict:
        params = {
            "action": "wbsearchentities",
            "search": text,
            "language": language,
            "format": "json",
            "limit": limit,
        }

        last_err: Optional[Exception] = None

        for attempt in range(self.max_retries):
            self._sleep_if_needed()
            resp = None
            try:
                resp = self.session.get(WIKI_API, params=params, timeout=self.timeout)
                self._last_request_ts = time.time()

                if resp.status_code in (429, 500, 502, 503, 504):
                    last_err = RuntimeError(f"Wikidata API transient HTTP {resp.status_code}")
                    self._sleep_backoff(attempt, resp)
                    continue

                resp.raise_for_status()
                try:
                    return resp.json()
                except json.JSONDecodeError:
                    return json.loads(resp.text or "", strict=False)

            except (requests.RequestException, ValueError, json.JSONDecodeError) as e:
                last_err = e
                if attempt == self.max_retries - 1:
                    raise RuntimeError(f"Wikidata API search failed after {self.max_retries} retries: {e}") from e
                self._sleep_backoff(attempt, resp)

        raise RuntimeError(f"Unreachable: {last_err}")


wd = WikidataClient()


wd = WikidataClient()


<a id="entity-parsing-helpers"></a>

## 5. Entity parsing helpers

Helpers for QIDs, URI parsing, Russian labels, aliases, and sitelinks.

In [5]:
QID_RE = re.compile(r"Q\d+")

def uri_to_qid(uri: str) -> Optional[str]:
    if not uri:
        return None
    m = QID_RE.search(uri)
    return m.group(0) if m else None

def rows_from_select(data: dict) -> List[Dict[str, str]]:
    bindings = data.get("results", {}).get("bindings", [])
    rows = []
    for b in bindings:
        row = {}
        for k, v in b.items():
            row[k] = v.get("value")
        rows.append(row)
    return rows

def resolve_qid(text: str, lang: str = "ru") -> Optional[str]:
    res = wd.api_search_entity(text, language=lang, limit=5)
    if "search" not in res or not res["search"]:
        return None
    return res["search"][0]["id"]

def ensure_qid(label: str, fallback_qid: Optional[str] = None) -> str:
    qid = resolve_qid(label, "ru") or resolve_qid(label, "en") or fallback_qid
    if not qid:
        raise ValueError(f"Cannot resolve QID for '{label}'")
    return qid


<a id="dataset-schema"></a>

## 6. Dataset schema

BenchmarkExample schema and append-only JSONL writing utilities.

In [6]:
@dataclass
class BenchmarkExample:
    id: str
    domain: str                 # e.g. cinema | geo_ru | books | videogames | music_albums | people
    complexity: str             # L1...L5
    query_text_ru: str
    constraints: Dict[str, Any]
    requested_count: int
    gold_answer_qids: List[str]
    gold_answer_labels_ru: List[str]
    sparql_query: str
    created_at: str


    is_advanced: bool = False
    template_id: Optional[str] = None
    template_family: str = "default"
    gold_truncated: bool = False                # True if the gold set is limited by LIMIT
    ask_validator_sparql: Optional[str] = None  # ASK template for validating one candidate

def save_jsonl(examples: List[BenchmarkExample], path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for ex in examples:
            f.write(json.dumps(asdict(ex), ensure_ascii=False) + "\n")


<a id="pool-persistence-helpers"></a>

## 7. Pool persistence helpers

Pool loading, saving, cache reuse, and safe numeric conversion helpers.

In [7]:
from pathlib import Path

def utc_now_z() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat().replace("+00:00", "Z")

def _pool_path(name: str) -> str:
    safe = re.sub(r"[^a-zA-Z0-9_\-\.]+", "_", name)
    return os.path.join(POOLS_DIR, f"{safe}.json")

def load_pool_df(name: str) -> Optional[pd.DataFrame]:
    path = _pool_path(name)
    if os.path.exists(path):
        try:
            return pd.read_json(path, lines=False)
        except ValueError:
            try:
                os.rename(path, path + ".bad")
            except Exception:
                pass
            return None
    return None


def save_pool_df(name: str, df: pd.DataFrame) -> None:
    path = _pool_path(name)
    tmp = path + ".tmp"
    df.to_json(tmp, force_ascii=False, orient="records")
    os.replace(tmp, path)


def load_or_build_pool(name: str, builder):
    """Загрузить пул из диска или построить.

    Важно: построение пулов — самый тяжёлый этап (WDQS).
    Поэтому здесь есть 'fail-fast' режим: на время построения пула
    снижаем max_retries/timeout/backoff, чтобы не висеть минутами.
    """
    df = load_pool_df(name)
    if df is not None and len(df) > 0:
        return df

    wd_obj = globals().get("wd", None)
    orig_retries = getattr(wd_obj, "max_retries", None) if wd_obj is not None else None
    orig_timeout = getattr(wd_obj, "timeout", None) if wd_obj is not None else None
    orig_backoff = globals().get("MAX_BACKOFF_SECONDS", None)

    try:
        if globals().get("POOL_FAIL_FAST", True) and wd_obj is not None:
            try:
                wd_obj.max_retries = int(globals().get("POOL_MAX_RETRIES", 2))
                wd_obj.timeout = int(globals().get("POOL_TIMEOUT_SECONDS", 25))
                if orig_backoff is not None:
                    globals()["MAX_BACKOFF_SECONDS"] = int(globals().get("POOL_MAX_BACKOFF_SECONDS", 6))
            except Exception:
                pass

        df = builder()

    except Exception as e:
        print(f"[WARN] pool '{name}' build failed: {e}")
        return pd.DataFrame()

    finally:
        if wd_obj is not None:
            if orig_retries is not None:
                wd_obj.max_retries = orig_retries
            if orig_timeout is not None:
                wd_obj.timeout = orig_timeout
        if orig_backoff is not None:
            globals()["MAX_BACKOFF_SECONDS"] = orig_backoff

    try:
        save_pool_df(name, df)
    except Exception as e:
        print(f"[WARN] pool '{name}' save failed: {e}")

    if 'POOL_BUILD_DELAY_S' in globals() and POOL_BUILD_DELAY_S and POOL_BUILD_DELAY_S > 0:
        time.sleep(float(POOL_BUILD_DELAY_S))
    return df


def safe_int(x, default: int = 0) -> int:
    try:
        return int(x)
    except Exception:
        try:
            return int(float(x))
        except Exception:
            return default

qid_from_uri = uri_to_qid


<a id="core-pool-builders"></a>

## 8. Core pool builders

Shared SPARQL builders, robust WDQS helpers, and reusable generation primitives.

In [8]:
def build_value_pool_ru(
    item_class_qid: str,
    pid: str,
    val_name: str,
    limit: int = 400,
) -> pd.DataFrame:
    """
    Лёгкий пул значений свойства pid, которые реально встречаются у item_class_qid.
    Без COUNT/GROUP BY (так меньше таймаутов). Возвращает QID + RU-лейбл.
    """
    sparql = f"""
    SELECT DISTINCT ?val ?valLabelRu WHERE {{
      ?item wdt:P31/wdt:P279* wd:{item_class_qid} ;
            wdt:{pid} ?val .
      ?val rdfs:label ?valLabelRu FILTER(LANG(?valLabelRu) = "ru") .
    }}
    LIMIT {int(limit)}
    """
    rows = rows_from_select(wd.sparql_select(sparql))
    data = []
    for r in rows:
        qid = uri_to_qid(r.get("val", ""))
        lbl = r.get("valLabelRu")
        if qid and lbl:
            data.append({f"{val_name}_qid": qid, f"{val_name}LabelRu": lbl})
    return pd.DataFrame(data).drop_duplicates().reset_index(drop=True)

def select_items_with_ru_label(
    item_class_qid: str,
    where_lines: List[str],
    limit: int = 300,
    item_var: str = "item",
) -> Tuple[str, List[Tuple[str,str]]]:
    """
    Выполняет SELECT для items и возвращает (sparql, [(qid, ru_label), ...]).
    """
    where = "\n      ".join(where_lines)
    sparql = f"""
    SELECT DISTINCT ?{item_var} ?{item_var}LabelRu WHERE {{
      ?{item_var} wdt:P31/wdt:P279* wd:{item_class_qid} .
      {where}
      ?{item_var} rdfs:label ?{item_var}LabelRu FILTER(LANG(?{item_var}LabelRu) = "ru") .
    }}
    LIMIT {int(limit)}
    """
    rows = rows_from_select(wd.sparql_select(sparql))
    items = []
    for r in rows:
        qid = uri_to_qid(r.get(item_var, ""))
        lbl = r.get(f"{item_var}LabelRu")
        if qid and lbl:
            items.append((qid, lbl))
    return sparql.strip(), items

def build_ask_validator(item_class_qid: str, where_lines: List[str], item_var: str = "item") -> str:
    """
    Возвращает ASK запрос, где item подставляется как wd:{{ITEM}}.
    Так можно валидировать ответы модели честно, даже если gold был LIMIT-нут.
    """
    where = "\n      ".join(where_lines)
    sparql = f"""
    ASK WHERE {{
      BIND(wd:{{ITEM}} AS ?{item_var})
      ?{item_var} wdt:P31/wdt:P279* wd:{item_class_qid} .
      {where}
    }}
    """
    return sparql.strip()

def pick_from_df(df: pd.DataFrame, qid_col: str, label_col: str, rng: random.Random):
    row = df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
    return row[qid_col], row[label_col]

def build_value_pool_ru_direct(
    item_class_qid: str,
    pid: str,
    val_name: str,
    limit: int = 200,
) -> pd.DataFrame:
    """
    Ещё более лёгкий пул: только прямые instance of (P31=class), без P279*.
    Иногда это существенно снижает нагрузку и уменьшает 429/обрывы ответа.
    """
    sparql = f"""
    SELECT DISTINCT ?val ?valLabelRu WHERE {{
      ?item wdt:P31 wd:{item_class_qid} ;
            wdt:{pid} ?val .
      ?val rdfs:label ?valLabelRu FILTER(LANG(?valLabelRu) = "ru") .
    }}
    LIMIT {int(limit)}
    """
    rows = rows_from_select(wd.sparql_select(sparql))
    data = []
    for r in rows:
        qid = uri_to_qid(r.get("val"))
        lab = r.get("valLabelRu")
        if qid and lab:
            data.append({f"{val_name}_qid": qid, f"{val_name}LabelRu": lab, "n": 1})
    return pd.DataFrame(data).drop_duplicates()

def pool_from_anchor_labels(
    labels: List[str],
    qid_col: str,
    label_col: str,
    fallback_qids: Optional[Dict[str, str]] = None,
) -> pd.DataFrame:
    """Строит маленький пул через API-поиск QID по строкам (без WDQS)."""
    rows = []
    for lab in labels:
        fb = (fallback_qids or {}).get(lab)
        try:
            qid = ensure_qid(lab, fallback_qid=fb)
        except Exception:
            continue
        rows.append({qid_col: qid, label_col: lab, "n": 1})
    return pd.DataFrame(rows).drop_duplicates()


def build_class_entities_pool_ru(
    class_qid: str,
    val_name: str,
    limit: int = 300,
    subclasses: bool = True,
) -> pd.DataFrame:
    """
    Возвращает пул сущностей, которые являются instance of (или subclass chain) class_qid.
    Пример: class_qid="Q9143" (programming language) -> список языков.
    """
    path = "wdt:P31/wdt:P279* " if subclasses else "wdt:P31 "
    sparql = f"""
    SELECT DISTINCT ?val ?valLabelRu WHERE {{
      ?val {path}wd:{class_qid} .
      ?val rdfs:label ?valLabelRu FILTER(LANG(?valLabelRu) = "ru") .
    }}
    LIMIT {int(limit)}
    """
    rows = rows_from_select(wd.sparql_select(sparql))
    data = []
    for r in rows:
        qid = uri_to_qid(r.get("val", ""))
        lab = r.get("valLabelRu")
        if qid and lab:
            data.append({f"{val_name}_qid": qid, f"{val_name}LabelRu": lab, "n": 1})
    return pd.DataFrame(data).drop_duplicates()

def select_items_with_ru_label_direct(
    item_class_qid: str,
    where_lines: List[str],
    limit: int = 300,
    item_var: str = "item",
) -> Tuple[str, List[Tuple[str, str]]]:
    """
    Как select_items_with_ru_label, но БЕЗ /wdt:P279* (только прямой P31=class).
    Для широких классов (типа "software") это может резко снизить нагрузку.
    """
    where = "\n      ".join(where_lines)
    sparql = f"""
    SELECT DISTINCT ?{item_var} ?{item_var}LabelRu WHERE {{
      ?{item_var} wdt:P31 wd:{item_class_qid} .
      {where}
      ?{item_var} rdfs:label ?{item_var}LabelRu FILTER(LANG(?{item_var}LabelRu) = "ru") .
    }}
    LIMIT {int(limit)}
    """
    rows = rows_from_select(wd.sparql_select(sparql))
    items = []
    for r in rows:
        qid = uri_to_qid(r.get(item_var, ""))
        lbl = r.get(f"{item_var}LabelRu")
        if qid and lbl:
            items.append((qid, lbl))
    return sparql.strip(), items

import os, time, random, json, requests
from typing import Optional

class WDQSTransientError(RuntimeError):
    """Transient WDQS failure (5xx/429/timeouts). Safe to retry/skip."""
    pass

_TRANSIENT_SUBSTRINGS = [
    "transient HTTP 429",
    "transient HTTP 500",
    "transient HTTP 502",
    "transient HTTP 503",
    "transient HTTP 504",
    "Read timed out",
    "timed out",
    "Response ended prematurely",
    "Truncated JSON",
    "Non-JSON response",
    "RemoteDisconnected",
    "Connection aborted",
]

def is_transient_wdqs_error(err: Exception) -> bool:
    s = str(err)
    if any(t in s for t in _TRANSIENT_SUBSTRINGS):
        return True
    # requests.* errors are almost always transient for WDQS
    if isinstance(err, requests.RequestException):
        return True
    if isinstance(err, json.JSONDecodeError):
        return True
    return False


OUT_DIR = globals().get("OUT_DIR", "out_wikidata_benchmark")

def _looks_like_complete_wdqs_json(text: str) -> bool:
    t = (text or "").strip()
    if not (t.startswith("{") and t.endswith("}")):
        return False
    return ('"results"' in t and '"bindings"' in t and '"head"' in t)

def _looks_like_html_or_text(text: str) -> bool:
    t = (text or "").lstrip().lower()
    return t.startswith("<!doctype") or t.startswith("<html") or t.startswith("<head") or t.startswith("<body")

def _dump_bad_response(key: str, text: str, suffix: str = "txt") -> None:
    bad_dir = os.path.join(OUT_DIR, "bad_responses")
    os.makedirs(bad_dir, exist_ok=True)
    path = os.path.join(bad_dir, f"{key}.{suffix}")
    max_chars = 250_000
    with open(path, "w", encoding="utf-8", errors="replace") as f:
        f.write((text or "")[:max_chars])

def _sleep_backoff(attempt: int, resp: Optional[requests.Response] = None) -> None:
    ra = None
    if resp is not None:
        ra = resp.headers.get("Retry-After")
    if ra and ra.strip().isdigit():
        time.sleep(float(ra.strip()))
        return
    time.sleep((2 ** attempt) + random.random())

def sparql_select_robust(self, query: str, use_cache: bool = True) -> dict:
    q = query.strip()
    key = _sha1("SELECT:" + q)

    if use_cache:
        cached = self.cache.get(key)
        if cached is not None:
            return cached

    last_err = None
    last_text = None
    last_resp = None

    for attempt in range(self.max_retries):
        self._sleep_if_needed()
        try:
            headers = {"Accept": "application/sparql-results+json", "Connection": "close", "Accept-Encoding": "identity"}
            resp = self.session.post(
                self.endpoint,
                params={"format": "json"},
                data={"query": q},
                headers=headers,
                timeout=self.timeout,
            )
            self._last_request_ts = time.time()
            last_resp = resp

            if resp.status_code in (429, 500, 502, 503, 504):
                last_err = RuntimeError(f"WDQS transient HTTP {resp.status_code}")
                last_text = resp.text
                _sleep_backoff(attempt, resp)
                continue

            resp.raise_for_status()
            text = resp.text
            last_text = text

            if _looks_like_html_or_text(text) or not text.strip().startswith("{"):
                last_err = ValueError(f"Non-JSON response, ct={resp.headers.get('Content-Type')}, http={resp.status_code}")
                _dump_bad_response(key, text, "nonjson.txt")
                _sleep_backoff(attempt, resp)
                continue

            if not text.strip().endswith("}"):
                last_err = json.JSONDecodeError("Truncated JSON (no closing brace)", text, len(text))
                _dump_bad_response(key, text, "truncated.txt")
                _sleep_backoff(attempt, resp)
                continue

            try:
                data = json.loads(text)
            except json.JSONDecodeError:
                data = json.loads(text, strict=False)

            if not isinstance(data, dict) or "results" not in data:
                last_err = ValueError("Parsed JSON but not WDQS results format")
                _dump_bad_response(key, text, "weird.json")
                _sleep_backoff(attempt, resp)
                continue

            if use_cache:
                self.cache.set(key, data)
            return data

        except (requests.RequestException, ValueError, json.JSONDecodeError) as e:
            last_err = e
            if last_text:
                _dump_bad_response(key, last_text, "bad.txt")
            if attempt == self.max_retries - 1:
                raise (WDQSTransientError(f"WDQS transient failure after {self.max_retries} retries: {e}") if is_transient_wdqs_error(e) else RuntimeError(f"WDQS failed after {self.max_retries} retries: {e}")) from e
            _sleep_backoff(attempt, last_resp)

    raise (WDQSTransientError(f"WDQS transient failure: {last_err}") if (last_err is not None and is_transient_wdqs_error(last_err)) else RuntimeError(f"WDQS failed: {last_err}"))

WikidataClient.sparql_select = sparql_select_robust


def load_or_build_pool_safe(name: str, builder):
    df = load_pool_df(name)
    if df is not None and len(df) > 0:
        return df
    try:
        df = builder()
    except Exception as e:
        print(f"[WARN] pool '{name}' build failed: {e}")
        return pd.DataFrame()
    try:
        save_pool_df(name, df)
    except Exception as e:
        print(f"[WARN] pool '{name}' save failed: {e}")
    return df

load_or_build_pool = load_or_build_pool_safe

print("✅ Patched: WikidataClient.sparql_select (robust) + load_or_build_pool (safe)")


def select_items_with_label_ru_en(
    item_class_qid: str,
    where_lines: List[str],
    limit: int = 300,
    item_var: str = "item",
    use_subclass_closure: bool = True,
) -> Tuple[str, List[Tuple[str,str]]]:
    """
    Выполняет SELECT для items и возвращает (sparql, [(qid, label_ru_or_en), ...]).
    Важно: НЕ требуем rdfs:label LANG='ru' — иначе много нулевых gold на доменах без RU-лейблов.
    """
    where = "\n      ".join(where_lines)
    class_line = (
        f"?{item_var} wdt:P31/wdt:P279* wd:{item_class_qid} ."
        if use_subclass_closure
        else f"?{item_var} wdt:P31 wd:{item_class_qid} ."
    )
    sparql = f"""
    SELECT DISTINCT ?{item_var} ?{item_var}Label WHERE {{
      {class_line}
      {where}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
    }}
    LIMIT {int(limit)}
    """
    rows = rows_from_select(wd.sparql_select(sparql))
    items: List[Tuple[str,str]] = []
    for r in rows:
        qid = uri_to_qid(r.get(item_var, ""))
        lbl = r.get(f"{item_var}Label")
        if qid and lbl:
            items.append((qid, lbl))
    return sparql.strip(), items

def select_items_with_ru_label(item_class_qid: str, where_lines: List[str], limit: int = 300, item_var: str = "item"):
    return select_items_with_label_ru_en(item_class_qid, where_lines, limit=limit, item_var=item_var, use_subclass_closure=True)

def select_items_with_ru_label_direct(item_class_qid: str, where_lines: List[str], limit: int = 300, item_var: str = "item"):
    return select_items_with_label_ru_en(item_class_qid, where_lines, limit=limit, item_var=item_var, use_subclass_closure=False)

_POOL_QID_RE = re.compile(r"^Q\d+$")

def _best_label_ru_en_from_entity(ent: Optional[dict]) -> Optional[str]:
    if not ent or not isinstance(ent, dict):
        return None
    labels = (ent.get("labels") or {})
    for lang in ("ru", "en"):
        val = ((labels.get(lang) or {}).get("value") or "").strip()
        if val and not _POOL_QID_RE.fullmatch(val):
            return val
    return None

def fetch_best_labels_ru_en(qids: List[str]) -> Dict[str, str]:
    """
    Добирает нормальные label'ы по QID через wbgetentities.
    Это чинит кейс, когда SERVICE wikibase:label возвращает сам QID (Q12345),
    и такой мусор потом попадает в query_text_ru / constraints.
    """
    out: Dict[str, str] = {}
    uniq = [str(q).strip() for q in dict.fromkeys(qids or []) if str(q).strip()]
    if not uniq:
        return out

    missing: List[str] = []
    cache = getattr(wd, "cache", None)

    for q in uniq:
        cache_key = f"wbterms_ruen_{q}"
        cached = cache.get(cache_key) if cache is not None else None
        if isinstance(cached, dict):
            lbl = str(cached.get("label") or "").strip()
            if lbl and not _POOL_QID_RE.fullmatch(lbl):
                out[q] = lbl
                continue
        missing.append(q)

    for i in range(0, len(missing), 50):
        chunk = missing[i:i+50]
        params = {
            "action": "wbgetentities",
            "ids": "|".join(chunk),
            "props": "labels",
            "languages": "ru|en",
            "format": "json",
        }
        try:
            resp = requests.get(WIKI_API, params=params, timeout=30)
            resp.raise_for_status()
            data = resp.json()
        except Exception:
            continue

        entities = (data or {}).get("entities", {}) or {}
        for q in chunk:
            ent = entities.get(q)
            lbl = _best_label_ru_en_from_entity(ent)
            if lbl:
                out[q] = lbl
                if cache is not None:
                    try:
                        cache.set(f"wbterms_ruen_{q}", {"label": lbl})
                    except Exception:
                        pass
    return out

def repair_pool_labels(df: pd.DataFrame, qid_col: str, label_col: str) -> pd.DataFrame:
    """
    Нормализует пул значений:
    - убирает пустые/missing labels
    - чинит строки вида Q12345 через wbgetentities (ru -> en)
    - выкидывает строки, которые так и остались QID'ами
    """
    if df is None or len(df) == 0:
        cols = list(df.columns) if isinstance(df, pd.DataFrame) else [qid_col, label_col]
        return pd.DataFrame(columns=cols)

    out = df.copy()

    if qid_col not in out.columns:
        out[qid_col] = None
    if label_col not in out.columns:
        out[label_col] = None

    out[qid_col] = out[qid_col].astype(str).str.strip()
    out[label_col] = out[label_col].fillna("").astype(str).str.strip()

    bad_mask = (
        out[qid_col].str.fullmatch(r"Q\d+").fillna(False)
        & (
            out[label_col].eq("")
            | out[label_col].str.fullmatch(r"Q\d+").fillna(False)
          )
    )

    bad_qids = [q for q in out.loc[bad_mask, qid_col].tolist() if q]
    if bad_qids:
        fixed = fetch_best_labels_ru_en(bad_qids)
        if fixed:
            repair_mask = bad_mask & out[qid_col].isin(fixed.keys())
            out.loc[repair_mask, label_col] = out.loc[repair_mask, qid_col].map(fixed)

    out[label_col] = out[label_col].fillna("").astype(str).str.strip()
    out = out[
        out[qid_col].str.fullmatch(r"Q\d+").fillna(False)
        & out[label_col].ne("")
        & ~out[label_col].str.fullmatch(r"Q\d+").fillna(False)
    ].copy()

    return out.drop_duplicates().reset_index(drop=True)

def build_value_pool_ru(item_class_qid: str, pid: str, val_name: str, limit: int = 400) -> pd.DataFrame:
    """
    Обновлённый пул значений: берём label с fallback "ru,en".
    Дополнительно ремонтируем кейсы, где label-сервис возвращает сам QID.
    """
    sparql = f"""
    SELECT DISTINCT ?val ?valLabel WHERE {{
      ?item wdt:P31/wdt:P279* wd:{item_class_qid} ;
            wdt:{pid} ?val .
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
    }}
    LIMIT {int(limit)}
    """
    rows = rows_from_select(wd.sparql_select(sparql))
    data = []
    for r in rows:
        qid = uri_to_qid(r.get("val", ""))
        lbl = r.get("valLabel")
        if qid:
            data.append({f"{val_name}_qid": qid, f"{val_name}LabelRu": lbl})
    df = pd.DataFrame(data)
    return repair_pool_labels(df, f"{val_name}_qid", f"{val_name}LabelRu")

print("✅ Patched: select_items_with_* используют ru/en fallback + repair_pool_labels чинит QID вместо label")


✅ Patched: WikidataClient.sparql_select (robust) + load_or_build_pool (safe)
✅ Patched: select_items_with_* используют ru/en fallback + repair_pool_labels чинит QID вместо label


<a id="cinema-domain"></a>

## 9. Cinema domain

Film and series templates with genre, country, year, rating, franchise, actor-bridge, and label filters.

In [9]:
Q_FILM = ensure_qid("фильм", fallback_qid="Q11424")
Q_FEMALE = ensure_qid("женский пол", fallback_qid="Q6581072")
Q_MALE   = ensure_qid("мужской пол",  fallback_qid="Q6581097")

genres_df = load_or_build_pool("cinema_genres_ru", lambda: build_value_pool_ru(Q_FILM, "P136", "genre", limit=250))
countries_df = load_or_build_pool("cinema_countries_ru", lambda: build_value_pool_ru(Q_FILM, "P495", "country", limit=150))

COMMON_GENRES_RU = ["драма", "комедия", "триллер", "ужасы", "боевик", "мелодрама", "криминальный фильм", "приключенческий фильм", "документальный фильм"]
COMMON_COUNTRIES_RU = ["Россия", "США", "Франция", "Великобритания", "Германия", "Италия", "Япония", "Канада", "Испания", "Китай"]

def _qid_from_label_in_df(df: pd.DataFrame, label_col: str, qid_col: str, label: str) -> Optional[str]:
    if df is None or len(df) == 0:
        return None
    m = df[df[label_col].str.lower() == label.lower()]
    if len(m) == 0:
        return None
    return m.iloc[0][qid_col]

def pick_genre_country(rng: random.Random) -> Tuple[str,str,str,str]:
    if rng.random() < 0.8:
        g_ru = rng.choice(COMMON_GENRES_RU)
        c_ru = rng.choice(COMMON_COUNTRIES_RU)
        g_qid = _qid_from_label_in_df(genres_df, "genreLabelRu", "genre_qid", g_ru) or ensure_qid(g_ru)
        c_qid = _qid_from_label_in_df(countries_df, "countryLabelRu", "country_qid", c_ru) or ensure_qid(c_ru)
        return g_qid, g_ru, c_qid, c_ru
    g_qid, g_ru = pick_from_df(genres_df, "genre_qid", "genreLabelRu", rng)
    c_qid, c_ru = pick_from_df(countries_df, "country_qid", "countryLabelRu", rng)
    return g_qid, g_ru, c_qid, c_ru

def run_cinema_query(
    genre_qid: str,
    country_qid: str,
    y1: int,
    y2: int,
    not_series: bool,
    director_gender_qid: Optional[str],
    limit: int = 250,
) -> Tuple[str, List[Tuple[str,str]]]:
    where = [
        f"?item wdt:P136 wd:{genre_qid} .",
        f"?item wdt:P495 wd:{country_qid} .",
        "?item wdt:P577 ?date .",
        "BIND(YEAR(?date) AS ?year) .",
        f"FILTER(?year >= {int(y1)} && ?year <= {int(y2)}) .",
    ]
    if not_series:
        where.append("FILTER NOT EXISTS { ?item wdt:P179 ?series . }")
    if director_gender_qid:
        where += [
            "?item wdt:P57 ?director .",
            f"?director wdt:P21 wd:{director_gender_qid} .",
        ]

    sparql, items = select_items_with_ru_label(Q_FILM, where_lines=where, limit=limit, item_var="item")
    return sparql, items

def nlg_cinema_ru(genre_ru: str, country_ru: str, y1: int, y2: int, not_series: bool, extra: Optional[str], k: int) -> str:
    cond = [f"жанра «{genre_ru}»", f"из страны: {country_ru}"]
    cond.append(f"в период {y1}–{y2} годов" if y1 != y2 else f"{y1} года")
    if not_series:
        cond.append("которые не являются частью франшизы/серии")
    if extra:
        cond.append(extra)
    return f"Назови {k} фильмов, " + ", ".join(cond) + "."

def generate_cinema_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 80) -> BenchmarkExample:
    for _ in range(max_attempts):
        g_qid, g_ru, c_qid, c_ru = pick_genre_country(rng)

        decade = rng.choice([(1980,1989),(1990,1999),(2000,2009),(2010,2019)])
        y1, y2 = decade
        not_series = False
        director_gender_qid = None
        extra = None
        k = 5

        if complexity == "L1":
            pass

        elif complexity == "L2":
            not_series = True

        elif complexity == "L3":
            not_series = True
            director_gender_qid = Q_FEMALE
            extra = "у которых режиссёр — женщина"

        elif complexity == "L4":
            y1, y2 = rng.choice([(1980,1984),(1985,1989),(1990,1994),(1995,1999),(2000,2004),(2005,2009)])
            not_series = True
            director_gender_qid = Q_FEMALE
            extra = "у которых режиссёр — женщина"
            k = 10

        elif complexity == "L5":
            y1, y2 = 2500, 2500
            not_series = True
            director_gender_qid = Q_FEMALE
            extra = "у которых режиссёр — женщина"
            k = 5

        else:
            raise ValueError(f"Unknown complexity: {complexity}")

        sparql, items = run_cinema_query(g_qid, c_qid, y1, y2, not_series, director_gender_qid, limit=300)

        gold = items[:300]
        gold_qids = [q for q,_ in gold]
        gold_lbls = [l for _,l in gold]
        truncated = len(items) >= 300

        where = [
            f"?item wdt:P136 wd:{g_qid} .",
            f"?item wdt:P495 wd:{c_qid} .",
            "?item wdt:P577 ?date .",
            "BIND(YEAR(?date) AS ?year) .",
            f"FILTER(?year >= {int(y1)} && ?year <= {int(y2)}) .",
        ]
        if not_series:
            where.append("FILTER NOT EXISTS { ?item wdt:P179 ?series . }")
        if director_gender_qid:
            where += [
                "?item wdt:P57 ?director .",
                f"?director wdt:P21 wd:{director_gender_qid} .",
            ]
        ask = build_ask_validator(Q_FILM, where, item_var="item")

        return BenchmarkExample(
            id=f"cinema_{complexity.lower()}_{idx:04d}",
            domain="cinema",
            complexity=complexity,
            query_text_ru=nlg_cinema_ru(g_ru, c_ru, y1, y2, not_series, extra, k),
            constraints={
                "genre_qid": g_qid, "genre_ru": g_ru,
                "country_qid": c_qid, "country_ru": c_ru,
                "year_from": y1, "year_to": y2,
                "not_series": bool(not_series),
                "director_gender_qid": director_gender_qid,
            },
            requested_count=k,
            gold_answer_qids=gold_qids,
            gold_answer_labels_ru=gold_lbls,
            sparql_query=sparql,
            created_at=utc_now_z(),
            gold_truncated=truncated,
            ask_validator_sparql=ask,
        )

    raise RuntimeError(f"Failed to generate cinema example {complexity} after {max_attempts} attempts")

try:
    _test = generate_cinema_example("L1", 0, random.Random(1))
    print(_test.query_text_ru)
    print("gold_size =", len(_test.gold_answer_qids), "truncated =", _test.gold_truncated)

except Exception as e:
    print("[WARN] sanity cinema skipped:", e)
if 'DOMAIN_PAUSE_S' in globals() and DOMAIN_PAUSE_S and DOMAIN_PAUSE_S > 0:
    time.sleep(float(DOMAIN_PAUSE_S))

# Advanced L4-L5 templates use bounded actor-bridge and exclusion constraints.

ADV_TEMPLATE_PROB_CINEMA = 0.15  # Share of advanced examples inside cinema

SEED_FILMS = [
    ("Начало", "Q43361"),              # Inception
    ("Матрица", "Q83495"),             # The Matrix
    ("Крёстный отец", "Q47703"),       # The Godfather
    ("Семь", "Q314829"),               # Se7en
]

Q_NOLAN = ensure_qid("Кристофер Нолан", fallback_qid="Q25189")

def _pick_seed_film(rng: random.Random) -> Tuple[str,str]:
    title, qid = rng.choice(SEED_FILMS)
    return title, qid

def run_cinema_actor_bridge_query(
    seed_film_qid: str,
    genre_qids: List[str],
    y1: int,
    y2: int,
    not_director_qid: Optional[str] = None,
    not_series: bool = True,
    limit: int = 250,
) -> Tuple[str, List[Tuple[str,str]]]:
    """
    Находит фильмы, где играет актёр из seed фильма.
    Фильтры: год, (OR) жанры, NOT director, NOT series.
    """
    genres_values = " ".join([f"wd:{g}" for g in genre_qids if g])
    genre_block = f"VALUES ?genre {{ {genres_values} }}" if genres_values else ""

    not_dir_filter = f"FILTER(?director != wd:{not_director_qid})" if not_director_qid else ""

    not_series_block = ""
    if not_series:
        not_series_block = """
        FILTER NOT EXISTS { ?film wdt:P179 ?series . }
        FILTER NOT EXISTS { ?film wdt:P155 ?prev . }
        FILTER NOT EXISTS { ?film wdt:P156 ?next . }
        """

    sparql = f"""
    SELECT DISTINCT ?film ?filmLabelRu WHERE {{
      BIND(wd:{seed_film_qid} AS ?seed) .
      ?seed wdt:P161 ?linkActor .

      ?film wdt:P31 wd:{Q_FILM} ;
            wdt:P161 ?linkActor ;
            wdt:P577 ?date ;
            wdt:P136 ?genre ;
            wdt:P57  ?director .

      FILTER(?film != ?seed)
      FILTER(YEAR(?date) >= {int(y1)} && YEAR(?date) <= {int(y2)})

      {genre_block}
      {not_dir_filter}
      {not_series_block}

      ?film wikibase:sitelinks ?sl . FILTER(?sl >= 1)
      ?film rdfs:label ?filmLabelRu FILTER(LANG(?filmLabelRu)='ru').
    }}
    LIMIT {int(limit)}
    """.strip()

    try:
        rows = rows_from_select(wd.sparql_select(sparql))
    except Exception:
        return sparql, []

    out=[]
    for r in rows:
        qid = uri_to_qid(r.get("film",""))
        lbl = r.get("filmLabelRu")
        if qid and lbl:
            out.append((qid,lbl))
    return sparql, out


def generate_cinema_example_advanced(complexity: str, idx: int, rng: random.Random, max_attempts: int = 60) -> BenchmarkExample:
    """
    Advanced template family:
      - L3: actor-bridge + OR genres + year range + NOT director + NOT series => >=5
      - L4: просим 12, но хотим 1..11 (insufficient)
      - L5: 0 ответов (невозможные условия)
    """
    template_id = "cinema_actor_bridge_or_not_range"

    if complexity == "L3":
        k=5
        y1,y2 = 2000, 2015
        require = lambda n: n >= k
        not_series = True
    elif complexity == "L4":
        k=4
        base = rng.randint(2000, 2015)
        y1,y2 = base, min(2015, base+1)
        require = lambda n: (0 < n < k)
        not_series = True
    elif complexity == "L5":
        k=3
        y1,y2 = 1200, 1200
        require = lambda n: n == 0
        not_series = True
    else:
        return _generate_cinema_example_default(complexity, idx, rng, max_attempts=max_attempts)

    for _ in range(max_attempts):
        seed_title, seed_qid = _pick_seed_film(rng)

        g1_qid, g1_ru = pick_from_df(genres_df, "genre_qid", "genreLabelRu", rng)
        g2_qid, g2_ru = pick_from_df(genres_df, "genre_qid", "genreLabelRu", rng)
        if g2_qid == g1_qid:
            continue

        not_dir = Q_NOLAN if rng.random() < 0.7 else None

        sparql, items = run_cinema_actor_bridge_query(
            seed_film_qid=seed_qid,
            genre_qids=[g1_qid, g2_qid],
            y1=y1, y2=y2,
            not_director_qid=not_dir,
            not_series=not_series,
            limit=300,
        )
        n=len(items)
        if not require(n):
            continue

        parts = [f"Подбери {k} фильмов"]
        parts.append(f"жанр: «{g1_ru}» или «{g2_ru}»")
        parts.append(f"год выхода: {y1}–{y2}")
        parts.append(f"в актёрском составе есть актёр из фильма «{seed_title}»")
        if not_dir:
            parts.append("режиссёр НЕ Кристофер Нолан")
        if not_series:
            parts.append("фильм НЕ является частью франшизы/серии")
        if complexity == "L4":
            parts.append("если таких фильмов меньше, чем просят — перечисли все")
        q_ru = ", ".join(parts) + "."

        gold_qids=[q for q,_ in items]
        gold_lbls=[l for _,l in items]

        return BenchmarkExample(
            id=f"cinema_adv_{complexity.lower()}_{idx:04d}",
            domain="cinema",
            complexity=complexity,
            query_text_ru=q_ru,
            constraints={
                "type": "cinema_actor_bridge",
                "template_id": template_id,
                "seed_film_qid": seed_qid,
                "seed_film_ru": seed_title,
                "genres_qids": [g1_qid, g2_qid],
                "genres_ru": [g1_ru, g2_ru],
                "year_from": y1,
                "year_to": y2,
                "not_director_qid": not_dir,
                "not_series": not_series,
            },
            requested_count=k,
            gold_answer_qids=gold_qids,
            gold_answer_labels_ru=gold_lbls,
            sparql_query=sparql,
            created_at=utc_now_z() if "utc_now_z" in globals() else None,
            is_advanced=True,
            template_id=template_id,
            template_family="advanced",
        )

    raise RuntimeError(f"Cinema advanced:{complexity} failed after {max_attempts} attempts")

_generate_cinema_example_default = generate_cinema_example

def generate_cinema_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 80) -> BenchmarkExample:
    if complexity in ("L3","L4","L5") and rng.random() < ADV_TEMPLATE_PROB_CINEMA:
        try:
            return generate_cinema_example_advanced(complexity, idx, rng, max_attempts=max_attempts)
        except Exception:
            pass
    return _generate_cinema_example_default(complexity, idx, rng, max_attempts=max_attempts)


def _maybe_ensure_qid(label: str, fallback: Optional[str] = None) -> Optional[str]:
    try:
        return ensure_qid(label, fallback_qid=fallback)
    except Exception:
        return fallback

Q_TV_SERIES = _maybe_ensure_qid("телесериал", fallback="Q5398426")  # television series
Q_IMDB = "Q37312"  # Stable IMDb QID
Q_KINOPOISK = _maybe_ensure_qid("Кинопоиск", fallback=None)         # May fail to resolve; handled below

# --- cinema tuning knobs (v12) ---
CINEMA_KINOPOISK_PROB = 0.55  # share of Kinopoisk rating constraints
CINEMA_FEMALE_DIRECTOR_PROB = {"L4": 0.20, "L5": 0.35}
CINEMA_ADV_NOT_NOLAN_PROB = 0.35
CINEMA_ADV_GODFATHER_BIAS = 0.35

CINEMA_ADV_PROB = float(globals().get("ADV_TEMPLATE_PROB_CINEMA", 0.12))   # Advanced-template share
CINEMA_COUNTRY_PROB = float(globals().get("CINEMA_COUNTRY_PROB", 0.40))  # Country-constraint share
CINEMA_SERIES_PROB = {"L1": 0.15, "L2": 0.20, "L3": 0.25, "L4": 0.30, "L5": 0.30}                 # Series-query share
CINEMA_NOT_FRANCHISE_PRELIMIT = int(globals().get("CINEMA_NOT_FRANCHISE_PRELIMIT", 2500))


# --- Rating config (diverse thresholds; supports >= and <=) ---
CINEMA_RATING_PROB = {"L2": 1.00, "L3": 1.00, "L4": 0.70, "L5": 0.80}  # Rating-constraint share

CINEMA_RATING_MIN  = {"L2": 6.8, "L3": 7.0, "L4": 7.2, "L5": 7.4}

CINEMA_RATING_LE_PROB = {
    "L2": float(globals().get("CINEMA_RATING_LE_PROB_L2", 0.08)),
    "L3": float(globals().get("CINEMA_RATING_LE_PROB_L3", 0.06)),
    "L4": float(globals().get("CINEMA_RATING_LE_PROB_L4", 0.05)),
    "L5": float(globals().get("CINEMA_RATING_LE_PROB_L5", 0.05)),
}

CINEMA_RATING_GE_LEVELS = {
    "L2": [(4.0, 1.2), (5.0, 1.5), (6.0, 2.0), (6.5, 1.8), (6.8, 1.4), (7.0, 1.2), (7.2, 0.7), (7.5, 0.5), (8.0, 0.3)],
    "L3": [(4.0, 0.6), (5.0, 1.0), (6.0, 1.6), (6.5, 2.0), (6.8, 1.6), (7.0, 1.4), (7.2, 1.1), (7.5, 0.8), (8.0, 0.4)],
    "L4": [(6.0, 0.8), (6.5, 1.2), (6.8, 1.4), (7.0, 1.4), (7.2, 1.2), (7.5, 0.9), (8.0, 0.6)],
    "L5": [(6.5, 0.8), (6.8, 1.0), (7.0, 1.2), (7.2, 1.2), (7.5, 1.0), (8.0, 0.9), (8.5, 0.4)],
}

CINEMA_RATING_LE_LEVELS = {
    "L2": [(6.0, 1.0), (5.5, 0.7), (5.0, 0.3)],
    "L3": [(6.0, 1.0), (5.5, 0.6)],
    "L4": [(6.5, 0.5), (6.0, 1.0), (5.5, 0.3)],
    "L5": [(6.5, 0.7), (6.0, 1.0), (5.5, 0.4)],
}
import datetime as _dt

CINEMA_MIN_YEAR = int(globals().get("CINEMA_MIN_YEAR", 2000))
_CINEMA_AUTO_MAX_YEAR = _dt.datetime.utcnow().year - 1
CINEMA_MAX_YEAR = int(globals().get("CINEMA_MAX_YEAR", _CINEMA_AUTO_MAX_YEAR))
CINEMA_MAX_YEAR = max(CINEMA_MIN_YEAR, CINEMA_MAX_YEAR)

CINEMA_EXACT_YEAR_PROB = {
    "L1": 0.55,
    "L2": 0.45,
    "L3": 0.45,
    "L4": 0.35,
    "L5": 0.30,
}

CINEMA_YEAR_ANCHOR_BUCKETS = {
    "L1": [((2023, CINEMA_MAX_YEAR), 0.35), ((2020, 2022), 0.20), ((2015, 2019), 0.25), ((2010, 2014), 0.12), ((2000, 2009), 0.08)],
    "L2": [((2023, CINEMA_MAX_YEAR), 0.38), ((2020, 2022), 0.22), ((2016, 2019), 0.22), ((2010, 2015), 0.13), ((2000, 2009), 0.05)],
    "L3": [((2023, CINEMA_MAX_YEAR), 0.42), ((2020, 2022), 0.23), ((2016, 2019), 0.22), ((2012, 2015), 0.10), ((2000, 2011), 0.03)],
    "L4": [((2021, CINEMA_MAX_YEAR), 0.70), ((2018, 2020), 0.20), ((2014, 2017), 0.08), ((2000, 2013), 0.02)],
    "L5": [((2023, CINEMA_MAX_YEAR), 0.38), ((2020, 2022), 0.22), ((2016, 2019), 0.22), ((2010, 2015), 0.15), ((2000, 2009), 0.03)],
}

CINEMA_RANGE_WIDTHS = {
    "L1": [2, 3, 4, 5, 7],
    "L2": [2, 3, 4, 5, 6, 7],
    "L3": [1, 2, 3, 4, 5, 6],
    "L4": [1, 2, 3, 4, 5],
    "L5": [2, 3, 4, 5, 6, 7],
}

CINEMA_RANGE_START_BUCKETS = {
    "L1": [((2000, 2007), 0.10), ((2008, 2013), 0.14), ((2014, 2017), 0.18), ((2018, 2021), 0.30), ((2022, CINEMA_MAX_YEAR), 0.28)],
    "L2": [((2000, 2009), 0.06), ((2010, 2014), 0.12), ((2015, 2018), 0.22), ((2019, 2021), 0.30), ((2022, CINEMA_MAX_YEAR), 0.30)],
    "L3": [((2000, 2011), 0.03), ((2012, 2015), 0.10), ((2016, 2018), 0.22), ((2019, 2021), 0.30), ((2022, CINEMA_MAX_YEAR), 0.35)],
    "L4": [((2000, 2013), 0.02), ((2014, 2017), 0.10), ((2018, 2019), 0.22), ((2020, 2021), 0.30), ((2022, CINEMA_MAX_YEAR), 0.36)],
    "L5": [((2000, 2009), 0.04), ((2010, 2014), 0.14), ((2015, 2018), 0.22), ((2019, 2021), 0.30), ((2022, CINEMA_MAX_YEAR), 0.30)],
}

def _weighted_choice(rng: random.Random, choices):
    total = sum(w for _, w in choices)
    x = rng.random() * total
    acc = 0.0
    for v, w in choices:
        acc += w
        if x <= acc:
            return v
    return choices[-1][0]

def _pick_year_range(rng: random.Random, complexity: str) -> Tuple[int,int]:
    c = complexity

    if rng.random() < CINEMA_EXACT_YEAR_PROB.get(c, 0.30):
        (a, b) = _weighted_choice(rng, CINEMA_YEAR_ANCHOR_BUCKETS.get(c, CINEMA_YEAR_ANCHOR_BUCKETS["L1"]))
        a = max(CINEMA_MIN_YEAR, int(a))
        b = min(CINEMA_MAX_YEAR, int(b))
        if a > b:
            a, b = CINEMA_MIN_YEAR, CINEMA_MAX_YEAR
        y = rng.randint(a, b)
        return y, y

    width = rng.choice(CINEMA_RANGE_WIDTHS.get(c, CINEMA_RANGE_WIDTHS["L1"]))
    buckets = CINEMA_RANGE_START_BUCKETS.get(c, CINEMA_RANGE_START_BUCKETS["L1"])
    for _ in range(25):
        (a, b) = _weighted_choice(rng, buckets)
        a = max(CINEMA_MIN_YEAR, int(a))
        b = min(CINEMA_MAX_YEAR, int(b))
        b2 = min(b, CINEMA_MAX_YEAR - width)
        if a <= b2:
            y1 = rng.randint(a, b2)
            y2 = y1 + width
            return int(y1), int(y2)

    # fallback
    y1 = max(CINEMA_MIN_YEAR, CINEMA_MAX_YEAR - width)
    return int(y1), int(CINEMA_MAX_YEAR)
def _pick_kind(rng: random.Random, complexity: str) -> str:
    """Выбираем: фильм или сериал. Доля сериалов задаётся таблицей CINEMA_SERIES_PROB."""
    p = CINEMA_SERIES_PROB.get(complexity, 0.0)
    if Q_TV_SERIES and rng.random() < p:
        return "series"
    return "film"

def _pick_rating_constraint(rng: random.Random, complexity: str, kind: str) -> Optional[Tuple[str, str, str, float]]:
    """Рейтинг начиная с L2.
    Возвращает (src_name, src_qid, op, score), где op ∈ {">=", "<="}.
    Для сериалов чуть реже (покрытие слабее).
    """
    if complexity not in ("L2","L3","L4","L5"):
        return None
    p = float(CINEMA_RATING_PROB.get(complexity, 0.0))
    if kind == "series":
        p *= 0.80
    if rng.random() > p:
        return None

    le_p = float(CINEMA_RATING_LE_PROB.get(complexity, 0.0))
    use_le = (rng.random() < le_p)

    if use_le:
        op = "<="
        score = _weighted_choice(rng, CINEMA_RATING_LE_LEVELS.get(complexity, CINEMA_RATING_LE_LEVELS["L2"]))
    else:
        op = ">="
        score = _weighted_choice(rng, CINEMA_RATING_GE_LEVELS.get(complexity, CINEMA_RATING_GE_LEVELS["L2"]))

    if Q_KINOPOISK and rng.random() < CINEMA_KINOPOISK_PROB:
        return ("kinopoisk", Q_KINOPOISK, op, float(score))
    return ("imdb", Q_IMDB, op, float(score))

def _cinema_limit_for_k(k: int, complexity: str) -> int:
    base = max(80, k * 20)
    if complexity in ("L4","L5"):
        base = max(base, 140)
    return min(base, 220)

CINEMA_MAX_GOLD = int(globals().get("CINEMA_MAX_GOLD", 25))
_CINEMA_BAD_LABEL_RE = re.compile(r"^Q\d+$")

# --- RU title resolution for cinema gold items ---
import re as _re

_CINEMA_CYR_RE = _re.compile(r"[А-Яа-яЁё]")

def _cinema_has_cyrillic(s: str) -> bool:
    return bool(_CINEMA_CYR_RE.search(s or ""))

_WD_API_URL = "https://www.wikidata.org/w/api.php"

def _chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def wd_get_entities_ru(qids: List[str]) -> Dict[str, dict]:
    """Fetch Wikidata entities with RU terms (labels/aliases/sitelinks/claims).
    Uses on-disk cache (wd.cache) per-QID to avoid repeated network calls.
    """
    out: Dict[str, dict] = {}
    if not qids:
        return out

    miss: List[str] = []
    for q in qids:
        key = f"wbget_ru_{q}"
        cached = getattr(wd, "cache", None).get(key) if getattr(wd, "cache", None) else None
        if cached and isinstance(cached, dict) and cached.get("entity"):
            out[q] = cached["entity"]
        else:
            miss.append(q)

    if not miss:
        return out

    for chunk in _chunks(miss, 50):
        params = {
            "action": "wbgetentities",
            "ids": "|".join(chunk),
            "props": "labels|aliases|sitelinks|claims",
            "languages": "ru|en",
            "format": "json",
        }
        try:
            r = requests.get(_WD_API_URL, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()
        except Exception:
            continue

        entities = (data or {}).get("entities", {}) or {}
        for q in chunk:
            ent = entities.get(q)
            if ent:
                out[q] = ent
                if getattr(wd, "cache", None):
                    wd.cache.set(f"wbget_ru_{q}", {"entity": ent})
    return out

def _cinema_best_ru_title(ent: Optional[dict], fallback_label: str = "") -> Optional[str]:
    """Pick the best RU title for an item.
    Preference: RU label/alias/title with Cyrillic; then RU wiki title; then fallback.
    """
    cands: List[str] = []

    if ent:
        # 1) RU label
        ru_lbl = ((ent.get("labels") or {}).get("ru") or {}).get("value")
        if ru_lbl:
            cands.append(str(ru_lbl))

        # 2) Monolingual titles (P1476) in RU
        for cl in (ent.get("claims") or {}).get("P1476", []) or []:
            try:
                dv = cl.get("mainsnak", {}).get("datavalue", {}).get("value", {})
                if isinstance(dv, dict) and dv.get("language") == "ru" and dv.get("text"):
                    cands.append(str(dv["text"]))
            except Exception:
                pass

        # 3) RU aliases
        for a in ((ent.get("aliases") or {}).get("ru") or []):
            v = a.get("value")
            if v:
                cands.append(str(v))

        # 4) RU Wikipedia sitelink title
        ru_wiki = ((ent.get("sitelinks") or {}).get("ruwiki") or {}).get("title")
        if ru_wiki:
            cands.append(str(ru_wiki))

    # Always allow fallback label as last resort
    if fallback_label:
        cands.append(str(fallback_label))

    # sanitize
    cands = [c.strip() for c in cands if c and str(c).strip()]
    # drop QID-like
    cands = [c for c in cands if not _CINEMA_BAD_LABEL_RE.fullmatch(c)]

    if not cands:
        return None

    # Prefer Cyrillic
    for c in cands:
        if _cinema_has_cyrillic(c):
            return c

    # Otherwise return the first candidate
    return cands[0]

def _cinema_has_ru_signal(ent: Optional[dict]) -> bool:
    """Heuristic: item is likely not a no-name for RU benchmark if it has RU wiki or Kinopoisk ID."""
    if not ent:
        return False
    if (ent.get("sitelinks") or {}).get("ruwiki"):
        return True
    claims = ent.get("claims") or {}
    if claims.get("P2603"):  # Kinopoisk film ID
        return True
    return False

def _cinema_postprocess_items(items: List[Tuple[str, str]]) -> Tuple[List[Tuple[str,str]], bool]:
    """Фильтруем/нормализуем gold items:
    - исправляем RU-лейблы через Wikidata API (labels/aliases/ruwiki/P1476)
    - выбрасываем элементы без нормального имени (и без RU-сигнала), чтобы не тащить ноунеймы
    - ограничиваем максимумом CINEMA_MAX_GOLD
    Возвращает (items_filtered, truncated_flag).
    """
    CINEMA_DROP_WEAK_RU = bool(globals().get("CINEMA_DROP_WEAK_RU", True))
    CINEMA_DROP_NON_CYRILLIC_TITLES = bool(globals().get("CINEMA_DROP_NON_CYRILLIC_TITLES", False))
    clean: List[Tuple[str,str]] = []
    seen: set = set()

    need_qids = [q for q,l in items if q and l and (not _cinema_has_cyrillic(str(l)))]
    ent_map = wd_get_entities_ru(list(dict.fromkeys(need_qids))) if need_qids else {}

    for q, l in items:
        if not q:
            continue
        if q in seen:
            continue

        lbl0 = str(l).strip() if l is not None else ""
        if lbl0 and _CINEMA_BAD_LABEL_RE.fullmatch(lbl0):
            lbl0 = ""

        ent = ent_map.get(q)
        best = _cinema_best_ru_title(ent, fallback_label=lbl0)

        if not best:
            continue

        if CINEMA_DROP_NON_CYRILLIC_TITLES and (not _cinema_has_cyrillic(best)):
            continue

        # If still no Cyrillic and looks like no-name (no RU wiki, no KP id) — drop
        if CINEMA_DROP_WEAK_RU and (not _cinema_has_cyrillic(best)):
            if not _cinema_has_ru_signal(ent):
                continue

        seen.add(q)
        clean.append((q, best))

        if len(clean) >= CINEMA_MAX_GOLD:
            break

    truncated = False
    # if there were more viable items than cap, mark truncated
    if len(clean) >= CINEMA_MAX_GOLD and len(items) > CINEMA_MAX_GOLD:
        truncated = True
    return clean, truncated

def _cinema_where_basic(kind: str, genre_qid: str, country_qid: Optional[str], y1: int, y2: int, not_franchise: bool):
    where = [
        f"?item wdt:P136 wd:{genre_qid} .",
        "?item wdt:P577 ?date .",
        "BIND(YEAR(?date) AS ?year) .",
        f"FILTER(?year >= {int(y1)} && ?year <= {int(y2)}) .",
    ]
    if country_qid:
        where.insert(1, f"?item wdt:P495 wd:{country_qid} .")
    if not_franchise:
        where.append("FILTER NOT EXISTS { ?item wdt:P179 ?series . }")
    return where

def _cinema_add_rating(where_lines: List[str], source_qid: str, op: str, score: float, require_id: bool):
    """Быстрый фильтр по рейтингу.
    Используем wdt:P444 (review score) без qualifiers — это стабильнее по WDQS.
    op ∈ {">=", "<="}. source_qid применяется только для опционального require_id (IMDb/Kinopoisk ID).
    """
    op = (op or ">=").strip()
    if op not in (">=", "<="):
        op = ">="
    where_lines += [
        "?item wdt:P444 ?_score_raw .",
        "BIND(xsd:decimal(?_score_raw) AS ?_score) .",
        f"FILTER(?_score {op} {float(score)}) .",
    ]
    if require_id:
        if source_qid == Q_IMDB:
            where_lines.append("?item wdt:P345 ?imdb_id .")
        elif Q_KINOPOISK and source_qid == Q_KINOPOISK:
            where_lines.append("?item wdt:P2603 ?kp_id .")
    return where_lines


def _cinema_wrap_not_franchise_subquery(where_lines: List[str], prelimit: int = None) -> List[str]:
    """Ускоритель для NOT EXISTS по франшизе/серии (P179).
    Идея: сначала берём ограниченное число кандидатов по остальным фильтрам,
    а уже потом применяем NOT EXISTS. Это резко ускоряет L3+ на WDQS.
    """
    pl = int(prelimit if prelimit is not None else CINEMA_NOT_FRANCHISE_PRELIMIT)
    inner = "\n        ".join(where_lines)
    return [
        "{ SELECT ?item WHERE {\n        " + inner + f"\n      }} LIMIT {pl} }}",
        "FILTER NOT EXISTS { ?item wdt:P179 ?series . }",
    ]
def _pick_seed_work(rng: random.Random) -> Tuple[str, str, str]:
    # returns (kind, title, qid)
    use_series = bool(SEED_SERIES_V2) and (rng.random() < 0.30)
    if use_series:
        title, qid = rng.choice(SEED_SERIES_V2)
        return "series", title, qid
    title, qid = rng.choice(SEED_FILMS_V2)
    return "film", title, qid

def _build_query_text(kind: str, k: int, genre_ru: str, country_ru: Optional[str], y1: int, y2: int,
                      not_franchise: bool, rating_info: Optional[Tuple[str,str,float]]):
    noun = "сериалов" if kind == "series" else "фильмов"
    parts = [f"Назови {k} {noun}"]
    parts.append(f"жанра «{genre_ru}»")
    if country_ru:
        parts.append(f"из страны: {country_ru}")
    parts.append(f"в период {y1}–{y2} годов" if y1 != y2 else f"{y1} года")
    if rating_info:
        src_name, op, score = rating_info
        src_ru = "IMDb" if src_name == "imdb" else "Кинопоиск"
        if op == "<=":
            parts.append(f"с рейтингом по {src_ru} не выше {float(score):g}")
        else:
            parts.append(f"с рейтингом по {src_ru} не ниже {float(score):g}")
    if not_franchise:
        parts.append("которые не являются частью франшизы/серии")
    return ", ".join(parts) + "."

def generate_cinema_example_advanced_v2(complexity: str, idx: int, rng: random.Random, max_attempts: int = 80) -> BenchmarkExample:
    """
    Advanced v2: multi-hop actor bridge.
    - seed (film/series) -> actor -> other films/series with same actor
    - + OR genres + year range + optional NOT director + optional rating
    """
    template_id = "cinema_actor_bridge_v2"

    if complexity == "L3":
        k = 5
        require = lambda n: n >= k
    elif complexity == "L4":
        k = 8
        require = lambda n: (0 < n < k)  # insufficient OK
    else:  # L5
        k = 5
        require = lambda n: n >= k

    gold_trunc = False

    best = None
    best_n = -1

    for _ in range(max_attempts):
        seed_kind, seed_title, seed_qid = _pick_seed_work(rng)
        # Bias: more often pick a well-known seed for RU evaluation
        if seed_kind == "film" and rng.random() < CINEMA_ADV_GODFATHER_BIAS:
            seed_title, seed_qid = "Крёстный отец", "Q47703"
        if not seed_qid:
            continue

        g1_qid, g1_ru, c_qid, c_ru = pick_genre_country(rng)
        g2_qid, g2_ru, _, _ = pick_genre_country(rng)

        y1, y2 = _pick_year_range(rng, complexity)
        kind = _pick_kind(rng, complexity)

        not_franchise = (kind == "film") and (rng.random() < 0.65)
        where = [
            "?item wdt:P577 ?date .",
            "BIND(YEAR(?date) AS ?year) .",
            f"FILTER(?year >= {int(y1)} && ?year <= {int(y2)}) .",
            f"VALUES ?g {{ wd:{g1_qid} wd:{g2_qid} }}",
            "?item wdt:P136 ?g .",
            f"wd:{seed_qid} wdt:P161 ?actor .",
            "?item wdt:P161 ?actor .",
            f"FILTER(?item != wd:{seed_qid})",
        ]
        if not_franchise and kind == "film":
            where.append("FILTER NOT EXISTS { ?item wdt:P179 ?series . }")

        not_dir = None
        if kind == "film" and _DIRECTORS_NOT_V2 and rng.random() < 0.55:
            not_dir = rng.choice(_DIRECTORS_NOT_V2)
            where += [
                "?item wdt:P57 ?dir .",
                f"FILTER(?dir != wd:{not_dir})",
            ]

        rating = _pick_rating_constraint(rng, complexity, kind)
        if complexity in ("L2","L3") and rating is None:
            # default: IMDb
            rating = ("imdb", Q_IMDB, ">=", float(CINEMA_RATING_MIN.get(complexity, 7.0)))
        rating_info = None

        if rating:
            src_name, src_qid, op, score = rating
            _cinema_add_rating(where, src_qid, op, score, require_id=False)
            rating_info = (src_name, op, float(score))

        item_class_qid = Q_FILM if kind == "film" else Q_TV_SERIES

        try:
            sparql, items = select_items_with_ru_label(item_class_qid, where_lines=where, limit=_cinema_limit_for_k(k, complexity), item_var="item")
            items, gold_trunc = _cinema_postprocess_items(items)
        except Exception as e:
            if globals().get("DEBUG_GENERATOR_ERRORS"):
                print("[cinema_adv_v2] wdqs error:", repr(e))
            continue

        n = len(items)
        if n > best_n:
            best_n = n
            best = (sparql, items, kind, seed_kind, seed_title, seed_qid, g1_ru, g2_ru, y1, y2, not_dir, not_franchise, rating_info)

        if not require(n):
            continue

        noun = "сериалов" if kind == "series" else "фильмов"
        parts = [f"Подбери {k} {noun}"]
        parts.append(f"жанр: «{g1_ru}» или «{g2_ru}»")
        parts.append(f"год выхода: {y1}–{y2}" if y1 != y2 else f"год выхода: {y1}")
        parts.append(f"в актёрском составе есть актёр из {'сериала' if seed_kind=='series' else 'фильма'} «{seed_title}»")
        if not_dir:
            parts.append(f"режиссёром которых не является {not_dir_title}")
        if not_franchise and kind == "film":
            parts.append("не является частью франшизы/серии")
        if rating_info:
            src_name, op, score = rating_info
            src_ru = "IMDb" if src_name == "imdb" else "Кинопоиск"
            if op == "<=":
                parts.append(f"рейтинг по {src_ru} <= {float(score):g}")
            else:
                parts.append(f"рейтинг по {src_ru} >= {float(score):g}")
        if complexity == "L4":
            parts.append("если таких меньше, чем просят — перечисли все")
        q_ru = ", ".join(parts) + "."

        gold_qids=[q for q,_ in items]
        gold_lbls=[l for _,l in items]

        return BenchmarkExample(
            id=f"cinema_adv2_{complexity.lower()}_{idx:04d}",
            domain="cinema",
            complexity=complexity,
            query_text_ru=q_ru,
            constraints={
                "type": "cinema_actor_bridge_v2",
                "template_id": template_id,
                "seed_kind": seed_kind,
                "seed_title": seed_title,
                "seed_qid": seed_qid,
                "genres_ru": [g1_ru, g2_ru],
                "year_from": y1, "year_to": y2,
                "kind": kind,
                "not_director_qid": not_dir,
                "not_franchise": bool(not_franchise),
                "female_director": bool(female_director),
                "rating": rating_info,
            },
            requested_count=k,
            gold_answer_qids=gold_qids,
            gold_answer_labels_ru=gold_lbls,
            gold_truncated=gold_trunc,
            sparql_query=sparql,
            created_at=utc_now_z() if "utc_now_z" in globals() else None,
            is_advanced=True,
            template_id=template_id,
            template_family="advanced",
        )

    if best is not None:
        sparql, items, kind, seed_kind, seed_title, seed_qid, g1_ru, g2_ru, y1, y2, not_dir, not_franchise, rating_info = best
        gold_qids=[q for q,_ in items]
        gold_lbls=[l for _,l in items]
        noun = "сериалов" if kind == "series" else "фильмов"
        q_ru = f"Подбери {min(5, max(1, len(items)))} {noun} (best-effort) по условиям: actor-bridge от «{seed_title}», жанры «{g1_ru}»/«{g2_ru}», {y1}–{y2}."
        return BenchmarkExample(
            id=f"cinema_adv2_{complexity.lower()}_{idx:04d}",
            domain="cinema",
            complexity=complexity,
            query_text_ru=q_ru,
            constraints={"type":"cinema_actor_bridge_v2_fallback"},
            requested_count=k,
            gold_answer_qids=gold_qids,
            gold_answer_labels_ru=gold_lbls,
            gold_truncated=gold_trunc,
            sparql_query=sparql,
            created_at=utc_now_z() if "utc_now_z" in globals() else None,
            is_advanced=True,
            template_id="cinema_actor_bridge_v2",
            template_family="advanced",
        )

    return _generate_cinema_example_default(complexity, idx, rng, max_attempts=60)

def generate_cinema_example_v2(complexity: str, idx: int, rng: random.Random, max_attempts: int = 90) -> BenchmarkExample:
    """
    Cinema generator v2: быстрый и устойчивый.
    - L1: жанр + годы (+ страна с вероятностью CINEMA_COUNTRY_PROB)
    - L2: (жанр + страна + годы + рейтинг) ИЛИ (жанр + годы + рейтинг)
    - L3: как L2, но + НЕ франшиза/серия
    - L4/L5: как раньше (advanced/сложнее)
    """
    if complexity in ("L3","L4","L5") and rng.random() < CINEMA_ADV_PROB:
        try:
            return generate_cinema_example_advanced_v2(complexity, idx, rng, max_attempts=max_attempts)
        except Exception:
            pass

    best = None
    best_n = -1

    best_gold_trunc = False
    for _ in range(max_attempts):
        g_qid, g_ru, c_qid, c_ru = pick_genre_country(rng)
        y1, y2 = _pick_year_range(rng, complexity)
        kind = _pick_kind(rng, complexity)

        if complexity == "L1":
            k = 5
            not_franchise = False
            use_country = (rng.random() < CINEMA_COUNTRY_PROB)
        elif complexity == "L2":
            k = 5
            not_franchise = False
            use_country = (rng.random() < CINEMA_COUNTRY_PROB)
        elif complexity == "L3":
            k = 5
            not_franchise = True
            use_country = (rng.random() < CINEMA_COUNTRY_PROB)
        elif complexity == "L4":
            k = 8
            not_franchise = True
            use_country = (rng.random() < CINEMA_COUNTRY_PROB)
        else:
            k = 5
            not_franchise = True
            use_country = (rng.random() < CINEMA_COUNTRY_PROB)

        rating = _pick_rating_constraint(rng, complexity, kind)
        if complexity in ("L2","L3") and rating is None:
            # default: IMDb
            rating = ("imdb", Q_IMDB, ">=", float(CINEMA_RATING_MIN.get(complexity, 7.0)))
        rating_info = None


        item_class_qid = Q_FILM if kind == "film" else Q_TV_SERIES
        if kind == "series" and not item_class_qid:
            kind = "film"
            item_class_qid = Q_FILM

        where = _cinema_where_basic(kind, g_qid, (c_qid if use_country else None), y1, y2, False, director_gender_qid=director_gender_qid)

        used_rating = False
        if rating:
            src_name, src_qid, op, score = rating
            require_id = (rng.random() < (0.35 if src_name == "imdb" else 0.12))
            _cinema_add_rating(where, src_qid, op, score, require_id=require_id)
            rating_info = (src_name, op, float(score))
            used_rating = True

        if not_franchise:
            where = _cinema_wrap_not_franchise_subquery(where)

        ask = build_ask_validator(item_class_qid, where, item_var="item")
        limit = _cinema_limit_for_k(k, complexity)

        try:
            sparql, items = select_items_with_ru_label(item_class_qid, where_lines=where, limit=limit, item_var="item")
            items, gold_trunc = _cinema_postprocess_items(items)
        except Exception as e:
            if globals().get("DEBUG_GENERATOR_ERRORS"):
                print("[cinema_v2] wdqs error:", repr(e))
            continue

        n = len(items)
        if n > best_n:
            best_n = n
            best = (sparql, items, kind, g_ru, (c_ru if use_country else None), y1, y2, not_franchise, rating_info, ask, k, use_country, c_qid)
            best_gold_trunc = gold_trunc

        if used_rating and n < k:
            if use_country:
                where2 = _cinema_where_basic(kind, g_qid, None, y1, y2, False)
                if rating:
                    src_name2, src_qid2, op2, score2 = rating
                    _cinema_add_rating(where2, src_qid2, op2, float(score2), require_id=False)
                if not_franchise:
                    where2 = _cinema_wrap_not_franchise_subquery(where2)
                ask2 = build_ask_validator(item_class_qid, where2, item_var="item")
                try:
                    sparql2, items2 = select_items_with_ru_label(item_class_qid, where_lines=where2, limit=limit, item_var="item")
                    items2, gold_trunc2 = _cinema_postprocess_items(items2)
                    if len(items2) > n:
                        sparql, items, ask = sparql2, items2, ask2
                        gold_trunc = gold_trunc2
                        n = len(items)
                        c_ru = None
                        use_country = False
                except Exception:
                    pass

            if n < k and rating and complexity in ("L2","L3"):
                src_name2, src_qid2, op2, score2 = rating
                if op2 == ">=":
                    score3 = max(4.0, float(score2) - 0.4)
                else:
                    score3 = min(9.5, float(score2) + 0.4)
                where3 = _cinema_where_basic(kind, g_qid, (c_qid if use_country else None), y1, y2, False)
                _cinema_add_rating(where3, src_qid2, op2, score3, require_id=False)
                if not_franchise:
                    where3 = _cinema_wrap_not_franchise_subquery(where3)
                ask3 = build_ask_validator(item_class_qid, where3, item_var="item")
                try:
                    sparql3, items3 = select_items_with_ru_label(item_class_qid, where_lines=where3, limit=limit, item_var="item")
                    items3, gold_trunc3 = _cinema_postprocess_items(items3)
                    if len(items3) > n:
                        sparql, items, ask = sparql3, items3, ask3
                        gold_trunc = gold_trunc3
                        n = len(items)
                        rating_info = (src_name2, op2, float(score3))
                except Exception:
                    pass


        if n < k:
            continue

        q_ru = _build_query_text(kind, k, g_ru, (c_ru if use_country else None), y1, y2, not_franchise, rating_info, female_director=female_director)

        gold_qids=[q for q,_ in items]
        gold_lbls=[l for _,l in items]

        return BenchmarkExample(
            id=f"cinema_{complexity.lower()}_{idx:04d}",
            domain="cinema",
            complexity=complexity,
            query_text_ru=q_ru,
            constraints={
                "kind": kind,
                "genre_ru": g_ru, "country_ru": c_ru,
                "year_from": y1, "year_to": y2,
                "not_franchise": bool(not_franchise),
            "not_director_qid": not_dir,
            "rating": rating_info,
            },
            requested_count=k,
            gold_answer_qids=gold_qids,
            gold_answer_labels_ru=gold_lbls,
            gold_truncated=gold_trunc,
            sparql_query=sparql,
            created_at=utc_now_z(),
            ask_validator_sparql=ask,
        )

    # best-effort fallback
    if best is not None:
        sparql, items, kind, g_ru, c_ru, y1, y2, not_franchise, rating_info, ask, k, use_country, c_qid_best = best
        gold_trunc = best_gold_trunc
        q_ru = _build_query_text(kind, min(k, max(1, len(items))), g_ru, c_ru, y1, y2, not_franchise, rating_info)
        gold_qids=[q for q,_ in items]
        gold_lbls=[l for _,l in items]
        return BenchmarkExample(
            id=f"cinema_{complexity.lower()}_{idx:04d}",
            domain="cinema",
            complexity=complexity,
            query_text_ru=q_ru,
            constraints={"fallback": True},
            requested_count=k,
            gold_answer_qids=gold_qids,
            gold_answer_labels_ru=gold_lbls,
            gold_truncated=gold_trunc,
            sparql_query=sparql,
            created_at=utc_now_z(),
            ask_validator_sparql=ask,
        )

    return BenchmarkExample(
        id=f"cinema_{complexity.lower()}_{idx:04d}",
        domain="cinema",
        complexity=complexity,
        query_text_ru="(Cinema generator failed to fetch any candidates; retry later.)",
        constraints={"failed": True},
        requested_count=5,
        gold_answer_qids=[],
        gold_answer_labels_ru=[],
        sparql_query="",
        created_at=utc_now_z(),
    )

_generate_cinema_example_default = generate_cinema_example_v2
generate_cinema_example = generate_cinema_example_v2

# L4-L5 cinema refinement: richer constraints, provider IDs, and bounded actor-bridge joins.

from typing import Optional, List, Tuple, Dict

# --- provider IDs ---
CINEMA_KINOPOISK_ID_PROP = "P2603"  # Kinopoisk film ID
CINEMA_IMDB_ID_PROP = "P345"        # IMDb title ID

# --- tuning knobs (you can tweak) ---
CINEMA_V3_KINOPOISK_PROB = 0.75  # share of "rating by Kinopoisk" in question wording (and optional ID constraint)
CINEMA_V3_REQUIRE_PROVIDER_ID_PROB = 0.22  # when mentioning a provider, how often we also require its ID to exist

CINEMA_V3_FEMALE_DIRECTOR_PROB = {"L4": 0.40, "L5": 0.60}
CINEMA_V3_NOT_DIRECTOR_PROB = {"L4": 0.30, "L5": 0.45}
CINEMA_V3_ACTOR_BRIDGE_PROB = {"L4": 0.45, "L5": 0.70}

# Limit actors to keep WDQS joins manageable
CINEMA_V3_ACTOR_SUBQUERY_LIMIT = 18

def _cinema_v3_maybe(label_ru: str, fallback_qid: Optional[str] = None) -> Optional[str]:
    try:
        return ensure_qid(label_ru, fallback_qid=fallback_qid)
    except Exception:
        return fallback_qid

# --- pools: directors to exclude (NOT ...) ---
_CINEMA_V3_DIRECTORS_NOT_RAW: List[Tuple[str, Optional[str]]] = [
    ("Кристофер Нолан", "Q25189"),
    ("Квентин Тарантино", None),
    ("Стивен Спилберг", "Q8877"),
    ("Мартин Скорсезе", None),
    ("Джеймс Кэмерон", None),
    ("Дэвид Финчер", None),
    ("Стэнли Кубрик", None),
]
_CINEMA_V3_DIRECTORS_NOT: List[Tuple[str, str]] = []
for _name, _fb in _CINEMA_V3_DIRECTORS_NOT_RAW:
    _q = _cinema_v3_maybe(_name, fallback_qid=_fb)
    if _q:
        _CINEMA_V3_DIRECTORS_NOT.append((_name, _q))

# --- pools: seed works for actor-bridge ---
_CINEMA_V3_SEED_FILMS_RAW: List[Tuple[str, Optional[str]]] = [
    ("Крёстный отец", "Q47703"),
    ("Начало", "Q43361"),
    ("Матрица", "Q83495"),
    ("Титаник", None),
    ("Интерстеллар", None),
    ("Побег из Шоушенка", None),
    ("Терминатор 2: Судный день", None),
]
_CINEMA_V3_SEED_SERIES_RAW: List[Tuple[str, Optional[str]]] = [
    ("Во все тяжкие", None),
    ("Игра престолов", None),
    ("Друзья", None),
]
_CINEMA_V3_SEED_FILMS: List[Tuple[str, str]] = []
_CINEMA_V3_SEED_SERIES: List[Tuple[str, str]] = []

for _name, _fb in _CINEMA_V3_SEED_FILMS_RAW:
    _q = _cinema_v3_maybe(_name, fallback_qid=_fb)
    if _q:
        _CINEMA_V3_SEED_FILMS.append((_name, _q))

for _name, _fb in _CINEMA_V3_SEED_SERIES_RAW:
    _q = _cinema_v3_maybe(_name, fallback_qid=_fb)
    if _q:
        _CINEMA_V3_SEED_SERIES.append((_name, _q))

def _cinema_v3_pick_seed(rng: random.Random) -> Tuple[str, str, str]:
    # returns (seed_kind, seed_title, seed_qid)
    use_series = bool(_CINEMA_V3_SEED_SERIES) and (rng.random() < 0.28)
    if use_series:
        title, qid = rng.choice(_CINEMA_V3_SEED_SERIES)
        return "series", title, qid
    if _CINEMA_V3_SEED_FILMS:
        title, qid = rng.choice(_CINEMA_V3_SEED_FILMS)
        return "film", title, qid
    return "film", "Крёстный отец", "Q47703"

def _cinema_v3_weighted_choice(rng: random.Random, pairs: List[Tuple[float, float]]) -> float:
    total = sum(w for _, w in pairs)
    r = rng.random() * (total if total > 0 else 1.0)
    s = 0.0
    for v, w in pairs:
        s += w
        if r <= s:
            return float(v)
    return float(pairs[-1][0])

def _cinema_v3_pick_rating(rng: random.Random, complexity: str) -> Tuple[str, Optional[str], str, float]:
    # returns (provider_name, provider_qid(optional), op, score)
    ge_tbl = globals().get("CINEMA_RATING_GE_LEVELS")
    le_tbl = globals().get("CINEMA_RATING_LE_LEVELS")
    if isinstance(ge_tbl, dict) and isinstance(le_tbl, dict):
        if rng.random() < 0.18:
            op = "<="
            score = _cinema_v3_weighted_choice(rng, le_tbl.get(complexity, le_tbl.get("L2")))
        else:
            op = ">="
            score = _cinema_v3_weighted_choice(rng, ge_tbl.get(complexity, ge_tbl.get("L2")))
    else:
        op = ">="
        score = float(rng.choice([6.0, 6.5, 6.8, 7.0, 7.2, 7.5, 8.0]))
    use_kp = (rng.random() < CINEMA_V3_KINOPOISK_PROB)
    if use_kp:
        return ("kinopoisk", globals().get("Q_KINOPOISK"), op, float(score))
    return ("imdb", globals().get("Q_IMDB", "Q37312"), op, float(score))

def _cinema_v3_add_rating(where: List[str], provider_name: str, op: str, score: float, require_id: bool) -> None:
    where += [
        "?item wdt:P444 ?_score_raw .",
        "BIND(xsd:decimal(?_score_raw) AS ?_score) .",
        "FILTER(?_score %s %s) ." % (op, float(score)),
    ]
    if not require_id:
        return
    if provider_name == "imdb":
        where.append("?item wdt:%s ?imdb_id ." % CINEMA_IMDB_ID_PROP)
    else:
        where.append("?item wdt:%s ?kp_id ." % CINEMA_KINOPOISK_ID_PROP)

def _cinema_v3_query_text(kind: str,
                          k: int,
                          genres_ru: List[str],
                          country_ru: Optional[str],
                          y1: int,
                          y2: int,
                          rating_info: Optional[Tuple[str, str, float]],
                          not_franchise: bool,
                          female_director: bool,
                          not_director_name: Optional[str],
                          actor_bridge_seed: Optional[Tuple[str, str]],
                          l4_allow_short: bool) -> str:
    noun = "сериалов" if kind == "series" else "фильмов"
    parts = ["Подбери %d %s" % (k, noun)]
    if len(genres_ru) == 1:
        parts.append("жанра «%s»" % genres_ru[0])
    elif len(genres_ru) >= 2:
        parts.append("жанр: «%s» или «%s»" % (genres_ru[0], genres_ru[1]))
    if country_ru:
        parts.append("страна: %s" % country_ru)
    parts.append("год выхода: %d–%d" % (y1, y2) if y1 != y2 else "год выхода: %d" % y1)
    if actor_bridge_seed:
        seed_kind, seed_title = actor_bridge_seed
        parts.append("в актёрском составе есть актёр из %s «%s»" % ("сериала" if seed_kind == "series" else "фильма", seed_title))
    if female_director:
        parts.append("режиссёр — женщина")
    if not_director_name:
        parts.append("режиссёр не %s" % not_director_name)
    if not_franchise and kind == "film":
        parts.append("не является частью франшизы/серии")
    if rating_info:
        src_name, op, score = rating_info
        src_ru = "IMDb" if src_name == "imdb" else "Кинопоиск"
        parts.append("рейтинг по %s %s %g" % (src_ru, op, float(score)))
    if l4_allow_short:
        parts.append("если таких меньше, чем просят — перечисли все")
    return ", ".join(parts) + "."

def generate_cinema_example_v3(complexity: str, idx: int, rng: random.Random, max_attempts: int = 140) -> BenchmarkExample:
    """
    Cinema v3 generator (override):
      - Fix KP property
      - Ensure L4/L5 actually produce examples by avoiding over-restrictive combos
      - Adds female-director, not-director, actor-bridge options
      - Boosts Kinopoisk rating mentions
    """
    # requested_count policy aligned with your buckets:
    if complexity == "L4":
        k = 6
        accept = lambda n: n > 0
    elif complexity == "L5":
        k = 5
        accept = lambda n: n > 0
    else:
        k = 5
        accept = lambda n: n >= k

    best = None
    best_n = -1

    for _ in range(max_attempts):
        kind = _pick_kind(rng, complexity) if "_pick_kind" in globals() else ("series" if rng.random() < 0.22 else "film")

        g1_qid, g1_ru, c_qid, c_ru = pick_genre_country(rng)
        genres_qids = [g1_qid]
        genres_ru = [g1_ru]

        if complexity in ("L4", "L5") and rng.random() < 0.55:
            g2_qid, g2_ru, _, _ = pick_genre_country(rng)
            if g2_qid and g2_qid != g1_qid:
                genres_qids.append(g2_qid)
                genres_ru.append(g2_ru)

        y1, y2 = _pick_year_range(rng, complexity) if "_pick_year_range" in globals() else (2000, 2023)
        use_country = (rng.random() < float(globals().get("CINEMA_COUNTRY_PROB", 0.40)))
        country_qid = c_qid if use_country else None
        country_ru = c_ru if use_country else None

        rating_info = None
        if complexity in ("L2", "L3", "L4", "L5"):
            src_name, _src_qid, op, score = _cinema_v3_pick_rating(rng, complexity)
            rating_info = (src_name, op, float(score))

        not_franchise = (kind == "film") and (complexity in ("L3", "L4", "L5")) and (rng.random() < 0.55)

        female_director = False
        not_director = None  # (name, qid)
        actor_bridge = False
        seed_kind = seed_title = seed_qid = None

        if complexity in ("L4", "L5"):
            if rng.random() < CINEMA_V3_FEMALE_DIRECTOR_PROB.get(complexity, 0.0):
                female_director = True

            if _CINEMA_V3_DIRECTORS_NOT and (rng.random() < CINEMA_V3_NOT_DIRECTOR_PROB.get(complexity, 0.0)):
                not_director = rng.choice(_CINEMA_V3_DIRECTORS_NOT)

            if _CINEMA_V3_SEED_FILMS and (rng.random() < CINEMA_V3_ACTOR_BRIDGE_PROB.get(complexity, 0.0)):
                actor_bridge = True
                seed_kind, seed_title, seed_qid = _cinema_v3_pick_seed(rng)

            # avoid over-restrictive triple combo too often
            if female_director and not_director and actor_bridge and rng.random() < 0.55:
                drop = rng.choice(["female", "notdir", "bridge"])
                if drop == "female":
                    female_director = False
                elif drop == "notdir":
                    not_director = None
                else:
                    actor_bridge = False
                    seed_kind = seed_title = seed_qid = None

        where = [
            "?item wdt:P577 ?date .",
            "BIND(YEAR(?date) AS ?year) .",
            "FILTER(?year >= %d && ?year <= %d) ." % (int(y1), int(y2)),
        ]
        if country_qid:
            where.insert(1, "?item wdt:P495 wd:%s ." % country_qid)

        if len(genres_qids) == 1:
            where.insert(0, "?item wdt:P136 wd:%s ." % genres_qids[0])
        else:
            where.insert(0, "VALUES ?g { %s }" % " ".join("wd:"+q for q in genres_qids[:2]))
            where.insert(1, "?item wdt:P136 ?g .")

        if actor_bridge and seed_qid:
            where += [
                "{",
                "  SELECT DISTINCT ?actor WHERE { wd:%s wdt:P161 ?actor . } LIMIT %d" % (seed_qid, int(CINEMA_V3_ACTOR_SUBQUERY_LIMIT)),
                "}",
                "?item wdt:P161 ?actor .",
                "FILTER(?item != wd:%s) ." % seed_qid,
            ]

        if female_director:
            where += [
                "?item wdt:P57 ?dir_f .",
                "?dir_f wdt:P21 wd:%s ." % Q_FEMALE,
            ]

        if not_director:
            nd_name, nd_qid = not_director
            where.append("FILTER NOT EXISTS { ?item wdt:P57 wd:%s }" % nd_qid)

        if not_franchise and kind == "film":
            where.append("FILTER NOT EXISTS { ?item wdt:P179 ?series . }")

        if rating_info:
            src_name, op, score = rating_info
            require_id = (rng.random() < CINEMA_V3_REQUIRE_PROVIDER_ID_PROB)
            _cinema_v3_add_rating(where, src_name, op, float(score), require_id=require_id)

        item_class_qid = Q_FILM if kind == "film" else (Q_TV_SERIES if globals().get("Q_TV_SERIES") else Q_FILM)
        try:
            sparql, items = select_items_with_ru_label(
                item_class_qid,
                where_lines=where,
                limit=_cinema_limit_for_k(k, complexity) if "_cinema_limit_for_k" in globals() else 220,
                item_var="item",
            )
            if "_cinema_postprocess_items" in globals():
                items, gold_trunc = _cinema_postprocess_items(items)
            else:
                gold_trunc = False
        except Exception:
            continue

        n = len(items)
        if n > best_n:
            best_n = n
            best = (sparql, items, gold_trunc, kind, genres_ru, country_ru, y1, y2, not_franchise, rating_info, female_director, not_director, actor_bridge, seed_kind, seed_title, seed_qid)

        if not accept(n):
            continue

        ask = None
        if "build_ask_validator" in globals():
            try:
                ask = build_ask_validator(item_class_qid, where, item_var="item")
            except Exception:
                ask = None

        q_ru = _cinema_v3_query_text(
            kind=kind,
            k=k,
            genres_ru=genres_ru,
            country_ru=country_ru,
            y1=y1,
            y2=y2,
            rating_info=rating_info,
            not_franchise=bool(not_franchise),
            female_director=bool(female_director),
            not_director_name=(not_director[0] if not_director else None),
            actor_bridge_seed=((seed_kind, seed_title) if actor_bridge and seed_title else None),
            l4_allow_short=(complexity == "L4"),
        )

        gold_qids = [q for q, _ in items]
        gold_lbls = [l for _, l in items]

        return BenchmarkExample(
            id="cinema_v3_%s_%04d" % (complexity.lower(), idx),
            domain="cinema",
            complexity=complexity,
            query_text_ru=q_ru,
            constraints={
                "type": "cinema_v3",
                "kind": kind,
                "genres_ru": genres_ru,
                "country_ru": country_ru,
                "year_from": y1, "year_to": y2,
                "not_franchise": bool(not_franchise),
                "rating": rating_info,
                "female_director": bool(female_director),
                "not_director": (not_director[0] if not_director else None),
                "actor_bridge_seed": (seed_title if actor_bridge else None),
            },
            requested_count=k,
            gold_answer_qids=gold_qids,
            gold_answer_labels_ru=gold_lbls,
            sparql_query=sparql,
            created_at=utc_now_z() if "utc_now_z" in globals() else "",
            is_advanced=bool(actor_bridge or female_director or not_director),
            template_id="cinema_v3",
            template_family="default" if not (actor_bridge or female_director or not_director) else "mixed",
            gold_truncated=bool(gold_trunc),
            ask_validator_sparql=ask,
        )

    if best is not None:
        sparql, items, gold_trunc, kind, genres_ru, country_ru, y1, y2, not_franchise, rating_info, female_director, not_director, actor_bridge, seed_kind, seed_title, seed_qid = best
        if len(items) > 0:
            q_ru = _cinema_v3_query_text(
                kind=kind,
                k=min(5, len(items)),
                genres_ru=genres_ru,
                country_ru=country_ru,
                y1=y1,
                y2=y2,
                rating_info=rating_info,
                not_franchise=bool(not_franchise),
                female_director=bool(female_director),
                not_director_name=(not_director[0] if not_director else None),
                actor_bridge_seed=((seed_kind, seed_title) if actor_bridge and seed_title else None),
                l4_allow_short=True,
            )
            gold_qids = [q for q, _ in items]
            gold_lbls = [l for _, l in items]
            return BenchmarkExample(
                id="cinema_v3_%s_%04d" % (complexity.lower(), idx),
                domain="cinema",
                complexity=complexity,
                query_text_ru=q_ru,
                constraints={"type": "cinema_v3_fallback"},
                requested_count=5,
                gold_answer_qids=gold_qids,
                gold_answer_labels_ru=gold_lbls,
                sparql_query=sparql,
                created_at=utc_now_z() if "utc_now_z" in globals() else "",
                is_advanced=True,
                template_id="cinema_v3_fallback",
                template_family="mixed",
                gold_truncated=bool(gold_trunc),
            )

    # placeholder (will be dropped by main builder)
    return BenchmarkExample(
        id="cinema_v3_fail_%s_%04d" % (complexity.lower(), idx),
        domain="cinema",
        complexity=complexity,
        query_text_ru="(Cinema v3 generator failed to fetch any candidates; retry later.)",
        constraints={"failed": True},
        requested_count=5,
        gold_answer_qids=[],
        gold_answer_labels_ru=[],
        sparql_query="",
        created_at=utc_now_z() if "utc_now_z" in globals() else "",
    )

# Override cinema generator used by DOMAIN_GENERATORS
generate_cinema_example = generate_cinema_example_v3

# Prefer Russian-facing labels when a stable RU title or sitelink exists.

RU_SITELINK_PRIORITY = [
    "ruwiki",
    "ruwiktionary",
    "ruwikivoyage",
    "ruwikiquote",
    "ruwikisource",
    "ruwikinews",
    "ruwikibooks",
]

def _get_ru_sitelink_title(ent: Optional[dict]) -> Optional[str]:
    if not ent:
        return None
    sl = ent.get("sitelinks") or {}
    for k in RU_SITELINK_PRIORITY:
        try:
            title = (sl.get(k) or {}).get("title")
            if title and str(title).strip():
                return str(title).strip()
        except Exception:
            continue
    return None

def _get_ru_monolingual_title(ent: Optional[dict], pid: str = "P1476") -> Optional[str]:
    if not ent:
        return None
    try:
        for cl in (ent.get("claims") or {}).get(pid, []) or []:
            dv = cl.get("mainsnak", {}).get("datavalue", {}).get("value", {})
            # monolingualtext -> {"text": "...", "language": "ru"}
            if isinstance(dv, dict) and dv.get("language") == "ru":
                txt = dv.get("text")
                if txt and str(txt).strip():
                    return str(txt).strip()
    except Exception:
        pass
    return None

def _has_popular_ru_anchor(ent: Optional[dict]) -> bool:
    """Heuristic: entity likely has a *real* RU name if it has a RU sitelink,
    or a RU monolingual title (P1476), or RU aliases (community-entered)."""
    if not ent:
        return False
    if _get_ru_sitelink_title(ent):
        return True
    if _get_ru_monolingual_title(ent, "P1476"):
        return True
    # RU aliases as an anchor signal
    try:
        als = ((ent.get("aliases") or {}).get("ru") or [])
        if isinstance(als, list) and len(als) > 0:
            for a in als:
                v = (a or {}).get("value")
                if v and str(v).strip():
                    return True
    except Exception:
        pass
    return False

def _best_popular_ru_name(ent: Optional[dict], fallback_label: str = "") -> Optional[str]:
    """Pick a *popular* RU name for gold display.
    Priority:
      1) RU sitelink title (ruwiki / ruwiktionary / ...)
      2) RU monolingual title P1476 (works)
      3) RU label
      4) RU alias
      5) fallback (only if looks Cyrillic enough)
    Also drops suspicious labels if there is no RU 'anchor'.
    """
    if not ent:
        # Keep fallback only if it looks RU enough
        if fallback_label and _cinema_has_cyrillic(str(fallback_label)):
            return str(fallback_label).strip()
        return None

    # If there is no RU anchor at all, we *prefer to drop* to avoid auto-translations.
    anchored = _has_popular_ru_anchor(ent)

    cands: List[str] = []

    # 1) RU sitelink title
    ru_sl = _get_ru_sitelink_title(ent)
    if ru_sl:
        cands.append(ru_sl)

    # 2) RU monolingual title (P1476)
    ru_t = _get_ru_monolingual_title(ent, "P1476")
    if ru_t:
        cands.append(ru_t)

    # 3) RU label
    ru_lbl = ((ent.get("labels") or {}).get("ru") or {}).get("value")
    if ru_lbl:
        cands.append(str(ru_lbl))

    # 4) RU aliases
    try:
        for a in ((ent.get("aliases") or {}).get("ru") or []) or []:
            v = (a or {}).get("value")
            if v:
                cands.append(str(v))
    except Exception:
        pass

    # normalize + filter
    cands = [c.strip() for c in cands if c and str(c).strip()]
    cands = [c for c in cands if not _CINEMA_BAD_LABEL_RE.fullmatch(c)]

    if not cands:
        return None

    # If anchored, take the first (sitelink/monolingual will dominate)
    if anchored:
        return cands[0]

    # Not anchored: keep only if candidate clearly Cyrillic (last resort)
    for c in cands:
        if _cinema_has_cyrillic(c):
            return c
    return None


Назови 5 фильмов, жанра «комедия», из страны: Германия, в период 1980–1989 годов.
gold_size = 1 truncated = False


<a id="russian-geography-domain"></a>

## 10. Russian geography domain

Stable local templates for Russian geographic entities.

In [10]:
import random
from typing import Optional, List, Tuple, Dict, Any

Q_RUSSIA = ensure_qid("Россия", fallback_qid="Q159")
Q_CITY   = ensure_qid("город", fallback_qid="Q515")
Q_MOSCOW = ensure_qid("Москва", fallback_qid="Q649")
Q_SPB    = ensure_qid("Санкт-Петербург", fallback_qid="Q656")
Q_SEV    = ensure_qid("Севастополь", fallback_qid="Q7525")


def _fmt_people(n: int) -> str:
    if n >= 1_000_000:
        v = n / 1_000_000
        if abs(v - int(v)) < 1e-9:
            return f"{int(v)} млн"
        return f"{v:.1f}".replace('.', ',') + " млн"
    if n % 1000 == 0:
        return f"{n // 1000} тысяч"
    return f"{n:,}".replace(',', ' ')


def _ru_count(n: int, one: str, few: str, many: str) -> str:
    n_abs = abs(int(n))
    n100 = n_abs % 100
    n10 = n_abs % 10
    if 11 <= n100 <= 14:
        return many
    if n10 == 1:
        return one
    if 2 <= n10 <= 4:
        return few
    return many


def _sparql_escape_ru_literal(s: str) -> str:
    return str(s).replace("\\", "\\\\").replace('"', '\\"')


def _values_label_ru(item_var: str, labels: List[str]) -> str:
    vals = " ".join(f'"{_sparql_escape_ru_literal(x)}"@ru' for x in labels)
    return f"VALUES ?{item_var}LabelRu {{ {vals} }}"


def geo_select_items(where_lines: List[str], limit: int = 100, item_var: str = "item") -> Tuple[str, List[Tuple[str, str]]]:
    where = "\n      ".join(where_lines)
    sparql = f"""
    SELECT DISTINCT ?{item_var} ?{item_var}LabelRu WHERE {{
      {where}
      ?{item_var} rdfs:label ?{item_var}LabelRu FILTER(LANG(?{item_var}LabelRu) = "ru") .
    }}
    LIMIT {int(limit)}
    """
    rows = rows_from_select(wd.sparql_select(sparql))
    items: List[Tuple[str, str]] = []
    for r in rows:
        qid = uri_to_qid(r.get(item_var, ""))
        lbl = r.get(f"{item_var}LabelRu")
        if qid and lbl:
            items.append((qid, lbl))
    return sparql.strip(), items


def geo_build_ask_validator(where_lines: List[str], item_var: str = "item") -> str:
    where = "\n      ".join(where_lines)
    return f"""
    ASK WHERE {{
      BIND(wd:{{ITEM}} AS ?{item_var})
      {where}
    }}
    """.strip()


# Curated candidate sets.

SEA_GROUPS: Dict[str, List[str]] = {
    "Северного Ледовитого океана": [
        "Баренцево море", "Белое море", "Карское море",
        "море Лаптевых", "Восточно-Сибирское море", "Чукотское море",
    ],
    "Тихого океана": [
        "Берингово море", "Охотское море", "Японское море",
    ],
    "Атлантического океана": [
        "Балтийское море", "Чёрное море", "Азовское море",
    ],
}

LAKE_GROUPS: List[Dict[str, Any]] = [
    {
        "region_nom": "Республика Карелия",
        "region_prep": "Республике Карелия",
        "labels": ["Сегозеро", "Выгозеро", "Топозеро", "Пяозеро", "Сямозеро", "Водлозеро"],
    },
    {
        "region_nom": "Северо-Запад России",
        "region_prep": "северо-западе России",
        "labels": ["Ладожское озеро", "Онежское озеро", "Ильмень", "Селигер", "Белое озеро"],
    },
]

VOLCANO_GROUP = {
    "region_nom": "Камчатский край",
    "region_prep": "Камчатском крае",
    "labels": [
        "Ключевская сопка", "Корякская сопка", "Авачинская сопка",
        "Кроноцкая сопка", "Мутновская сопка", "Вилючинская сопка",
        "Шивелуч", "Толбачик", "Безымянный",
    ],
}

MOUNTAIN_GROUPS: List[Dict[str, Any]] = [
    {
        "region_nom": "Кабардино-Балкарская Республика",
        "region_prep": "Кабардино-Балкарской Республике",
        "labels": ["Эльбрус", "Дыхтау", "Коштантау", "Пик Пушкина", "Джанги-тау"],
    },
    {
        "region_nom": "Карачаево-Черкесская Республика",
        "region_prep": "Карачаево-Черкесской Республике",
        "labels": ["Домбай-Ульген", "Софруджу", "Семёнов-Баши", "Белалакая"],
    },
    {
        "region_nom": "Республика Алтай",
        "region_prep": "Республике Алтай",
        "labels": ["Белуха", "Маашей-Баш", "Актру-Баш", "Караташ"],
    },
]

PROTECTED_GROUPS: List[Dict[str, Any]] = [
    {
        "location_nom": "Приморский край",
        "location_prep": "Приморском крае",
        "labels": ["Земля леопарда", "Кедровая Падь", "Сихотэ-Алинский заповедник", "Лазовский заповедник"],
    },
    {
        "location_nom": "Краснодарский край",
        "location_prep": "Краснодарском крае",
        "labels": ["Сочинский национальный парк", "Кавказский заповедник", "Утриш"],
    },
    {
        "location_nom": "Республика Карелия",
        "location_prep": "Республике Карелия",
        "labels": ["Паанаярви", "Водлозерский национальный парк", "Кивач", "Костомукшский заповедник"],
    },
    {
        "location_nom": "Камчатский край",
        "location_prep": "Камчатском крае",
        "labels": ["Кроноцкий заповедник", "Налычево", "Быстринский природный парк"],
    },
]

MAJOR_RIVER_CITY_GROUPS: List[Dict[str, Any]] = [
    {
        "river_ru": "Волга",
        "river_prep": "Волге",
        "labels": ["Тверь", "Ярославль", "Кострома", "Нижний Новгород", "Казань", "Самара", "Саратов", "Волгоград", "Астрахань"],
    },
    {
        "river_ru": "Обь",
        "river_prep": "Оби",
        "labels": ["Барнаул", "Новосибирск", "Сургут", "Нижневартовск", "Салехард"],
    },
    {
        "river_ru": "Енисей",
        "river_prep": "Енисее",
        "labels": ["Красноярск", "Енисейск", "Игарка", "Дудинка"],
    },
    {
        "river_ru": "Амур",
        "river_prep": "Амуре",
        "labels": ["Благовещенск", "Хабаровск", "Комсомольск-на-Амуре", "Николаевск-на-Амуре"],
    },
]

TRIBUTARY_GROUPS: List[Dict[str, Any]] = [
    {
        "main_river_ru": "Волга",
        "tributary_labels": ["Кама", "Белая", "Вятка", "Ока", "Которосль"],
        "city_labels": ["Пермь", "Набережные Челны", "Уфа", "Киров", "Калуга", "Рязань", "Ярославль"],
        "admin_center_labels": ["Пермь", "Уфа", "Киров", "Калуга", "Рязань", "Ярославль"],
    },
    {
        "main_river_ru": "Обь",
        "tributary_labels": ["Иртыш", "Томь", "Тобол"],
        "city_labels": ["Омск", "Томск", "Кемерово", "Тобольск"],
        "admin_center_labels": ["Омск", "Томск", "Кемерово"],
    },
]


def _geo_finalize_example(
    *,
    idx: int,
    complexity: str,
    template_id: str,
    template_family: str,
    q_ru: str,
    constraints: Dict[str, Any],
    where_lines: List[str],
    requested_count: int,
    item_var: str = "item",
    limit: int = 100,
) -> Optional[BenchmarkExample]:
    try:
        sparql, items = geo_select_items(where_lines=where_lines, limit=limit, item_var=item_var)
    except Exception:
        return None

    # defensive: dedup by qid
    seen = set()
    items = [(q, l) for q, l in items if q and l and not (q in seen or seen.add(q))]

    n = len(items)
    if complexity in ("L1", "L2", "L3") and n < requested_count:
        return None
    if complexity in ("L4", "L5") and n == 0:
        return None

    actual_k = min(requested_count, n) if n > 0 else requested_count
    q_text = q_ru
    if n > 0:
        if n > actual_k:
            q_text += " Перечисли любые из них."
        else:
            q_text += " Перечисли все подходящие."

    ask = None
    try:
        ask = geo_build_ask_validator(where_lines=where_lines, item_var=item_var)
    except Exception:
        pass

    return BenchmarkExample(
        id=f"geo_ru_{complexity.lower()}_{idx:04d}",
        domain="geo_ru",
        complexity=complexity,
        query_text_ru=q_text,
        constraints={**constraints, "template_id": template_id, "template_family": template_family},
        requested_count=actual_k,
        gold_answer_qids=[q for q, _ in items[:limit]],
        gold_answer_labels_ru=[l for _, l in items[:limit]],
        sparql_query=sparql,
        created_at=utc_now_z(),
        gold_truncated=(n >= limit),
        ask_validator_sparql=ask,
        template_id=template_id,
        template_family=template_family,
        is_advanced=(complexity in ("L3", "L4", "L5")),
    )


def _tpl_seas_by_basin(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in ("L1", "L2"):
        return None
    available = [(basin, labels) for basin, labels in SEA_GROUPS.items() if len(labels) >= 3]
    if not available:
        return None

    basin_name, labels = rng.choice(available)
    k = min(rng.choice([3, 4]), len(labels))
    q_ru = f"Назови {k} моря, которые омывают Россию и относятся к бассейну {basin_name}."
    where = [_values_label_ru("item", labels)]

    return _geo_finalize_example(
        idx=idx,
        complexity=complexity,
        template_id="geo_sea_basin_values",
        template_family="seas",
        q_ru=q_ru,
        constraints={"basin_ru": basin_name, "candidate_labels": labels},
        where_lines=where,
        requested_count=k,
        item_var="item",
        limit=40,
    )


def _tpl_lakes_region(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L1":
        return None
    grp = rng.choice(LAKE_GROUPS)
    labels = list(grp["labels"])
    if len(labels) < 4:
        return None

    k = rng.choice([4, 5])
    q_ru = f"Назови {k} озёр России, расположенных в {grp['region_prep']}."
    where = [_values_label_ru("item", labels)]

    return _geo_finalize_example(
        idx=idx,
        complexity=complexity,
        template_id="geo_lake_region_candidates",
        template_family="lakes",
        q_ru=q_ru,
        constraints={"region_ru": grp["region_nom"], "candidate_labels": labels},
        where_lines=where,
        requested_count=min(k, len(labels)),
        item_var="item",
        limit=40,
    )


def _tpl_volcanoes_height(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    labels = list(VOLCANO_GROUP["labels"])
    if len(labels) < 3:
        return None

    if complexity == "L1":
        elev_min = rng.choice([1200, 1500, 1800])
        k = rng.choice([3, 4])
        template_id = "geo_volcano_kamchatka_height_easy"
    elif complexity == "L3":
        elev_min = rng.choice([2000, 2500, 3000])
        k = rng.choice([3, 4])
        template_id = "geo_volcano_kamchatka_height_mid"
    elif complexity == "L5":
        elev_min = rng.choice([3500, 4000])
        k = rng.choice([2, 3])
        template_id = "geo_volcano_kamchatka_height_hard"
    else:
        return None

    noun = _ru_count(k, "вулкан", "вулкана", "вулканов")
    q_ru = f"Перечисли {k} {noun} России, расположенных в {VOLCANO_GROUP['region_prep']} и имеющих высоту не ниже {elev_min} м."
    where = [
        _values_label_ru("item", labels),
        "?item wdt:P2044 ?elev .",
        f"FILTER(?elev >= {int(elev_min)}) .",
    ]

    return _geo_finalize_example(
        idx=idx,
        complexity=complexity,
        template_id=template_id,
        template_family="volcanoes",
        q_ru=q_ru,
        constraints={"region_ru": VOLCANO_GROUP["region_nom"], "elev_min_m": elev_min, "candidate_labels": labels},
        where_lines=where,
        requested_count=k,
        item_var="item",
        limit=40,
    )


def _tpl_mountains_height(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in ("L2", "L4"):
        return None

    grp = rng.choice([g for g in MOUNTAIN_GROUPS if len(g["labels"]) >= 3])
    labels = list(grp["labels"])

    if complexity == "L2":
        elev_min = rng.choice([3000, 3500])
        k = rng.choice([3, 4])
        template_id = "geo_mountain_region_height"
    else:
        elev_min = rng.choice([3800, 4200])
        k = rng.choice([2, 3])
        template_id = "geo_mountain_region_height_hard"

    noun = _ru_count(k, "гору", "горы", "гор")
    q_ru = f"Перечисли {k} {noun} России, расположенных в {grp['region_prep']} и имеющих высоту не ниже {elev_min} м."
    where = [
        _values_label_ru("item", labels),
        "?item wdt:P2044 ?elev .",
        f"FILTER(?elev >= {int(elev_min)}) .",
    ]

    return _geo_finalize_example(
        idx=idx,
        complexity=complexity,
        template_id=template_id,
        template_family="mountains",
        q_ru=q_ru,
        constraints={"region_ru": grp["region_nom"], "elev_min_m": elev_min, "candidate_labels": labels},
        where_lines=where,
        requested_count=k,
        item_var="item",
        limit=40,
    )


def _tpl_protected_areas(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in ("L2", "L4"):
        return None

    grp = rng.choice([g for g in PROTECTED_GROUPS if len(g["labels"]) >= 3])
    labels = list(grp["labels"])

    if complexity == "L2":
        year_max = rng.choice([1995, 2005, 2010])
        k = rng.choice([3, 4])
        template_id = "geo_protected_area_before_year"
    else:
        year_max = rng.choice([1990, 2000])
        k = rng.choice([2, 3])
        template_id = "geo_protected_area_before_year_hard"

    q_ru = f"Назови {k} национальных парков или заповедников России, расположенных в {grp['location_prep']} и созданных не позже {year_max} года."
    where = [
        _values_label_ru("item", labels),
        "?item wdt:P571 ?date .",
        f"FILTER(YEAR(?date) <= {int(year_max)}) .",
    ]

    return _geo_finalize_example(
        idx=idx,
        complexity=complexity,
        template_id=template_id,
        template_family="protected_areas",
        q_ru=q_ru,
        constraints={"location_ru": grp["location_nom"], "year_max": year_max, "candidate_labels": labels},
        where_lines=where,
        requested_count=min(k, len(labels)),
        item_var="item",
        limit=40,
    )


def _tpl_cities_on_major_river(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in ("L1", "L2"):
        return None

    grp = rng.choice([g for g in MAJOR_RIVER_CITY_GROUPS if len(g["labels"]) >= 4])
    labels = list(grp["labels"])

    pop_min = rng.choice([100_000, 200_000, 300_000])
    k = rng.choice([4, 5])

    if complexity == "L1":
        q_ru = f"Назови {k} городов России, стоящих на {grp['river_prep']}, с населением не менее {_fmt_people(pop_min)} человек."
        template_id = "geo_city_major_river_pop"
    else:
        q_ru = f"Назови {k} городов России, стоящих на {grp['river_prep']}, с населением не менее {_fmt_people(pop_min)} человек, которые не являются городами федерального значения."
        template_id = "geo_city_major_river_pop_not_federal"

    where = [
        _values_label_ru("city", labels),
        f"?city wdt:P17 wd:{Q_RUSSIA} .",
        f"?city wdt:P31/wdt:P279* wd:{Q_CITY} .",
        "?city wdt:P1082 ?pop .",
        f"FILTER(?pop >= {int(pop_min)}) .",
    ]
    if complexity == "L2":
        where.append(f"FILTER(?city != wd:{Q_MOSCOW} && ?city != wd:{Q_SPB} && ?city != wd:{Q_SEV}) .")

    return _geo_finalize_example(
        idx=idx,
        complexity=complexity,
        template_id=template_id,
        template_family="rivers",
        q_ru=q_ru,
        constraints={"river_ru": grp["river_ru"], "pop_min": pop_min, "candidate_labels": labels},
        where_lines=where,
        requested_count=min(k, len(labels)),
        item_var="city",
        limit=50,
    )


def _tpl_cities_on_tributaries(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in ("L3", "L5"):
        return None

    grp = rng.choice(TRIBUTARY_GROUPS)
    main_river_ru = grp["main_river_ru"]

    if complexity == "L3":
        labels = list(grp["city_labels"])
        pop_min = rng.choice([80_000, 100_000, 150_000])
        k = rng.choice([3, 4])
        q_ru = f"Назови {k} городов России с населением не менее {_fmt_people(pop_min)} человек, стоящих на реках, которые впадают в {main_river_ru}."
        template_id = "geo_city_tributaries_pop"
        where = [
            _values_label_ru("city", labels),
            f"?city wdt:P17 wd:{Q_RUSSIA} .",
            f"?city wdt:P31/wdt:P279* wd:{Q_CITY} .",
            "?city wdt:P1082 ?pop .",
            f"FILTER(?pop >= {int(pop_min)}) .",
        ]
        constraints = {"main_river_ru": main_river_ru, "pop_min": pop_min, "candidate_labels": labels, "tributary_labels": grp["tributary_labels"]}
        req = min(k, len(labels))
    else:
        labels = list(grp["admin_center_labels"])
        pop_max = rng.choice([900_000, 1_000_000, 1_200_000])
        k = rng.choice([2, 3])
        q_ru = f"Назови {k} административных центров субъектов РФ, стоящих на реках, которые впадают в {main_river_ru}, если население города меньше {_fmt_people(pop_max)} человек."
        template_id = "geo_admin_center_tributaries_popmax"
        where = [
            _values_label_ru("city", labels),
            f"?city wdt:P17 wd:{Q_RUSSIA} .",
            f"?city wdt:P31/wdt:P279* wd:{Q_CITY} .",
            "?city wdt:P1082 ?pop .",
            f"FILTER(?pop < {int(pop_max)}) .",
        ]
        constraints = {"main_river_ru": main_river_ru, "pop_max": pop_max, "candidate_labels": labels, "tributary_labels": grp["tributary_labels"]}
        req = min(k, len(labels))

    return _geo_finalize_example(
        idx=idx,
        complexity=complexity,
        template_id=template_id,
        template_family="rivers_multihop",
        q_ru=q_ru,
        constraints=constraints,
        where_lines=where,
        requested_count=req,
        item_var="city",
        limit=50,
    )


GEO_RU_TEMPLATE_FNS: Dict[str, List[Any]] = {
    "L1": [_tpl_seas_by_basin, _tpl_lakes_region, _tpl_volcanoes_height, _tpl_cities_on_major_river],
    "L2": [_tpl_seas_by_basin, _tpl_mountains_height, _tpl_protected_areas, _tpl_cities_on_major_river],
    "L3": [_tpl_cities_on_tributaries, _tpl_volcanoes_height],
    "L4": [_tpl_mountains_height, _tpl_protected_areas],
    "L5": [_tpl_cities_on_tributaries, _tpl_volcanoes_height],
}


def generate_geo_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 60) -> BenchmarkExample:
    template_fns = list(GEO_RU_TEMPLATE_FNS.get(complexity, []))
    if not template_fns:
        raise ValueError(f"Unknown complexity: {complexity}")

    for _ in range(max_attempts):
        rng.shuffle(template_fns)
        for fn in template_fns:
            try:
                ex = fn(complexity, idx, rng)
            except Exception:
                ex = None
            if ex is not None and getattr(ex, "gold_answer_qids", None):
                return ex

    return BenchmarkExample(
        id=f"geo_ru_fail_{complexity.lower()}_{idx:04d}",
        domain="geo_ru",
        complexity=complexity,
        query_text_ru="(ошибка генерации geo_ru)",
        constraints={"failed": True, "complexity": complexity},
        requested_count=3,
        gold_answer_qids=[],
        gold_answer_labels_ru=[],
        sparql_query="",
        created_at=utc_now_z(),
        template_id="geo_ru_failed",
        template_family="failed",
    )


if "DOMAIN_GENERATORS" in globals():
    DOMAIN_GENERATORS["geo_ru"] = generate_geo_example
print("geo_ru generator ready: fast curated templates")

# Geo_RU PATCH NOTE

generate_geo_ru_example = generate_geo_example
if "DOMAIN_GENERATORS" in globals():
    DOMAIN_GENERATORS["geo_ru"] = generate_geo_example
print("geo_ru generator synced -> generate_geo_example")


geo_ru generator ready: fast curated templates
geo_ru generator synced -> generate_geo_example


<a id="international-geography-domain"></a>

## 11. International geography domain

Global geographic templates for oceans, seas, lakes, waterfalls, deserts, mountains, rivers, and islands.

In [11]:
import random
from typing import Optional, List, Tuple, Dict, Any
import re

Q_GEO_OCEAN = ensure_qid("океан", fallback_qid="Q9430")
Q_GEO_SEA = ensure_qid("море", fallback_qid="Q165")
Q_GEO_LAKE = ensure_qid("озеро", fallback_qid="Q23397")
Q_GEO_MOUNTAIN = ensure_qid("гора", fallback_qid="Q8502")
Q_GEO_VOLCANO = ensure_qid("вулкан", fallback_qid="Q8072")
Q_GEO_WATERFALL = ensure_qid("водопад", fallback_qid="Q34038")
Q_GEO_DESERT = ensure_qid("пустыня", fallback_qid="Q8514")
Q_GEO_FOREST = ensure_qid("лес", fallback_qid="Q4421")
Q_GEO_CANYON = ensure_qid("каньон", fallback_qid="Q150784")

_GW_QID_CACHE: Dict[str, Optional[str]] = {}


def _gw_norm_label(lbl: str) -> str:
    s = (lbl or "").strip()
    s = re.sub(r"\s+", " ", s)
    s = s.replace("’", "'").replace("`", "'")
    return s.casefold()


def _gw_ru_count(n: int, one: str, few: str, many: str) -> str:
    n_abs = abs(int(n))
    n100 = n_abs % 100
    n10 = n_abs % 10
    if 11 <= n100 <= 14:
        return many
    if n10 == 1:
        return one
    if 2 <= n10 <= 4:
        return few
    return many


def _gw_escape_ru_literal(s: str) -> str:
    return str(s).replace("\\", "\\\\").replace('"', '\\"')


def _gw_resolve_qid_cached(search_label: str, fallback_qid: Optional[str] = None) -> Optional[str]:
    key = (search_label or "").strip()
    if not key:
        return fallback_qid
    if key in _GW_QID_CACHE:
        return _GW_QID_CACHE[key] or fallback_qid
    qid = None
    try:
        qid = resolve_qid(key, "ru") or resolve_qid(key, "en") or fallback_qid
    except Exception:
        qid = fallback_qid
    _GW_QID_CACHE[key] = qid
    return qid


def _gw_build_items(raw_items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    seen_qids = set()
    seen_labels = set()
    for it in raw_items:
        search_label = (it.get("search_label") or it.get("label") or "").strip()
        label = (it.get("label") or search_label).strip()
        qid = it.get("qid") or _gw_resolve_qid_cached(search_label, it.get("fallback_qid"))
        if not qid or not label:
            continue
        nl = _gw_norm_label(label)
        if qid in seen_qids or nl in seen_labels:
            continue
        seen_qids.add(qid)
        seen_labels.add(nl)
        obj = dict(it)
        obj["qid"] = qid
        obj["label"] = label
        obj["search_label"] = search_label
        out.append(obj)
    return out


def _gw_pick_rotated(seq: List[Any], idx: int, rng: random.Random, salt: int = 0) -> Any:
    if not seq:
        raise ValueError("empty sequence")
    base = (idx + salt) % len(seq)
    shift = rng.randrange(len(seq)) if len(seq) > 1 else 0
    return seq[(base + shift) % len(seq)]


def _gw_make_values_sparql(
    *,
    item_var: str,
    items: List[Dict[str, Any]],
    type_qid: Optional[str] = None,
    extra_lines: Optional[List[str]] = None,
    limit: Optional[int] = None,
) -> str:
    values_rows = []
    for it in items:
        label = _gw_escape_ru_literal(it["label"])
        values_rows.append(f'(wd:{it["qid"]} "{label}"@ru)')
    values_block = "\n        ".join(values_rows)
    extra = "\n      ".join(extra_lines or [])
    type_line = f"?{item_var} wdt:P31/wdt:P279* wd:{type_qid} ." if type_qid else ""
    limit_line = f"\n    LIMIT {int(limit)}" if limit else ""
    return f"""
    SELECT DISTINCT ?{item_var} ?{item_var}Label WHERE {{
      VALUES (?{item_var} ?{item_var}Label) {{
        {values_block}
      }}
      {type_line}
      {extra}
    }}{limit_line}
    """.strip()


def _gw_make_values_ask(
    *,
    item_var: str,
    items: List[Dict[str, Any]],
    type_qid: Optional[str] = None,
    extra_lines: Optional[List[str]] = None,
) -> str:
    values_rows = " ".join(f"wd:{it['qid']}" for it in items)
    extra = "\n      ".join(extra_lines or [])
    type_line = f"?{item_var} wdt:P31/wdt:P279* wd:{type_qid} ." if type_qid else ""
    return f"""
    ASK WHERE {{
      BIND(wd:{{ITEM}} AS ?{item_var})
      VALUES ?{item_var} {{ {values_rows} }}
      {type_line}
      {extra}
    }}
    """.strip()


def _gw_finalize_static_example(
    *,
    idx: int,
    complexity: str,
    template_id: str,
    template_family: str,
    q_ru: str,
    constraints: Dict[str, Any],
    eligible_items: List[Dict[str, Any]],
    requested_count: int,
    type_qid: Optional[str] = None,
    extra_lines: Optional[List[str]] = None,
) -> Optional[BenchmarkExample]:
    seen_qids = set()
    seen_labels = set()
    items: List[Dict[str, Any]] = []
    for it in eligible_items:
        qid = it.get("qid")
        lbl = it.get("label")
        if not qid or not lbl:
            continue
        nl = _gw_norm_label(lbl)
        if qid in seen_qids or nl in seen_labels:
            continue
        seen_qids.add(qid)
        seen_labels.add(nl)
        items.append(it)

    n = len(items)
    if complexity in ("L1", "L2", "L3") and n < requested_count:
        return None
    if complexity in ("L4", "L5") and n == 0:
        return None

    actual_k = min(requested_count, n) if n > 0 else requested_count
    q_text = q_ru

    sparql = _gw_make_values_sparql(
        item_var="item",
        items=items,
        type_qid=type_qid,
        extra_lines=extra_lines,
        limit=max(10, len(items)),
    )
    ask = _gw_make_values_ask(
        item_var="item",
        items=items,
        type_qid=type_qid,
        extra_lines=extra_lines,
    )

    return BenchmarkExample(
        id=f"geo_{complexity.lower()}_{idx:04d}",
        domain="geo",
        complexity=complexity,
        query_text_ru=q_text,
        constraints={**constraints, "template_id": template_id, "template_family": template_family},
        requested_count=actual_k,
        gold_answer_qids=[it["qid"] for it in items],
        gold_answer_labels_ru=[it["label"] for it in items],
        sparql_query=sparql,
        created_at=utc_now_z(),
        gold_truncated=False,
        ask_validator_sparql=ask,
        template_id=template_id,
        template_family=template_family,
        is_advanced=(complexity in ("L3", "L4", "L5")),
    )


def _gw_with_items(group: Dict[str, Any]) -> Dict[str, Any]:
    g = dict(group)
    g["items"] = _gw_build_items(group.get("items_raw", []))
    return g


OCEAN_BY_CONTINENT_GROUPS = [
    _gw_with_items({
        "continent_acc": "Африку",
        "continent_nom": "Африка",
        "items_raw": [
            {"search_label": "Атлантический океан", "label": "Атлантический океан"},
            {"search_label": "Индийский океан", "label": "Индийский океан"},
        ],
    }),
    _gw_with_items({
        "continent_acc": "Европу",
        "continent_nom": "Европа",
        "items_raw": [
            {"search_label": "Атлантический океан", "label": "Атлантический океан"},
            {"search_label": "Северный Ледовитый океан", "label": "Северный Ледовитый океан"},
        ],
    }),
    _gw_with_items({
        "continent_acc": "Азию",
        "continent_nom": "Азия",
        "items_raw": [
            {"search_label": "Тихий океан", "label": "Тихий океан"},
            {"search_label": "Индийский океан", "label": "Индийский океан"},
            {"search_label": "Северный Ледовитый океан", "label": "Северный Ледовитый океан"},
        ],
    }),
    _gw_with_items({
        "continent_acc": "Северную Америку",
        "continent_nom": "Северная Америка",
        "items_raw": [
            {"search_label": "Тихий океан", "label": "Тихий океан"},
            {"search_label": "Атлантический океан", "label": "Атлантический океан"},
            {"search_label": "Северный Ледовитый океан", "label": "Северный Ледовитый океан"},
        ],
    }),
    _gw_with_items({
        "continent_acc": "Южную Америку",
        "continent_nom": "Южная Америка",
        "items_raw": [
            {"search_label": "Тихий океан", "label": "Тихий океан"},
            {"search_label": "Атлантический океан", "label": "Атлантический океан"},
        ],
    }),
    _gw_with_items({
        "continent_acc": "Австралию",
        "continent_nom": "Австралия",
        "items_raw": [
            {"search_label": "Тихий океан", "label": "Тихий океан"},
            {"search_label": "Индийский океан", "label": "Индийский океан"},
        ],
    }),
]

OCEAN_MULTI_CONTINENT_GROUPS = [
    _gw_with_items({
        "continent1_acc": "Северную Америку",
        "continent2_acc": "Азию",
        "continent1_nom": "Северная Америка",
        "continent2_nom": "Азия",
        "items_raw": [
            {"search_label": "Тихий океан", "label": "Тихий океан"},
            {"search_label": "Северный Ледовитый океан", "label": "Северный Ледовитый океан"},
        ],
    }),
    _gw_with_items({
        "continent1_acc": "Европу",
        "continent2_acc": "Северную Америку",
        "continent1_nom": "Европа",
        "continent2_nom": "Северная Америка",
        "items_raw": [
            {"search_label": "Атлантический океан", "label": "Атлантический океан"},
            {"search_label": "Северный Ледовитый океан", "label": "Северный Ледовитый океан"},
        ],
    }),
    _gw_with_items({
        "continent1_acc": "Северную Америку",
        "continent2_acc": "Южную Америку",
        "continent1_nom": "Северная Америка",
        "continent2_nom": "Южная Америка",
        "items_raw": [
            {"search_label": "Тихий океан", "label": "Тихий океан"},
            {"search_label": "Атлантический океан", "label": "Атлантический океан"},
        ],
    }),
]

SEA_REGION_GROUPS = [
    _gw_with_items({
        "region_nom": "Средиземноморье",
        "region_prep": "в Средиземноморье",
        "basin_gen": "Атлантического океана",
        "descriptor_clause_l4": "которые обычно рассматривают как основные моря Средиземноморского региона",
        "items_raw": [
            {"search_label": "Адриатическое море", "label": "Адриатическое море"},
            {"search_label": "Эгейское море", "label": "Эгейское море"},
            {"search_label": "Ионическое море", "label": "Ионическое море"},
            {"search_label": "Лигурийское море", "label": "Лигурийское море"},
            {"search_label": "Тирренское море", "label": "Тирренское море"},
            {"search_label": "Балеарское море", "label": "Балеарское море"},
            {"search_label": "Альборанское море", "label": "Альборанское море"},
            {"search_label": "Средиземное море", "label": "Средиземное море"},
        ],
    }),
    _gw_with_items({
        "region_nom": "Восточная и Юго-Восточная Азия",
        "region_prep": "у берегов Восточной и Юго-Восточной Азии",
        "basin_gen": "Тихого океана",
        "descriptor_clause_l4": "которые обычно относят к окраинным морям Тихого океана",
        "items_raw": [
            {"search_label": "Японское море", "label": "Японское море"},
            {"search_label": "Охотское море", "label": "Охотское море"},
            {"search_label": "Жёлтое море", "label": "Жёлтое море"},
            {"search_label": "Восточно-Китайское море", "label": "Восточно-Китайское море"},
            {"search_label": "Южно-Китайское море", "label": "Южно-Китайское море"},
            {"search_label": "Филиппинское море", "label": "Филиппинское море"},
        ],
    }),
    _gw_with_items({
        "region_nom": "северо-запад Индийского океана",
        "region_prep": "в северо-западной части Индийского океана",
        "basin_gen": "Индийского океана",
        "descriptor_clause_l4": "которые обычно относят к морям северной части Индийского океана",
        "items_raw": [
            {"search_label": "Красное море", "label": "Красное море"},
            {"search_label": "Аравийское море", "label": "Аравийское море"},
            {"search_label": "Андаманское море", "label": "Андаманское море"},
            {"search_label": "Лаккадивское море", "label": "Лаккадивское море"},
        ],
    }),
    _gw_with_items({
        "region_nom": "Карибский бассейн",
        "region_prep": "в Карибском бассейне",
        "basin_gen": "Атлантического океана",
        "descriptor_clause_l4": "которые обычно относят к морям Атлантического бассейна Карибского региона",
        "items_raw": [
            {"search_label": "Карибское море", "label": "Карибское море"},
            {"search_label": "Саргассово море", "label": "Саргассово море"},
        ],
    }),
]

LAKE_WORLD_GROUPS = [
    _gw_with_items({
        "region_nom": "Северная Америка",
        "region_prep": "в Северной Америке",
        "group_name": "Великие озёра Северной Америки",
        "descriptor_clause_l3": "входящих в число крупнейших пресноводных озёр этого макрорегиона",
        "items_raw": [
            {"search_label": "озеро Верхнее", "label": "Верхнее"},
            {"search_label": "озеро Мичиган", "label": "Мичиган"},
            {"search_label": "озеро Гурон", "label": "Гурон"},
            {"search_label": "озеро Эри", "label": "Эри"},
            {"search_label": "озеро Онтарио", "label": "Онтарио"},
        ],
    }),
    _gw_with_items({
        "region_nom": "Восточная Африка",
        "region_prep": "в Восточной Африке",
        "group_name": "Великие Африканские озёра",
        "descriptor_clause_l3": "входящих в число крупнейших озёр Восточной Африки",
        "items_raw": [
            {"search_label": "озеро Виктория", "label": "Виктория"},
            {"search_label": "озеро Танганьика", "label": "Танганьика"},
            {"search_label": "озеро Малави", "label": "Малави"},
            {"search_label": "озеро Киву", "label": "Киву"},
            {"search_label": "озеро Альберт", "label": "Альберт"},
            {"search_label": "озеро Эдуард", "label": "Эдуард"},
            {"search_label": "озеро Туркана", "label": "Туркана"},
        ],
    }),
    _gw_with_items({
        "region_nom": "Скандинавия",
        "region_prep": "в Скандинавии",
        "group_name": "крупные озёра Скандинавии",
        "descriptor_clause_l3": "которые обычно относят к крупнейшим озёрам Скандинавии",
        "items_raw": [
            {"search_label": "озеро Венерн", "label": "Венерн"},
            {"search_label": "озеро Веттерн", "label": "Веттерн"},
            {"search_label": "озеро Сайма", "label": "Сайма"},
            {"search_label": "озеро Меларен", "label": "Меларен"},
            {"search_label": "озеро Инари", "label": "Инари"},
        ],
    }),
]

WATERFALL_GROUPS = [
    _gw_with_items({
        "region_nom": "Южная Америка",
        "region_prep": "в Южной Америке",
        "descriptor_clause_l2": "входящих в число наиболее известных водопадов Южной Америки",
        "descriptor_clause_l3": "которые обычно относят к крупным и наиболее известным водопадам континента",
        "items_raw": [
            {"search_label": "водопад Анхель", "label": "Анхель"},
            {"search_label": "водопады Игуасу", "label": "Игуасу"},
            {"search_label": "водопад Кайетур", "label": "Кайетур"},
        ],
    }),
    _gw_with_items({
        "region_nom": "Африка",
        "region_prep": "в Африке",
        "descriptor_clause_l2": "входящих в число наиболее известных водопадов Африки",
        "descriptor_clause_l3": "которые обычно относят к крупным и наиболее известным водопадам континента",
        "items_raw": [
            {"search_label": "водопад Виктория", "label": "Водопад Виктория"},
            {"search_label": "водопад Тугела", "label": "Тугела"},
            {"search_label": "водопад Каламбо", "label": "Каламбо"},
        ],
    }),
    _gw_with_items({
        "region_nom": "Северная Америка",
        "region_prep": "в Северной Америке",
        "descriptor_clause_l2": "входящих в число наиболее известных водопадов Северной Америки",
        "descriptor_clause_l3": "которые обычно относят к наиболее известным природным водопадам континента",
        "items_raw": [
            {"search_label": "Ниагарский водопад", "label": "Ниагара"},
            {"search_label": "водопад Йосемити", "label": "Йосемити"},
            {"search_label": "водопад Малтнома", "label": "Малтнома"},
        ],
    }),
]

MOUNTAIN_WORLD_GROUPS = [
    _gw_with_items({
        "system_nom": "Альпы",
        "system_gen": "Альп",
        "system_prep": "в Альпах",
        "macroregion_nom": "Европа",
        "macroregion_prep": "в Европе",
        "items_raw": [
            {"search_label": "Монблан", "label": "Монблан", "elev_m": 4808},
            {"search_label": "Маттерхорн", "label": "Маттерхорн", "elev_m": 4478},
            {"search_label": "Дюфуршпитце", "label": "Дюфуршпитце", "elev_m": 4634},
            {"search_label": "Дом", "label": "Дом", "elev_m": 4545},
            {"search_label": "Вайсхорн", "label": "Вайсхорн", "elev_m": 4506},
            {"search_label": "Финстераархорн", "label": "Финстераархорн", "elev_m": 4274},
            {"search_label": "Юнгфрау", "label": "Юнгфрау", "elev_m": 4158},
            {"search_label": "Гран-Парадизо", "label": "Гран-Парадизо", "elev_m": 4061},
        ],
    }),
    _gw_with_items({
        "system_nom": "Гималаи",
        "system_gen": "Гималаев",
        "system_prep": "в Гималаях",
        "macroregion_nom": "Азия",
        "macroregion_prep": "в Азии",
        "items_raw": [
            {"search_label": "Эверест", "label": "Эверест", "elev_m": 8848},
            {"search_label": "Канченджанга", "label": "Канченджанга", "elev_m": 8586},
            {"search_label": "Лхоцзе", "label": "Лхоцзе", "elev_m": 8516},
            {"search_label": "Макалу", "label": "Макалу", "elev_m": 8485},
            {"search_label": "Чо-Ойю", "label": "Чо-Ойю", "elev_m": 8188},
            {"search_label": "Дхаулагири", "label": "Дхаулагири", "elev_m": 8167},
            {"search_label": "Манаслу", "label": "Манаслу", "elev_m": 8163},
            {"search_label": "Нангапарбат", "label": "Нангапарбат", "elev_m": 8126},
            {"search_label": "Аннапурна", "label": "Аннапурна", "elev_m": 8091},
            {"search_label": "Шишабангма", "label": "Шишабангма", "elev_m": 8027},
        ],
    }),
    _gw_with_items({
        "system_nom": "Анды",
        "system_gen": "Анд",
        "system_prep": "в Андах",
        "macroregion_nom": "Южная Америка",
        "macroregion_prep": "в Южной Америке",
        "items_raw": [
            {"search_label": "Аконкагуа", "label": "Аконкагуа", "elev_m": 6961},
            {"search_label": "Охос-дель-Саладо", "label": "Охос-дель-Саладо", "elev_m": 6893},
            {"search_label": "Монте-Писсис", "label": "Монте-Писсис", "elev_m": 6795},
            {"search_label": "Уаскаран", "label": "Уаскаран", "elev_m": 6768},
            {"search_label": "Сахама", "label": "Сахама", "elev_m": 6542},
            {"search_label": "Ильямпу", "label": "Ильямпу", "elev_m": 6368},
            {"search_label": "Чимборасо", "label": "Чимборасо", "elev_m": 6263},
        ],
    }),
]

VOLCANO_WORLD_GROUPS = [
    _gw_with_items({
        "region_nom": "Япония",
        "region_prep": "в Японии",
        "context_clause": "относящихся к Тихоокеанскому вулканическому поясу",
        "items_raw": [
            {"search_label": "Фудзи", "label": "Фудзи", "elev_m": 3776},
            {"search_label": "Асо", "label": "Асо", "elev_m": 1592},
            {"search_label": "Асама", "label": "Асама", "elev_m": 2568},
            {"search_label": "Онтакэ", "label": "Онтакэ", "elev_m": 3067},
            {"search_label": "Сакурадзима", "label": "Сакурадзима", "elev_m": 1117},
            {"search_label": "Ундзэн", "label": "Ундзэн", "elev_m": 1483},
        ],
    }),
    _gw_with_items({
        "region_nom": "Индонезия",
        "region_prep": "в Индонезии",
        "context_clause": "относящихся к Тихоокеанскому вулканическому поясу",
        "items_raw": [
            {"search_label": "Мерапи", "label": "Мерапи", "elev_m": 2930},
            {"search_label": "Семеру", "label": "Семеру", "elev_m": 3676},
            {"search_label": "Тамбора", "label": "Тамбора", "elev_m": 2850},
            {"search_label": "Ринджани", "label": "Ринджани", "elev_m": 3726},
            {"search_label": "Агунг", "label": "Агунг", "elev_m": 3031},
            {"search_label": "Бромо", "label": "Бромо", "elev_m": 2329},
        ],
    }),
    _gw_with_items({
        "region_nom": "Италия",
        "region_prep": "в Италии",
        "context_clause": "относящихся к Средиземноморскому вулканическому региону",
        "items_raw": [
            {"search_label": "Этна", "label": "Этна", "elev_m": 3357},
            {"search_label": "Везувий", "label": "Везувий", "elev_m": 1281},
            {"search_label": "Стромболи", "label": "Стромболи", "elev_m": 924},
            {"search_label": "Вулькано", "label": "Вулькано", "elev_m": 500},
        ],
    }),
    _gw_with_items({
        "region_nom": "Исландия",
        "region_prep": "в Исландии",
        "context_clause": "относящихся к Северо-Атлантической рифтовой зоне",
        "items_raw": [
            {"search_label": "Гекла", "label": "Гекла", "elev_m": 1491},
            {"search_label": "Катла", "label": "Катла", "elev_m": 1512},
            {"search_label": "Аскья", "label": "Аскья", "elev_m": 1516},
            {"search_label": "Эйяфьятлайокудль", "label": "Эйяфьятлайокудль", "elev_m": 1651},
            {"search_label": "Снайфедльсйёкюдль", "label": "Снайфедльсйёкюдль", "elev_m": 1446},
        ],
    }),
]

DESERT_GROUPS = [
    _gw_with_items({
        "region_nom": "Африка",
        "region_prep": "в Африке",
        "descriptor_clause_l2": "относящихся к крупнейшим пустыням континента",
        "descriptor_clause_l3": "считающихся одними из самых известных пустынь Африки",
        "items_raw": [
            {"search_label": "Сахара", "label": "Сахара"},
            {"search_label": "Калахари", "label": "Калахари"},
            {"search_label": "Намиб", "label": "Намиб"},
            {"search_label": "Данакильская пустыня", "label": "Данакильская пустыня"},
        ],
    }),
    _gw_with_items({
        "region_nom": "Центральная и Восточная Азия",
        "region_prep": "в Центральной и Восточной Азии",
        "descriptor_clause_l2": "относящихся к крупным внутриконтинентальным пустыням Евразии",
        "descriptor_clause_l3": "считающихся одними из самых известных пустынь этого макрорегиона",
        "items_raw": [
            {"search_label": "Гоби", "label": "Гоби"},
            {"search_label": "Такла-Макан", "label": "Такла-Макан"},
            {"search_label": "Каракумы", "label": "Каракумы"},
            {"search_label": "Кызылкум", "label": "Кызылкум"},
        ],
    }),
    _gw_with_items({
        "region_nom": "Северная Америка",
        "region_prep": "в Северной Америке",
        "descriptor_clause_l2": "относящихся к основным пустыням засушливого юго-запада континента",
        "descriptor_clause_l3": "считающихся одними из самых известных пустынь региона",
        "items_raw": [
            {"search_label": "пустыня Мохаве", "label": "Мохаве"},
            {"search_label": "пустыня Сонора", "label": "Сонора"},
            {"search_label": "Чиуауанская пустыня", "label": "Чиуауанская пустыня"},
            {"search_label": "пустыня Большого Бассейна", "label": "Пустыня Большого Бассейна"},
        ],
    }),
    _gw_with_items({
        "region_nom": "Южная Америка",
        "region_prep": "в Южной Америке",
        "descriptor_clause_l2": "относящихся к крупным пустыням западной и южной части континента",
        "descriptor_clause_l3": "считающихся одними из самых известных пустынь Южной Америки",
        "items_raw": [
            {"search_label": "Атакама", "label": "Атакама"},
            {"search_label": "Патагонская пустыня", "label": "Патагонская пустыня"},
        ],
    }),
]

FOREST_GROUPS = [
    _gw_with_items({
        "region_nom": "Европа",
        "region_prep": "в Европе",
        "biome_clause_l2": "относящихся к известным лесным регионам умеренной зоны Европы",
        "descriptor_clause_l3": "часто рассматриваемых как заметные природные регионы континента",
        "descriptor_clause_l4": "которые одновременно относятся к известным лесным регионам умеренной зоны и часто упоминаются как заметные природные территории Европы",
        "items_raw": [
            {"search_label": "Шварцвальд", "label": "Шварцвальд"},
            {"search_label": "Беловежская пуща", "label": "Беловежская пуща"},
            {"search_label": "Шервудский лес", "label": "Шервудский лес"},
            {"search_label": "Баварский лес", "label": "Баварский лес"},
            {"search_label": "лес Иррати", "label": "Иррати"},
        ],
    }),
    _gw_with_items({
        "region_nom": "тропические регионы мира",
        "region_prep": "в тропических регионах мира",
        "biome_clause_l2": "относящихся к влажным тропическим лесам",
        "descriptor_clause_l3": "часто рассматриваемых как наиболее известные тропические лесные регионы мира",
        "descriptor_clause_l4": "которые одновременно относятся к влажным тропическим лесам и часто упоминаются как известные природные регионы мира",
        "items_raw": [
            {"search_label": "Амазонский дождевой лес", "label": "Амазонский дождевой лес"},
            {"search_label": "Конголезские дождевые леса", "label": "Конголезские дождевые леса"},
            {"search_label": "Сундарбан", "label": "Сундарбан"},
            {"search_label": "Атлантический лес", "label": "Атлантический лес"},
        ],
    }),
    _gw_with_items({
        "region_nom": "Северная Америка",
        "region_prep": "в Северной Америке",
        "biome_clause_l2": "относящихся к крупным лесным массивам континента",
        "descriptor_clause_l3": "часто рассматриваемых как заметные природные регионы Северной Америки",
        "descriptor_clause_l4": "которые одновременно относятся к крупным лесным массивам континента и часто упоминаются как заметные природные территории Северной Америки",
        "items_raw": [
            {"search_label": "Йеллоустонский лес", "label": "Йеллоустонский лес"},
            {"search_label": "Тонгасс", "label": "Тонгасс"},
            {"search_label": "лес Редвуд", "label": "Редвуд"},
        ],
    }),
]

CANYON_GROUPS = [
    _gw_with_items({
        "region_nom": "Северная Америка",
        "region_prep": "в Северной Америке",
        "descriptor_clause_l3": "относящихся к наиболее известным каньонам континента",
        "descriptor_clause_l5": "которые одновременно считаются одними из самых узнаваемых каньонов региона",
        "items_raw": [
            {"search_label": "Гранд-Каньон", "label": "Гранд-Каньон"},
            {"search_label": "Каньон Антилопы", "label": "Каньон Антилопы"},
            {"search_label": "Медный каньон", "label": "Медный каньон"},
        ],
    }),
    _gw_with_items({
        "region_nom": "Южная Америка",
        "region_prep": "в Южной Америке",
        "descriptor_clause_l3": "относящихся к крупным каньонам Андского региона",
        "descriptor_clause_l5": "которые одновременно считаются одними из самых известных каньонов западной части континента",
        "items_raw": [
            {"search_label": "Каньон Колка", "label": "Каньон Колка"},
            {"search_label": "Каньон Котауаси", "label": "Каньон Котауаси"},
        ],
    }),
    _gw_with_items({
        "region_nom": "Африка",
        "region_prep": "в Африке",
        "descriptor_clause_l3": "относящихся к крупным каньонам юга и востока Африки",
        "descriptor_clause_l5": "которые одновременно считаются одними из самых известных каньонов континента",
        "items_raw": [
            {"search_label": "Каньон реки Блайд", "label": "Каньон реки Блайд"},
            {"search_label": "Каньон Фиш-Ривер", "label": "Каньон Фиш-Ривер"},
        ],
    }),
]


def _gw_filter_items_by_min_elev(items: List[Dict[str, Any]], elev_min: int) -> List[Dict[str, Any]]:
    out = []
    for it in items:
        if int(it.get("elev_m", -10**9)) >= int(elev_min):
            out.append(it)
    return out


def _gw_filter_items_by_elev_range(items: List[Dict[str, Any]], elev_min: int, elev_max: int) -> List[Dict[str, Any]]:
    out = []
    for it in items:
        elev = int(it.get("elev_m", -10**9))
        if elev >= int(elev_min) and elev <= int(elev_max):
            out.append(it)
    return out


def _gw_tpl_oceans_basic(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L1":
        return None
    grp = _gw_pick_rotated(OCEAN_BY_CONTINENT_GROUPS, idx, rng, salt=1)
    items = grp["items"]
    if not items:
        return None
    k = len(items)
    q_ru = f"Назови {k} океана, которые омывают {grp['continent_acc']}."
    return _gw_finalize_static_example(
        idx=idx,
        complexity=complexity,
        template_id="geo_ocean_continent_curated",
        template_family="oceans",
        q_ru=q_ru,
        constraints={"continent_ru": grp["continent_nom"], "candidate_labels": [it["label"] for it in items]},
        eligible_items=items,
        requested_count=k,
        type_qid=Q_GEO_OCEAN,
    )


def _gw_tpl_oceans_bridge(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L3":
        return None
    grp = _gw_pick_rotated(OCEAN_MULTI_CONTINENT_GROUPS, idx, rng, salt=2)
    items = grp["items"]
    if len(items) < 2:
        return None
    k = min(len(items), 2)
    q_ru = f"Назови {k} океана, которые одновременно омывают {grp['continent1_acc']} и {grp['continent2_acc']}."
    return _gw_finalize_static_example(
        idx=idx,
        complexity=complexity,
        template_id="geo_ocean_two_continents_curated",
        template_family="oceans",
        q_ru=q_ru,
        constraints={
            "continent1_ru": grp["continent1_nom"],
            "continent2_ru": grp["continent2_nom"],
            "candidate_labels": [it["label"] for it in items],
        },
        eligible_items=items,
        requested_count=k,
        type_qid=Q_GEO_OCEAN,
    )


def _gw_tpl_seas_basic(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in ("L1", "L2", "L4"):
        return None
    groups = [g for g in SEA_REGION_GROUPS if len(g["items"]) >= 2]
    grp = _gw_pick_rotated(groups, idx, rng, salt=3)
    items = grp["items"]
    if complexity == "L1":
        k = min(len(items), rng.choice([2, 3, 4]))
        q_ru = f"Назови {k} моря, расположенных {grp['region_prep']}."
        template_id = "geo_sea_region_curated"
    elif complexity == "L2":
        k = min(len(items), rng.choice([2, 3, 4]))
        q_ru = f"Назови {k} моря, расположенных {grp['region_prep']} и относящихся к бассейну {grp['basin_gen']}."
        template_id = "geo_sea_region_basin_curated"
    else:
        k = min(len(items), rng.choice([2, 3]))
        q_ru = f"Перечисли {k} моря, расположенных {grp['region_prep']}, относящихся к бассейну {grp['basin_gen']} и {grp['descriptor_clause_l4']}."
        template_id = "geo_sea_region_basin_advanced_curated"
    return _gw_finalize_static_example(
        idx=idx,
        complexity=complexity,
        template_id=template_id,
        template_family="seas",
        q_ru=q_ru,
        constraints={
            "region_ru": grp["region_nom"],
            "basin_ru": grp.get("basin_gen"),
            "candidate_labels": [it["label"] for it in items],
        },
        eligible_items=items,
        requested_count=k,
        type_qid=Q_GEO_SEA,
    )


def _gw_tpl_lakes_basic(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in ("L1", "L2", "L3"):
        return None
    grp = _gw_pick_rotated(LAKE_WORLD_GROUPS, idx, rng, salt=5)
    items = grp["items"]
    if not items:
        return None
    if complexity == "L1":
        k = min(len(items), rng.choice([3, 4, 5]))
        q_ru = f"Назови {k} озёр, расположенных {grp['region_prep']}."
        template_id = "geo_lake_region_curated"
    elif complexity == "L2":
        k = min(len(items), rng.choice([3, 4, 5]))
        q_ru = f"Назови {k} озёр, расположенных {grp['region_prep']} и относящихся к группе «{grp['group_name']}»."
        template_id = "geo_lake_region_group_curated"
    else:
        k = min(len(items), rng.choice([3, 4]))
        q_ru = f"Перечисли {k} озёр, расположенных {grp['region_prep']}, относящихся к группе «{grp['group_name']}» и {grp['descriptor_clause_l3']}."
        template_id = "geo_lake_region_group_advanced_curated"
    return _gw_finalize_static_example(
        idx=idx,
        complexity=complexity,
        template_id=template_id,
        template_family="lakes",
        q_ru=q_ru,
        constraints={
            "region_ru": grp["region_nom"],
            "group_name_ru": grp.get("group_name"),
            "candidate_labels": [it["label"] for it in items],
        },
        eligible_items=items,
        requested_count=k,
        type_qid=Q_GEO_LAKE,
    )


def _gw_tpl_waterfalls_basic(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in ("L1", "L2", "L3"):
        return None
    grp = _gw_pick_rotated(WATERFALL_GROUPS, idx, rng, salt=7)
    items = grp["items"]
    if len(items) < 2:
        return None
    k = min(len(items), rng.choice([2, 3]))
    if complexity == "L1":
        q_ru = f"Назови {k} водопада, расположенных {grp['region_prep']}."
        template_id = "geo_waterfall_region_curated"
    elif complexity == "L2":
        q_ru = f"Назови {k} водопада, расположенных {grp['region_prep']} и {grp['descriptor_clause_l2']}."
        template_id = "geo_waterfall_region_famous_curated"
    else:
        q_ru = f"Перечисли {k} водопада, расположенных {grp['region_prep']}, {grp['descriptor_clause_l2']} и {grp['descriptor_clause_l3']}."
        template_id = "geo_waterfall_region_advanced_curated"
    return _gw_finalize_static_example(
        idx=idx,
        complexity=complexity,
        template_id=template_id,
        template_family="waterfalls",
        q_ru=q_ru,
        constraints={"region_ru": grp["region_nom"], "candidate_labels": [it["label"] for it in items]},
        eligible_items=items,
        requested_count=k,
        type_qid=Q_GEO_WATERFALL,
    )


def _gw_tpl_deserts_basic(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in ("L1", "L2", "L3"):
        return None
    groups = [g for g in DESERT_GROUPS if len(g["items"]) >= 2]
    grp = _gw_pick_rotated(groups, idx, rng, salt=9)
    items = grp["items"]
    if complexity == "L1":
        k = min(len(items), rng.choice([2, 3]))
        q_ru = f"Назови {k} пустыни, расположенных {grp['region_prep']}."
        template_id = "geo_desert_region_curated"
    elif complexity == "L2":
        k = min(len(items), rng.choice([2, 3, 4]))
        q_ru = f"Назови {k} пустыни, расположенных {grp['region_prep']} и {grp['descriptor_clause_l2']}."
        template_id = "geo_desert_region_descriptor_curated"
    else:
        k = min(len(items), rng.choice([2, 3]))
        q_ru = f"Перечисли {k} пустыни, расположенных {grp['region_prep']}, {grp['descriptor_clause_l2']} и {grp['descriptor_clause_l3']}."
        template_id = "geo_desert_region_advanced_curated"
    return _gw_finalize_static_example(
        idx=idx,
        complexity=complexity,
        template_id=template_id,
        template_family="deserts",
        q_ru=q_ru,
        constraints={"region_ru": grp["region_nom"], "candidate_labels": [it["label"] for it in items]},
        eligible_items=items,
        requested_count=k,
        type_qid=Q_GEO_DESERT,
    )


def _gw_tpl_forests_basic(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in ("L1", "L2", "L3", "L4"):
        return None
    grp = _gw_pick_rotated(FOREST_GROUPS, idx, rng, salt=11)
    items = grp["items"]
    if len(items) < 2:
        return None
    k = min(len(items), rng.choice([2, 3]))
    if complexity == "L1":
        q_ru = f"Назови {k} известных лесных массива, расположенных {grp['region_prep']}."
        template_id = "geo_forest_region_curated"
    elif complexity == "L2":
        q_ru = f"Назови {k} лесных массива, расположенных {grp['region_prep']} и {grp['biome_clause_l2']}."
        template_id = "geo_forest_region_biome_curated"
    elif complexity == "L3":
        q_ru = f"Перечисли {k} лесных массива, расположенных {grp['region_prep']}, {grp['biome_clause_l2']} и {grp['descriptor_clause_l3']}."
        template_id = "geo_forest_region_biome_advanced_curated"
    else:
        q_ru = f"Перечисли {k} лесных массива, расположенных {grp['region_prep']} и {grp['descriptor_clause_l4']}."
        template_id = "geo_forest_region_hard_curated"
    return _gw_finalize_static_example(
        idx=idx,
        complexity=complexity,
        template_id=template_id,
        template_family="forests",
        q_ru=q_ru,
        constraints={"region_ru": grp["region_nom"], "candidate_labels": [it["label"] for it in items]},
        eligible_items=items,
        requested_count=k,
        type_qid=Q_GEO_FOREST,
    )


def _gw_tpl_canyons_basic(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in ("L1", "L3", "L5"):
        return None
    groups = [g for g in CANYON_GROUPS if len(g["items"]) >= 2]
    grp = _gw_pick_rotated(groups, idx, rng, salt=13)
    items = grp["items"]
    k = min(len(items), rng.choice([2, 3]))
    if complexity == "L1":
        q_ru = f"Назови {k} каньона, расположенных {grp['region_prep']}."
        template_id = "geo_canyon_region_curated"
    elif complexity == "L3":
        q_ru = f"Назови {k} каньона, расположенных {grp['region_prep']} и {grp['descriptor_clause_l3']}."
        template_id = "geo_canyon_region_descriptor_curated"
    else:
        q_ru = f"Перечисли {k} каньона, расположенных {grp['region_prep']}, {grp['descriptor_clause_l3']} и {grp['descriptor_clause_l5']}."
        template_id = "geo_canyon_region_hard_curated"
    return _gw_finalize_static_example(
        idx=idx,
        complexity=complexity,
        template_id=template_id,
        template_family="canyons",
        q_ru=q_ru,
        constraints={"region_ru": grp["region_nom"], "candidate_labels": [it["label"] for it in items]},
        eligible_items=items,
        requested_count=k,
        type_qid=Q_GEO_CANYON,
    )


def _gw_tpl_mountains_height(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in ("L1", "L2", "L3", "L4"):
        return None
    grp = _gw_pick_rotated(MOUNTAIN_WORLD_GROUPS, idx, rng, salt=15)
    items = grp["items"]
    if len(items) < 2:
        return None

    if complexity == "L1":
        eligible = items
        k = min(len(eligible), rng.choice([2, 3, 4]))
        q_ru = f"Назови {k} горы, расположенных {grp['system_prep']}."
        template_id = "geo_mountain_system_basic_curated"
        extra_lines = None
        constraints_extra = {}
    elif complexity == "L2":
        elev_min = rng.choice([4000, 4300, 4500]) if grp["system_nom"] == "Альпы" else (rng.choice([6500, 7000, 7500]) if grp["system_nom"] == "Гималаи" else rng.choice([6000, 6300, 6500]))
        eligible = _gw_filter_items_by_min_elev(items, elev_min)
        k = min(len(eligible), rng.choice([2, 3, 4]))
        noun = _gw_ru_count(k, "гору", "горы", "гор")
        q_ru = f"Перечисли {k} {noun}, расположенных {grp['system_prep']}, {grp['macroregion_prep']}, и имеющих высоту не ниже {int(elev_min)} м."
        template_id = "geo_mountain_system_macroregion_height_curated"
        extra_lines = [f"?item wdt:P2044 ?elev .", f"FILTER(?elev >= {int(elev_min)}) ."]
        constraints_extra = {"elev_min_m": elev_min}
    elif complexity == "L3":
        elev_min = 4400 if grp["system_nom"] == "Альпы" else (8200 if grp["system_nom"] == "Гималаи" else 6500)
        eligible = _gw_filter_items_by_min_elev(items, elev_min)
        k = min(len(eligible), rng.choice([2, 3, 4]))
        noun = _gw_ru_count(k, "гору", "горы", "гор")
        q_ru = f"Перечисли {k} {noun}, расположенных {grp['system_prep']}, {grp['macroregion_prep']}, входящих в число высочайших вершин {grp['system_gen']} и имеющих высоту не ниже {int(elev_min)} м."
        template_id = "geo_mountain_system_toppeaks_curated"
        extra_lines = [f"?item wdt:P2044 ?elev .", f"FILTER(?elev >= {int(elev_min)}) ."]
        constraints_extra = {"elev_min_m": elev_min}
    else:
        if grp["system_nom"] == "Альпы":
            elev_min, elev_max = 4300, 4700
        elif grp["system_nom"] == "Гималаи":
            elev_min, elev_max = 8100, 8600
        else:
            elev_min, elev_max = 6300, 7000
        eligible = _gw_filter_items_by_elev_range(items, elev_min, elev_max)
        k = min(len(eligible), rng.choice([2, 3]))
        noun = _gw_ru_count(k, "гору", "горы", "гор")
        q_ru = f"Перечисли {k} {noun}, расположенных {grp['system_prep']}, {grp['macroregion_prep']}, входящих в число высочайших вершин {grp['system_gen']} и имеющих высоту от {int(elev_min)} до {int(elev_max)} м."
        template_id = "geo_mountain_system_toprange_curated"
        extra_lines = [
            f"?item wdt:P2044 ?elev .",
            f"FILTER(?elev >= {int(elev_min)} && ?elev <= {int(elev_max)}) .",
        ]
        constraints_extra = {"elev_min_m": elev_min, "elev_max_m": elev_max}

    return _gw_finalize_static_example(
        idx=idx,
        complexity=complexity,
        template_id=template_id,
        template_family="mountains",
        q_ru=q_ru,
        constraints={
            "mountain_system_ru": grp["system_nom"],
            "macroregion_ru": grp["macroregion_nom"],
            "candidate_labels": [it["label"] for it in items],
            **constraints_extra,
        },
        eligible_items=eligible,
        requested_count=k,
        type_qid=Q_GEO_MOUNTAIN,
        extra_lines=extra_lines,
    )


def _gw_tpl_volcanoes_height(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in ("L1", "L2", "L3", "L4", "L5"):
        return None
    grp = _gw_pick_rotated(VOLCANO_WORLD_GROUPS, idx, rng, salt=17)
    items = grp["items"]
    if len(items) < 2:
        return None

    if complexity == "L1":
        eligible = items
        k = min(len(eligible), rng.choice([2, 3]))
        q_ru = f"Назови {k} вулкана, расположенных {grp['region_prep']}."
        template_id = "geo_volcano_region_basic_curated"
        extra_lines = None
        constraints_extra = {}
    elif complexity == "L2":
        elev_min = rng.choice([1500, 2000, 2500])
        eligible = _gw_filter_items_by_min_elev(items, elev_min)
        k = min(len(eligible), rng.choice([2, 3]))
        noun = _gw_ru_count(k, "вулкан", "вулкана", "вулканов")
        q_ru = f"Назови {k} {noun}, расположенных {grp['region_prep']} и имеющих высоту не ниже {int(elev_min)} м."
        template_id = "geo_volcano_region_height_curated"
        extra_lines = [f"?item wdt:P2044 ?elev .", f"FILTER(?elev >= {int(elev_min)}) ."]
        constraints_extra = {"elev_min_m": elev_min}
    elif complexity == "L3":
        elev_min = rng.choice([1800, 2200, 2500])
        eligible = _gw_filter_items_by_min_elev(items, elev_min)
        k = min(len(eligible), rng.choice([2, 3]))
        noun = _gw_ru_count(k, "вулкан", "вулкана", "вулканов")
        q_ru = f"Перечисли {k} {noun}, расположенных {grp['region_prep']}, {grp['context_clause']} и имеющих высоту не ниже {int(elev_min)} м."
        template_id = "geo_volcano_region_context_height_curated"
        extra_lines = [f"?item wdt:P2044 ?elev .", f"FILTER(?elev >= {int(elev_min)}) ."]
        constraints_extra = {"elev_min_m": elev_min}
    elif complexity == "L4":
        elev_min = rng.choice([2200, 2500, 3000])
        eligible = _gw_filter_items_by_min_elev(items, elev_min)
        k = min(len(eligible), rng.choice([2, 3]))
        noun = _gw_ru_count(k, "вулкан", "вулкана", "вулканов")
        q_ru = f"Перечисли {k} {noun}, расположенных {grp['region_prep']}, {grp['context_clause']} и имеющих высоту не ниже {int(elev_min)} м."
        template_id = "geo_volcano_region_context_hard_curated"
        extra_lines = [f"?item wdt:P2044 ?elev .", f"FILTER(?elev >= {int(elev_min)}) ."]
        constraints_extra = {"elev_min_m": elev_min}
    else:
        if grp["region_nom"] == "Япония":
            elev_min, elev_max = 2500, 3800
        elif grp["region_nom"] == "Индонезия":
            elev_min, elev_max = 2800, 3800
        elif grp["region_nom"] == "Италия":
            elev_min, elev_max = 900, 3400
        else:
            elev_min, elev_max = 1400, 1700
        eligible = _gw_filter_items_by_elev_range(items, elev_min, elev_max)
        k = min(len(eligible), rng.choice([2, 3]))
        noun = _gw_ru_count(k, "вулкан", "вулкана", "вулканов")
        q_ru = f"Перечисли {k} {noun}, расположенных {grp['region_prep']}, {grp['context_clause']}, входящих в число наиболее высоких вулканов региона и имеющих высоту от {int(elev_min)} до {int(elev_max)} м."
        template_id = "geo_volcano_region_context_range_expert_curated"
        extra_lines = [
            f"?item wdt:P2044 ?elev .",
            f"FILTER(?elev >= {int(elev_min)} && ?elev <= {int(elev_max)}) .",
        ]
        constraints_extra = {"elev_min_m": elev_min, "elev_max_m": elev_max}

    return _gw_finalize_static_example(
        idx=idx,
        complexity=complexity,
        template_id=template_id,
        template_family="volcanoes",
        q_ru=q_ru,
        constraints={
            "region_ru": grp["region_nom"],
            "tectonic_context_ru": grp["context_clause"],
            "candidate_labels": [it["label"] for it in items],
            **constraints_extra,
        },
        eligible_items=eligible,
        requested_count=k,
        type_qid=Q_GEO_VOLCANO,
        extra_lines=extra_lines,
    )


GEO_WORLD_TEMPLATE_FNS: Dict[str, List[Any]] = {
    "L1": [
        _gw_tpl_oceans_basic,
        _gw_tpl_seas_basic,
        _gw_tpl_lakes_basic,
        _gw_tpl_waterfalls_basic,
        _gw_tpl_deserts_basic,
        _gw_tpl_forests_basic,
        _gw_tpl_canyons_basic,
        _gw_tpl_mountains_height,
        _gw_tpl_volcanoes_height,
    ],
    "L2": [
        _gw_tpl_seas_basic,
        _gw_tpl_lakes_basic,
        _gw_tpl_waterfalls_basic,
        _gw_tpl_deserts_basic,
        _gw_tpl_forests_basic,
        _gw_tpl_mountains_height,
        _gw_tpl_volcanoes_height,
    ],
    "L3": [
        _gw_tpl_oceans_bridge,
        _gw_tpl_lakes_basic,
        _gw_tpl_waterfalls_basic,
        _gw_tpl_deserts_basic,
        _gw_tpl_forests_basic,
        _gw_tpl_canyons_basic,
        _gw_tpl_mountains_height,
        _gw_tpl_volcanoes_height,
    ],
    "L4": [
        _gw_tpl_seas_basic,
        _gw_tpl_forests_basic,
        _gw_tpl_mountains_height,
        _gw_tpl_volcanoes_height,
    ],
    "L5": [
        _gw_tpl_canyons_basic,
        _gw_tpl_volcanoes_height,
    ],
}


def generate_geo_world_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 12) -> BenchmarkExample:
    template_fns = list(GEO_WORLD_TEMPLATE_FNS.get(complexity, []))
    if not template_fns:
        raise ValueError(f"Unknown complexity: {complexity}")

    primary_pos = (idx - 1) % len(template_fns)
    ordered_fns = template_fns[primary_pos:] + template_fns[:primary_pos]

    if len(ordered_fns) > 2:
        head = ordered_fns[:1]
        tail = ordered_fns[1:]
        rng.shuffle(tail)
        ordered_fns = head + tail

    for _ in range(max_attempts):
        for fn in ordered_fns:
            try:
                ex = fn(complexity, idx, rng)
            except Exception:
                ex = None
            if ex is not None and getattr(ex, "gold_answer_qids", None):
                return ex

    return BenchmarkExample(
        id=f"geo_fail_{complexity.lower()}_{idx:04d}",
        domain="geo",
        complexity=complexity,
        query_text_ru="(ошибка генерации geo)",
        constraints={"failed": True, "complexity": complexity},
        requested_count=3,
        gold_answer_qids=[],
        gold_answer_labels_ru=[],
        sparql_query="",
        created_at=utc_now_z(),
        template_id="geo_failed",
        template_family="failed",
    )


if "DOMAIN_GENERATORS" in globals():
    DOMAIN_GENERATORS["geo"] = generate_geo_world_example
print("geo generator ready: richer L2+ world geography")


geo generator ready: richer L2+ world geography


<a id="books-domain"></a>

## 12. Books domain

Book templates with genre, language, time, setting, adaptation, and author-related constraints.

In [12]:
Q_BOOK = ensure_qid("книга", fallback_qid="Q571")

book_genres_df = load_or_build_pool("books_genres_ru", lambda: build_value_pool_ru(Q_BOOK, "P136", "genre", limit=400))
book_countries_df = load_or_build_pool("books_countries_ru", lambda: build_value_pool_ru(Q_BOOK, "P495", "country", limit=250))
book_langs_df = load_or_build_pool("books_langs_ru", lambda: build_value_pool_ru(Q_BOOK, "P407", "lang", limit=250))

COMMON_BOOK_GENRES_RU = ["фэнтези", "детектив", "научная фантастика", "роман", "повесть", "поэзия", "драма", "биография"]
COMMON_LANGS_RU = ["русский язык", "английский язык", "французский язык", "немецкий язык", "японский язык"]
COMMON_COUNTRIES_RU = ["Россия", "США", "Франция", "Великобритания", "Германия", "Италия", "Япония", "Канада"]

LANG_COUNTRY_HINTS_RU = {
    "русский язык": ["Россия"],
    "английский язык": ["США", "Великобритания", "Канада"],
    "французский язык": ["Франция", "Канада"],
    "немецкий язык": ["Германия", "Австрия"],
    "японский язык": ["Япония"],
}

BOOK_YEAR_WINDOWS_L2 = [(1950, 1969), (1970, 1989), (1990, 2009), (2000, 2019)]
BOOK_YEAR_WINDOWS_L3 = [(1900, 1949), (1950, 1989), (1990, 2019)]
BOOK_YEAR_WINDOWS_L5 = [(2500, 2500)]


BOOKS_MIN_RU_ANCHOR_SCORE = 5

def _book_ru_anchor_score(ent: Optional[dict]) -> int:
    """
    Грубая эвристика "популярности" книги в RU-пространстве.
    Чем выше score, тем надёжнее, что у книги есть нормальное русское название.
      +6 : есть ruwiki/ru-* sitelink
      +5 : есть русскоязычный monolingual title (P1476)
      +3 : есть русский label с кириллицей
      +2 : есть русский alias с кириллицей
    """
    if not ent:
        return 0

    score = 0

    if _get_ru_sitelink_title(ent):
        score += 6

    if _get_ru_monolingual_title(ent, "P1476"):
        score += 5

    ru_lbl = ((ent.get("labels") or {}).get("ru") or {}).get("value")
    if ru_lbl and _cinema_has_cyrillic(str(ru_lbl)):
        score += 3

    try:
        aliases = ((ent.get("aliases") or {}).get("ru") or []) or []
        if any((a or {}).get("value") and _cinema_has_cyrillic(str((a or {}).get("value"))) for a in aliases):
            score += 2
    except Exception:
        pass

    return score


def _postfilter_books_items_popular_ru(
    items: List[Tuple[str, str]],
    min_anchor_score: int = BOOKS_MIN_RU_ANCHOR_SCORE,
) -> List[Tuple[str, str]]:
    """
    Оставляем только книги с хорошим русским названием и ранжируем так,
    чтобы вверх поднимались более "популярные" / лучше закреплённые в RU.
    """
    if not items:
        return []

    uniq_items: List[Tuple[str, str]] = []
    seen = set()
    for q, l in items:
        if q and q not in seen:
            uniq_items.append((q, l))
            seen.add(q)

    ent_map = wd_get_entities_ru([q for q, _ in uniq_items])
    scored: List[Tuple[int, str, str]] = []

    for q, raw_label in uniq_items:
        ent = ent_map.get(q)
        best_ru = _best_popular_ru_name(ent, fallback_label="")
        if not best_ru:
            continue
        if not _cinema_has_cyrillic(best_ru):
            continue

        score = _book_ru_anchor_score(ent)
        if score < int(min_anchor_score):
            continue

        quality = score
        if len(best_ru) <= 80:
            quality += 1
        if re.fullmatch(r"Q\d+", best_ru):
            continue

        scored.append((quality, q, best_ru))

    scored.sort(key=lambda t: (-t[0], t[2].lower(), t[1]))
    return [(q, lbl) for _, q, lbl in scored]

def pick_book_constraints(rng: random.Random, prefer_coherent: bool = False):
    """
    prefer_coherent=True:
      выбираем согласованные пары (язык -> типичная страна),
      чтобы чаще находились >= requested_count.
    """
    if prefer_coherent and rng.random() < 0.9:
        l_ru = rng.choice(COMMON_LANGS_RU)
        c_ru = rng.choice(LANG_COUNTRY_HINTS_RU.get(l_ru, COMMON_COUNTRIES_RU))
        g_ru = rng.choice(COMMON_BOOK_GENRES_RU)
        g_qid = ensure_qid(g_ru)
        c_qid = ensure_qid(c_ru)
        l_qid = ensure_qid(l_ru)
        return g_qid, g_ru, c_qid, c_ru, l_qid, l_ru

    if rng.random() < 0.8:
        g_ru = rng.choice(COMMON_BOOK_GENRES_RU)
        c_ru = rng.choice(COMMON_COUNTRIES_RU)
        l_ru = rng.choice(COMMON_LANGS_RU)
        g_qid = ensure_qid(g_ru)
        c_qid = ensure_qid(c_ru)
        l_qid = ensure_qid(l_ru)
    else:
        g_qid, g_ru = pick_from_df(book_genres_df, "genre_qid", "genreLabelRu", rng)
        c_qid, c_ru = pick_from_df(book_countries_df, "country_qid", "countryLabelRu", rng)
        l_qid, l_ru = pick_from_df(book_langs_df, "lang_qid", "langLabelRu", rng)
    return g_qid, g_ru, c_qid, c_ru, l_qid, l_ru


def run_books_query(
    genre_qid: str,
    country_qid: str,
    lang_qid: Optional[str],
    y1: Optional[int],
    y2: Optional[int],
    not_series: bool,
    author_gender_qid: Optional[str],
    country_mode: str = "author_citizenship",  # "work_origin" | "author_citizenship"
    limit: int = 250,
):
    """
    country_mode:
      - work_origin: ?item wdt:P495 country
      - author_citizenship: ?item wdt:P50 ?author ; ?author wdt:P27 country

    ВАЖНО для books:
      даже если WDQS отдаёт ru-label, мы потом дополнительно фильтруем кандидатов,
      чтобы остались только книги с сильным русским якорем и нормальным RU названием.
    """
    where = [
        f"?item wdt:P136 wd:{genre_qid} .",
    ]

    needs_author = (country_mode == "author_citizenship") or (author_gender_qid is not None)
    if needs_author:
        where.append("?item wdt:P50 ?author .")

    if country_mode == "work_origin":
        where.append(f"?item wdt:P495 wd:{country_qid} .")
    elif country_mode == "author_citizenship":
        where.append(f"?author wdt:P27 wd:{country_qid} .")
    else:
        raise ValueError(f"Unknown country_mode: {country_mode}")

    if y1 is not None and y2 is not None:
        where += [
            "?item wdt:P577 ?date .",
            "BIND(YEAR(?date) AS ?year) .",
            f"FILTER(?year >= {int(y1)} && ?year <= {int(y2)}) .",
        ]

    if lang_qid:
        where.append(f"?item wdt:P407 wd:{lang_qid} .")
    if not_series:
        where.append("FILTER NOT EXISTS { ?item wdt:P179 ?series . }")
    if author_gender_qid:
        where.append(f"?author wdt:P21 wd:{author_gender_qid} .")

    raw_limit = max(int(limit), 500)
    sparql, items = select_items_with_ru_label(Q_BOOK, where, limit=raw_limit, item_var="item")
    items = _postfilter_books_items_popular_ru(items, min_anchor_score=BOOKS_MIN_RU_ANCHOR_SCORE)

    if limit is not None and int(limit) > 0:
        items = items[:int(limit)]
    return sparql, items, where


def nlg_books_ru(
    genre_ru: str,
    country_ru: str,
    lang_ru: Optional[str],
    y1: Optional[int],
    y2: Optional[int],
    not_series: bool,
    extra: Optional[str],
    k: int,
    country_mode: str = "author_citizenship",
    prompt_head: str = "Назови",
) -> str:
    if country_mode == "author_citizenship":
        country_part = f"у авторов гражданство: {country_ru}"
    else:
        country_part = f"из страны: {country_ru}"

    cond = [f"жанра «{genre_ru}»", country_part]
    if lang_ru:
        cond.append(f"на языке: {lang_ru}")
    if y1 is not None and y2 is not None:
        cond.append(f"в период {y1}–{y2} годов" if y1 != y2 else f"{y1} года")
    if not_series:
        cond.append("которые не входят в книжную серию")
    if extra:
        cond.append(extra)
    return f"{prompt_head} {k} книг, " + ", ".join(cond) + "."


def _books_plan_for_complexity(complexity: str, rng: random.Random):
    """
    Разводим уровни по разным типам критериев:
      L1: жанр + гражданство автора
      L2: жанр + гражданство автора + годы
      L3: жанр + гражданство автора + годы + язык
      L4: жанр + гражданство автора + язык + автор-женщина + не серия
      L5: как L4, но с невозможным годом => zero-case
    """
    g_qid, g_ru, c_qid, c_ru, l_qid, l_ru = pick_book_constraints(
        rng, prefer_coherent=(complexity in ("L2", "L3", "L4", "L5"))
    )

    plan = {
        "genre_qid": g_qid,
        "genre_ru": g_ru,
        "country_qid": c_qid,
        "country_ru": c_ru,
        "lang_qid": None,
        "lang_ru": None,
        "y1": None,
        "y2": None,
        "not_series": False,
        "author_gender_qid": None,
        "country_mode": "author_citizenship",
        "k": 5,
        "extra": None,
        "template_id": None,
    }

    if complexity == "L1":
        plan["template_id"] = "books_genre_author_country"

    elif complexity == "L2":
        y1, y2 = rng.choice(BOOK_YEAR_WINDOWS_L2)
        plan.update({
            "y1": y1, "y2": y2,
            "template_id": "books_genre_author_country_year",
        })

    elif complexity == "L3":
        y1, y2 = rng.choice(BOOK_YEAR_WINDOWS_L3)
        plan.update({
            "lang_qid": l_qid, "lang_ru": l_ru,
            "y1": y1, "y2": y2,
            "template_id": "books_genre_author_country_year_language",
        })

    elif complexity == "L4":
        plan.update({
            "lang_qid": l_qid, "lang_ru": l_ru,
            "not_series": True,
            "author_gender_qid": Q_FEMALE,
            "extra": "у которых автор — женщина",
            "k": 10,
            "template_id": "books_genre_author_country_language_female_noseries",
        })

    elif complexity == "L5":
        y1, y2 = rng.choice(BOOK_YEAR_WINDOWS_L5)
        plan.update({
            "lang_qid": l_qid, "lang_ru": l_ru,
            "y1": y1, "y2": y2,
            "not_series": True,
            "author_gender_qid": Q_FEMALE,
            "extra": "у которых автор — женщина",
            "template_id": "books_zero_future_year_country_language_female_noseries",
        })
    else:
        raise ValueError(f"Unknown complexity: {complexity}")

    return plan


def generate_books_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 140) -> BenchmarkExample:
    """
    Главный фикс для books:
      1) L1 больше НЕ требует year/P577 — это сильно расширяет домен.
      2) Базовая страна для книг теперь author_citizenship, а не work_origin.
      3) Уровни разведены по разным критериям.
    """
    for _ in range(max_attempts):
        plan = _books_plan_for_complexity(complexity, rng)

        try:
            sparql, items, where = run_books_query(
                plan["genre_qid"],
                plan["country_qid"],
                plan["lang_qid"],
                plan["y1"],
                plan["y2"],
                plan["not_series"],
                plan["author_gender_qid"],
                country_mode=plan["country_mode"],
                limit=300,
            )
        except WDQSTransientError:
            continue

        if complexity in ("L1", "L2", "L3") and len(items) < plan["k"]:
            try:
                sparql, items, where = run_books_query(
                    plan["genre_qid"],
                    plan["country_qid"],
                    plan["lang_qid"],
                    plan["y1"],
                    plan["y2"],
                    plan["not_series"],
                    plan["author_gender_qid"],
                    country_mode="work_origin",
                    limit=300,
                )
                effective_country_mode = "work_origin"
            except WDQSTransientError:
                continue
        else:
            effective_country_mode = plan["country_mode"]

        # MAIN filter
        if complexity in ("L1", "L2", "L3") and len(items) < plan["k"]:
            continue
        if complexity == "L4" and len(items) == 0:
            continue
        if complexity == "L5" and len(items) != 0:
            continue

        gold = items[:300]
        gold_qids = [q for q, _ in gold]
        gold_lbls = [l for _, l in gold]
        truncated = len(items) >= 300
        ask = build_ask_validator(Q_BOOK, where, item_var="item")

        ex = BenchmarkExample(
            id=f"books_{complexity.lower()}_{idx:04d}",
            domain="books",
            complexity=complexity,
            query_text_ru=nlg_books_ru(
                plan["genre_ru"],
                plan["country_ru"],
                plan["lang_ru"],
                plan["y1"],
                plan["y2"],
                plan["not_series"],
                plan["extra"],
                plan["k"],
                country_mode=effective_country_mode,
            ),
            constraints={
                "genre_qid": plan["genre_qid"], "genre_ru": plan["genre_ru"],
                "country_qid": plan["country_qid"], "country_ru": plan["country_ru"],
                "country_mode": effective_country_mode,
                "lang_qid": plan["lang_qid"], "lang_ru": plan["lang_ru"],
                "year_from": plan["y1"], "year_to": plan["y2"],
                "not_series": plan["not_series"],
                "author_gender_qid": plan["author_gender_qid"],
                "template_id": plan["template_id"],
            },
            requested_count=plan["k"],
            gold_answer_qids=gold_qids,
            gold_answer_labels_ru=gold_lbls,
            sparql_query=sparql,
            created_at=utc_now_z(),
            gold_truncated=truncated,
            ask_validator_sparql=ask,
            template_id=plan["template_id"],
        )
        return ex

    raise RuntimeError(f"Failed to generate books example {complexity} after {max_attempts} attempts")


ADV_TEMPLATE_PROB_BOOKS = 0.20

BOOK_SETTING_CITIES = [
    ("Санкт-Петербург", "Q656"),
    ("Москва", "Q649"),
    ("Лондон", "Q84"),
    ("Париж", "Q90"),
]

Q_RUSSIAN_LANG = ensure_qid("русский язык", fallback_qid="Q7737")

def _pick_setting_city(rng: random.Random):
    name, qid = rng.choice(BOOK_SETTING_CITIES)
    return name, qid

def run_books_setting_adaptation_query(
    city_qid: str,
    pub_before_year: int,
    adapt_after_year: int,
    require_russian: bool = True,
    limit: int = 250,
) -> Tuple[str, List[Tuple[str,str]]]:
    """
    Книги/литературные произведения, где:
      - публикация раньше pub_before_year
      - место действия связано с city (P840 / P131*)
      - существует экранизация после adapt_after_year (у адаптации P144 based on)
    """
    lang_clause = f"?work wdt:P407 wd:{Q_RUSSIAN_LANG} ." if require_russian else ""
    sparql = f"""
    SELECT DISTINCT ?work ?workLabelRu WHERE {{
      ?work wdt:P31/wdt:P279* wd:{Q_BOOK} ;
            wdt:P577 ?pub ;
            wdt:P840/wdt:P131* wd:{city_qid} .
      {lang_clause}
      FILTER(YEAR(?pub) < {int(pub_before_year)})

      ?adapt wdt:P144 ?work ;
             wdt:P577 ?adate .
      FILTER(YEAR(?adate) >= {int(adapt_after_year)})

      ?work wikibase:sitelinks ?sl . FILTER(?sl >= 1)
      ?work rdfs:label ?workLabelRu FILTER(LANG(?workLabelRu)='ru').
    }}
    LIMIT {int(limit)}
    """.strip()

    try:
        rows = rows_from_select(wd.sparql_select(sparql))
    except Exception:
        return sparql, []

    out = []
    for r in rows:
        qid = uri_to_qid(r.get("work", ""))
        lbl = r.get("workLabelRu")
        if qid and lbl:
            out.append((qid, lbl))
    return sparql, out


def generate_books_example_advanced(complexity: str, idx: int, rng: random.Random, max_attempts: int = 60) -> BenchmarkExample:
    """
    Дополнительный более "сюжетный" шаблон:
      L3: публикация до 1917 + место действия + поздняя экранизация => >=5
      L4: просим 12, но хотим 1..11
      L5: 0 ответов (невозможный ранний cutoff)
    """
    template_id = "books_setting_inverse_adaptation_range"

    if complexity == "L3":
        k = 5
        pub_before = 1917
        adapt_after = 2000
        require = lambda n: n >= k
    elif complexity == "L4":
        k = 12
        pub_before = 1917
        adapt_after = 2015
        require = lambda n: (0 < n < k)
    elif complexity == "L5":
        k = 5
        pub_before = 1200
        adapt_after = 2000
        require = lambda n: n == 0
    else:
        return _generate_books_example_default(complexity, idx, rng, max_attempts=max_attempts)

    for _ in range(max_attempts):
        city_ru, city_qid = _pick_setting_city(rng)
        sparql, items = run_books_setting_adaptation_query(
            city_qid=city_qid,
            pub_before_year=pub_before,
            adapt_after_year=adapt_after,
            require_russian=True,
            limit=300,
        )
        n = len(items)
        if not require(n):
            continue

        q_ru = (
            f"Подбери {k} произведений (книг), которые одновременно: "
            f"(1) опубликованы до {pub_before} года; "
            f"(2) действие заметной части связано с городом {city_ru}; "
            f"(3) у произведения есть экранизация (фильм или сериал) после {adapt_after} года. "
        )
        if complexity == "L4":
            q_ru += "Если таких произведений меньше, чем просят — перечисли все."

        gold_qids = [q for q, _ in items]
        gold_lbls = [l for _, l in items]

        return BenchmarkExample(
            id=f"books_adv_{complexity.lower()}_{idx:04d}",
            domain="books",
            complexity=complexity,
            query_text_ru=q_ru,
            constraints={
                "type": "books_setting_adaptation",
                "template_id": template_id,
                "setting_city_qid": city_qid,
                "setting_city_ru": city_ru,
                "pub_before_year": pub_before,
                "adapt_after_year": adapt_after,
                "require_russian": True,
            },
            requested_count=k,
            gold_answer_qids=gold_qids,
            gold_answer_labels_ru=gold_lbls,
            sparql_query=sparql,
            created_at=utc_now_z() if "utc_now_z" in globals() else None,
            is_advanced=True,
            template_id=template_id,
            template_family="advanced",
        )

    raise RuntimeError(f"Books advanced:{complexity} failed after {max_attempts} attempts")


_generate_books_example_default = generate_books_example

def generate_books_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 100) -> BenchmarkExample:
    if complexity in ("L3", "L4", "L5") and rng.random() < ADV_TEMPLATE_PROB_BOOKS:
        try:
            return generate_books_example_advanced(complexity, idx, rng, max_attempts=max_attempts)
        except Exception:
            pass
    return _generate_books_example_default(complexity, idx, rng, max_attempts=max_attempts)


<a id="video-games-domain"></a>

## 13. Video games domain

Cached video-game pools and bounded templates over platform, genre, developer, publisher, and release year.

In [13]:
Q_VIDEO_GAME = ensure_qid("видеоигра", fallback_qid="Q7889")

VG_GENRE_ANCHORS = [
    "ролевая игра", "платформер", "стратегия", "шутер",
    "головоломка", "приключенческая игра", "симулятор",
    "гонка", "стелс-игра", "survival horror", "файтинг",
    "roguelike", "tower defense", "immersive sim"
]
VG_PLATFORM_ANCHORS = [
    "Windows", "PlayStation 4", "PlayStation 5", "Nintendo Switch",
    "Xbox One", "Xbox Series X and Series S", "Android", "iOS"
]
VG_DEVELOPER_ANCHORS = [
    "Nintendo", "Ubisoft", "Konami", "Capcom", "Valve",
    "Bethesda Game Studios", "id Software", "Rockstar Games",
    "CD Projekt Red", "Kojima Productions", "FromSoftware"
]
VG_PUBLISHER_ANCHORS = [
    "Nintendo", "Electronic Arts", "Ubisoft", "Square Enix",
    "Sony Interactive Entertainment", "Xbox Game Studios",
    "Activision", "Sega", "Konami", "Bandai Namco Entertainment"
]
VG_SERIES_ANCHORS = [
    "Mario", "The Legend of Zelda", "Metal Gear", "Final Fantasy",
    "Resident Evil", "Pokémon", "Assassin's Creed", "Halo",
    "Civilization", "Call of Duty", "Street Fighter", "Mortal Kombat"
]

VG_PLATFORM_BLOCKLIST = {
    "DOS", "MS-DOS", "Linux", "macOS", "Microsoft Windows",
    "Arcade game", "Arcade", "browser", "Web browser"
}
VG_PLATFORM_PREFERRED = {
    "Windows", "PlayStation 4", "PlayStation 5", "Nintendo Switch",
    "Xbox One", "Xbox Series X and Series S", "Android", "iOS"
}
VG_GENRE_PREFERRED = {
    "ролевая игра", "платформер", "стратегия", "шутер",
    "приключенческая игра", "головоломка", "симулятор", "гонка",
    "файтинг", "roguelike", "survival horror", "tower defense", "immersive sim"
}
VG_DEVELOPER_PREFERRED = {
    "Nintendo", "Ubisoft", "Konami", "Capcom", "Valve",
    "Bethesda Game Studios", "id Software", "Rockstar Games",
    "CD Projekt Red", "Kojima Productions", "FromSoftware"
}
VG_PUBLISHER_PREFERRED = {
    "Nintendo", "Electronic Arts", "Ubisoft", "Square Enix",
    "Sony Interactive Entertainment", "Xbox Game Studios",
    "Activision", "Sega", "Konami", "Bandai Namco Entertainment"
}
VG_SERIES_PREFERRED = {
    "Mario", "The Legend of Zelda", "Metal Gear", "Final Fantasy",
    "Resident Evil", "Pokémon", "Assassin's Creed", "Halo",
    "Civilization", "Call of Duty", "Street Fighter", "Mortal Kombat"
}

def _vg_pool_or_anchors(name: str, builder, anchors: List[str], qid_col: str, label_col: str) -> pd.DataFrame:
    df = load_or_build_pool(name, builder)
    if df is None or len(df) == 0:
        df = pd.DataFrame(columns=[qid_col, label_col])
    else:
        df = repair_pool_labels(df, qid_col, label_col)

    try:
        anchor_df = pool_from_anchor_labels(anchors, qid_col, label_col)
    except Exception:
        anchor_df = pd.DataFrame(columns=[qid_col, label_col])

    if len(anchor_df):
        anchor_df = repair_pool_labels(anchor_df, qid_col, label_col)

    if len(df) == 0 and len(anchor_df) == 0:
        return pd.DataFrame(columns=[qid_col, label_col])

    parts = []
    if len(df):
        parts.append(df[[qid_col, label_col]].copy())
    if len(anchor_df):
        parts.append(anchor_df[[qid_col, label_col]].copy())

    out = pd.concat(parts, ignore_index=True).dropna().drop_duplicates(subset=[qid_col, label_col]).reset_index(drop=True)
    out = repair_pool_labels(out, qid_col, label_col)
    return out

def _vg_filter_pool(df: pd.DataFrame, qid_col: str, label_col: str, preferred: Optional[set] = None, blocklist: Optional[set] = None) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=[qid_col, label_col])
    out = df.copy()
    out[label_col] = out[label_col].fillna("").astype(str).str.strip()
    if blocklist:
        out = out[~out[label_col].isin(blocklist)].copy()
    if preferred:
        sub = out[out[label_col].isin(preferred)].copy()
        if len(sub) >= max(4, min(8, len(out))):
            out = sub
    out = out.drop_duplicates(subset=[qid_col, label_col]).reset_index(drop=True)
    return out

vg_genres_df = _vg_pool_or_anchors(
    "videogames_genres_ru_v3",
    lambda: build_value_pool_ru(Q_VIDEO_GAME, "P136", "genre", limit=320),
    VG_GENRE_ANCHORS,
    "genre_qid", "genreLabelRu",
)
vg_platforms_df = _vg_pool_or_anchors(
    "videogames_platforms_ru_v3",
    lambda: build_value_pool_ru(Q_VIDEO_GAME, "P400", "platform", limit=320),
    VG_PLATFORM_ANCHORS,
    "platform_qid", "platformLabelRu",
)
vg_developers_df = _vg_pool_or_anchors(
    "videogames_developers_ru_v3",
    lambda: build_value_pool_ru(Q_VIDEO_GAME, "P178", "developer", limit=320),
    VG_DEVELOPER_ANCHORS,
    "developer_qid", "developerLabelRu",
)
vg_publishers_df = _vg_pool_or_anchors(
    "videogames_publishers_ru_v3",
    lambda: build_value_pool_ru(Q_VIDEO_GAME, "P123", "publisher", limit=320),
    VG_PUBLISHER_ANCHORS,
    "publisher_qid", "publisherLabelRu",
)
vg_series_df = _vg_pool_or_anchors(
    "videogames_series_ru_v3",
    lambda: build_value_pool_ru(Q_VIDEO_GAME, "P179", "series", limit=320),
    VG_SERIES_ANCHORS,
    "series_qid", "seriesLabelRu",
)

vg_genres_pick_df = _vg_filter_pool(vg_genres_df, "genre_qid", "genreLabelRu", preferred=VG_GENRE_PREFERRED)
vg_platforms_pick_df = _vg_filter_pool(vg_platforms_df, "platform_qid", "platformLabelRu", preferred=VG_PLATFORM_PREFERRED, blocklist=VG_PLATFORM_BLOCKLIST)
vg_developers_pick_df = _vg_filter_pool(vg_developers_df, "developer_qid", "developerLabelRu", preferred=VG_DEVELOPER_PREFERRED)
vg_publishers_pick_df = _vg_filter_pool(vg_publishers_df, "publisher_qid", "publisherLabelRu", preferred=VG_PUBLISHER_PREFERRED)
vg_series_pick_df = _vg_filter_pool(vg_series_df, "series_qid", "seriesLabelRu", preferred=VG_SERIES_PREFERRED)

def _vg_pick(df: pd.DataFrame, qid_col: str, label_col: str, rng: random.Random):
    if df is None or len(df) == 0:
        raise ValueError(f"Empty videogames pool for {qid_col}/{label_col}")
    row = df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
    qid = str(row[qid_col]).strip()
    lbl = str(row[label_col]).strip()
    if not lbl or re.fullmatch(r"Q\d+", lbl):
        fixed = fetch_best_labels_ru_en([qid]).get(qid, "")
        if fixed:
            lbl = fixed
    if not lbl or re.fullmatch(r"Q\d+", lbl):
        raise ValueError(f"Bad videogames label for {qid_col}/{label_col}: {qid=} {lbl=}")
    return qid, lbl

def _vg_year_window_text(y1: Optional[int], y2: Optional[int]) -> Optional[str]:
    if y1 is None or y2 is None:
        return None
    if y1 == y2:
        return f"{y1} года"
    return f"в период {y1}–{y2} годов"

def _vg_gold_limit(k: int, complexity: str) -> int:
    k = int(k)
    c = str(complexity)
    if c in ("L1", "L2"):
        return max(k + 5, 12)
    if c == "L3":
        return max(k + 6, 15)
    if c == "L4":
        return max(k + 8, 24)
    if c == "L5":
        return max(k + 5, 12)
    return max(k + 5, 12)

_VIDEOGAME_QUERY_CACHE: Dict[Tuple[Any, ...], Tuple[str, List[Tuple[str, str]], List[str]]] = {}

def _vg_with_fail_fast(callable_obj):
    wd_obj = globals().get("wd", None)
    orig_retries = getattr(wd_obj, "max_retries", None) if wd_obj is not None else None
    orig_timeout = getattr(wd_obj, "timeout", None) if wd_obj is not None else None
    orig_backoff = globals().get("MAX_BACKOFF_SECONDS", None)

    try:
        if wd_obj is not None:
            try:
                wd_obj.max_retries = int(globals().get("VIDEOGAMES_QUERY_MAX_RETRIES", 2))
                wd_obj.timeout = int(globals().get("VIDEOGAMES_QUERY_TIMEOUT_SECONDS", 18))
            except Exception:
                pass
        if orig_backoff is not None:
            try:
                globals()["MAX_BACKOFF_SECONDS"] = int(globals().get("VIDEOGAMES_QUERY_MAX_BACKOFF_SECONDS", 4))
            except Exception:
                pass
        return callable_obj()
    finally:
        if wd_obj is not None:
            if orig_retries is not None:
                wd_obj.max_retries = orig_retries
            if orig_timeout is not None:
                wd_obj.timeout = orig_timeout
        if orig_backoff is not None:
            globals()["MAX_BACKOFF_SECONDS"] = orig_backoff

def run_videogames_query(
    *,
    genre_qid: Optional[str] = None,
    platform_qid: Optional[str] = None,
    developer_qid: Optional[str] = None,
    publisher_qid: Optional[str] = None,
    series_qid: Optional[str] = None,
    y1: Optional[int] = None,
    y2: Optional[int] = None,
    limit: int = 60,
):
    key = (
        str(genre_qid or ""),
        str(platform_qid or ""),
        str(developer_qid or ""),
        str(publisher_qid or ""),
        str(series_qid or ""),
        int(y1) if y1 is not None else None,
        int(y2) if y2 is not None else None,
        int(limit),
    )
    cached = _VIDEOGAME_QUERY_CACHE.get(key)
    if cached is not None:
        return cached

    where: List[str] = []

    if genre_qid:
        where.append(f"?item wdt:P136 wd:{genre_qid} .")
    if platform_qid:
        where.append(f"?item wdt:P400 wd:{platform_qid} .")
    if developer_qid:
        where.append(f"?item wdt:P178 wd:{developer_qid} .")
    if publisher_qid:
        where.append(f"?item wdt:P123 wd:{publisher_qid} .")
    if series_qid:
        where.append(f"?item wdt:P179 wd:{series_qid} .")
    if y1 is not None and y2 is not None:
        where += [
            "?item wdt:P577 ?date .",
            "BIND(YEAR(?date) AS ?year) .",
            f"FILTER(?year >= {int(y1)} && ?year <= {int(y2)}) .",
        ]

    def _run():
        return select_items_with_label_ru_en(
            Q_VIDEO_GAME,
            where,
            limit=int(limit),
            item_var="item",
            use_subclass_closure=True,
        )

    sparql, items = _vg_with_fail_fast(_run)
    res = (sparql, items, where)
    _VIDEOGAME_QUERY_CACHE[key] = res
    return res

def _vg_nlg(template_id: str, k: int, *, genre_ru=None, platform_ru=None, developer_ru=None, publisher_ru=None, series_ru=None, y1=None, y2=None) -> str:
    period_text = _vg_year_window_text(y1, y2)

    if template_id == "vg_genre_platform":
        cond = [f"жанра «{genre_ru}»", f"на платформе: {platform_ru}"]
        if period_text:
            cond.append(period_text)
        return f"Назови {k} видеоигр, " + ", ".join(cond) + "."

    if template_id == "vg_genre_platform_year":
        cond = [f"жанра «{genre_ru}»", f"на платформе: {platform_ru}"]
        if period_text:
            cond.append(period_text)
        return f"Назови {k} видеоигр, " + ", ".join(cond) + "."

    if template_id == "vg_developer_platform_year":
        cond = [f"где разработчик: {developer_ru}", f"на платформе: {platform_ru}"]
        if period_text:
            cond.append(period_text)
        return f"Назови {k} игр, " + ", ".join(cond) + "."

    if template_id == "vg_series_platform_year":
        cond = [f"из серии «{series_ru}»", f"на платформе: {platform_ru}"]
        if period_text:
            cond.append(period_text)
        return f"Назови {k} игр, " + ", ".join(cond) + "."

    if template_id == "vg_genre_publisher_platform_year":
        cond = [f"жанра «{genre_ru}»", f"издатель: {publisher_ru}", f"на платформе: {platform_ru}"]
        if period_text:
            cond.append(period_text)
        return f"Назови {k} видеоигр, " + ", ".join(cond) + "."

    if template_id == "vg_series_developer_platform_year":
        cond = [f"из серии «{series_ru}»", f"где разработчик: {developer_ru}", f"на платформе: {platform_ru}"]
        if period_text:
            cond.append(period_text)
        return f"Назови {k} игр, " + ", ".join(cond) + "."

    if template_id == "vg_genre_developer_publisher_platform_year":
        cond = [f"жанра «{genre_ru}»", f"где разработчик: {developer_ru}", f"издатель: {publisher_ru}", f"на платформе: {platform_ru}"]
        if period_text:
            cond.append(period_text)
        return f"Назови {k} видеоигр, " + ", ".join(cond) + "."

    raise ValueError(f"Unknown videogames template_id: {template_id}")

def _vg_year_choices(complexity: str) -> List[Tuple[Optional[int], Optional[int]]]:
    if complexity == "L1":
        return [(None, None)]
    if complexity == "L2":
        return [(1990, 2005), (2000, 2010), (2010, 2018), (2016, 2025)]
    if complexity == "L3":
        return [(1995, 2008), (2006, 2014), (2012, 2020), (2018, 2025)]
    if complexity == "L4":
        return [(2000, 2008), (2008, 2014), (2014, 2020), (2020, 2025)]
    if complexity == "L5":
        return [(1998, 2006), (2005, 2011), (2010, 2016), (2016, 2023)]
    return [(2000, 2010)]

def _vg_k_for_complexity(complexity: str) -> int:
    return {
        "L1": 5,
        "L2": 5,
        "L3": 4,
        "L4": 3,
        "L5": 3,
    }.get(complexity, 4)

def _vg_template_pool(complexity: str) -> List[str]:
    if complexity == "L1":
        return ["vg_genre_platform"]
    if complexity == "L2":
        return ["vg_genre_platform_year"]
    if complexity == "L3":
        return ["vg_developer_platform_year", "vg_series_platform_year", "vg_genre_publisher_platform_year"]
    if complexity == "L4":
        return ["vg_series_platform_year", "vg_genre_publisher_platform_year", "vg_series_developer_platform_year"]
    if complexity == "L5":
        return ["vg_series_developer_platform_year", "vg_genre_developer_publisher_platform_year"]
    raise ValueError(f"Unknown complexity: {complexity}")

def generate_videogames_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 70) -> BenchmarkExample:
    k = _vg_k_for_complexity(complexity)
    min_needed = k if complexity in ("L1", "L2", "L3") else 1
    query_limit = _vg_gold_limit(k, complexity)

    for _ in range(max_attempts):
        template_id = rng.choice(_vg_template_pool(complexity))
        y1, y2 = rng.choice(_vg_year_choices(complexity))

        genre_qid = genre_ru = None
        platform_qid = platform_ru = None
        developer_qid = developer_ru = None
        publisher_qid = publisher_ru = None
        series_qid = series_ru = None

        if template_id in ("vg_genre_platform", "vg_genre_platform_year", "vg_genre_publisher_platform_year", "vg_genre_developer_publisher_platform_year"):
            genre_qid, genre_ru = _vg_pick(vg_genres_pick_df if len(vg_genres_pick_df) else vg_genres_df, "genre_qid", "genreLabelRu", rng)

        if template_id in (
            "vg_genre_platform", "vg_genre_platform_year", "vg_developer_platform_year",
            "vg_series_platform_year", "vg_genre_publisher_platform_year",
            "vg_series_developer_platform_year", "vg_genre_developer_publisher_platform_year"
        ):
            platform_qid, platform_ru = _vg_pick(vg_platforms_pick_df if len(vg_platforms_pick_df) else vg_platforms_df, "platform_qid", "platformLabelRu", rng)

        if template_id in ("vg_developer_platform_year", "vg_series_developer_platform_year", "vg_genre_developer_publisher_platform_year"):
            developer_qid, developer_ru = _vg_pick(vg_developers_pick_df if len(vg_developers_pick_df) else vg_developers_df, "developer_qid", "developerLabelRu", rng)

        if template_id in ("vg_genre_publisher_platform_year", "vg_genre_developer_publisher_platform_year"):
            publisher_qid, publisher_ru = _vg_pick(vg_publishers_pick_df if len(vg_publishers_pick_df) else vg_publishers_df, "publisher_qid", "publisherLabelRu", rng)

        if template_id in ("vg_series_platform_year", "vg_series_developer_platform_year"):
            series_qid, series_ru = _vg_pick(vg_series_pick_df if len(vg_series_pick_df) else vg_series_df, "series_qid", "seriesLabelRu", rng)

        if genre_qid and (not genre_ru or re.fullmatch(r"Q\d+", str(genre_ru).strip())):
            genre_ru = fetch_best_labels_ru_en([genre_qid]).get(genre_qid, genre_ru)
        if platform_qid and (not platform_ru or re.fullmatch(r"Q\d+", str(platform_ru).strip())):
            platform_ru = fetch_best_labels_ru_en([platform_qid]).get(platform_qid, platform_ru)
        if developer_qid and (not developer_ru or re.fullmatch(r"Q\d+", str(developer_ru).strip())):
            developer_ru = fetch_best_labels_ru_en([developer_qid]).get(developer_qid, developer_ru)
        if publisher_qid and (not publisher_ru or re.fullmatch(r"Q\d+", str(publisher_ru).strip())):
            publisher_ru = fetch_best_labels_ru_en([publisher_qid]).get(publisher_qid, publisher_ru)
        if series_qid and (not series_ru or re.fullmatch(r"Q\d+", str(series_ru).strip())):
            series_ru = fetch_best_labels_ru_en([series_qid]).get(series_qid, series_ru)

        if complexity == "L1":
            y1 = y2 = None

        sparql, items, where = run_videogames_query(
            genre_qid=genre_qid,
            platform_qid=platform_qid,
            developer_qid=developer_qid,
            publisher_qid=publisher_qid,
            series_qid=series_qid,
            y1=y1,
            y2=y2,
            limit=query_limit,
        )

        gold = items[:query_limit]
        if len(gold) < min_needed:
            continue

        gold_qids = [q for q, _ in gold]
        gold_lbls = [l for _, l in gold]
        truncated = len(items) >= query_limit
        ask = build_ask_validator(Q_VIDEO_GAME, where, item_var="item")

        return BenchmarkExample(
            id=f"videogames_{complexity.lower()}_{idx:04d}",
            domain="videogames",
            complexity=complexity,
            query_text_ru=_vg_nlg(
                template_id,
                k,
                genre_ru=genre_ru,
                platform_ru=platform_ru,
                developer_ru=developer_ru,
                publisher_ru=publisher_ru,
                series_ru=series_ru,
                y1=y1,
                y2=y2,
            ),
            constraints={
                "genre_qid": genre_qid, "genre_ru": genre_ru,
                "platform_qid": platform_qid, "platform_ru": platform_ru,
                "developer_qid": developer_qid, "developer_ru": developer_ru,
                "publisher_qid": publisher_qid, "publisher_ru": publisher_ru,
                "series_qid": series_qid, "series_ru": series_ru,
                "year_from": y1, "year_to": y2,
                "template_id": template_id,
            },
            requested_count=k,
            gold_answer_qids=gold_qids,
            gold_answer_labels_ru=gold_lbls,
            sparql_query=sparql,
            created_at=utc_now_z(),
            gold_truncated=truncated,
            ask_validator_sparql=ask,
            is_advanced=(complexity in ("L4", "L5")),
            template_id=template_id,
            template_family="videogames_v3_fast",
        )

    raise RuntimeError(f"Failed to generate videogames example {complexity} after {max_attempts} attempts")

try:
    _test_vg = generate_videogames_example("L1", 0, random.Random(1))
    print(_test_vg.query_text_ru)
    print("gold_size =", len(_test_vg.gold_answer_qids))
except Exception as e:
    print("[WARN] sanity videogames skipped:", e)

if 'DOMAIN_PAUSE_S' in globals() and DOMAIN_PAUSE_S and DOMAIN_PAUSE_S > 0:
    time.sleep(float(DOMAIN_PAUSE_S))


Назови 5 видеоигр, жанра «ролевая игра», на платформе: Xbox Series X and Series S.
gold_size = 12


<a id="music-albums-domain"></a>

## 14. Music albums domain

Album templates with artist, genre, language or country, and release-period constraints.

In [14]:
Q_MUSIC_ALBUM = ensure_qid("музыкальный альбом", fallback_qid="Q482994")

ma_genres_df = load_or_build_pool("music_albums_genres_ru", lambda: build_value_pool_ru(Q_MUSIC_ALBUM, "P136", "genre", limit=350))
ma_countries_df = load_or_build_pool("music_albums_countries_ru", lambda: build_value_pool_ru(Q_MUSIC_ALBUM, "P495", "country", limit=250))
ma_performers_df = load_or_build_pool("music_albums_performers_ru", lambda: build_value_pool_ru(Q_MUSIC_ALBUM, "P175", "performer", limit=350))
ma_labels_df = load_or_build_pool("music_albums_labels_ru", lambda: build_value_pool_ru(Q_MUSIC_ALBUM, "P264", "label", limit=250))

COMMON_MA_GENRES_RU = ["рок", "поп-музыка", "джаз", "хип-хоп", "классическая музыка", "электронная музыка", "метал"]
COMMON_COUNTRIES_RU = ["США", "Россия", "Великобритания", "Франция", "Германия", "Япония", "Канада"]

def pick_music_constraints(rng: random.Random):
    if rng.random() < 0.6:
        g_ru = rng.choice(COMMON_MA_GENRES_RU)
        c_ru = rng.choice(COMMON_COUNTRIES_RU)
        g_qid = ensure_qid(g_ru)
        c_qid = ensure_qid(c_ru)
    else:
        g_qid, g_ru = pick_from_df(ma_genres_df, "genre_qid", "genreLabelRu", rng)
        c_qid, c_ru = pick_from_df(ma_countries_df, "country_qid", "countryLabelRu", rng)
    perf_qid, perf_ru = pick_from_df(ma_performers_df, "performer_qid", "performerLabelRu", rng)
    lab_qid, lab_ru = pick_from_df(ma_labels_df, "label_qid", "labelLabelRu", rng)
    return g_qid, g_ru, c_qid, c_ru, perf_qid, perf_ru, lab_qid, lab_ru

def run_music_album_query(
    genre_qid: str,
    country_qid: str,
    performer_qid: Optional[str],
    label_qid: Optional[str],
    y1: int,
    y2: int,
    limit: int = 250,
):
    where = [
        f"?item wdt:P136 wd:{genre_qid} .",
        f"?item wdt:P495 wd:{country_qid} .",
        "?item wdt:P577 ?date .",
        "BIND(YEAR(?date) AS ?year) .",
        f"FILTER(?year >= {int(y1)} && ?year <= {int(y2)}) .",
    ]
    if performer_qid:
        where.append(f"?item wdt:P175 wd:{performer_qid} .")
    if label_qid:
        where.append(f"?item wdt:P264 wd:{label_qid} .")
    sparql, items = select_items_with_ru_label(Q_MUSIC_ALBUM, where, limit=limit, item_var="item")
    return sparql, items, where

def nlg_music_album_ru(genre_ru: str, country_ru: str, performer_ru: Optional[str], label_ru: Optional[str], y1: int, y2: int, k: int) -> str:
    cond = [f"жанра «{genre_ru}»", f"из страны: {country_ru}"]
    if performer_ru:
        cond.append(f"исполнитель: {performer_ru}")
    if label_ru:
        cond.append(f"лейбл: {label_ru}")
    cond.append(f"в период {y1}–{y2} годов" if y1 != y2 else f"{y1} года")
    return f"Назови {k} музыкальных альбомов, " + ", ".join(cond) + "."

def generate_music_albums_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 90) -> BenchmarkExample:
    for _ in range(max_attempts):
        g_qid, g_ru, c_qid, c_ru, perf_qid, perf_ru, lab_qid, lab_ru = pick_music_constraints(rng)

        decade = rng.choice([(1970,1979),(1980,1989),(1990,1999),(2000,2009),(2010,2019)])
        y1, y2 = decade
        performer_qid = None
        performer_ru = None
        label_qid = None
        label_ru = None
        k = 5

        if complexity == "L1":
            pass
        elif complexity == "L2":
            performer_qid, performer_ru = perf_qid, perf_ru
        elif complexity == "L3":
            performer_qid, performer_ru = perf_qid, perf_ru
            label_qid, label_ru = lab_qid, lab_ru
        elif complexity == "L4":
            performer_qid, performer_ru = perf_qid, perf_ru
            label_qid, label_ru = lab_qid, lab_ru
            y1, y2 = rng.choice([(1970,1974),(1980,1984),(1990,1994),(2000,2004),(2010,2014)])
            k = 12
        elif complexity == "L5":
            performer_qid, performer_ru = perf_qid, perf_ru
            label_qid, label_ru = lab_qid, lab_ru
            y1, y2 = 2500, 2500
            k = 5
        else:
            raise ValueError(f"Unknown complexity: {complexity}")

        sparql, items, where = run_music_album_query(g_qid, c_qid, performer_qid, label_qid, y1, y2, limit=300)

        gold = items[:300]
        gold_qids = [q for q,_ in gold]
        gold_lbls = [l for _,l in gold]
        truncated = len(items) >= 300
        ask = build_ask_validator(Q_MUSIC_ALBUM, where, item_var="item")

        return BenchmarkExample(
            id=f"music_albums_{complexity.lower()}_{idx:04d}",
            domain="music_albums",
            complexity=complexity,
            query_text_ru=nlg_music_album_ru(g_ru, c_ru, performer_ru, label_ru, y1, y2, k),
            constraints={
                "genre_qid": g_qid, "genre_ru": g_ru,
                "country_qid": c_qid, "country_ru": c_ru,
                "performer_qid": performer_qid, "performer_ru": performer_ru,
                "label_qid": label_qid, "label_ru": label_ru,
                "year_from": y1, "year_to": y2,
            },
            requested_count=k,
            gold_answer_qids=gold_qids,
            gold_answer_labels_ru=gold_lbls,
            sparql_query=sparql,
            created_at=utc_now_z(),
            gold_truncated=truncated,
            ask_validator_sparql=ask,
        )

    raise RuntimeError(f"Failed to generate music_albums example {complexity} after {max_attempts} attempts")

try:
    _test_ma = generate_music_albums_example("L1", 0, random.Random(1))
    print(_test_ma.query_text_ru)
    print("gold_size =", len(_test_ma.gold_answer_qids))


except Exception as e:
    print("[WARN] sanity music_albums skipped:", e)
if 'DOMAIN_PAUSE_S' in globals() and DOMAIN_PAUSE_S and DOMAIN_PAUSE_S > 0:
    time.sleep(float(DOMAIN_PAUSE_S))


Назови 5 музыкальных альбомов, жанра «метал», из страны: Канада, в период 1990–1999 годов.
gold_size = 0


<a id="software-domain"></a>

## 15. Software domain

Software templates using class, developer, platform, license, programming language, and release period.

In [15]:
Q_SOFTWARE = ensure_qid("программное обеспечение", fallback_qid="Q7397")
Q_PROG_LANG = ensure_qid("язык программирования", fallback_qid="Q9143")
Q_OPERATING_SYSTEM = ensure_qid("операционная система", fallback_qid="Q9135")
Q_SOFTWARE_LICENSE = ensure_qid("лицензия на программное обеспечение", fallback_qid="Q207621")


COMMON_SW_LANGS = ["Python", "C++", "Java", "JavaScript", "C#", "Go"]
COMMON_SW_OS = ["Windows", "Linux", "macOS", "Android", "iOS"]
COMMON_SW_LIC = ["GNU General Public License", "MIT License", "Apache License 2.0", "BSD licenses"]

sw_langs_df = load_or_build_pool("software_languages_ru", lambda: build_class_entities_pool_ru(Q_PROG_LANG, "lang", limit=260))
sw_oses_df = load_or_build_pool("software_os_ru", lambda: build_class_entities_pool_ru(Q_OPERATING_SYSTEM, "os", limit=220))
sw_licenses_df = load_or_build_pool("software_licenses_ru", lambda: build_class_entities_pool_ru(Q_SOFTWARE_LICENSE, "lic", limit=260))

if sw_langs_df is None or len(sw_langs_df) == 0:
    sw_langs_df = pool_from_anchor_labels(COMMON_SW_LANGS, "lang_qid", "langLabelRu")
if sw_oses_df is None or len(sw_oses_df) == 0:
    sw_oses_df = pool_from_anchor_labels(COMMON_SW_OS, "os_qid", "osLabelRu")
if sw_licenses_df is None or len(sw_licenses_df) == 0:
    sw_licenses_df = pool_from_anchor_labels(COMMON_SW_LIC, "lic_qid", "licLabelRu")

def pick_sw_constraints(rng: random.Random):

    if rng.random() < 0.5:
        l_ru = rng.choice(COMMON_SW_LANGS)
        o_ru = rng.choice(COMMON_SW_OS)
        lic_ru = rng.choice(COMMON_SW_LIC)
        l_qid = ensure_qid(l_ru)
        o_qid = ensure_qid(o_ru)
        lic_qid = ensure_qid(lic_ru)
    else:
        l_qid, l_ru = pick_from_df(sw_langs_df, "lang_qid", "langLabelRu", rng)
        o_qid, o_ru = pick_from_df(sw_oses_df, "os_qid", "osLabelRu", rng)
        lic_qid, lic_ru = pick_from_df(sw_licenses_df, "lic_qid", "licLabelRu", rng)
    return l_qid, l_ru, o_qid, o_ru, lic_qid, lic_ru

def run_software_query(
    lang_qid: str,
    os_qid: str,
    lic_qid: Optional[str],
    y1: Optional[int],
    y2: Optional[int],
    limit: int = 250,
):
    where = [
        f"?item wdt:P277 wd:{lang_qid} .",
        f"?item wdt:P306 wd:{os_qid} .",
    ]
    if lic_qid:
        where.append(f"?item wdt:P275 wd:{lic_qid} .")
    if y1 is not None and y2 is not None:
        where += [
            "?item wdt:P577 ?date .",
            "BIND(YEAR(?date) AS ?year) .",
            f"FILTER(?year >= {int(y1)} && ?year <= {int(y2)}) .",
        ]
    sparql, items = select_items_with_ru_label_direct(Q_SOFTWARE, where, limit=limit, item_var="item")
    return sparql, items, where

def nlg_software_ru(lang_ru: str, os_ru: str, lic_ru: Optional[str], y1: Optional[int], y2: Optional[int], k: int) -> str:
    cond = [f"на языке/стеке: {lang_ru}", f"для ОС: {os_ru}"]
    if lic_ru:
        cond.append(f"лицензия: {lic_ru}")
    if y1 is not None and y2 is not None:
        cond.append(f"в период {y1}–{y2} годов" if y1 != y2 else f"{y1} года")
    return f"Назови {k} программ/пакетов, " + ", ".join(cond) + "."

def generate_software_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 90) -> BenchmarkExample:
    for _ in range(max_attempts):
        lang_qid, lang_ru, os_qid, os_ru, lic_qid, lic_ru = pick_sw_constraints(rng)

        use_lic = False
        y1 = y2 = None
        k = 3

        if complexity == "L1":
            pass
        elif complexity == "L2":
            use_lic = True
        elif complexity == "L3":
            use_lic = True
            y1, y2 = rng.choice([(2000,2009),(2010,2019)])
        elif complexity == "L4":
            use_lic = True
            y1, y2 = rng.choice([(2000,2004),(2010,2014)])
            k = 12
        elif complexity == "L5":
            use_lic = True
            y1, y2 = 2500, 2500
            k = 5
        else:
            raise ValueError(f"Unknown complexity: {complexity}")

        sparql, items, where = run_software_query(
            lang_qid=lang_qid,
            os_qid=os_qid,
            lic_qid=(lic_qid if use_lic else None),
            y1=y1, y2=y2,
            limit=300
        )

        gold = items[:300]
        gold_qids = [q for q,_ in gold]
        gold_lbls = [l for _,l in gold]
        truncated = len(items) >= 300
        ask = build_ask_validator(Q_SOFTWARE, where, item_var="item")

        return BenchmarkExample(
            id=f"software_{complexity.lower()}_{idx:04d}",
            domain="software",
            complexity=complexity,
            query_text_ru=nlg_software_ru(lang_ru, os_ru, (lic_ru if use_lic else None), y1, y2, k),
            constraints={
                "lang_qid": lang_qid, "lang_ru": lang_ru,
                "os_qid": os_qid, "os_ru": os_ru,
                "license_qid": (lic_qid if use_lic else None),
                "license_ru": (lic_ru if use_lic else None),
                "year_from": y1, "year_to": y2,
            },
            requested_count=k,
            gold_answer_qids=gold_qids,
            gold_answer_labels_ru=gold_lbls,
            sparql_query=sparql,
            created_at=utc_now_z(),
            gold_truncated=truncated,
            ask_validator_sparql=ask,
        )

    raise RuntimeError(f"Failed to generate software example {complexity} after {max_attempts} attempts")

try:
    _test_sw = generate_software_example("L1", 0, random.Random(1))
    print(_test_sw.query_text_ru)
    print("gold_size =", len(_test_sw.gold_answer_qids))


except Exception as e:
    print("[WARN] sanity software skipped:", e)
if 'DOMAIN_PAUSE_S' in globals() and DOMAIN_PAUSE_S and DOMAIN_PAUSE_S > 0:
    time.sleep(float(DOMAIN_PAUSE_S))


Назови 3 программ/пакетов, на языке/стеке: Python, для ОС: macOS.
gold_size = 0


<a id="people-domain"></a>

## 16. People domain

Fast people templates based on occupation, citizenship, and birth-year anchors.

In [16]:
import random
from typing import List, Tuple, Optional

Q_HUMAN = "Q5"
Q_MALE = ensure_qid("мужской пол", fallback_qid="Q6581097")
Q_FEMALE = ensure_qid("женский пол", fallback_qid="Q6581072")

PEOPLE_OCCUPATIONS = [
    ("актёр", "Q33999"),
    ("писатель", "Q36180"),
    ("певец", "Q177220"),
    ("композитор", "Q36834"),
    ("режиссёр", "Q2526255"),
    ("спортсмен", "Q2066131"),
    ("футболист", "Q937857"),
    ("математик", "Q170790"),
    ("физик", "Q169470"),
    ("художник", "Q483501"),
    ("журналист", "Q1930187"),
    ("политик", "Q82955"),
]

PEOPLE_COUNTRIES = [
    ("Россия", "Q159"),
    ("США", "Q30"),
    ("Франция", "Q142"),
    ("Германия", "Q183"),
    ("Великобритания", "Q145"),
    ("Италия", "Q38"),
    ("Испания", "Q29"),
    ("Япония", "Q17"),
    ("Китай", "Q148"),
    ("Канада", "Q16"),
]

def _pick_anchor(rng: random.Random, anchors: List[Tuple[str,str]]) -> Tuple[str,str]:
    return rng.choice(anchors)

def run_people_query(
    occ_qid: str,
    country_qid: str,
    year: Optional[int] = None,
    sex_qid: Optional[str] = None,
    limit: int = 200,
) -> Tuple[str, List[Tuple[str,str]]]:
    """
    Возвращает (sparql, [(person_qid, ru_label), ...])
    Ускорение:
      - FILTER по sitelinks (wikibase:sitelinks)
      - год рождения через SUBSTR(STR(?dob),1,4)
      - без ORDER BY RAND()
    """
    year_filter = ""
    if year is not None:
        year_filter = f"""
        BIND(STR(?dob) AS ?dobS)
        FILTER(SUBSTR(?dobS,1,4) = "{int(year)}")
        """

    sex_clause = ""
    if sex_qid is not None:
        sex_clause = f"?person wdt:P21 wd:{sex_qid} ."

    sparql = f"""
    SELECT DISTINCT ?person ?personLabelRu WHERE {{
      ?person wdt:P31 wd:{Q_HUMAN} ;
              wdt:P106 wd:{occ_qid} ;
              wdt:P27 wd:{country_qid} ;
              wdt:P569 ?dob .
      {sex_clause}
      {year_filter}

      ?person wikibase:sitelinks ?sl .
      FILTER(?sl >= 1)

      ?person rdfs:label ?personLabelRu FILTER(LANG(?personLabelRu)='ru').
    }}
    LIMIT {int(limit)}
    """.strip()

    rows = rows_from_select(wd.sparql_select(sparql))
    out = []
    for r in rows:
        qid = uri_to_qid(r.get("person",""))
        lbl = r.get("personLabelRu")
        if qid and lbl:
            out.append((qid, lbl))
    return sparql, out


def generate_people_example_fast(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 40,
) -> BenchmarkExample:
    """
    L1: профессия + гражданство -> 5 персон
    L2: + год рождения -> 5 персон
    L3: + год рождения + пол -> 5 персон
    L4: просим 12, но ответов должно быть 1..11 (insufficient)
    L5: 0 ответов (несовместимые условия)
    """
    if complexity == "L1":
        k = 5
        year = None
        sex = None
        require = lambda n: n >= k
    elif complexity == "L2":
        k = 5
        year = rng.randint(1940, 2000)
        sex = None
        require = lambda n: n >= k
    elif complexity == "L3":
        k = 5
        year = rng.randint(1940, 2000)
        sex = rng.choice([Q_MALE, Q_FEMALE])
        require = lambda n: n >= k
    elif complexity == "L4":
        k = 12
        year = rng.randint(1850, 2005)
        sex = rng.choice([Q_MALE, Q_FEMALE])
        require = lambda n: (0 < n < k)
    elif complexity == "L5":
        k = 5
        year = 1200
        sex = Q_FEMALE
        require = lambda n: n == 0
    else:
        raise ValueError(complexity)

    for _ in range(max_attempts):
        occ_ru, occ_qid = _pick_anchor(rng, PEOPLE_OCCUPATIONS)
        c_ru, c_qid = _pick_anchor(rng, PEOPLE_COUNTRIES)

        sparql, items = run_people_query(
            occ_qid=occ_qid,
            country_qid=c_qid,
            year=year,
            sex_qid=sex,
            limit=250,
        )
        n = len(items)
        if not require(n):
            continue

        parts = [f"Назови {k} известных персон"]
        parts.append(f"профессии: «{occ_ru}»")
        parts.append(f"гражданство: {c_ru}")
        if year is not None and complexity in ("L2","L3","L4","L5"):
            parts.append(f"год рождения: {year}")
        if sex is not None and complexity in ("L3","L4","L5"):
            parts.append("пол: " + ("женский" if sex == Q_FEMALE else "мужской"))
        if complexity == "L4":
            parts.append("если таких персон меньше, чем просят — перечисли всех")
        q_ru = ", ".join(parts) + "."

        gold_qids = [q for q,_ in items]
        gold_lbls = [l for _,l in items]

        return BenchmarkExample(
            id=f"people_{complexity.lower()}_{idx:04d}",
            domain="people",
            complexity=complexity,
            query_text_ru=q_ru,
            constraints={
                "type": "people_list",
                "occupation_qid": occ_qid,
                "occupation_ru": occ_ru,
                "citizenship_qid": c_qid,
                "citizenship_ru": c_ru,
                "birth_year": year,
                "sex_qid": sex,
            },
            requested_count=k,
            gold_answer_qids=gold_qids,
            gold_answer_labels_ru=gold_lbls,
            sparql_query=sparql,
            created_at=utc_now_z() if "utc_now_z" in globals() else None,
        )

    raise RuntimeError(f"People:{complexity} failed to sample after {max_attempts} attempts (WDQS slow/empty).")

try:
    _t = generate_people_example_fast("L1", 0, random.Random(1))
    print(_t.query_text_ru, "gold=", len(_t.gold_answer_qids))
except Exception as e:
    print("[WARN] people sanity skipped:", e)


[WARN] people sanity skipped: WDQS transient failure after 4 retries: HTTPSConnectionPool(host='query.wikidata.org', port=443): Read timed out. (read timeout=30)


<a id="scientists-and-mathematicians-domains"></a>

## 17. Scientists and mathematicians domains

Specialized people templates with occupation chains and award-style constraints.

In [17]:
Q_HUMAN = "Q5"
Q_SCIENTIST = ensure_qid("учёный", fallback_qid="Q901")
Q_PHYSICIST = ensure_qid("физик", fallback_qid="Q169470")
Q_MATHEMATICIAN = ensure_qid("математик", fallback_qid="Q170790")

Q_MALE_LOCAL = globals().get("Q_MALE", ensure_qid("мужской пол", fallback_qid="Q6581097"))
Q_FEMALE_LOCAL = globals().get("Q_FEMALE", ensure_qid("женский пол", fallback_qid="Q6581072"))

PEOPLE_COUNTRIES_LOCAL = globals().get("PEOPLE_COUNTRIES", [
    ("Россия", "Q159"),
    ("США", "Q30"),
    ("Франция", "Q142"),
    ("Германия", "Q183"),
    ("Великобритания", "Q145"),
    ("Италия", "Q38"),
    ("Испания", "Q29"),
    ("Япония", "Q17"),
    ("Китай", "Q148"),
    ("Канада", "Q16"),
])

def _pick_anchor_local(rng: random.Random, anchors: List[Tuple[str,str]]) -> Tuple[str,str]:
    return rng.choice(anchors)

def run_people_occ_query(
    occ_qid: str,
    country_qid: str,
    year: Optional[int] = None,
    sex_qid: Optional[str] = None,
    limit: int = 250,
):
    """
    Быстрый select персон:
      - occupation через P106/P279*
      - citizenship через P27
      - (опц.) год рождения через P569
      - (опц.) пол через P21
    """
    where = [
        f"?person wdt:P106/wdt:P279* wd:{occ_qid} .",
        f"?person wdt:P27 wd:{country_qid} .",
    ]
    if year is not None:
        where += [
            "?person wdt:P569 ?dob .",
            "BIND(YEAR(?dob) AS ?byear) .",
            f"FILTER(?byear = {int(year)}) .",
        ]
    if sex_qid is not None:
        where.append(f"?person wdt:P21 wd:{sex_qid} .")

    sparql, items = select_items_with_ru_label(Q_HUMAN, where, limit=limit, item_var="person")
    return sparql, items, where

def nlg_people_occ_ru(role_inst_pl: str, country_ru: str, year: Optional[int], sex_qid: Optional[str], k: int) -> str:
    cond = [f"которые являются {role_inst_pl}", f"и имеют гражданство: {country_ru}"]
    if year is not None:
        cond.append(f"и родились в {int(year)} году")
    if sex_qid is not None:
        sex_ru = "женского пола" if sex_qid == Q_FEMALE_LOCAL else "мужского пола"
        cond.append(f"и {sex_ru}")
    return f"Назови {k} известных персон, " + ", ".join(cond) + "."

def _make_people_occ_generator(domain_key: str, role_inst_pl: str, occ_qid: str):
    def _gen(complexity: str, idx: int, rng: random.Random, max_attempts: int = 40) -> BenchmarkExample:
        if complexity == "L1":
            k = 5
            year = None
            sex = None
            require = lambda n: n >= k
        elif complexity == "L2":
            k = 5
            year = rng.randint(1940, 2000)
            sex = None
            require = lambda n: n >= k
        elif complexity == "L3":
            k = 5
            year = rng.randint(1940, 2000)
            sex = rng.choice([Q_MALE_LOCAL, Q_FEMALE_LOCAL])
            require = lambda n: n >= k
        elif complexity == "L4":
            k = 12
            year = rng.randint(1850, 2005)
            sex = rng.choice([Q_MALE_LOCAL, Q_FEMALE_LOCAL])
            require = lambda n: (0 < n < k)
        elif complexity == "L5":
            k = 5
            year = 2500
            sex = Q_FEMALE_LOCAL
            require = lambda n: n == 0
        else:
            raise ValueError(complexity)

        for _ in range(max_attempts):
            c_ru, c_qid = _pick_anchor_local(rng, PEOPLE_COUNTRIES_LOCAL)

            sparql, items, where = run_people_occ_query(
                occ_qid=occ_qid,
                country_qid=c_qid,
                year=year,
                sex_qid=sex,
                limit=300,
            )
            n = len(items)
            if not require(n):
                continue

            gold = items[:300]
            gold_qids = [q for q,_ in gold]
            gold_lbls = [l for _,l in gold]
            truncated = len(items) >= 300
            ask = build_ask_validator(Q_HUMAN, where, item_var="person")

            qtext = nlg_people_occ_ru(role_inst_pl, c_ru, year, sex, k)

            return BenchmarkExample(
                id=f"{domain_key}_{complexity.lower()}_{idx:04d}",
                domain=domain_key,
                complexity=complexity,
                query_text_ru=qtext,
                constraints={
                    "occupation_qid": occ_qid,
                    "occupation_ru": role_inst_pl,
                    "citizenship_qid": c_qid,
                    "citizenship_ru": c_ru,
                    "birth_year": year,
                    "sex_qid": sex,
                },
                requested_count=k,
                gold_answer_qids=gold_qids,
                gold_answer_labels_ru=gold_lbls,
                sparql_query=sparql,
                created_at=utc_now_z(),
                gold_truncated=truncated,
                ask_validator_sparql=ask,
            )

        sparql, items, where = run_people_occ_query(
            occ_qid=occ_qid,
            country_qid=PEOPLE_COUNTRIES_LOCAL[0][1],
            year=year,
            sex_qid=sex,
            limit=300,
        )
        gold = items[:300]
        ask = build_ask_validator(Q_HUMAN, where, item_var="person")
        return BenchmarkExample(
            id=f"{domain_key}_{complexity.lower()}_{idx:04d}",
            domain=domain_key,
            complexity=complexity,
            query_text_ru=nlg_people_occ_ru(role_inst_pl, PEOPLE_COUNTRIES_LOCAL[0][0], year, sex, k),
            constraints={
                "occupation_qid": occ_qid,
                "occupation_ru": role_inst_pl,
                "citizenship_qid": PEOPLE_COUNTRIES_LOCAL[0][1],
                "citizenship_ru": PEOPLE_COUNTRIES_LOCAL[0][0],
                "birth_year": year,
                "sex_qid": sex,
            },
            requested_count=k,
            gold_answer_qids=[q for q,_ in gold],
            gold_answer_labels_ru=[l for _,l in gold],
            sparql_query=sparql,
            created_at=utc_now_z(),
            gold_truncated=len(items) >= 300,
            ask_validator_sparql=ask,
        )
    return _gen

generate_scientists_example = _make_people_occ_generator("scientists", "учёными", Q_SCIENTIST)
generate_physicists_example = _make_people_occ_generator("physicists", "физиками", Q_PHYSICIST)
generate_mathematicians_example = _make_people_occ_generator("mathematicians", "математиками", Q_MATHEMATICIAN)

# ADVANCED templates for Scientists/Physicists/Mathematicians
#  - award qualifiers: P166 statement with pq:P585 (point in time)

ADV_TEMPLATE_PROB_SCI = 0.15

Q_NOBEL = ensure_qid("Нобелевская премия", fallback_qid="Q7191")
Q_FIELDS = ensure_qid("Медаль Филдса", fallback_qid="Q170172")

def run_people_award_qual_query(
    occ_qid: str,
    award_qid: str,
    y1: int,
    y2: int,
    country_qid: Optional[str] = None,
    sex_qid: Optional[str] = None,
    limit: int = 250,
) -> Tuple[str, List[Tuple[str,str]], List[str]]:
    """
    Ищем людей с профессией (occupation-chain), у которых есть награда award_qid
    со временем награждения pq:P585 в диапазоне [y1,y2].
    Возвращает: (sparql, items[(qid,label)], where_lines_for_ask)
    """
    where = [
        f"?person wdt:P31 wd:{Q_HUMAN} .",
        f"?person wdt:P106/wdt:P279* wd:{occ_qid} .",
    ]
    if country_qid:
        where.append(f"?person wdt:P27 wd:{country_qid} .")
    if sex_qid:
        where.append(f"?person wdt:P21 wd:{sex_qid} .")

    # statement with qualifiers
    where += [
        "?person p:P166 ?st .",
        f"?st ps:P166 wd:{award_qid} .",
        "?st pq:P585 ?t .",
        "BIND(YEAR(?t) AS ?ay) .",
        f"FILTER(?ay >= {int(y1)} && ?ay <= {int(y2)}) .",
        "?person wikibase:sitelinks ?sl .",
        "FILTER(?sl >= 1) .",
        "?person rdfs:label ?personLabelRu FILTER(LANG(?personLabelRu)='ru') .",
    ]
    where_block = (chr(10) + "      ").join(where)
    sparql = f"""
    SELECT DISTINCT ?person ?personLabelRu WHERE {{
      {where_block}
    }}
    LIMIT {int(limit)}
    """.strip()

    try:
        rows = rows_from_select(wd.sparql_select(sparql))
    except Exception:
        return sparql, [], where

    items=[]
    for r in rows:
        qid=uri_to_qid(r.get("person",""))
        lbl=r.get("personLabelRu")
        if qid and lbl:
            items.append((qid,lbl))
    return sparql, items, where

def _make_people_occ_generator_with_advanced(domain_key: str, who_ru: str, occ_qid: str, default_gen):
    def _gen(complexity: str, idx: int, rng: random.Random, max_attempts: int = 80) -> BenchmarkExample:
        if complexity in ("L3","L4","L5") and rng.random() < ADV_TEMPLATE_PROB_SCI:
            template_id = f"{domain_key}_award_qualifiers"
            if domain_key == "mathematicians":
                award_qid = Q_FIELDS if rng.random() < 0.8 else Q_NOBEL
                award_ru = "Медаль Филдса" if award_qid == Q_FIELDS else "Нобелевская премия"
            else:
                award_qid = Q_NOBEL if rng.random() < 0.85 else Q_FIELDS
                award_ru = "Нобелевская премия" if award_qid == Q_NOBEL else "Медаль Филдса"

            if complexity == "L3":
                k=5; y1,y2 = 1950, 2020; require=lambda n: n>=k
            elif complexity == "L4":
                k=12; y1,y2 = rng.randint(1950, 2010), rng.randint(1950, 2010)
                if y2 < y1: y1,y2 = y2,y1
                require=lambda n: (0 < n < k)
            else:  # L5
                k=5; y1,y2 = 1200, 1200; require=lambda n: n==0

            for _ in range(max_attempts):
                c_qid = None
                if rng.random() < 0.5 and 'PEOPLE_COUNTRIES_LOCAL' in globals():
                    _, c_qid = rng.choice(PEOPLE_COUNTRIES_LOCAL)
                sex_qid = rng.choice([Q_MALE_LOCAL, Q_FEMALE_LOCAL]) if rng.random() < 0.4 else None

                sparql, items, where = run_people_award_qual_query(
                    occ_qid=occ_qid,
                    award_qid=award_qid,
                    y1=y1, y2=y2,
                    country_qid=c_qid,
                    sex_qid=sex_qid,
                    limit=300,
                )
                n=len(items)
                if not require(n):
                    continue

                q_parts = [f"Назови {k} {who_ru}"]
                q_parts.append(f"которые получали награду «{award_ru}» в период {y1}–{y2}")
                if c_qid:
                    q_parts.append("с заданным гражданством")
                if sex_qid:
                    q_parts.append("и с заданным полом")
                if complexity == "L4":
                    q_parts.append("если таких людей меньше, чем просят — перечисли всех")
                q_ru = ", ".join(q_parts) + "."

                ask = build_ask_validator(Q_HUMAN, where, item_var="person")

                return BenchmarkExample(
                    id=f"{domain_key}_adv_{complexity.lower()}_{idx:04d}",
                    domain=domain_key,
                    complexity=complexity,
                    query_text_ru=q_ru,
                    constraints={
                        "type": "people_award_qual",
                        "template_id": template_id,
                        "occupation_qid": occ_qid,
                        "award_qid": award_qid,
                        "award_ru": award_ru,
                        "award_year_from": y1,
                        "award_year_to": y2,
                        "citizenship_qid": c_qid,
                        "sex_qid": sex_qid,
                    },
                    requested_count=k,
                    gold_answer_qids=[q for q,_ in items],
                    gold_answer_labels_ru=[l for _,l in items],
                    sparql_query=sparql,
                    created_at=utc_now_z(),
                    gold_truncated=len(items) >= 300,
                    ask_validator_sparql=ask,
                    is_advanced=True,
                    template_id=template_id,
                    template_family="advanced",
                )
        return default_gen(complexity, idx, rng, max_attempts=max_attempts)
    return _gen

_generate_scientists_example_default = generate_scientists_example
_generate_physicists_example_default = generate_physicists_example
_generate_mathematicians_example_default = generate_mathematicians_example

generate_scientists_example = _make_people_occ_generator_with_advanced("scientists", "учёных", Q_SCIENTIST, _generate_scientists_example_default)
generate_physicists_example = _make_people_occ_generator_with_advanced("physicists", "физиков", Q_PHYSICIST, _generate_physicists_example_default)
generate_mathematicians_example = _make_people_occ_generator_with_advanced("mathematicians", "математиков", Q_MATHEMATICIAN, _generate_mathematicians_example_default)


<a id="paintings-and-museums-domains"></a>

## 18. Paintings and museums domains

Artwork and museum templates with collection, author, location, and stable multihop refinements.

In [18]:
Q_PAINTING = ensure_qid("картина", fallback_qid="Q3305213")

#

def _paint_parse_year(val) -> Optional[int]:
    if val is None:
        return None
    s = str(val).strip()
    if not s:
        return None
    m = re.search(r"(-?\d{3,4})", s)
    if not m:
        return None
    try:
        y = int(m.group(1))
    except Exception:
        return None
    if y < 1000 or y > 2100:
        return None
    return y

def _paint_year_bucket_meta(y: Optional[int]) -> Tuple[Optional[str], Optional[int], Optional[int]]:
    if y is None:
        return None, None, None
    if y <= 1599:
        return "1400_1599", 1400, 1599
    if y <= 1799:
        return "1600_1799", 1600, 1799
    if y <= 1899:
        return "1800_1899", 1800, 1899
    return "1900_2024", 1900, 2024

def _paint_first_nonempty(*vals):
    for v in vals:
        if v is not None and str(v).strip():
            return str(v).strip()
    return None

def build_paintings_seed_rows(limit: int = 1600) -> pd.DataFrame:
    """
    Собирает sample реальных картин и их свойств.
    Это не пул "возможных значений", а именно реальные строки:
      item, genre, creator, collection, country(collection), year
    """
    def _run(class_clause: str):
        sparql = f"""
        SELECT DISTINCT
          ?item ?itemLabelRu
          ?genre ?genreLabelRu
          ?creator ?creatorLabelRu
          ?collection ?collectionLabelRu
          ?country ?countryLabelRu
          ?date
        WHERE {{
          {class_clause}
          ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") .

          OPTIONAL {{
            ?item wdt:P136 ?genre .
            ?genre rdfs:label ?genreLabelRu FILTER(LANG(?genreLabelRu) = "ru") .
          }}
          OPTIONAL {{
            ?item wdt:P170 ?creator .
            ?creator rdfs:label ?creatorLabelRu FILTER(LANG(?creatorLabelRu) = "ru") .
          }}
          OPTIONAL {{
            ?item wdt:P195 ?collection .
            ?collection rdfs:label ?collectionLabelRu FILTER(LANG(?collectionLabelRu) = "ru") .
            OPTIONAL {{
              ?collection wdt:P17 ?country .
              ?country rdfs:label ?countryLabelRu FILTER(LANG(?countryLabelRu) = "ru") .
            }}
          }}
          OPTIONAL {{ ?item wdt:P571 ?date . }}
        }}
        LIMIT {int(limit)}
        """
        rows = rows_from_select(wd.sparql_select(sparql))
        data = []
        for r in rows:
            item_qid = uri_to_qid(r.get("item", ""))
            if not item_qid:
                continue
            year = _paint_parse_year(r.get("date"))
            yb, y1, y2 = _paint_year_bucket_meta(year)
            data.append({
                "item_qid": item_qid,
                "itemLabelRu": _paint_first_nonempty(r.get("itemLabelRu")),
                "genre_qid": uri_to_qid(r.get("genre", "")),
                "genreLabelRu": _paint_first_nonempty(r.get("genreLabelRu")),
                "creator_qid": uri_to_qid(r.get("creator", "")),
                "creatorLabelRu": _paint_first_nonempty(r.get("creatorLabelRu")),
                "collection_qid": uri_to_qid(r.get("collection", "")),
                "collectionLabelRu": _paint_first_nonempty(r.get("collectionLabelRu")),
                "country_qid": uri_to_qid(r.get("country", "")),
                "countryLabelRu": _paint_first_nonempty(r.get("countryLabelRu")),
                "year": year,
                "year_bucket": yb,
                "year_from": y1,
                "year_to": y2,
            })
        return pd.DataFrame(data)

    try:
        df = _run(f"?item wdt:P31/wdt:P279* wd:{Q_PAINTING} .")
        if df is not None and len(df) > 0:
            return df.drop_duplicates().reset_index(drop=True)
    except Exception:
        pass

    try:
        df = _run(f"?item wdt:P31 wd:{Q_PAINTING} .")
        if df is not None:
            return df.drop_duplicates().reset_index(drop=True)
    except Exception:
        pass

    return pd.DataFrame()

paintings_seed_df = load_or_build_pool("paintings_seed_rows_v5", lambda: build_paintings_seed_rows(limit=1600))

def _paint_group_candidates(
    df: pd.DataFrame,
    dims: List[str],
    min_n: int = 1,
    max_n: Optional[int] = None,
    max_rows: int = 500,
) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame()

    sub = df.copy()

    group_cols = []
    agg_spec = {"n": ("item_qid", "nunique")}

    for d in dims:
        if d == "year_bucket":
            sub = sub[sub["year_bucket"].notna()]
            group_cols.append("year_bucket")
            agg_spec["year_from"] = ("year_from", "first")
            agg_spec["year_to"] = ("year_to", "first")
        else:
            qcol = f"{d}_qid"
            lcol = f"{d}LabelRu"
            if qcol not in sub.columns or lcol not in sub.columns:
                return pd.DataFrame()
            sub = sub[sub[qcol].notna() & sub[lcol].notna()]
            group_cols.append(qcol)
            agg_spec[lcol] = (lcol, "first")

    if len(sub) == 0:
        return pd.DataFrame()

    grp = sub.groupby(group_cols, dropna=False).agg(**agg_spec).reset_index()

    if max_n is None:
        grp = grp[grp["n"] >= int(min_n)]
    else:
        grp = grp[(grp["n"] >= int(min_n)) & (grp["n"] < int(max_n))]

    if len(grp) == 0:
        return pd.DataFrame()

    grp = grp.sort_values(["n"], ascending=[False]).reset_index(drop=True)
    if max_rows and len(grp) > max_rows:
        grp = grp.head(int(max_rows)).copy()
    return grp.reset_index(drop=True)

PAINT_L1_TABLES = [
    ("creator_only",        _paint_group_candidates(paintings_seed_df, ["creator"], min_n=5, max_rows=300)),
    ("genre_only",          _paint_group_candidates(paintings_seed_df, ["genre"], min_n=5, max_rows=300)),
    ("collection_only",     _paint_group_candidates(paintings_seed_df, ["collection"], min_n=5, max_rows=300)),
    ("country_only",        _paint_group_candidates(paintings_seed_df, ["country"], min_n=5, max_rows=300)),
]

PAINT_L2_TABLES = [
    ("genre_creator",       _paint_group_candidates(paintings_seed_df, ["genre", "creator"], min_n=3, max_rows=500)),
    ("genre_country",       _paint_group_candidates(paintings_seed_df, ["genre", "country"], min_n=3, max_rows=500)),
    ("creator_collection",  _paint_group_candidates(paintings_seed_df, ["creator", "collection"], min_n=3, max_rows=500)),
    ("collection_country",  _paint_group_candidates(paintings_seed_df, ["collection", "country"], min_n=3, max_rows=500)),
]

PAINT_L3_TABLES = [
    ("genre_collection_year", _paint_group_candidates(paintings_seed_df, ["genre", "collection", "year_bucket"], min_n=2, max_rows=500)),
    ("genre_country_year",    _paint_group_candidates(paintings_seed_df, ["genre", "country", "year_bucket"], min_n=2, max_rows=500)),
    ("creator_country_year",  _paint_group_candidates(paintings_seed_df, ["creator", "country", "year_bucket"], min_n=2, max_rows=500)),
]

PAINT_L4_TABLES = [
    ("genre_creator_collection", _paint_group_candidates(paintings_seed_df, ["genre", "creator", "collection"], min_n=1, max_n=5, max_rows=500)),
    ("genre_creator_year",       _paint_group_candidates(paintings_seed_df, ["genre", "creator", "year_bucket"], min_n=1, max_n=5, max_rows=500)),
    ("creator_collection_year",  _paint_group_candidates(paintings_seed_df, ["creator", "collection", "year_bucket"], min_n=1, max_n=5, max_rows=500)),
]

PAINT_L5_BASE_TABLES = [
    ("genre_creator",       _paint_group_candidates(paintings_seed_df, ["genre", "creator"], min_n=1, max_rows=500)),
    ("creator_collection",  _paint_group_candidates(paintings_seed_df, ["creator", "collection"], min_n=1, max_rows=500)),
    ("genre_country",       _paint_group_candidates(paintings_seed_df, ["genre", "country"], min_n=1, max_rows=500)),
]

def _paint_pick_candidate(tables: List[Tuple[str, pd.DataFrame]], rng: random.Random) -> Tuple[Optional[str], Optional[dict]]:
    available = [(name, df) for name, df in tables if df is not None and len(df) > 0]
    if not available:
        return None, None
    name, df = rng.choice(available)
    row = df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0].to_dict()
    return name, row

def _paint_candidate_to_params(mode: str, row: dict) -> dict:
    params = {
        "genre_qid": None, "genre_ru": None,
        "movement_qid": None, "movement_ru": None,
        "country_qid": None, "country_ru": None,
        "creator_qid": None, "creator_ru": None,
        "collection_qid": None, "collection_ru": None,
        "year_from": None, "year_to": None,
        "mode": mode,
    }

    if row is None:
        return params

    if row.get("genre_qid"):
        params["genre_qid"] = row.get("genre_qid")
        params["genre_ru"] = row.get("genreLabelRu")
    if row.get("creator_qid"):
        params["creator_qid"] = row.get("creator_qid")
        params["creator_ru"] = row.get("creatorLabelRu")
    if row.get("collection_qid"):
        params["collection_qid"] = row.get("collection_qid")
        params["collection_ru"] = row.get("collectionLabelRu")
    if row.get("country_qid"):
        params["country_qid"] = row.get("country_qid")
        params["country_ru"] = row.get("countryLabelRu")
    if row.get("year_from") and row.get("year_to"):
        params["year_from"] = int(row.get("year_from"))
        params["year_to"] = int(row.get("year_to"))

    return params

def run_paintings_query(
    genre_qid: Optional[str] = None,
    movement_qid: Optional[str] = None,
    country_qid: Optional[str] = None,
    creator_qid: Optional[str] = None,
    collection_qid: Optional[str] = None,
    y1: Optional[int] = None,
    y2: Optional[int] = None,
    limit: int = 300,
):
    where: List[str] = []

    if genre_qid:
        where.append(f"?item wdt:P136 wd:{genre_qid} .")
    if movement_qid:
        where.append(f"?item wdt:P135 wd:{movement_qid} .")
    if creator_qid:
        where.append(f"?item wdt:P170 wd:{creator_qid} .")

    if collection_qid:
        where.append(f"?item wdt:P195 wd:{collection_qid} .")
        if country_qid:
            where.append(f"wd:{collection_qid} wdt:P17 wd:{country_qid} .")
    elif country_qid:
        where += [
            "?item wdt:P195 ?collectionCountryFilter .",
            f"?collectionCountryFilter wdt:P17 wd:{country_qid} .",
        ]

    if y1 is not None and y2 is not None:
        where += [
            "?item wdt:P571 ?date .",
            "BIND(YEAR(?date) AS ?year) .",
            f"FILTER(?year >= {int(y1)} && ?year <= {int(y2)}) .",
        ]

    sparql, items = select_items_with_ru_label(Q_PAINTING, where, limit=limit, item_var="item")
    return sparql, items, where

def nlg_paintings_ru(
    genre_ru: Optional[str],
    movement_ru: Optional[str],
    country_ru: Optional[str],
    creator_ru: Optional[str],
    collection_ru: Optional[str],
    y1: Optional[int],
    y2: Optional[int],
    k: int,
) -> str:
    cond = []
    if genre_ru:
        cond.append(f"жанра «{genre_ru}»")
    if movement_ru:
        cond.append(f"относящиеся к направлению «{movement_ru}»")
    if creator_ru:
        cond.append(f"созданные автором: {creator_ru}")
    if collection_ru:
        cond.append(f"находящиеся в коллекции: {collection_ru}")
    elif country_ru:
        cond.append(f"находящиеся в коллекциях или музеях в стране: {country_ru}")
    if y1 is not None and y2 is not None:
        cond.append(f"созданные в период {int(y1)}–{int(y2)}")

    if not cond:
        return f"Назови {k} известных картин."
    return f"Назови {k} известных картин, " + ", ".join(cond) + "."

def _paint_fallback_from_seed(rng: random.Random) -> dict:
    if paintings_seed_df is not None and len(paintings_seed_df) > 0:
        row = paintings_seed_df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0].to_dict()
        params = _paint_candidate_to_params("seed_fallback", row)
        for keep in ["creator_qid", "genre_qid", "collection_qid", "country_qid"]:
            if params.get(keep):
                short = {
                    "genre_qid": None, "genre_ru": None,
                    "movement_qid": None, "movement_ru": None,
                    "country_qid": None, "country_ru": None,
                    "creator_qid": None, "creator_ru": None,
                    "collection_qid": None, "collection_ru": None,
                    "year_from": None, "year_to": None,
                    "mode": f"fallback_{keep}",
                }
                if keep == "creator_qid":
                    short["creator_qid"] = params["creator_qid"]; short["creator_ru"] = params["creator_ru"]
                elif keep == "genre_qid":
                    short["genre_qid"] = params["genre_qid"]; short["genre_ru"] = params["genre_ru"]
                elif keep == "collection_qid":
                    short["collection_qid"] = params["collection_qid"]; short["collection_ru"] = params["collection_ru"]
                elif keep == "country_qid":
                    short["country_qid"] = params["country_qid"]; short["country_ru"] = params["country_ru"]
                return short
    return {
        "genre_qid": None, "genre_ru": None,
        "movement_qid": None, "movement_ru": None,
        "country_qid": None, "country_ru": None,
        "creator_qid": None, "creator_ru": None,
        "collection_qid": None, "collection_ru": None,
        "year_from": None, "year_to": None,
        "mode": "fallback_empty",
    }

def generate_paintings_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 60) -> BenchmarkExample:
    if complexity == "L1":
        k = 5
        tables = PAINT_L1_TABLES
        require = lambda n: n >= k
    elif complexity == "L2":
        k = 3
        tables = PAINT_L2_TABLES
        require = lambda n: n >= k
    elif complexity == "L3":
        k = 2
        tables = PAINT_L3_TABLES
        require = lambda n: n >= k
    elif complexity == "L4":
        k = 5
        tables = PAINT_L4_TABLES
        require = lambda n: (0 < n < k)
    elif complexity == "L5":
        k = 5
        tables = PAINT_L5_BASE_TABLES
        require = lambda n: n == 0
    else:
        raise ValueError(complexity)

    last_payload = None

    for _ in range(max_attempts):
        mode, row = _paint_pick_candidate(tables, rng)
        params = _paint_candidate_to_params(mode or "unknown", row)

        if row is None:
            params = _paint_fallback_from_seed(rng)

        if complexity == "L5":
            params["year_from"] = 2500
            params["year_to"] = 2510
            params["mode"] = f"{params.get('mode', 'base')}_future"

        sparql, items, where = run_paintings_query(
            genre_qid=params["genre_qid"],
            movement_qid=params["movement_qid"],
            country_qid=params["country_qid"],
            creator_qid=params["creator_qid"],
            collection_qid=params["collection_qid"],
            y1=params["year_from"],
            y2=params["year_to"],
            limit=300,
        )
        n = len(items)

        last_payload = {**params, "sparql": sparql, "items": items, "where": where}

        if not require(n):
            continue

        gold = items[:300]
        ask = build_ask_validator(Q_PAINTING, where, item_var="item")
        return BenchmarkExample(
            id=f"paintings_{complexity.lower()}_{idx:04d}",
            domain="paintings",
            complexity=complexity,
            query_text_ru=nlg_paintings_ru(
                genre_ru=params["genre_ru"],
                movement_ru=params["movement_ru"],
                country_ru=params["country_ru"],
                creator_ru=params["creator_ru"],
                collection_ru=params["collection_ru"],
                y1=params["year_from"],
                y2=params["year_to"],
                k=k,
            ),
            constraints={
                "genre_qid": params["genre_qid"], "genre_ru": params["genre_ru"],
                "movement_qid": params["movement_qid"], "movement_ru": params["movement_ru"],
                "country_qid": params["country_qid"], "country_ru": params["country_ru"],
                "creator_qid": params["creator_qid"], "creator_ru": params["creator_ru"],
                "collection_qid": params["collection_qid"], "collection_ru": params["collection_ru"],
                "year_from": params["year_from"], "year_to": params["year_to"],
                "mode": params["mode"],
            },
            requested_count=k,
            gold_answer_qids=[q for q, _ in gold],
            gold_answer_labels_ru=[l for _, l in gold],
            sparql_query=sparql,
            created_at=utc_now_z(),
            gold_truncated=len(items) >= 300,
            ask_validator_sparql=ask,
        )

    if last_payload is None:
        params = _paint_fallback_from_seed(rng)
        sparql, items, where = run_paintings_query(
            genre_qid=params["genre_qid"],
            movement_qid=params["movement_qid"],
            country_qid=params["country_qid"],
            creator_qid=params["creator_qid"],
            collection_qid=params["collection_qid"],
            y1=params["year_from"],
            y2=params["year_to"],
            limit=300,
        )
        last_payload = {**params, "sparql": sparql, "items": items, "where": where}

    ask = build_ask_validator(Q_PAINTING, last_payload["where"], item_var="item")
    return BenchmarkExample(
        id=f"paintings_{complexity.lower()}_{idx:04d}",
        domain="paintings",
        complexity=complexity,
        query_text_ru=nlg_paintings_ru(
            last_payload["genre_ru"],
            last_payload["movement_ru"],
            last_payload["country_ru"],
            last_payload["creator_ru"],
            last_payload["collection_ru"],
            last_payload["year_from"],
            last_payload["year_to"],
            k,
        ),
        constraints={
            "genre_qid": last_payload["genre_qid"], "genre_ru": last_payload["genre_ru"],
            "movement_qid": last_payload["movement_qid"], "movement_ru": last_payload["movement_ru"],
            "country_qid": last_payload["country_qid"], "country_ru": last_payload["country_ru"],
            "creator_qid": last_payload["creator_qid"], "creator_ru": last_payload["creator_ru"],
            "collection_qid": last_payload["collection_qid"], "collection_ru": last_payload["collection_ru"],
            "year_from": last_payload["year_from"], "year_to": last_payload["year_to"],
            "mode": last_payload["mode"],
        },
        requested_count=k,
        gold_answer_qids=[q for q, _ in last_payload["items"][:300]],
        gold_answer_labels_ru=[l for _, l in last_payload["items"][:300]],
        sparql_query=last_payload["sparql"],
        created_at=utc_now_z(),
        gold_truncated=len(last_payload["items"]) >= 300,
        ask_validator_sparql=ask,
    )

def paintings_debug_tables() -> pd.DataFrame:
    rows = []
    for level, tables in [
        ("L1", PAINT_L1_TABLES),
        ("L2", PAINT_L2_TABLES),
        ("L3", PAINT_L3_TABLES),
        ("L4", PAINT_L4_TABLES),
        ("L5", PAINT_L5_BASE_TABLES),
    ]:
        for name, df in tables:
            rows.append({
                "complexity": level,
                "table": name,
                "rows": int(len(df)) if df is not None else 0,
            })
    return pd.DataFrame(rows)

def smoke_test_paintings(seed: int = 42, per_complexity: int = 1) -> pd.DataFrame:
    rows = []
    local_rng = random.Random(seed)

    for complexity in ["L1", "L2", "L3", "L4", "L5"]:
        for j in range(per_complexity):
            ex = generate_paintings_example(complexity, j + 1, local_rng, max_attempts=20)
            rows.append({
                "complexity": complexity,
                "mode": ex.constraints.get("mode"),
                "gold_count": len(ex.gold_answer_qids),
                "requested_count": ex.requested_count,
                "query_text_ru": ex.query_text_ru,
            })
    return pd.DataFrame(rows)


# --- Museums ---
Q_MUSEUM = ensure_qid("музей", fallback_qid="Q33506")
museum_countries_df = load_or_build_pool("museums_countries_ru", lambda: build_value_pool_ru(Q_MUSEUM, "P17", "country", limit=180))
museum_types_df = load_or_build_pool("museums_types_ru", lambda: build_value_pool_ru(Q_MUSEUM, "P31", "type", limit=220))
museum_cities_df = load_or_build_pool("museums_cities_ru", lambda: build_value_pool_ru(Q_MUSEUM, "P131", "city", limit=240))

COMMON_MUSEUM_TYPES_RU = ["художественный музей", "музей современного искусства", "научный музей", "исторический музей"]
COMMON_MUSEUM_COUNTRIES_RU = ["Россия", "США", "Франция", "Великобритания", "Германия", "Италия", "Испания", "Япония", "Нидерланды"]

def pick_museums_constraints(rng: random.Random):
    # country
    if rng.random() < 0.8:
        c_ru = rng.choice(COMMON_MUSEUM_COUNTRIES_RU)
        c_qid = _qid_from_label_in_df(museum_countries_df, "countryLabelRu", "country_qid", c_ru) or ensure_qid(c_ru)
    else:
        c_qid, c_ru = pick_from_df(museum_countries_df, "country_qid", "countryLabelRu", rng)

    # type
    if rng.random() < 0.8:
        t_ru = rng.choice(COMMON_MUSEUM_TYPES_RU)
        t_qid = _qid_from_label_in_df(museum_types_df, "typeLabelRu", "type_qid", t_ru) or ensure_qid(t_ru)
    else:
        t_qid, t_ru = pick_from_df(museum_types_df, "type_qid", "typeLabelRu", rng)

    # city
    city_qid, city_ru = pick_from_df(museum_cities_df, "city_qid", "cityLabelRu", rng)
    return c_qid, c_ru, t_qid, t_ru, city_qid, city_ru

def run_museums_query(
    country_qid: str,
    type_qid: str,
    city_qid: Optional[str],
    y1: Optional[int],
    y2: Optional[int],
    limit: int = 300,
):
    where = [
        f"?item wdt:P17 wd:{country_qid} .",
        f"?item wdt:P31 wd:{type_qid} .",
    ]
    if city_qid:
        where.append(f"?item wdt:P131 wd:{city_qid} .")
    if y1 is not None and y2 is not None:
        where += [
            "?item wdt:P571 ?date .",
            "BIND(YEAR(?date) AS ?year) .",
            f"FILTER(?year >= {int(y1)} && ?year <= {int(y2)}) .",
        ]
    sparql, items = select_items_with_ru_label(Q_MUSEUM, where, limit=limit, item_var="item")
    return sparql, items, where

def nlg_museums_ru(country_ru: str, type_ru: str, city_ru: Optional[str], y1: Optional[int], y2: Optional[int], k: int) -> str:
    cond = [f"которые находятся в стране: {country_ru}", f"и относятся к типу: «{type_ru}»"]
    if city_ru:
        cond.append(f"и расположены в: {city_ru}")
    if y1 is not None and y2 is not None:
        cond.append(f"основаны в период {int(y1)}–{int(y2)}")
    return f"Назови {k} известных музеев, " + ", ".join(cond) + "."

def generate_museums_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 60) -> BenchmarkExample:
    if complexity == "L1":
        k=5; use_city=False; y1=y2=None; require=lambda n: n>=k
    elif complexity == "L2":
        k=3; use_city=True; y1=y2=None; require=lambda n: n>=k
    elif complexity == "L3":
        k=2; use_city=True; y1,y2=rng.choice([(1700,1799),(1800,1899),(1900,1999),(2000,2024)]); require=lambda n: n>=k
    elif complexity == "L4":
        k=5; use_city=True; y1,y2=rng.choice([(1900,1905),(1905,1910),(1910,1915),(1915,1920)]); require=lambda n: (0<n<k)
    elif complexity == "L5":
        k=5; use_city=True; y1,y2=2500,2510; require=lambda n: n==0
    else:
        raise ValueError(complexity)

    for _ in range(max_attempts):
        c_qid, c_ru, t_qid, t_ru, city_qid, city_ru = pick_museums_constraints(rng)
        if not use_city:
            city_qid = city_ru = None

        sparql, items, where = run_museums_query(c_qid, t_qid, city_qid, y1, y2, limit=300)
        n=len(items)
        if not require(n):
            continue

        gold=items[:300]
        ask=build_ask_validator(Q_MUSEUM, where, item_var="item")
        return BenchmarkExample(
            id=f"museums_{complexity.lower()}_{idx:04d}",
            domain="museums",
            complexity=complexity,
            query_text_ru=nlg_museums_ru(c_ru, t_ru, city_ru, y1, y2, k),
            constraints={
                "country_qid": c_qid, "country_ru": c_ru,
                "type_qid": t_qid, "type_ru": t_ru,
                "city_qid": city_qid, "city_ru": city_ru,
                "year_from": y1, "year_to": y2,
            },
            requested_count=k,
            gold_answer_qids=[q for q,_ in gold],
            gold_answer_labels_ru=[l for _,l in gold],
            sparql_query=sparql,
            created_at=utc_now_z(),
            gold_truncated=len(items) >= 300,
            ask_validator_sparql=ask,
        )

    sparql, items, where = run_museums_query(c_qid, t_qid, city_qid, y1, y2, limit=300)
    ask = build_ask_validator(Q_MUSEUM, where, item_var="item")
    return BenchmarkExample(
        id=f"museums_{complexity.lower()}_{idx:04d}",
        domain="museums",
        complexity=complexity,
        query_text_ru=nlg_museums_ru(c_ru, t_ru, city_ru, y1, y2, k),
        constraints={"country_qid": c_qid, "type_qid": t_qid, "city_qid": city_qid, "year_from": y1, "year_to": y2},
        requested_count=k,
        gold_answer_qids=[q for q,_ in items[:300]],
        gold_answer_labels_ru=[l for _,l in items[:300]],
        sparql_query=sparql,
        created_at=utc_now_z(),
        gold_truncated=len(items) >= 300,
        ask_validator_sparql=ask,
    )

# PATCH v3: Paintings L4/L5 non-redundant positive-gold multihop
#

PAINTINGS_ADVANCED_MAX_ATTEMPTS = 140
PAINTINGS_POSITIVE_GOLD_LIMIT = 300


def _paint_norm_label(x):
    if x is None:
        return None
    s = str(x).strip()
    if not s or re.fullmatch(r"Q\d+", s):
        return None
    return s


def _paint_prepare_positive_seed(df: pd.DataFrame) -> pd.DataFrame:
    """Лёгкая чистка уже построенного paintings_seed_df без новых тяжёлых WDQS-запросов."""
    if df is None or len(df) == 0:
        return pd.DataFrame()
    out = df.copy()
    for col in [c for c in out.columns if c.endswith("LabelRu") or c == "itemLabelRu"]:
        out[col] = out[col].map(_paint_norm_label)
    if "item_qid" in out.columns:
        out = out[out["item_qid"].notna()]
    return out.drop_duplicates().reset_index(drop=True)


paintings_seed_positive_df = _paint_prepare_positive_seed(globals().get("paintings_seed_df", pd.DataFrame()))


PAINTINGS_EXTRA_MAX_ITEMS = 900
PAINTINGS_EXTRA_BATCH_SIZE = 180


def _paint_fetch_item_property_values(
    item_qids: List[str],
    pid: str,
    val_name: str,
    max_items: int = PAINTINGS_EXTRA_MAX_ITEMS,
    batch_size: int = PAINTINGS_EXTRA_BATCH_SIZE,
) -> pd.DataFrame:
    item_qids = [q for q in dict.fromkeys(item_qids or []) if isinstance(q, str) and re.fullmatch(r"Q\d+", q)]
    item_qids = item_qids[:int(max_items)]
    if not item_qids:
        return pd.DataFrame(columns=["item_qid", f"{val_name}_qid", f"{val_name}LabelRu"])

    rows_all = []
    for start in range(0, len(item_qids), int(batch_size)):
        chunk = item_qids[start:start + int(batch_size)]
        values = " ".join(f"wd:{q}" for q in chunk)
        sparql = f"""
        SELECT DISTINCT ?item ?val ?valLabel WHERE {{
          VALUES ?item {{ {values} }}
          ?item wdt:{pid} ?val .
          SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
        }}
        """
        try:
            got = rows_from_select(wd.sparql_select(sparql))
        except Exception as e:
            print(f"[WARN] paintings enrichment {val_name}/{pid} failed on batch {start}: {e}")
            continue
        for r in got:
            item = uri_to_qid(r.get("item", ""))
            val = uri_to_qid(r.get("val", ""))
            lbl = _paint_norm_label(r.get("valLabel"))
            if item and val and lbl:
                rows_all.append({"item_qid": item, f"{val_name}_qid": val, f"{val_name}LabelRu": lbl})

    if not rows_all:
        return pd.DataFrame(columns=["item_qid", f"{val_name}_qid", f"{val_name}LabelRu"])
    return pd.DataFrame(rows_all).drop_duplicates().reset_index(drop=True)


def _paint_add_extra_property(seed_df: pd.DataFrame, prop_df: pd.DataFrame) -> pd.DataFrame:
    """
    Аккуратно расширяет seed. Если у картины несколько depicts/material, строки размножаются,
    что нормально: candidate tables потом группируются и сами считают gold-list.
    """
    if seed_df is None or len(seed_df) == 0:
        return pd.DataFrame()
    if prop_df is None or len(prop_df) == 0 or "item_qid" not in prop_df.columns:
        return seed_df.copy()
    return seed_df.merge(prop_df, on="item_qid", how="left").drop_duplicates().reset_index(drop=True)


def _paint_build_enriched_seed(seed_df: pd.DataFrame) -> pd.DataFrame:
    if seed_df is None or len(seed_df) == 0:
        return pd.DataFrame()
    item_qids = seed_df["item_qid"].dropna().astype(str).drop_duplicates().tolist()

    depicts_df = load_or_build_pool(
        "paintings_item_depicts_values_v3",
        lambda: _paint_fetch_item_property_values(item_qids, "P180", "depicts"),
    )
    movement_df = load_or_build_pool(
        "paintings_item_movement_values_v3",
        lambda: _paint_fetch_item_property_values(item_qids, "P135", "movement"),
    )
    material_df = load_or_build_pool(
        "paintings_item_material_values_v3",
        lambda: _paint_fetch_item_property_values(item_qids, "P186", "material"),
    )

    out = seed_df.copy()
    out = _paint_add_extra_property(out, depicts_df)
    out = _paint_add_extra_property(out, movement_df)
    out = _paint_add_extra_property(out, material_df)
    return out.drop_duplicates().reset_index(drop=True)


paintings_seed_enriched_df = _paint_build_enriched_seed(paintings_seed_positive_df)
if paintings_seed_enriched_df is None or len(paintings_seed_enriched_df) == 0:
    paintings_seed_enriched_df = paintings_seed_positive_df


def _paint_positive_group_candidates(
    df: pd.DataFrame,
    dims: List[str],
    min_n: int = 1,
    max_n: Optional[int] = None,
    max_rows: int = 900,
) -> pd.DataFrame:
    """
    Строит candidate-комбинации по реальным строкам seed.
    Важно: max_n здесь включительный, то есть max_n=4 допускает ровно 4 gold.
    """
    if df is None or len(df) == 0:
        return pd.DataFrame()

    sub = df.copy()
    group_cols: List[str] = []
    agg_spec: Dict[str, Any] = {"n": ("item_qid", "nunique")}

    for d in dims:
        if d == "year_bucket":
            if "year_bucket" not in sub.columns:
                return pd.DataFrame()
            sub = sub[sub["year_bucket"].notna()]
            group_cols.append("year_bucket")
            agg_spec["year_from"] = ("year_from", "first")
            agg_spec["year_to"] = ("year_to", "first")
            continue

        qcol = f"{d}_qid"
        lcol = f"{d}LabelRu"
        if qcol not in sub.columns or lcol not in sub.columns:
            return pd.DataFrame()
        sub = sub[sub[qcol].notna() & sub[lcol].notna()]
        group_cols.append(qcol)
        agg_spec[lcol] = (lcol, "first")

    if len(sub) == 0 or not group_cols:
        return pd.DataFrame()

    grp = sub.groupby(group_cols, dropna=False).agg(**agg_spec).reset_index()
    if max_n is None:
        grp = grp[grp["n"] >= int(min_n)]
    else:
        grp = grp[(grp["n"] >= int(min_n)) & (grp["n"] <= int(max_n))]

    if len(grp) == 0:
        return pd.DataFrame()

    grp = grp.sort_values(["n"], ascending=[True]).reset_index(drop=True)
    if max_rows and len(grp) > max_rows:
        grp = grp.head(int(max_rows)).copy()
    return grp.reset_index(drop=True)


PAINT_ADV_L4_TABLES = [
    ("genre_creator_collection", _paint_positive_group_candidates(paintings_seed_enriched_df, ["genre", "creator", "collection"], min_n=1, max_n=5, max_rows=700)),
    ("genre_creator_year", _paint_positive_group_candidates(paintings_seed_enriched_df, ["genre", "creator", "year_bucket"], min_n=1, max_n=5, max_rows=700)),
    ("creator_collection_year", _paint_positive_group_candidates(paintings_seed_enriched_df, ["creator", "collection", "year_bucket"], min_n=1, max_n=5, max_rows=700)),
    ("genre_depicts_year", _paint_positive_group_candidates(paintings_seed_enriched_df, ["genre", "depicts", "year_bucket"], min_n=1, max_n=5, max_rows=700)),
    ("creator_depicts_year", _paint_positive_group_candidates(paintings_seed_enriched_df, ["creator", "depicts", "year_bucket"], min_n=1, max_n=5, max_rows=700)),
    ("genre_movement_year", _paint_positive_group_candidates(paintings_seed_enriched_df, ["genre", "movement", "year_bucket"], min_n=1, max_n=5, max_rows=700)),
    ("creator_material_year", _paint_positive_group_candidates(paintings_seed_enriched_df, ["creator", "material", "year_bucket"], min_n=1, max_n=5, max_rows=700)),
    ("collection_depicts_year", _paint_positive_group_candidates(paintings_seed_enriched_df, ["collection", "depicts", "year_bucket"], min_n=1, max_n=5, max_rows=700)),
]

PAINT_ADV_L5_TABLES = [
    ("genre_creator_collection_year", _paint_positive_group_candidates(paintings_seed_enriched_df, ["genre", "creator", "collection", "year_bucket"], min_n=1, max_n=4, max_rows=900)),
    ("genre_creator_depicts_year", _paint_positive_group_candidates(paintings_seed_enriched_df, ["genre", "creator", "depicts", "year_bucket"], min_n=1, max_n=4, max_rows=900)),
    ("creator_collection_depicts_year", _paint_positive_group_candidates(paintings_seed_enriched_df, ["creator", "collection", "depicts", "year_bucket"], min_n=1, max_n=4, max_rows=900)),
    ("genre_collection_depicts_year", _paint_positive_group_candidates(paintings_seed_enriched_df, ["genre", "collection", "depicts", "year_bucket"], min_n=1, max_n=4, max_rows=900)),
    ("genre_movement_collection_year", _paint_positive_group_candidates(paintings_seed_enriched_df, ["genre", "movement", "collection", "year_bucket"], min_n=1, max_n=4, max_rows=900)),
    ("genre_creator_movement_year", _paint_positive_group_candidates(paintings_seed_enriched_df, ["genre", "creator", "movement", "year_bucket"], min_n=1, max_n=4, max_rows=900)),
    ("creator_material_collection_year", _paint_positive_group_candidates(paintings_seed_enriched_df, ["creator", "material", "collection", "year_bucket"], min_n=1, max_n=4, max_rows=900)),
    ("genre_material_creator_year", _paint_positive_group_candidates(paintings_seed_enriched_df, ["genre", "material", "creator", "year_bucket"], min_n=1, max_n=4, max_rows=900)),
]


def _paint_adv_pick_candidate(tables: List[Tuple[str, pd.DataFrame]], rng: random.Random) -> Tuple[Optional[str], Optional[dict]]:
    available = [(name, df) for name, df in tables if df is not None and len(df) > 0]
    if not available:
        return None, None
    name, df = rng.choice(available)
    top = df.head(min(len(df), 350))
    row = top.sample(1, random_state=rng.randint(0, 10**9)).iloc[0].to_dict()
    return name, row


def _paint_adv_candidate_to_params(mode: str, row: Optional[dict]) -> dict:
    params = {
        "genre_qid": None, "genre_ru": None,
        "movement_qid": None, "movement_ru": None,
        "creator_qid": None, "creator_ru": None,
        "creator_country_qid": None, "creator_country_ru": None,
        "collection_qid": None, "collection_ru": None,
        "collection_country_qid": None, "collection_country_ru": None,
        "depicts_qid": None, "depicts_ru": None,
        "material_qid": None, "material_ru": None,
        "year_from": None, "year_to": None,
        "mode": mode,
    }
    if row is None:
        return params

    if row.get("genre_qid"):
        params["genre_qid"] = row.get("genre_qid")
        params["genre_ru"] = row.get("genreLabelRu")
    if row.get("movement_qid"):
        params["movement_qid"] = row.get("movement_qid")
        params["movement_ru"] = row.get("movementLabelRu")
    if row.get("depicts_qid"):
        params["depicts_qid"] = row.get("depicts_qid")
        params["depicts_ru"] = row.get("depictsLabelRu")
    if row.get("material_qid"):
        params["material_qid"] = row.get("material_qid")
        params["material_ru"] = row.get("materialLabelRu")
    if row.get("creator_qid"):
        params["creator_qid"] = row.get("creator_qid")
        params["creator_ru"] = row.get("creatorLabelRu")
    if row.get("collection_qid"):
        params["collection_qid"] = row.get("collection_qid")
        params["collection_ru"] = row.get("collectionLabelRu")
    if row.get("country_qid"):
        params["collection_country_qid"] = row.get("country_qid")
        params["collection_country_ru"] = row.get("countryLabelRu")
    if row.get("year_from") is not None and row.get("year_to") is not None:
        params["year_from"] = int(row.get("year_from"))
        params["year_to"] = int(row.get("year_to"))
    return params


def run_paintings_query_v2(
    genre_qid: Optional[str] = None,
    movement_qid: Optional[str] = None,
    creator_qid: Optional[str] = None,
    creator_country_qid: Optional[str] = None,
    collection_qid: Optional[str] = None,
    collection_country_qid: Optional[str] = None,
    depicts_qid: Optional[str] = None,
    material_qid: Optional[str] = None,
    y1: Optional[int] = None,
    y2: Optional[int] = None,
    limit: int = PAINTINGS_POSITIVE_GOLD_LIMIT,
):
    where: List[str] = []

    if genre_qid:
        where.append(f"?item wdt:P136 wd:{genre_qid} .")
    if movement_qid:
        where.append(f"?item wdt:P135 wd:{movement_qid} .")
    if depicts_qid:
        where.append(f"?item wdt:P180 wd:{depicts_qid} .")
    if material_qid:
        where.append(f"?item wdt:P186 wd:{material_qid} .")

    if creator_qid:
        where.append(f"?item wdt:P170 wd:{creator_qid} .")
    elif creator_country_qid:
        where += [
            "?item wdt:P170 ?creatorFilter .",
            f"?creatorFilter wdt:P27 wd:{creator_country_qid} .",
        ]

    if collection_qid:
        where.append(f"?item wdt:P195 wd:{collection_qid} .")
        if collection_country_qid:
            where.append(f"wd:{collection_qid} wdt:P17 wd:{collection_country_qid} .")
    elif collection_country_qid:
        where += [
            "?item wdt:P195 ?collectionFilter .",
            f"?collectionFilter wdt:P17 wd:{collection_country_qid} .",
        ]

    if y1 is not None and y2 is not None:
        where += [
            "?item wdt:P571 ?date .",
            "BIND(YEAR(?date) AS ?year) .",
            f"FILTER(?year >= {int(y1)} && ?year <= {int(y2)}) .",
        ]

    sparql, items = select_items_with_ru_label(Q_PAINTING, where, limit=limit, item_var="item")
    return sparql, items, where


def nlg_paintings_ru_v2(
    genre_ru: Optional[str] = None,
    movement_ru: Optional[str] = None,
    creator_ru: Optional[str] = None,
    creator_country_ru: Optional[str] = None,
    collection_ru: Optional[str] = None,
    collection_country_ru: Optional[str] = None,
    depicts_ru: Optional[str] = None,
    material_ru: Optional[str] = None,
    y1: Optional[int] = None,
    y2: Optional[int] = None,
    k: int = 5,
) -> str:
    cond = []
    if genre_ru:
        cond.append(f"относятся к жанру «{genre_ru}»")
    if movement_ru:
        cond.append(f"связаны с художественным направлением «{movement_ru}»")
    if creator_ru:
        cond.append(f"созданы художником {creator_ru}")
    elif creator_country_ru:
        cond.append(f"созданы художником, связанным со страной «{creator_country_ru}»")
    if collection_ru:
        cond.append(f"хранятся в коллекции «{collection_ru}»")
    elif collection_country_ru:
        cond.append(f"хранятся в коллекции или музее в стране «{collection_country_ru}»")
    if depicts_ru:
        cond.append(f"изображают «{depicts_ru}»")
    if material_ru:
        cond.append(f"выполнены с использованием материала или техники «{material_ru}»")
    if y1 is not None and y2 is not None:
        cond.append(f"созданы в период {int(y1)}–{int(y2)}")

    if not cond:
        return f"Назови {k} известных картин."
    return "Назови " + str(k) + " картин, которые " + ", ".join(cond) + "."


def _paint_adv_fallback_params(rng: random.Random, complexity: str) -> dict:
    """Fallback на старые таблицы, если по какой-то причине новые candidate tables пустые."""
    tables = PAINT_L4_TABLES + PAINT_L3_TABLES + PAINT_L2_TABLES
    mode, row = _paint_pick_candidate(tables, rng)
    base = _paint_candidate_to_params(mode or "fallback_old_paintings", row)
    return {
        "genre_qid": base.get("genre_qid"), "genre_ru": base.get("genre_ru"),
        "movement_qid": base.get("movement_qid"), "movement_ru": base.get("movement_ru"),
        "creator_qid": base.get("creator_qid"), "creator_ru": base.get("creator_ru"),
        "creator_country_qid": None, "creator_country_ru": None,
        "collection_qid": base.get("collection_qid"), "collection_ru": base.get("collection_ru"),
        "collection_country_qid": base.get("country_qid"), "collection_country_ru": base.get("country_ru"),
        "depicts_qid": None, "depicts_ru": None,
        "material_qid": None, "material_ru": None,
        "year_from": base.get("year_from"), "year_to": base.get("year_to"),
        "mode": f"fallback_{base.get('mode', 'old')}",
    }


def generate_paintings_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = PAINTINGS_ADVANCED_MAX_ATTEMPTS) -> BenchmarkExample:
    """
    Переопределённый генератор paintings.
    L1-L3 оставлены на старой логике, L4-L5 — positive-gold only.
    """
    if complexity in {"L1", "L2", "L3"}:
        if complexity == "L1":
            k, tables, require = 5, PAINT_L1_TABLES, (lambda n: n >= 5)
        elif complexity == "L2":
            k, tables, require = 3, PAINT_L2_TABLES, (lambda n: n >= 3)
        else:
            k, tables, require = 2, PAINT_L3_TABLES, (lambda n: n >= 2)

        last_payload = None
        for _ in range(max_attempts):
            mode, row = _paint_pick_candidate(tables, rng)
            params = _paint_candidate_to_params(mode or "unknown", row) if row is not None else _paint_fallback_from_seed(rng)
            sparql, items, where = run_paintings_query(
                genre_qid=params["genre_qid"],
                movement_qid=params["movement_qid"],
                country_qid=params["country_qid"],
                creator_qid=params["creator_qid"],
                collection_qid=params["collection_qid"],
                y1=params["year_from"],
                y2=params["year_to"],
                limit=PAINTINGS_POSITIVE_GOLD_LIMIT,
            )
            last_payload = {**params, "sparql": sparql, "items": items, "where": where}
            if require(len(items)):
                ask = build_ask_validator(Q_PAINTING, where, item_var="item")
                return BenchmarkExample(
                    id=f"paintings_{complexity.lower()}_{idx:04d}",
                    domain="paintings",
                    complexity=complexity,
                    query_text_ru=nlg_paintings_ru(params["genre_ru"], params["movement_ru"], params["country_ru"], params["creator_ru"], params["collection_ru"], params["year_from"], params["year_to"], k),
                    constraints={
                        "genre_qid": params["genre_qid"], "genre_ru": params["genre_ru"],
                        "movement_qid": params["movement_qid"], "movement_ru": params["movement_ru"],
                        "country_qid": params["country_qid"], "country_ru": params["country_ru"],
                        "creator_qid": params["creator_qid"], "creator_ru": params["creator_ru"],
                        "collection_qid": params["collection_qid"], "collection_ru": params["collection_ru"],
                        "year_from": params["year_from"], "year_to": params["year_to"],
                        "mode": params["mode"],
                    },
                    requested_count=k,
                    gold_answer_qids=[q for q, _ in items[:PAINTINGS_POSITIVE_GOLD_LIMIT]],
                    gold_answer_labels_ru=[l for _, l in items[:PAINTINGS_POSITIVE_GOLD_LIMIT]],
                    sparql_query=sparql,
                    created_at=utc_now_z(),
                    gold_truncated=len(items) >= PAINTINGS_POSITIVE_GOLD_LIMIT,
                    ask_validator_sparql=ask,
                )

        last_n = 0 if last_payload is None else len(last_payload.get("items", []))
        raise RuntimeError(f"paintings {complexity}: failed to generate positive-gold example; last_gold_count={last_n}")

    if complexity == "L4":
        k = 4
        tables = PAINT_ADV_L4_TABLES
        require = lambda n: 1 <= n <= 5
    elif complexity == "L5":
        k = 3
        tables = PAINT_ADV_L5_TABLES
        require = lambda n: 1 <= n <= 4
    else:
        raise ValueError(complexity)

    last_payload = None
    for _ in range(max_attempts):
        mode, row = _paint_adv_pick_candidate(tables, rng)
        params = _paint_adv_candidate_to_params(mode or "advanced_empty", row)
        if row is None:
            params = _paint_adv_fallback_params(rng, complexity)

        sparql, items, where = run_paintings_query_v2(
            genre_qid=params["genre_qid"],
            movement_qid=params["movement_qid"],
            creator_qid=params["creator_qid"],
            creator_country_qid=params["creator_country_qid"],
            collection_qid=params["collection_qid"],
            collection_country_qid=params["collection_country_qid"],
            depicts_qid=params["depicts_qid"],
            material_qid=params["material_qid"],
            y1=params["year_from"],
            y2=params["year_to"],
            limit=PAINTINGS_POSITIVE_GOLD_LIMIT,
        )
        n = len(items)
        last_payload = {**params, "sparql": sparql, "items": items, "where": where}
        if not require(n):
            continue

        ask = build_ask_validator(Q_PAINTING, where, item_var="item")
        return BenchmarkExample(
            id=f"paintings_{complexity.lower()}_{idx:04d}",
            domain="paintings",
            complexity=complexity,
            query_text_ru=nlg_paintings_ru_v2(
                genre_ru=params["genre_ru"],
                movement_ru=params["movement_ru"],
                creator_ru=params["creator_ru"],
                creator_country_ru=params["creator_country_ru"],
                collection_ru=params["collection_ru"],
                collection_country_ru=params["collection_country_ru"],
                depicts_ru=params["depicts_ru"],
                material_ru=params["material_ru"],
                y1=params["year_from"], y2=params["year_to"], k=k,
            ),
            constraints={
                "genre_qid": params["genre_qid"], "genre_ru": params["genre_ru"],
                "movement_qid": params["movement_qid"], "movement_ru": params["movement_ru"],
                "creator_qid": params["creator_qid"], "creator_ru": params["creator_ru"],
                "creator_country_qid": params["creator_country_qid"], "creator_country_ru": params["creator_country_ru"],
                "collection_qid": params["collection_qid"], "collection_ru": params["collection_ru"],
                "collection_country_qid": params["collection_country_qid"], "collection_country_ru": params["collection_country_ru"],
                "depicts_qid": params["depicts_qid"], "depicts_ru": params["depicts_ru"],
                "material_qid": params["material_qid"], "material_ru": params["material_ru"],
                "year_from": params["year_from"], "year_to": params["year_to"],
                "mode": params["mode"],
            },
            requested_count=k,
            gold_answer_qids=[q for q, _ in items[:PAINTINGS_POSITIVE_GOLD_LIMIT]],
            gold_answer_labels_ru=[l for _, l in items[:PAINTINGS_POSITIVE_GOLD_LIMIT]],
            sparql_query=sparql,
            created_at=utc_now_z(),
            gold_truncated=len(items) >= PAINTINGS_POSITIVE_GOLD_LIMIT,
            ask_validator_sparql=ask,
            is_advanced=True,
            template_id=params["mode"],
            template_family="paintings_positive_multihop_stable",
        )

    last_n = 0 if last_payload is None else len(last_payload.get("items", []))
    last_mode = None if last_payload is None else last_payload.get("mode")
    raise RuntimeError(
        f"paintings {complexity}: failed to generate a positive-gold multihop example "
        f"after {max_attempts} attempts; last_mode={last_mode}, last_gold_count={last_n}"
    )


def paintings_advanced_debug_tables() -> pd.DataFrame:
    rows = []
    for level, tables in [("L4", PAINT_ADV_L4_TABLES), ("L5", PAINT_ADV_L5_TABLES)]:
        for name, df in tables:
            rows.append({
                "complexity": level,
                "mode": name,
                "candidate_rows": 0 if df is None else len(df),
                "min_gold_est": None if df is None or len(df) == 0 else int(df["n"].min()),
                "max_gold_est": None if df is None or len(df) == 0 else int(df["n"].max()),
            })
    return pd.DataFrame(rows)


print("✅ Patched paintings v3: non-redundant positive-gold L4/L5 enabled; depicts/movement/material enrichment added")
try:
    display(paintings_advanced_debug_tables())
except Exception:
    print(paintings_advanced_debug_tables())

# Museum refinement: seed-based templates with local post-filtering.

import random
import re
import pandas as pd
from typing import Optional, Tuple, Dict, Any, List

Q_MUSEUM = ensure_qid("музей", fallback_qid="Q33506")

MUSEUM_QUERY_LIMIT = 500

MUSEUM_REQUESTED_COUNT = {
    "L1": 5,
    "L2": 5,
    "L3": 4,
    "L4": 3,
    "L5": 2,
}

MUSEUM_MAX_GOLD_BY_COMPLEXITY = {
    "L1": 350,
    "L2": 260,
    "L3": 180,
    "L4": 140,
    "L5": 100,
}


def _museum_good_qid(x):
    return isinstance(x, str) and x.startswith("Q")


def _museum_good_label(x):
    if x is None:
        return False
    try:
        if pd.isna(x):
            return False
    except Exception:
        pass
    return isinstance(x, str) and len(x.strip()) > 0


def _museum_qid(x):
    if not x:
        return None
    try:
        return uri_to_qid(str(x))
    except Exception:
        return None


def _museum_parse_year(x):
    if x is None:
        return None
    s = str(x)
    m = re.search(r"(-?\d{3,4})", s)
    if not m:
        return None
    y = int(m.group(1))
    if 1200 <= y <= 2026:
        return y
    return None


def _museum_year_window(center_year, half_width=60):
    if center_year is None:
        return None, None
    y1 = max(1200, int(center_year) - int(half_width))
    y2 = min(2026, int(center_year) + int(half_width))
    return y1, y2


def _museum_year_filter(y1: int, y2: int) -> List[str]:
    return [
        "?item wdt:P571 ?inception_date .",
        "BIND(YEAR(?inception_date) AS ?inception_year) .",
        f"FILTER(?inception_year >= {int(y1)} && ?inception_year <= {int(y2)}) .",
    ]


def _museum_count_phrase(k: int) -> str:
    if k == 1:
        return "Назови 1 музей"
    if 2 <= k <= 4:
        return f"Назови {k} музея"
    return f"Назови {k} музеев"


def _build_museums_seed_pool_v2(limit: int = 1800) -> pd.DataFrame:
    """
    Seed-pool реальных музеев с русскими label'ами и основными свойствами.

    Используем свойства:
      P17  — страна
      P31  — тип/класс музея
      P131 — административная единица / город
      P571 — дата основания
      P84  — архитектор
      P127 — владелец
      P137 — оператор
      P138 — назван в честь
    """
    sparql = f"""
    SELECT DISTINCT
      ?item ?itemLabelRu
      ?country ?countryLabelRu
      ?type ?typeLabelRu
      ?city ?cityLabelRu
      ?inception_date
      ?architect ?architectLabelRu
      ?owner ?ownerLabelRu
      ?operator ?operatorLabelRu
      ?named_after ?namedAfterLabelRu
    WHERE {{
      ?item wdt:P31/wdt:P279* wd:{Q_MUSEUM} .
      ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") .

      OPTIONAL {{
        ?item wdt:P17 ?country .
        ?country rdfs:label ?countryLabelRu FILTER(LANG(?countryLabelRu) = "ru") .
      }}

      OPTIONAL {{
        ?item wdt:P31 ?type .
        ?type wdt:P279* wd:{Q_MUSEUM} .
        FILTER(?type != wd:{Q_MUSEUM})
        ?type rdfs:label ?typeLabelRu FILTER(LANG(?typeLabelRu) = "ru") .
      }}

      OPTIONAL {{
        ?item wdt:P131 ?city .
        ?city rdfs:label ?cityLabelRu FILTER(LANG(?cityLabelRu) = "ru") .
      }}

      OPTIONAL {{
        ?item wdt:P571 ?inception_date .
      }}

      OPTIONAL {{
        ?item wdt:P84 ?architect .
        ?architect rdfs:label ?architectLabelRu FILTER(LANG(?architectLabelRu) = "ru") .
      }}

      OPTIONAL {{
        ?item wdt:P127 ?owner .
        ?owner rdfs:label ?ownerLabelRu FILTER(LANG(?ownerLabelRu) = "ru") .
      }}

      OPTIONAL {{
        ?item wdt:P137 ?operator .
        ?operator rdfs:label ?operatorLabelRu FILTER(LANG(?operatorLabelRu) = "ru") .
      }}

      OPTIONAL {{
        ?item wdt:P138 ?named_after .
        ?named_after rdfs:label ?namedAfterLabelRu FILTER(LANG(?namedAfterLabelRu) = "ru") .
      }}

      FILTER(
        BOUND(?country) ||
        BOUND(?type) ||
        BOUND(?city)
      )
    }}
    LIMIT {int(limit)}
    """

    rows = rows_from_select(wd.sparql_select(sparql))
    data = []

    for r in rows:
        item_qid = _museum_qid(r.get("item"))
        item_ru = r.get("itemLabelRu")

        if not (_museum_good_qid(item_qid) and _museum_good_label(item_ru)):
            continue

        data.append({
            "museum_qid": item_qid,
            "museum_ru": item_ru,

            "country_qid": _museum_qid(r.get("country")),
            "country_ru": r.get("countryLabelRu"),

            "type_qid": _museum_qid(r.get("type")),
            "type_ru": r.get("typeLabelRu"),

            "city_qid": _museum_qid(r.get("city")),
            "city_ru": r.get("cityLabelRu"),

            "inception_year": _museum_parse_year(r.get("inception_date")),

            "architect_qid": _museum_qid(r.get("architect")),
            "architect_ru": r.get("architectLabelRu"),

            "owner_qid": _museum_qid(r.get("owner")),
            "owner_ru": r.get("ownerLabelRu"),

            "operator_qid": _museum_qid(r.get("operator")),
            "operator_ru": r.get("operatorLabelRu"),

            "named_after_qid": _museum_qid(r.get("named_after")),
            "named_after_ru": r.get("namedAfterLabelRu"),
        })

    df = pd.DataFrame(data).drop_duplicates().reset_index(drop=True)

    interesting_tokens = [
        "Лувр", "Эрмитаж", "Метрополитен", "Британский музей", "Прадо",
        "Гуггенхайм", "Тейт", "Орсе", "Уффици", "Ватикан",
        "национальный", "художественный", "современного искусства",
        "исторический", "археологический", "естественной истории",
    ]

    def weight_row(row):
        text = " ".join(str(row.get(c, "")) for c in [
            "museum_ru", "country_ru", "type_ru", "city_ru",
            "architect_ru", "owner_ru", "operator_ru", "named_after_ru",
        ])
        w = 1.0
        if any(t in text for t in interesting_tokens):
            w += 3.0
        filled = sum(
            _museum_good_qid(row.get(c))
            for c in [
                "country_qid", "type_qid", "city_qid",
                "architect_qid", "owner_qid", "operator_qid", "named_after_qid",
            ]
        )
        w += 0.35 * filled
        if row.get("inception_year") is not None:
            w += 0.5
        return w

    if len(df):
        df["sample_weight"] = df.apply(weight_row, axis=1)

    print("[Museums v2] seed pool size:", len(df))
    return df


museums_seed_df_v2 = load_or_build_pool(
    "museums_seed_pool_v2_robust",
    lambda: _build_museums_seed_pool_v2(limit=1800),
)


def _museum_row_has(row, fields):
    for qid_col, label_col in fields:
        if not _museum_good_qid(row.get(qid_col)):
            return False
        if not _museum_good_label(row.get(label_col)):
            return False
    return True


def _museum_sample_target(fields, rng, require_year=False, max_tries=700):
    if museums_seed_df_v2.empty:
        raise RuntimeError("Museums seed pool is empty.")

    df = museums_seed_df_v2

    for _ in range(max_tries):
        if "sample_weight" in df.columns:
            row = df.sample(
                1,
                weights=df["sample_weight"],
                random_state=rng.randint(0, 10**9),
            ).iloc[0].to_dict()
        else:
            row = df.iloc[rng.randrange(len(df))].to_dict()

        if not _museum_row_has(row, fields):
            continue

        if require_year and row.get("inception_year") is None:
            continue

        return row

    return None


def _museum_pick_ref(prop_col, value_qid, exclude_qids, rng):
    """
    Локально выбираем другой музей с тем же свойством.
    Нужно для bridge-паттернов L4/L5.
    """
    if not _museum_good_qid(value_qid):
        return None

    exclude_qids = {q for q in exclude_qids if _museum_good_qid(q)}

    sub = museums_seed_df_v2[
        (museums_seed_df_v2[prop_col] == value_qid)
        & (~museums_seed_df_v2["museum_qid"].isin(exclude_qids))
        & (museums_seed_df_v2["museum_qid"].apply(_museum_good_qid))
        & (museums_seed_df_v2["museum_ru"].apply(_museum_good_label))
    ]

    if len(sub) == 0:
        return None

    row = sub.sample(1, random_state=rng.randint(0, 10**9)).iloc[0].to_dict()
    return row["museum_qid"], row["museum_ru"]


def _run_museums_query_v2(where_lines: List[str], limit: int = MUSEUM_QUERY_LIMIT):
    where = "\n      ".join(where_lines)

    sparql = f"""
    SELECT DISTINCT ?item ?itemLabelRu WHERE {{
      ?item wdt:P31/wdt:P279* wd:{Q_MUSEUM} .
      {where}
      ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") .
      ?item wikibase:sitelinks ?sitelinks .
      FILTER(?sitelinks >= 1)
    }}
    LIMIT {int(limit)}
    """

    rows = rows_from_select(wd.sparql_select(sparql))
    items = []
    seen = set()

    for r in rows:
        qid = uri_to_qid(r.get("item", ""))
        lbl = r.get("itemLabelRu")
        if qid and lbl and qid not in seen:
            items.append((qid, lbl))
            seen.add(qid)

    return sparql.strip(), items


def _museum_accept_count(complexity: str, n: int, k: int) -> bool:
    return n >= k and n <= MUSEUM_MAX_GOLD_BY_COMPLEXITY[complexity]


def _museum_make_candidate(complexity: str, rng: random.Random):
    k = MUSEUM_REQUESTED_COUNT[complexity]

    if complexity == "L1":
        patterns = []

        def p_country():
            row = _museum_sample_target(
                [("country_qid", "country_ru")],
                rng,
                require_year=False,
            )
            if not row:
                return None

            country_qid, country_ru = row["country_qid"], row["country_ru"]

            return {
                "template_id": "museums_L1_country",
                "query_text_ru": f"{_museum_count_phrase(k)}, находящихся в стране «{country_ru}».",
                "where_lines": [
                    f"?item wdt:P17 wd:{country_qid} .",
                ],
                "constraints": {
                    "country_qid": country_qid,
                    "country_ru": country_ru,
                    "seed_qid": row["museum_qid"],
                    "seed_ru": row["museum_ru"],
                },
            }

        def p_type():
            row = _museum_sample_target(
                [("type_qid", "type_ru")],
                rng,
                require_year=False,
            )
            if not row:
                return None

            type_qid, type_ru = row["type_qid"], row["type_ru"]

            return {
                "template_id": "museums_L1_type",
                "query_text_ru": f"{_museum_count_phrase(k)}, относящихся к типу «{type_ru}».",
                "where_lines": [
                    f"?item wdt:P31/wdt:P279* wd:{type_qid} .",
                ],
                "constraints": {
                    "type_qid": type_qid,
                    "type_ru": type_ru,
                    "seed_qid": row["museum_qid"],
                    "seed_ru": row["museum_ru"],
                },
            }

        patterns = [p_country, p_type]

    elif complexity == "L2":
        patterns = []

        def p_country_type():
            row = _museum_sample_target(
                [("country_qid", "country_ru"), ("type_qid", "type_ru")],
                rng,
                require_year=False,
            )
            if not row:
                return None

            country_qid, country_ru = row["country_qid"], row["country_ru"]
            type_qid, type_ru = row["type_qid"], row["type_ru"]

            return {
                "template_id": "museums_L2_country+type",
                "query_text_ru": (
                    f"{_museum_count_phrase(k)}, которые находятся в стране «{country_ru}» "
                    f"и относятся к типу «{type_ru}»."
                ),
                "where_lines": [
                    f"?item wdt:P17 wd:{country_qid} .",
                    f"?item wdt:P31/wdt:P279* wd:{type_qid} .",
                ],
                "constraints": {
                    "country_qid": country_qid,
                    "country_ru": country_ru,
                    "type_qid": type_qid,
                    "type_ru": type_ru,
                    "seed_qid": row["museum_qid"],
                    "seed_ru": row["museum_ru"],
                },
            }

        def p_city_type():
            row = _museum_sample_target(
                [("city_qid", "city_ru"), ("type_qid", "type_ru")],
                rng,
                require_year=False,
            )
            if not row:
                return None

            city_qid, city_ru = row["city_qid"], row["city_ru"]
            type_qid, type_ru = row["type_qid"], row["type_ru"]

            return {
                "template_id": "museums_L2_city+type",
                "query_text_ru": (
                    f"{_museum_count_phrase(k)}, которые расположены в «{city_ru}» "
                    f"и относятся к типу «{type_ru}»."
                ),
                "where_lines": [
                    f"?item wdt:P131 wd:{city_qid} .",
                    f"?item wdt:P31/wdt:P279* wd:{type_qid} .",
                ],
                "constraints": {
                    "city_qid": city_qid,
                    "city_ru": city_ru,
                    "type_qid": type_qid,
                    "type_ru": type_ru,
                    "seed_qid": row["museum_qid"],
                    "seed_ru": row["museum_ru"],
                },
            }

        def p_country_year():
            row = _museum_sample_target(
                [("country_qid", "country_ru")],
                rng,
                require_year=True,
            )
            if not row:
                return None

            country_qid, country_ru = row["country_qid"], row["country_ru"]
            y1, y2 = _museum_year_window(row["inception_year"], half_width=80)

            return {
                "template_id": "museums_L2_country+inception_period",
                "query_text_ru": (
                    f"{_museum_count_phrase(k)}, которые находятся в стране «{country_ru}» "
                    f"и были основаны в период {y1}–{y2} годов."
                ),
                "where_lines": [
                    f"?item wdt:P17 wd:{country_qid} .",
                    *_museum_year_filter(y1, y2),
                ],
                "constraints": {
                    "country_qid": country_qid,
                    "country_ru": country_ru,
                    "year_from": y1,
                    "year_to": y2,
                    "seed_qid": row["museum_qid"],
                    "seed_ru": row["museum_ru"],
                },
            }

        patterns = [p_country_type, p_city_type, p_country_year]

    elif complexity == "L3":
        patterns = []

        def p_country_type_year():
            row = _museum_sample_target(
                [("country_qid", "country_ru"), ("type_qid", "type_ru")],
                rng,
                require_year=True,
            )
            if not row:
                return None

            country_qid, country_ru = row["country_qid"], row["country_ru"]
            type_qid, type_ru = row["type_qid"], row["type_ru"]
            y1, y2 = _museum_year_window(row["inception_year"], half_width=70)

            return {
                "template_id": "museums_L3_country+type+inception_period",
                "query_text_ru": (
                    f"{_museum_count_phrase(k)}, которые находятся в стране «{country_ru}», "
                    f"относятся к типу «{type_ru}» и были основаны в период {y1}–{y2} годов."
                ),
                "where_lines": [
                    f"?item wdt:P17 wd:{country_qid} .",
                    f"?item wdt:P31/wdt:P279* wd:{type_qid} .",
                    *_museum_year_filter(y1, y2),
                ],
                "constraints": {
                    "country_qid": country_qid,
                    "country_ru": country_ru,
                    "type_qid": type_qid,
                    "type_ru": type_ru,
                    "year_from": y1,
                    "year_to": y2,
                    "seed_qid": row["museum_qid"],
                    "seed_ru": row["museum_ru"],
                },
            }

        def p_city_type_year():
            row = _museum_sample_target(
                [("city_qid", "city_ru"), ("type_qid", "type_ru")],
                rng,
                require_year=True,
            )
            if not row:
                return None

            city_qid, city_ru = row["city_qid"], row["city_ru"]
            type_qid, type_ru = row["type_qid"], row["type_ru"]
            y1, y2 = _museum_year_window(row["inception_year"], half_width=90)

            return {
                "template_id": "museums_L3_city+type+inception_period",
                "query_text_ru": (
                    f"{_museum_count_phrase(k)}, расположенных в «{city_ru}», "
                    f"относящихся к типу «{type_ru}» и основанных в период {y1}–{y2} годов."
                ),
                "where_lines": [
                    f"?item wdt:P131 wd:{city_qid} .",
                    f"?item wdt:P31/wdt:P279* wd:{type_qid} .",
                    *_museum_year_filter(y1, y2),
                ],
                "constraints": {
                    "city_qid": city_qid,
                    "city_ru": city_ru,
                    "type_qid": type_qid,
                    "type_ru": type_ru,
                    "year_from": y1,
                    "year_to": y2,
                    "seed_qid": row["museum_qid"],
                    "seed_ru": row["museum_ru"],
                },
            }

        def p_country_type_owner_or_operator():
            row = _museum_sample_target(
                [("country_qid", "country_ru"), ("type_qid", "type_ru")],
                rng,
                require_year=False,
            )
            if not row:
                return None

            extra_kind = None
            if _museum_good_qid(row.get("owner_qid")) and _museum_good_label(row.get("owner_ru")):
                extra_kind = "owner"
            elif _museum_good_qid(row.get("operator_qid")) and _museum_good_label(row.get("operator_ru")):
                extra_kind = "operator"
            else:
                return None

            country_qid, country_ru = row["country_qid"], row["country_ru"]
            type_qid, type_ru = row["type_qid"], row["type_ru"]

            if extra_kind == "owner":
                extra_qid, extra_ru = row["owner_qid"], row["owner_ru"]
                extra_text = f"владельцем которых является «{extra_ru}»"
                extra_line = f"?item wdt:P127 wd:{extra_qid} ."
                extra_key = "owner"
            else:
                extra_qid, extra_ru = row["operator_qid"], row["operator_ru"]
                extra_text = f"оператором которых является «{extra_ru}»"
                extra_line = f"?item wdt:P137 wd:{extra_qid} ."
                extra_key = "operator"

            return {
                "template_id": f"museums_L3_country+type+{extra_key}",
                "query_text_ru": (
                    f"{_museum_count_phrase(k)}, которые находятся в стране «{country_ru}», "
                    f"относятся к типу «{type_ru}» и {extra_text}."
                ),
                "where_lines": [
                    f"?item wdt:P17 wd:{country_qid} .",
                    f"?item wdt:P31/wdt:P279* wd:{type_qid} .",
                    extra_line,
                ],
                "constraints": {
                    "country_qid": country_qid,
                    "country_ru": country_ru,
                    "type_qid": type_qid,
                    "type_ru": type_ru,
                    f"{extra_key}_qid": extra_qid,
                    f"{extra_key}_ru": extra_ru,
                    "seed_qid": row["museum_qid"],
                    "seed_ru": row["museum_ru"],
                },
            }

        patterns = [p_country_type_year, p_city_type_year, p_country_type_owner_or_operator]

    elif complexity == "L4":
        patterns = []

        def p_same_country_as_seed_type_year():
            row = _museum_sample_target(
                [("country_qid", "country_ru"), ("type_qid", "type_ru")],
                rng,
                require_year=True,
            )
            if not row:
                return None

            seed_a = _museum_pick_ref(
                "country_qid",
                row["country_qid"],
                exclude_qids=[row["museum_qid"]],
                rng=rng,
            )
            if not seed_a:
                return None

            seed_a_qid, seed_a_ru = seed_a
            type_qid, type_ru = row["type_qid"], row["type_ru"]
            y1, y2 = _museum_year_window(row["inception_year"], half_width=70)

            return {
                "template_id": "museums_L4_same_country_as_seed+type+period",
                "query_text_ru": (
                    f"{_museum_count_phrase(k)}, находящихся в той же стране, что и «{seed_a_ru}», "
                    f"относящихся к типу «{type_ru}» и основанных в период {y1}–{y2} годов. "
                    f"Не включай музей «{seed_a_ru}» в ответ."
                ),
                "where_lines": [
                    f"BIND(wd:{seed_a_qid} AS ?seedA) .",
                    "?seedA wdt:P17 ?country .",
                    "?item wdt:P17 ?country .",
                    f"?item wdt:P31/wdt:P279* wd:{type_qid} .",
                    f"FILTER(?item != wd:{seed_a_qid}) .",
                    *_museum_year_filter(y1, y2),
                ],
                "constraints": {
                    "seed_country_qid": seed_a_qid,
                    "seed_country_ru": seed_a_ru,
                    "type_qid": type_qid,
                    "type_ru": type_ru,
                    "bridge": "same_country_as_seed",
                    "year_from": y1,
                    "year_to": y2,
                    "target_probe_qid": row["museum_qid"],
                    "target_probe_ru": row["museum_ru"],
                },
            }

        def p_same_city_as_seed_type():
            row = _museum_sample_target(
                [("city_qid", "city_ru"), ("type_qid", "type_ru")],
                rng,
                require_year=False,
            )
            if not row:
                return None

            seed_a = _museum_pick_ref(
                "city_qid",
                row["city_qid"],
                exclude_qids=[row["museum_qid"]],
                rng=rng,
            )
            if not seed_a:
                return None

            seed_a_qid, seed_a_ru = seed_a
            type_qid, type_ru = row["type_qid"], row["type_ru"]

            return {
                "template_id": "museums_L4_same_city_as_seed+type",
                "query_text_ru": (
                    f"{_museum_count_phrase(k)}, расположенных в той же административной единице или городе, "
                    f"что и «{seed_a_ru}», и относящихся к типу «{type_ru}». "
                    f"Не включай музей «{seed_a_ru}» в ответ."
                ),
                "where_lines": [
                    f"BIND(wd:{seed_a_qid} AS ?seedA) .",
                    "?seedA wdt:P131 ?city .",
                    "?item wdt:P131 ?city .",
                    f"?item wdt:P31/wdt:P279* wd:{type_qid} .",
                    f"FILTER(?item != wd:{seed_a_qid}) .",
                ],
                "constraints": {
                    "seed_city_qid": seed_a_qid,
                    "seed_city_ru": seed_a_ru,
                    "type_qid": type_qid,
                    "type_ru": type_ru,
                    "bridge": "same_city_as_seed",
                    "target_probe_qid": row["museum_qid"],
                    "target_probe_ru": row["museum_ru"],
                },
            }

        def p_same_owner_or_operator_as_seed_type_country():
            row = _museum_sample_target(
                [("country_qid", "country_ru"), ("type_qid", "type_ru")],
                rng,
                require_year=False,
            )
            if not row:
                return None

            if _museum_good_qid(row.get("owner_qid")) and _museum_good_label(row.get("owner_ru")):
                prop_col = "owner_qid"
                prop_pid = "P127"
                bridge_name = "same_owner_as_seed"
                text_name = "владелец"
            elif _museum_good_qid(row.get("operator_qid")) and _museum_good_label(row.get("operator_ru")):
                prop_col = "operator_qid"
                prop_pid = "P137"
                bridge_name = "same_operator_as_seed"
                text_name = "оператор"
            else:
                return None

            seed_a = _museum_pick_ref(
                prop_col,
                row[prop_col],
                exclude_qids=[row["museum_qid"]],
                rng=rng,
            )
            if not seed_a:
                return None

            seed_a_qid, seed_a_ru = seed_a
            country_qid, country_ru = row["country_qid"], row["country_ru"]
            type_qid, type_ru = row["type_qid"], row["type_ru"]

            return {
                "template_id": f"museums_L4_{bridge_name}+country+type",
                "query_text_ru": (
                    f"{_museum_count_phrase(k)}, у которых {text_name} совпадает с {text_name}ом музея "
                    f"«{seed_a_ru}», которые находятся в стране «{country_ru}» "
                    f"и относятся к типу «{type_ru}». Не включай музей «{seed_a_ru}» в ответ."
                ),
                "where_lines": [
                    f"BIND(wd:{seed_a_qid} AS ?seedA) .",
                    f"?seedA wdt:{prop_pid} ?shared_value .",
                    f"?item wdt:{prop_pid} ?shared_value .",
                    f"?item wdt:P17 wd:{country_qid} .",
                    f"?item wdt:P31/wdt:P279* wd:{type_qid} .",
                    f"FILTER(?item != wd:{seed_a_qid}) .",
                ],
                "constraints": {
                    "seed_shared_property_qid": seed_a_qid,
                    "seed_shared_property_ru": seed_a_ru,
                    "country_qid": country_qid,
                    "country_ru": country_ru,
                    "type_qid": type_qid,
                    "type_ru": type_ru,
                    "bridge": bridge_name,
                    "target_probe_qid": row["museum_qid"],
                    "target_probe_ru": row["museum_ru"],
                },
            }

        patterns = [
            p_same_country_as_seed_type_year,
            p_same_city_as_seed_type,
            p_same_owner_or_operator_as_seed_type_country,
        ]

    elif complexity == "L5":
        patterns = []

        def p_country_from_seedA_type_from_seedB_period():
            row = _museum_sample_target(
                [("country_qid", "country_ru"), ("type_qid", "type_ru")],
                rng,
                require_year=True,
            )
            if not row:
                return None

            seed_a = _museum_pick_ref(
                "country_qid",
                row["country_qid"],
                exclude_qids=[row["museum_qid"]],
                rng=rng,
            )
            seed_b = _museum_pick_ref(
                "type_qid",
                row["type_qid"],
                exclude_qids=[row["museum_qid"], seed_a[0] if seed_a else None],
                rng=rng,
            )

            if not seed_a or not seed_b:
                return None

            seed_a_qid, seed_a_ru = seed_a
            seed_b_qid, seed_b_ru = seed_b
            y1, y2 = _museum_year_window(row["inception_year"], half_width=80)

            return {
                "template_id": "museums_L5_country_from_seedA+type_from_seedB+period",
                "query_text_ru": (
                    f"Назови {k} музея, которые находятся в той же стране, что и «{seed_a_ru}», "
                    f"относятся к тому же типу, что и «{seed_b_ru}», "
                    f"и были основаны в период {y1}–{y2} годов. "
                    f"Не включай музеи «{seed_a_ru}» и «{seed_b_ru}» в ответ."
                ),
                "where_lines": [
                    f"BIND(wd:{seed_a_qid} AS ?seedA) .",
                    f"BIND(wd:{seed_b_qid} AS ?seedB) .",
                    "?seedA wdt:P17 ?country .",
                    "?seedB wdt:P31 ?type .",
                    "?item wdt:P17 ?country .",
                    "?item wdt:P31/wdt:P279* ?type .",
                    f"FILTER(?item != wd:{seed_a_qid} && ?item != wd:{seed_b_qid}) .",
                    *_museum_year_filter(y1, y2),
                ],
                "constraints": {
                    "seed_country_qid": seed_a_qid,
                    "seed_country_ru": seed_a_ru,
                    "seed_type_qid": seed_b_qid,
                    "seed_type_ru": seed_b_ru,
                    "bridge": "country_from_seedA_type_from_seedB",
                    "year_from": y1,
                    "year_to": y2,
                    "target_probe_qid": row["museum_qid"],
                    "target_probe_ru": row["museum_ru"],
                },
            }

        def p_city_from_seedA_type_from_seedB():
            row = _museum_sample_target(
                [("city_qid", "city_ru"), ("type_qid", "type_ru")],
                rng,
                require_year=False,
            )
            if not row:
                return None

            seed_a = _museum_pick_ref(
                "city_qid",
                row["city_qid"],
                exclude_qids=[row["museum_qid"]],
                rng=rng,
            )
            seed_b = _museum_pick_ref(
                "type_qid",
                row["type_qid"],
                exclude_qids=[row["museum_qid"], seed_a[0] if seed_a else None],
                rng=rng,
            )

            if not seed_a or not seed_b:
                return None

            seed_a_qid, seed_a_ru = seed_a
            seed_b_qid, seed_b_ru = seed_b

            return {
                "template_id": "museums_L5_city_from_seedA+type_from_seedB",
                "query_text_ru": (
                    f"Подбери {k} музея: они должны быть расположены в той же административной единице "
                    f"или городе, что и «{seed_a_ru}», и относиться к тому же типу, что и «{seed_b_ru}». "
                    f"Не включай музеи «{seed_a_ru}» и «{seed_b_ru}» в ответ."
                ),
                "where_lines": [
                    f"BIND(wd:{seed_a_qid} AS ?seedA) .",
                    f"BIND(wd:{seed_b_qid} AS ?seedB) .",
                    "?seedA wdt:P131 ?city .",
                    "?seedB wdt:P31 ?type .",
                    "?item wdt:P131 ?city .",
                    "?item wdt:P31/wdt:P279* ?type .",
                    f"FILTER(?item != wd:{seed_a_qid} && ?item != wd:{seed_b_qid}) .",
                ],
                "constraints": {
                    "seed_city_qid": seed_a_qid,
                    "seed_city_ru": seed_a_ru,
                    "seed_type_qid": seed_b_qid,
                    "seed_type_ru": seed_b_ru,
                    "bridge": "city_from_seedA_type_from_seedB",
                    "target_probe_qid": row["museum_qid"],
                    "target_probe_ru": row["museum_ru"],
                },
            }

        def p_country_from_seedA_owner_or_operator_from_seedB_type():
            row = _museum_sample_target(
                [("country_qid", "country_ru"), ("type_qid", "type_ru")],
                rng,
                require_year=False,
            )
            if not row:
                return None

            if _museum_good_qid(row.get("owner_qid")) and _museum_good_label(row.get("owner_ru")):
                prop_col = "owner_qid"
                prop_pid = "P127"
                bridge_name = "country_from_seedA_owner_from_seedB_type"
                text_name = "владелец"
            elif _museum_good_qid(row.get("operator_qid")) and _museum_good_label(row.get("operator_ru")):
                prop_col = "operator_qid"
                prop_pid = "P137"
                bridge_name = "country_from_seedA_operator_from_seedB_type"
                text_name = "оператор"
            else:
                return None

            seed_a = _museum_pick_ref(
                "country_qid",
                row["country_qid"],
                exclude_qids=[row["museum_qid"]],
                rng=rng,
            )
            seed_b = _museum_pick_ref(
                prop_col,
                row[prop_col],
                exclude_qids=[row["museum_qid"], seed_a[0] if seed_a else None],
                rng=rng,
            )

            if not seed_a or not seed_b:
                return None

            seed_a_qid, seed_a_ru = seed_a
            seed_b_qid, seed_b_ru = seed_b
            type_qid, type_ru = row["type_qid"], row["type_ru"]

            return {
                "template_id": f"museums_L5_{bridge_name}",
                "query_text_ru": (
                    f"Найди {k} музея, которые находятся в той же стране, что и «{seed_a_ru}», "
                    f"имеют того же {text_name}а, что и «{seed_b_ru}», "
                    f"и относятся к типу «{type_ru}». "
                    f"Не включай музеи «{seed_a_ru}» и «{seed_b_ru}» в ответ."
                ),
                "where_lines": [
                    f"BIND(wd:{seed_a_qid} AS ?seedA) .",
                    f"BIND(wd:{seed_b_qid} AS ?seedB) .",
                    "?seedA wdt:P17 ?country .",
                    f"?seedB wdt:{prop_pid} ?shared_value .",
                    "?item wdt:P17 ?country .",
                    f"?item wdt:{prop_pid} ?shared_value .",
                    f"?item wdt:P31/wdt:P279* wd:{type_qid} .",
                    f"FILTER(?item != wd:{seed_a_qid} && ?item != wd:{seed_b_qid}) .",
                ],
                "constraints": {
                    "seed_country_qid": seed_a_qid,
                    "seed_country_ru": seed_a_ru,
                    "seed_property_qid": seed_b_qid,
                    "seed_property_ru": seed_b_ru,
                    "type_qid": type_qid,
                    "type_ru": type_ru,
                    "bridge": bridge_name,
                    "target_probe_qid": row["museum_qid"],
                    "target_probe_ru": row["museum_ru"],
                },
            }

        patterns = [
            p_country_from_seedA_type_from_seedB_period,
            p_city_from_seedA_type_from_seedB,
            p_country_from_seedA_owner_or_operator_from_seedB_type,
        ]

    else:
        raise ValueError(complexity)

    for _ in range(120):
        cand = rng.choice(patterns)()
        if cand is not None:
            return cand

    raise RuntimeError(f"Museums {complexity}: could not build candidate.")


def generate_museums_example(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 160,
) -> BenchmarkExample:
    k = MUSEUM_REQUESTED_COUNT[complexity]
    last_debug = None

    for _ in range(max_attempts):
        try:
            cand = _museum_make_candidate(complexity, rng)
        except Exception as e:
            last_debug = f"candidate error: {type(e).__name__}: {e}"
            continue

        try:
            sparql, items = _run_museums_query_v2(
                cand["where_lines"],
                limit=MUSEUM_QUERY_LIMIT,
            )
        except Exception as e:
            last_debug = f"{cand.get('template_id')}: SPARQL error {type(e).__name__}: {e}"
            continue

        n = len(items)
        last_debug = f"{cand.get('template_id')}: n={n}"

        if not _museum_accept_count(complexity, n, k):
            continue

        ask = build_ask_validator(Q_MUSEUM, cand["where_lines"], item_var="item")

        return BenchmarkExample(
            id=f"museums_{complexity.lower()}_{idx:04d}",
            domain="museums",
            complexity=complexity,
            query_text_ru=cand["query_text_ru"],
            constraints=cand["constraints"],
            requested_count=k,
            gold_answer_qids=[q for q, _ in items],
            gold_answer_labels_ru=[l for _, l in items],
            sparql_query=sparql,
            created_at=utc_now_z(),
            is_advanced=(complexity in {"L4", "L5"}),
            template_id=cand["template_id"],
            template_family="museums_v2_seeded_robust",
            gold_truncated=(len(items) >= MUSEUM_QUERY_LIMIT),
            ask_validator_sparql=ask,
        )

    raise RuntimeError(
        f"Museums {complexity}: failed after {max_attempts} attempts. Last: {last_debug}"
    )


print("✅ Museums HOTFIX loaded: seed-based robust L1-L5 generation.")

# Museum L4-L5 refinement over stable local pools.

import random
import re
import json
import pandas as pd

if "_MUSEUMS_GENERATOR_BEFORE_STABLE_L45" not in globals():
    _MUSEUMS_GENERATOR_BEFORE_STABLE_L45 = generate_museums_example

Q_MUSEUM = ensure_qid("музей", fallback_qid="Q33506")

MUSEUM_STABLE_REQUESTED_COUNT = {
    "L4": 2,
    "L5": 1,
}

MUSEUM_STABLE_MAX_LOCAL_GOLD = {
    "L4": 160,
    "L5": 120,
}

MUSEUM_STABLE_MIN_LOCAL_GOLD = {
    "L4": 2,
    "L5": 1,
}


def _museum_stable_good_qid(x):
    return isinstance(x, str) and x.startswith("Q")


def _museum_stable_good_label(x):
    if x is None:
        return False
    try:
        if pd.isna(x):
            return False
    except Exception:
        pass
    return isinstance(x, str) and len(x.strip()) > 0


def _museum_stable_parse_year(x):
    if x is None:
        return None
    if isinstance(x, (int, float)) and not pd.isna(x):
        y = int(x)
        return y if 1200 <= y <= 2026 else None
    s = str(x)
    m = re.search(r"(-?\d{3,4})", s)
    if not m:
        return None
    y = int(m.group(1))
    return y if 1200 <= y <= 2026 else None


def _museum_stable_year_bucket(y):
    y = _museum_stable_parse_year(y)
    if y is None:
        return None, None
    if y < 1800:
        return 1200, 1799
    if y < 1900:
        return 1800, 1899
    if y < 1950:
        return 1900, 1949
    if y < 2000:
        return 1950, 1999
    return 2000, 2026


def _museum_stable_year_filter(y1, y2):
    return [
        "?item wdt:P571 ?inception_date .",
        "BIND(YEAR(?inception_date) AS ?inception_year) .",
        f"FILTER(?inception_year >= {int(y1)} && ?inception_year <= {int(y2)}) .",
    ]


def _museum_stable_count_phrase(k):
    if k == 1:
        return "Назови 1 музей"
    if 2 <= k <= 4:
        return f"Назови {k} музея"
    return f"Назови {k} музеев"


def _museum_stable_get_pool():
    """
    Берём уже построенный локальный pool из предыдущих блоков.
    Никаких SPARQL-запросов здесь не делаем.
    """
    candidates = [
        "museums_rich_df",
        "museums_fast_df",
        "museums_seed_df_v2",
        "museums_seed_df",
    ]

    for name in candidates:
        if name in globals():
            df = globals()[name]
            if isinstance(df, pd.DataFrame) and len(df) > 0:
                df = df.copy()

                rename_map = {
                    "item_qid": "museum_qid",
                    "item_ru": "museum_ru",
                    "itemLabelRu": "museum_ru",
                    "countryLabelRu": "country_ru",
                    "continentLabelRu": "continent_ru",
                    "typeLabelRu": "type_ru",
                    "cityLabelRu": "city_ru",
                }

                for old, new in rename_map.items():
                    if old in df.columns and new not in df.columns:
                        df[new] = df[old]

                required = [
                    "museum_qid", "museum_ru",
                    "country_qid", "country_ru",
                    "continent_qid", "continent_ru",
                    "city_qid", "city_ru",
                    "type_qid", "type_ru",
                    "inception_year",
                ]

                for c in required:
                    if c not in df.columns:
                        df[c] = None

                df["inception_year"] = df["inception_year"].apply(_museum_stable_parse_year)

                df = df[
                    df["museum_qid"].apply(_museum_stable_good_qid)
                    & df["museum_ru"].apply(_museum_stable_good_label)
                ].copy()

                df = df.drop_duplicates(subset=["museum_qid"]).reset_index(drop=True)

                print(f"[Museums stable] using pool {name}: rows={len(df)}")
                return df

    raise RuntimeError(
        "No museums pool found. Сначала запусти предыдущий Museums-блок, который строит museums_fast_df или museums_seed_df_v2."
    )


museums_stable_df = _museum_stable_get_pool()


def _museum_stable_pick_ref(prop_col, value_qid, exclude_qids, rng):
    if not _museum_stable_good_qid(value_qid):
        return None

    exclude_qids = {q for q in exclude_qids if _museum_stable_good_qid(q)}

    sub = museums_stable_df[
        (museums_stable_df[prop_col] == value_qid)
        & (~museums_stable_df["museum_qid"].isin(exclude_qids))
        & (museums_stable_df["museum_qid"].apply(_museum_stable_good_qid))
        & (museums_stable_df["museum_ru"].apply(_museum_stable_good_label))
    ]

    if len(sub) == 0:
        return None

    row = sub.sample(1, random_state=rng.randint(0, 10**9)).iloc[0].to_dict()
    return row["museum_qid"], row["museum_ru"]


def _museum_stable_local_items(spec):
    df = museums_stable_df.copy()

    for col in ["country_qid", "continent_qid", "city_qid", "type_qid"]:
        if spec.get(col):
            df = df[df[col] == spec[col]]

    if spec.get("year_from") is not None and spec.get("year_to") is not None:
        df = df[
            df["inception_year"].notna()
            & (df["inception_year"] >= int(spec["year_from"]))
            & (df["inception_year"] <= int(spec["year_to"]))
        ]

    exclude_qids = set(spec.get("exclude_qids") or [])
    if exclude_qids:
        df = df[~df["museum_qid"].isin(exclude_qids)]

    items = []
    for _, row in df.drop_duplicates(subset=["museum_qid"]).iterrows():
        qid = row.get("museum_qid")
        label = row.get("museum_ru")
        if _museum_stable_good_qid(qid) and _museum_stable_good_label(label):
            items.append((qid, label))

    return items


def _museum_stable_build_sparql(where_lines, limit=700):
    where = "\n      ".join(where_lines)
    return f"""
    SELECT DISTINCT ?item ?itemLabelRu WHERE {{
      ?item wdt:P31/wdt:P279* wd:{Q_MUSEUM} .
      {where}
      ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") .
    }}
    LIMIT {int(limit)}
    """.strip()


def _museum_stable_candidate_ok(complexity, items):
    n = len(items)
    return (
        n >= MUSEUM_STABLE_MIN_LOCAL_GOLD[complexity]
        and n >= MUSEUM_STABLE_REQUESTED_COUNT[complexity]
        and n <= MUSEUM_STABLE_MAX_LOCAL_GOLD[complexity]
    )


def _build_museums_stable_l4_candidates(max_candidates=500):
    """
    L4: один seed-bridge + 1-2 явных фильтра.
    """
    candidates = []
    rng = random.Random(2026)

    df = museums_stable_df.copy()

    # L4-A: same country as seed + type + period
    rows = df[
        df["country_qid"].apply(_museum_stable_good_qid)
        & df["country_ru"].apply(_museum_stable_good_label)
        & df["type_qid"].apply(_museum_stable_good_qid)
        & df["type_ru"].apply(_museum_stable_good_label)
        & df["inception_year"].notna()
    ].copy()

    rows["year_from"], rows["year_to"] = zip(*rows["inception_year"].apply(_museum_stable_year_bucket))

    for _, row in rows.sample(frac=1, random_state=1).iterrows():
        seed = _museum_stable_pick_ref(
            "country_qid",
            row["country_qid"],
            exclude_qids=[row["museum_qid"]],
            rng=rng,
        )
        if not seed:
            continue

        seed_qid, seed_ru = seed

        spec = {
            "country_qid": row["country_qid"],
            "type_qid": row["type_qid"],
            "year_from": row["year_from"],
            "year_to": row["year_to"],
            "exclude_qids": [seed_qid],
        }
        items = _museum_stable_local_items(spec)

        if not _museum_stable_candidate_ok("L4", items):
            continue

        where_lines = [
            f"BIND(wd:{seed_qid} AS ?seedA) .",
            "?seedA wdt:P17 ?country .",
            "?item wdt:P17 ?country .",
            f"?item wdt:P31/wdt:P279* wd:{row['type_qid']} .",
            f"FILTER(?item != wd:{seed_qid}) .",
            *_museum_stable_year_filter(row["year_from"], row["year_to"]),
        ]

        candidates.append({
            "template_id": "museums_stable_L4_same_country_seed+type+period",
            "query_text_ru": (
                f"{_museum_stable_count_phrase(MUSEUM_STABLE_REQUESTED_COUNT['L4'])}, находящихся в той же стране, "
                f"что и «{seed_ru}», относящихся к типу «{row['type_ru']}» "
                f"и основанных в период {int(row['year_from'])}–{int(row['year_to'])} годов. "
                f"Не включай музей «{seed_ru}» в ответ."
            ),
            "constraints": {
                "seed_country_qid": seed_qid,
                "seed_country_ru": seed_ru,
                "type_qid": row["type_qid"],
                "type_ru": row["type_ru"],
                "year_from": int(row["year_from"]),
                "year_to": int(row["year_to"]),
                "bridge": "same_country_as_seed+type+period",
            },
            "where_lines": where_lines,
            "items": items,
        })

        if len(candidates) >= max_candidates:
            return candidates

    # L4-B: same city as seed + type
    rows = df[
        df["city_qid"].apply(_museum_stable_good_qid)
        & df["city_ru"].apply(_museum_stable_good_label)
        & df["type_qid"].apply(_museum_stable_good_qid)
        & df["type_ru"].apply(_museum_stable_good_label)
    ].copy()

    for _, row in rows.sample(frac=1, random_state=2).iterrows():
        seed = _museum_stable_pick_ref(
            "city_qid",
            row["city_qid"],
            exclude_qids=[row["museum_qid"]],
            rng=rng,
        )
        if not seed:
            continue

        seed_qid, seed_ru = seed

        spec = {
            "city_qid": row["city_qid"],
            "type_qid": row["type_qid"],
            "exclude_qids": [seed_qid],
        }
        items = _museum_stable_local_items(spec)

        if not _museum_stable_candidate_ok("L4", items):
            continue

        where_lines = [
            f"BIND(wd:{seed_qid} AS ?seedA) .",
            "?seedA wdt:P131 ?city .",
            "?item wdt:P131 ?city .",
            f"?item wdt:P31/wdt:P279* wd:{row['type_qid']} .",
            f"FILTER(?item != wd:{seed_qid}) .",
        ]

        candidates.append({
            "template_id": "museums_stable_L4_same_city_seed+type",
            "query_text_ru": (
                f"{_museum_stable_count_phrase(MUSEUM_STABLE_REQUESTED_COUNT['L4'])}, расположенных в той же "
                f"административной единице или городе, что и «{seed_ru}», "
                f"и относящихся к типу «{row['type_ru']}». "
                f"Не включай музей «{seed_ru}» в ответ."
            ),
            "constraints": {
                "seed_city_qid": seed_qid,
                "seed_city_ru": seed_ru,
                "type_qid": row["type_qid"],
                "type_ru": row["type_ru"],
                "bridge": "same_city_as_seed+type",
            },
            "where_lines": where_lines,
            "items": items,
        })

        if len(candidates) >= max_candidates:
            return candidates

    return candidates


def _build_museums_stable_l5_candidates(max_candidates=500):
    """
    L5: два seed-bridge + дополнительный фильтр.
    """
    candidates = []
    rng = random.Random(2027)

    df = museums_stable_df.copy()

    # L5-A: country from seedA + type from seedB + period
    rows = df[
        df["country_qid"].apply(_museum_stable_good_qid)
        & df["country_ru"].apply(_museum_stable_good_label)
        & df["type_qid"].apply(_museum_stable_good_qid)
        & df["type_ru"].apply(_museum_stable_good_label)
        & df["inception_year"].notna()
    ].copy()

    rows["year_from"], rows["year_to"] = zip(*rows["inception_year"].apply(_museum_stable_year_bucket))

    for _, row in rows.sample(frac=1, random_state=3).iterrows():
        seed_a = _museum_stable_pick_ref(
            "country_qid",
            row["country_qid"],
            exclude_qids=[row["museum_qid"]],
            rng=rng,
        )
        if not seed_a:
            continue

        seed_b = _museum_stable_pick_ref(
            "type_qid",
            row["type_qid"],
            exclude_qids=[row["museum_qid"], seed_a[0]],
            rng=rng,
        )
        if not seed_b:
            continue

        seed_a_qid, seed_a_ru = seed_a
        seed_b_qid, seed_b_ru = seed_b

        spec = {
            "country_qid": row["country_qid"],
            "type_qid": row["type_qid"],
            "year_from": row["year_from"],
            "year_to": row["year_to"],
            "exclude_qids": [seed_a_qid, seed_b_qid],
        }
        items = _museum_stable_local_items(spec)

        if not _museum_stable_candidate_ok("L5", items):
            continue

        where_lines = [
            f"BIND(wd:{seed_a_qid} AS ?seedA) .",
            f"BIND(wd:{seed_b_qid} AS ?seedB) .",
            "?seedA wdt:P17 ?country .",
            "?seedB wdt:P31 ?type .",
            "?item wdt:P17 ?country .",
            "?item wdt:P31/wdt:P279* ?type .",
            f"FILTER(?item != wd:{seed_a_qid} && ?item != wd:{seed_b_qid}) .",
            *_museum_stable_year_filter(row["year_from"], row["year_to"]),
        ]

        candidates.append({
            "template_id": "museums_stable_L5_country_seedA+type_seedB+period",
            "query_text_ru": (
                f"Назови {_museum_stable_count_phrase(MUSEUM_STABLE_REQUESTED_COUNT['L5']).replace('Назови ', '')}, "
                f"который находится в той же стране, что и «{seed_a_ru}», "
                f"относится к тому же типу, что и «{seed_b_ru}», "
                f"и был основан в период {int(row['year_from'])}–{int(row['year_to'])} годов. "
                f"Не включай музеи «{seed_a_ru}» и «{seed_b_ru}» в ответ."
            ),
            "constraints": {
                "seed_country_qid": seed_a_qid,
                "seed_country_ru": seed_a_ru,
                "seed_type_qid": seed_b_qid,
                "seed_type_ru": seed_b_ru,
                "year_from": int(row["year_from"]),
                "year_to": int(row["year_to"]),
                "bridge": "country_from_seedA_type_from_seedB_period",
            },
            "where_lines": where_lines,
            "items": items,
        })

        if len(candidates) >= max_candidates:
            return candidates

    # L5-B: city from seedA + type from seedB
    rows = df[
        df["city_qid"].apply(_museum_stable_good_qid)
        & df["city_ru"].apply(_museum_stable_good_label)
        & df["type_qid"].apply(_museum_stable_good_qid)
        & df["type_ru"].apply(_museum_stable_good_label)
    ].copy()

    for _, row in rows.sample(frac=1, random_state=4).iterrows():
        seed_a = _museum_stable_pick_ref(
            "city_qid",
            row["city_qid"],
            exclude_qids=[row["museum_qid"]],
            rng=rng,
        )
        if not seed_a:
            continue

        seed_b = _museum_stable_pick_ref(
            "type_qid",
            row["type_qid"],
            exclude_qids=[row["museum_qid"], seed_a[0]],
            rng=rng,
        )
        if not seed_b:
            continue

        seed_a_qid, seed_a_ru = seed_a
        seed_b_qid, seed_b_ru = seed_b

        spec = {
            "city_qid": row["city_qid"],
            "type_qid": row["type_qid"],
            "exclude_qids": [seed_a_qid, seed_b_qid],
        }
        items = _museum_stable_local_items(spec)

        if not _museum_stable_candidate_ok("L5", items):
            continue

        where_lines = [
            f"BIND(wd:{seed_a_qid} AS ?seedA) .",
            f"BIND(wd:{seed_b_qid} AS ?seedB) .",
            "?seedA wdt:P131 ?city .",
            "?seedB wdt:P31 ?type .",
            "?item wdt:P131 ?city .",
            "?item wdt:P31/wdt:P279* ?type .",
            f"FILTER(?item != wd:{seed_a_qid} && ?item != wd:{seed_b_qid}) .",
        ]

        candidates.append({
            "template_id": "museums_stable_L5_city_seedA+type_seedB",
            "query_text_ru": (
                f"Подбери {_museum_stable_count_phrase(MUSEUM_STABLE_REQUESTED_COUNT['L5']).replace('Назови ', '')}, "
                f"расположенный в той же административной единице или городе, что и «{seed_a_ru}», "
                f"и относящийся к тому же типу, что и «{seed_b_ru}». "
                f"Не включай музеи «{seed_a_ru}» и «{seed_b_ru}» в ответ."
            ),
            "constraints": {
                "seed_city_qid": seed_a_qid,
                "seed_city_ru": seed_a_ru,
                "seed_type_qid": seed_b_qid,
                "seed_type_ru": seed_b_ru,
                "bridge": "city_from_seedA_type_from_seedB",
            },
            "where_lines": where_lines,
            "items": items,
        })

        if len(candidates) >= max_candidates:
            return candidates

    return candidates


MUSEUMS_STABLE_L4_CANDIDATES = _build_museums_stable_l4_candidates()
MUSEUMS_STABLE_L5_CANDIDATES = _build_museums_stable_l5_candidates()

print("[Museums stable] L4 candidates:", len(MUSEUMS_STABLE_L4_CANDIDATES))
print("[Museums stable] L5 candidates:", len(MUSEUMS_STABLE_L5_CANDIDATES))


def _museum_stable_sample_candidate(complexity, rng):
    if complexity == "L4":
        pool = MUSEUMS_STABLE_L4_CANDIDATES
    elif complexity == "L5":
        pool = MUSEUMS_STABLE_L5_CANDIDATES
    else:
        raise ValueError(complexity)

    if not pool:
        raise RuntimeError(f"No stable museums candidates for {complexity}")

    return rng.choice(pool)


def generate_museums_example(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 120,
) -> BenchmarkExample:
    if complexity not in {"L4", "L5"}:
        return _MUSEUMS_GENERATOR_BEFORE_STABLE_L45(
            complexity,
            idx,
            rng,
            max_attempts=max_attempts,
        )

    cand = _museum_stable_sample_candidate(complexity, rng)
    k = MUSEUM_STABLE_REQUESTED_COUNT[complexity]

    sparql = _museum_stable_build_sparql(cand["where_lines"], limit=700)
    ask = build_ask_validator(Q_MUSEUM, cand["where_lines"], item_var="item")

    return BenchmarkExample(
        id=f"museums_{complexity.lower()}_{idx:04d}",
        domain="museums",
        complexity=complexity,
        query_text_ru=cand["query_text_ru"],
        constraints=cand["constraints"],
        requested_count=k,
        gold_answer_qids=[q for q, _ in cand["items"]],
        gold_answer_labels_ru=[l for _, l in cand["items"]],
        sparql_query=sparql,
        created_at=utc_now_z(),
        is_advanced=True,
        template_id=cand["template_id"],
        template_family="museums_stable_l45_multihop",
        gold_truncated=True,
        ask_validator_sparql=ask,
    )


print("✅ Museums STABLE L4/L5 HOTFIX loaded.")


✅ Patched paintings v3: non-redundant positive-gold L4/L5 enabled; depicts/movement/material enrichment added


,complexity,mode,candidate_rows,min_gold_est,max_gold_est
0,L4,genre_creator_collection,700,1,1
1,L4,genre_creator_year,521,1,5
2,L4,creator_collection_year,699,1,5
3,L4,genre_depicts_year,700,1,1
4,L4,creator_depicts_year,700,1,1
5,L4,genre_movement_year,151,1,5
6,L4,creator_material_year,477,1,5
7,L4,collection_depicts_year,700,1,1
8,L5,genre_creator_collection_year,873,1,4
9,L5,genre_creator_depicts_year,900,1,1


✅ Museums HOTFIX loaded: seed-based robust L1-L5 generation.
[Museums stable] using pool museums_seed_df_v2: rows=1244
[Museums stable] L4 candidates: 238
[Museums stable] L5 candidates: 500
✅ Museums STABLE L4/L5 HOTFIX loaded.


<a id="spacecraft-domain"></a>

## 19. Spacecraft domain

Spacecraft templates with manufacturer, operator, program, launch period, and hard L5 bridge patterns.

In [19]:
Q_SPACECRAFT = ensure_qid("космический аппарат", fallback_qid="Q40218")
space_manu_df     = load_or_build_pool("spacecraft_manufacturers_ru", lambda: build_value_pool_ru(Q_SPACECRAFT, "P176", "manu",    limit=240))
space_operator_df = load_or_build_pool("spacecraft_operators_ru",     lambda: build_value_pool_ru(Q_SPACECRAFT, "P137", "operator",limit=240))
space_countries_df= load_or_build_pool("spacecraft_countries_ru",     lambda: build_value_pool_ru(Q_SPACECRAFT, "P495", "country", limit=160))

def pick_spacecraft_constraints(rng: random.Random):
    manu_qid, manu_ru = pick_from_df(space_manu_df, "manu_qid", "manuLabelRu", rng)
    op_qid, op_ru = pick_from_df(space_operator_df, "operator_qid", "operatorLabelRu", rng)
    c_qid, c_ru = pick_from_df(space_countries_df, "country_qid", "countryLabelRu", rng)
    return manu_qid, manu_ru, op_qid, op_ru, c_qid, c_ru

def run_spacecraft_query(manu_qid: str, op_qid: str, country_qid: Optional[str], y1: Optional[int], y2: Optional[int], limit: int = 300):
    where = [
        f"?item wdt:P176 wd:{manu_qid} .",
        f"?item wdt:P137 wd:{op_qid} .",
    ]
    if country_qid:
        where.append(f"?item wdt:P495 wd:{country_qid} .")
    if y1 is not None and y2 is not None:
        where += [
            "?item wdt:P619 ?date .",
            "BIND(YEAR(?date) AS ?year) .",
            f"FILTER(?year >= {int(y1)} && ?year <= {int(y2)}) .",
        ]
    sparql, items = select_items_with_ru_label(Q_SPACECRAFT, where, limit=limit, item_var="item")
    return sparql, items, where

def nlg_spacecraft_ru(manu_ru: str, op_ru: str, country_ru: Optional[str], y1: Optional[int], y2: Optional[int], k: int) -> str:
    cond = [f"производителя: {manu_ru}", f"оператора: {op_ru}"]
    if country_ru:
        cond.append(f"страны происхождения: {country_ru}")
    if y1 is not None and y2 is not None:
        cond.append(f"запущенные в период {int(y1)}–{int(y2)}")
    return f"Назови {k} космических аппаратов, " + ", ".join(cond) + "."

def generate_spacecraft_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 70) -> BenchmarkExample:
    if complexity == "L1":
        k=5; use_country=False; y1=y2=None; require=lambda n: n>=k
    elif complexity == "L2":
        k=3; use_country=False; y1,y2=rng.choice([(1957,1970),(1971,1989),(1990,2004),(2005,2024)]); require=lambda n: n>=k
    elif complexity == "L3":
        k=3; use_country=True; y1,y2=rng.choice([(1957,1970),(1971,1989),(1990,2004),(2005,2024)]); require=lambda n: n>=k
    elif complexity == "L4":
        k=12; use_country=True; y1,y2=rng.choice([(1965,1967),(1967,1969),(1969,1971),(1971,1973)]); require=lambda n: (0<n<k)
    elif complexity == "L5":
        k=5; use_country=True; y1,y2=2500,2510; require=lambda n: n==0
    else:
        raise ValueError(complexity)

    for _ in range(max_attempts):
        manu_qid, manu_ru, op_qid, op_ru, c_qid, c_ru = pick_spacecraft_constraints(rng)
        if not use_country:
            c_qid = c_ru = None

        sparql, items, where = run_spacecraft_query(manu_qid, op_qid, c_qid, y1, y2, limit=300)
        n=len(items)
        if not require(n):
            continue

        gold=items[:300]
        ask=build_ask_validator(Q_SPACECRAFT, where, item_var="item")
        return BenchmarkExample(
            id=f"spacecraft_{complexity.lower()}_{idx:04d}",
            domain="spacecraft",
            complexity=complexity,
            query_text_ru=nlg_spacecraft_ru(manu_ru, op_ru, c_ru, y1, y2, k),
            constraints={
                "manufacturer_qid": manu_qid, "manufacturer_ru": manu_ru,
                "operator_qid": op_qid, "operator_ru": op_ru,
                "country_qid": c_qid, "country_ru": c_ru,
                "year_from": y1, "year_to": y2,
            },
            requested_count=k,
            gold_answer_qids=[q for q,_ in gold],
            gold_answer_labels_ru=[l for _,l in gold],
            sparql_query=sparql,
            created_at=utc_now_z(),
            gold_truncated=len(items) >= 300,
            ask_validator_sparql=ask,
        )

    sparql, items, where = run_spacecraft_query(manu_qid, op_qid, c_qid, y1, y2, limit=300)
    ask = build_ask_validator(Q_SPACECRAFT, where, item_var="item")
    return BenchmarkExample(
        id=f"spacecraft_{complexity.lower()}_{idx:04d}",
        domain="spacecraft",
        complexity=complexity,
        query_text_ru=nlg_spacecraft_ru(manu_ru, op_ru, c_ru, y1, y2, k),
        constraints={"manufacturer_qid": manu_qid, "operator_qid": op_qid, "country_qid": c_qid, "year_from": y1, "year_to": y2},
        requested_count=k,
        gold_answer_qids=[q for q,_ in items[:300]],
        gold_answer_labels_ru=[l for _,l in items[:300]],
        sparql_query=sparql,
        created_at=utc_now_z(),
        gold_truncated=len(items) >= 300,
        ask_validator_sparql=ask,
    )


from typing import Optional, List, Tuple, Dict, Any
import random
import pandas as pd

Q_SPACECRAFT = ensure_qid("космический аппарат", fallback_qid="Q40218")

SPACECRAFT_QUERY_LIMIT = 700

SPACECRAFT_MAX_GOLD_BY_COMPLEXITY = {
    "L1": 260,
    "L2": 180,
    "L3": 120,
    "L4": 80,
    "L5": 60,
}

SPACECRAFT_REQUESTED_COUNT = {
    "L1": 5,
    "L2": 5,
    "L3": 4,
    "L4": 3,
    "L5": 2,
}

SPACECRAFT_YEAR_RANGES = {
    "broad": [
        (1957, 1975),
        (1976, 1995),
        (1996, 2010),
        (2011, 2025),
    ],
    "medium": [
        (1957, 1969),
        (1970, 1985),
        (1986, 2000),
        (2001, 2012),
        (2013, 2025),
    ],
    "narrow": [
        (1961, 1969),
        (1970, 1978),
        (1979, 1988),
        (1989, 1999),
        (2000, 2008),
        (2009, 2016),
        (2017, 2025),
    ],
}


def _spacecraft_build_value_pool_ru(pid: str, val_name: str, limit: int = 400) -> pd.DataFrame:
    """
    Пул значений свойства pid у космических аппаратов.
    Берём только значения с русским label, чтобы потом не ловить странные QID/англ. подписи.
    """
    sparql = f"""
    SELECT DISTINCT ?val ?valLabelRu WHERE {{
      ?item wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ;
            wdt:{pid} ?val .
      ?val rdfs:label ?valLabelRu FILTER(LANG(?valLabelRu) = "ru") .
    }}
    LIMIT {int(limit)}
    """
    rows = rows_from_select(wd.sparql_select(sparql))
    data = []
    for r in rows:
        qid = uri_to_qid(r.get("val", ""))
        lbl = r.get("valLabelRu")
        if qid and lbl:
            data.append({f"{val_name}_qid": qid, f"{val_name}LabelRu": lbl})
    return pd.DataFrame(data).drop_duplicates().reset_index(drop=True)


def _spacecraft_build_type_pool_ru(limit: int = 300) -> pd.DataFrame:
    """
    Пул типов/подклассов космических аппаратов, реально встречающихся в Wikidata.
    Например: искусственный спутник, космический зонд, орбитальная станция и т.п.
    """
    sparql = f"""
    SELECT DISTINCT ?type ?typeLabelRu WHERE {{
      ?item wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ;
            wdt:P31 ?type .
      ?type wdt:P279* wd:{Q_SPACECRAFT} .
      FILTER(?type != wd:{Q_SPACECRAFT})
      ?type rdfs:label ?typeLabelRu FILTER(LANG(?typeLabelRu) = "ru") .
    }}
    LIMIT {int(limit)}
    """
    rows = rows_from_select(wd.sparql_select(sparql))
    data = []
    for r in rows:
        qid = uri_to_qid(r.get("type", ""))
        lbl = r.get("typeLabelRu")
        if qid and lbl:
            data.append({"type_qid": qid, "typeLabelRu": lbl})
    return pd.DataFrame(data).drop_duplicates().reset_index(drop=True)


def _spacecraft_build_seed_pool_ru(limit: int = 700) -> pd.DataFrame:
    """
    Пул известных космических аппаратов для bridge-паттернов.
    Не требуем здесь конкретные свойства: они проверяются уже в самих SPARQL-шаблонах.
    """
    sparql = f"""
    SELECT DISTINCT ?seed ?seedLabelRu WHERE {{
      ?seed wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} .
      ?seed rdfs:label ?seedLabelRu FILTER(LANG(?seedLabelRu) = "ru") .
      FILTER EXISTS {{
        {{ ?seed wdt:P176 ?x . }}
        UNION {{ ?seed wdt:P137 ?x . }}
        UNION {{ ?seed wdt:P375 ?x . }}
        UNION {{ ?seed wdt:P361 ?x . }}
      }}
    }}
    LIMIT {int(limit)}
    """
    rows = rows_from_select(wd.sparql_select(sparql))
    data = []
    for r in rows:
        qid = uri_to_qid(r.get("seed", ""))
        lbl = r.get("seedLabelRu")
        if qid and lbl:
            data.append({"seed_qid": qid, "seedLabelRu": lbl})
    return pd.DataFrame(data).drop_duplicates().reset_index(drop=True)


spacecraft_type_df = load_or_build_pool(
    "spacecraft_types_ru_v2",
    lambda: _spacecraft_build_type_pool_ru(limit=350),
)

spacecraft_manu_df = load_or_build_pool(
    "spacecraft_manufacturers_ru_v2",
    lambda: _spacecraft_build_value_pool_ru("P176", "manufacturer", limit=350),
)

spacecraft_operator_df = load_or_build_pool(
    "spacecraft_operators_ru_v2",
    lambda: _spacecraft_build_value_pool_ru("P137", "operator", limit=350),
)

spacecraft_country_df = load_or_build_pool(
    "spacecraft_countries_ru_v2",
    lambda: _spacecraft_build_value_pool_ru("P495", "country", limit=220),
)

spacecraft_launch_vehicle_df = load_or_build_pool(
    "spacecraft_launch_vehicles_ru_v2",
    lambda: _spacecraft_build_value_pool_ru("P375", "launch_vehicle", limit=350),
)

spacecraft_program_df = load_or_build_pool(
    "spacecraft_programs_ru_v2",
    lambda: _spacecraft_build_value_pool_ru("P361", "program", limit=350),
)

spacecraft_seed_df = load_or_build_pool(
    "spacecraft_seed_items_ru_v2",
    lambda: _spacecraft_build_seed_pool_ru(limit=800),
)


def _space_pool_ok(df: pd.DataFrame, *cols: str) -> bool:
    return isinstance(df, pd.DataFrame) and len(df) > 0 and all(c in df.columns for c in cols)


def _space_pick(df: pd.DataFrame, qid_col: str, label_col: str, rng: random.Random) -> Tuple[str, str]:
    if not _space_pool_ok(df, qid_col, label_col):
        raise RuntimeError(f"Empty spacecraft pool or missing columns: {qid_col}, {label_col}")
    row = df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
    return str(row[qid_col]), str(row[label_col])


def _space_year_range(kind: str, rng: random.Random) -> Tuple[int, int]:
    return rng.choice(SPACECRAFT_YEAR_RANGES[kind])


def _space_year_filter(y1: int, y2: int) -> List[str]:
    return [
        "?item wdt:P619 ?launch_date .",
        "BIND(YEAR(?launch_date) AS ?launch_year) .",
        f"FILTER(?launch_year >= {int(y1)} && ?launch_year <= {int(y2)}) .",
    ]


def _run_spacecraft_query(where_lines: List[str], limit: int = SPACECRAFT_QUERY_LIMIT) -> Tuple[str, List[Tuple[str, str]]]:
    """
    SELECT с русским label.
    LIMIT здесь нужен только для скорости генерации.
    После генерации можно дособрать full-gold через отдельный enrich-notebook.
    """
    where = "\n      ".join(where_lines)
    sparql = f"""
    SELECT DISTINCT ?item ?itemLabelRu WHERE {{
      ?item wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} .
      {where}
      ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") .
    }}
    LIMIT {int(limit)}
    """
    rows = rows_from_select(wd.sparql_select(sparql))
    items = []
    seen = set()
    for r in rows:
        qid = uri_to_qid(r.get("item", ""))
        lbl = r.get("itemLabelRu")
        if qid and lbl and qid not in seen:
            items.append((qid, lbl))
            seen.add(qid)
    return sparql.strip(), items


def _space_nlg_count_prefix(k: int) -> str:
    if k == 1:
        return "Назови 1 космический аппарат"
    if 2 <= k <= 4:
        return f"Назови {k} космических аппарата"
    return f"Назови {k} космических аппаратов"


def _space_accept_count(complexity: str, n: int, k: int) -> bool:
    return (n >= k) and (n <= SPACECRAFT_MAX_GOLD_BY_COMPLEXITY[complexity])


def _space_make_candidate(complexity: str, rng: random.Random) -> Dict[str, Any]:
    """
    Возвращает один случайный шаблон:
      where_lines, query_text_ru, constraints, template_id.
    Сам SPARQL здесь ещё не запускается.
    """
    k = SPACECRAFT_REQUESTED_COUNT[complexity]

    patterns = []

    if complexity == "L1":
        if _space_pool_ok(spacecraft_type_df, "type_qid", "typeLabelRu"):
            def p_type_year():
                type_qid, type_ru = _space_pick(spacecraft_type_df, "type_qid", "typeLabelRu", rng)
                y1, y2 = _space_year_range("broad", rng)
                where = [
                    f"?item wdt:P31/wdt:P279* wd:{type_qid} .",
                    *_space_year_filter(y1, y2),
                ]
                return {
                    "template_id": "spacecraft_L1_type+launch_period",
                    "where_lines": where,
                    "query_text_ru": f"{_space_nlg_count_prefix(k)} типа «{type_ru}», запущенных в период {y1}–{y2} годов.",
                    "constraints": {
                        "type_qid": type_qid,
                        "type_ru": type_ru,
                        "year_from": y1,
                        "year_to": y2,
                    },
                }
            patterns.append(p_type_year)

        if _space_pool_ok(spacecraft_operator_df, "operator_qid", "operatorLabelRu"):
            def p_operator():
                op_qid, op_ru = _space_pick(spacecraft_operator_df, "operator_qid", "operatorLabelRu", rng)
                where = [f"?item wdt:P137 wd:{op_qid} ."]
                return {
                    "template_id": "spacecraft_L1_operator",
                    "where_lines": where,
                    "query_text_ru": f"{_space_nlg_count_prefix(k)}, оператором которых является «{op_ru}».",
                    "constraints": {
                        "operator_qid": op_qid,
                        "operator_ru": op_ru,
                    },
                }
            patterns.append(p_operator)

        if _space_pool_ok(spacecraft_manu_df, "manufacturer_qid", "manufacturerLabelRu"):
            def p_manufacturer():
                manu_qid, manu_ru = _space_pick(spacecraft_manu_df, "manufacturer_qid", "manufacturerLabelRu", rng)
                where = [f"?item wdt:P176 wd:{manu_qid} ."]
                return {
                    "template_id": "spacecraft_L1_manufacturer",
                    "where_lines": where,
                    "query_text_ru": f"{_space_nlg_count_prefix(k)}, произведённых организацией «{manu_ru}».",
                    "constraints": {
                        "manufacturer_qid": manu_qid,
                        "manufacturer_ru": manu_ru,
                    },
                }
            patterns.append(p_manufacturer)

    elif complexity == "L2":
        if _space_pool_ok(spacecraft_operator_df, "operator_qid", "operatorLabelRu"):
            def p_operator_year():
                op_qid, op_ru = _space_pick(spacecraft_operator_df, "operator_qid", "operatorLabelRu", rng)
                y1, y2 = _space_year_range("medium", rng)
                where = [
                    f"?item wdt:P137 wd:{op_qid} .",
                    *_space_year_filter(y1, y2),
                ]
                return {
                    "template_id": "spacecraft_L2_operator+launch_period",
                    "where_lines": where,
                    "query_text_ru": f"{_space_nlg_count_prefix(k)}, оператором которых является «{op_ru}», запущенных в период {y1}–{y2} годов.",
                    "constraints": {
                        "operator_qid": op_qid,
                        "operator_ru": op_ru,
                        "year_from": y1,
                        "year_to": y2,
                    },
                }
            patterns.append(p_operator_year)

        if _space_pool_ok(spacecraft_manu_df, "manufacturer_qid", "manufacturerLabelRu"):
            def p_manufacturer_year():
                manu_qid, manu_ru = _space_pick(spacecraft_manu_df, "manufacturer_qid", "manufacturerLabelRu", rng)
                y1, y2 = _space_year_range("medium", rng)
                where = [
                    f"?item wdt:P176 wd:{manu_qid} .",
                    *_space_year_filter(y1, y2),
                ]
                return {
                    "template_id": "spacecraft_L2_manufacturer+launch_period",
                    "where_lines": where,
                    "query_text_ru": f"{_space_nlg_count_prefix(k)}, произведённых организацией «{manu_ru}» и запущенных в период {y1}–{y2} годов.",
                    "constraints": {
                        "manufacturer_qid": manu_qid,
                        "manufacturer_ru": manu_ru,
                        "year_from": y1,
                        "year_to": y2,
                    },
                }
            patterns.append(p_manufacturer_year)

        if _space_pool_ok(spacecraft_launch_vehicle_df, "launch_vehicle_qid", "launch_vehicleLabelRu"):
            def p_launch_vehicle_year():
                lv_qid, lv_ru = _space_pick(spacecraft_launch_vehicle_df, "launch_vehicle_qid", "launch_vehicleLabelRu", rng)
                y1, y2 = _space_year_range("medium", rng)
                where = [
                    f"?item wdt:P375 wd:{lv_qid} .",
                    *_space_year_filter(y1, y2),
                ]
                return {
                    "template_id": "spacecraft_L2_launch_vehicle+launch_period",
                    "where_lines": where,
                    "query_text_ru": f"{_space_nlg_count_prefix(k)}, запущенных ракетой-носителем «{lv_ru}» в период {y1}–{y2} годов.",
                    "constraints": {
                        "launch_vehicle_qid": lv_qid,
                        "launch_vehicle_ru": lv_ru,
                        "year_from": y1,
                        "year_to": y2,
                    },
                }
            patterns.append(p_launch_vehicle_year)

        if (
            _space_pool_ok(spacecraft_type_df, "type_qid", "typeLabelRu")
            and _space_pool_ok(spacecraft_operator_df, "operator_qid", "operatorLabelRu")
        ):
            def p_type_operator():
                type_qid, type_ru = _space_pick(spacecraft_type_df, "type_qid", "typeLabelRu", rng)
                op_qid, op_ru = _space_pick(spacecraft_operator_df, "operator_qid", "operatorLabelRu", rng)
                where = [
                    f"?item wdt:P31/wdt:P279* wd:{type_qid} .",
                    f"?item wdt:P137 wd:{op_qid} .",
                ]
                return {
                    "template_id": "spacecraft_L2_type+operator",
                    "where_lines": where,
                    "query_text_ru": f"{_space_nlg_count_prefix(k)} типа «{type_ru}», оператором которых является «{op_ru}».",
                    "constraints": {
                        "type_qid": type_qid,
                        "type_ru": type_ru,
                        "operator_qid": op_qid,
                        "operator_ru": op_ru,
                    },
                }
            patterns.append(p_type_operator)

    elif complexity == "L3":
        if (
            _space_pool_ok(spacecraft_type_df, "type_qid", "typeLabelRu")
            and _space_pool_ok(spacecraft_operator_df, "operator_qid", "operatorLabelRu")
        ):
            def p_type_operator_year():
                type_qid, type_ru = _space_pick(spacecraft_type_df, "type_qid", "typeLabelRu", rng)
                op_qid, op_ru = _space_pick(spacecraft_operator_df, "operator_qid", "operatorLabelRu", rng)
                y1, y2 = _space_year_range("medium", rng)
                where = [
                    f"?item wdt:P31/wdt:P279* wd:{type_qid} .",
                    f"?item wdt:P137 wd:{op_qid} .",
                    *_space_year_filter(y1, y2),
                ]
                return {
                    "template_id": "spacecraft_L3_type+operator+launch_period",
                    "where_lines": where,
                    "query_text_ru": f"{_space_nlg_count_prefix(k)} типа «{type_ru}», оператором которых является «{op_ru}», запущенных в период {y1}–{y2} годов.",
                    "constraints": {
                        "type_qid": type_qid,
                        "type_ru": type_ru,
                        "operator_qid": op_qid,
                        "operator_ru": op_ru,
                        "year_from": y1,
                        "year_to": y2,
                    },
                }
            patterns.append(p_type_operator_year)

        if (
            _space_pool_ok(spacecraft_manu_df, "manufacturer_qid", "manufacturerLabelRu")
            and _space_pool_ok(spacecraft_launch_vehicle_df, "launch_vehicle_qid", "launch_vehicleLabelRu")
        ):
            def p_manufacturer_launch_vehicle_year():
                manu_qid, manu_ru = _space_pick(spacecraft_manu_df, "manufacturer_qid", "manufacturerLabelRu", rng)
                lv_qid, lv_ru = _space_pick(spacecraft_launch_vehicle_df, "launch_vehicle_qid", "launch_vehicleLabelRu", rng)
                y1, y2 = _space_year_range("medium", rng)
                where = [
                    f"?item wdt:P176 wd:{manu_qid} .",
                    f"?item wdt:P375 wd:{lv_qid} .",
                    *_space_year_filter(y1, y2),
                ]
                return {
                    "template_id": "spacecraft_L3_manufacturer+launch_vehicle+launch_period",
                    "where_lines": where,
                    "query_text_ru": f"{_space_nlg_count_prefix(k)}, произведённых организацией «{manu_ru}», запущенных ракетой-носителем «{lv_ru}» в период {y1}–{y2} годов.",
                    "constraints": {
                        "manufacturer_qid": manu_qid,
                        "manufacturer_ru": manu_ru,
                        "launch_vehicle_qid": lv_qid,
                        "launch_vehicle_ru": lv_ru,
                        "year_from": y1,
                        "year_to": y2,
                    },
                }
            patterns.append(p_manufacturer_launch_vehicle_year)

        if (
            _space_pool_ok(spacecraft_program_df, "program_qid", "programLabelRu")
            and _space_pool_ok(spacecraft_type_df, "type_qid", "typeLabelRu")
        ):
            def p_program_type_year():
                program_qid, program_ru = _space_pick(spacecraft_program_df, "program_qid", "programLabelRu", rng)
                type_qid, type_ru = _space_pick(spacecraft_type_df, "type_qid", "typeLabelRu", rng)
                y1, y2 = _space_year_range("medium", rng)
                where = [
                    f"?item wdt:P361 wd:{program_qid} .",
                    f"?item wdt:P31/wdt:P279* wd:{type_qid} .",
                    *_space_year_filter(y1, y2),
                ]
                return {
                    "template_id": "spacecraft_L3_program+type+launch_period",
                    "where_lines": where,
                    "query_text_ru": f"{_space_nlg_count_prefix(k)} типа «{type_ru}», относящихся к программе/серии «{program_ru}» и запущенных в период {y1}–{y2} годов.",
                    "constraints": {
                        "program_qid": program_qid,
                        "program_ru": program_ru,
                        "type_qid": type_qid,
                        "type_ru": type_ru,
                        "year_from": y1,
                        "year_to": y2,
                    },
                }
            patterns.append(p_program_type_year)

        if (
            _space_pool_ok(spacecraft_country_df, "country_qid", "countryLabelRu")
            and _space_pool_ok(spacecraft_manu_df, "manufacturer_qid", "manufacturerLabelRu")
        ):
            def p_country_manufacturer_year():
                country_qid, country_ru = _space_pick(spacecraft_country_df, "country_qid", "countryLabelRu", rng)
                manu_qid, manu_ru = _space_pick(spacecraft_manu_df, "manufacturer_qid", "manufacturerLabelRu", rng)
                y1, y2 = _space_year_range("medium", rng)
                where = [
                    f"?item wdt:P495 wd:{country_qid} .",
                    f"?item wdt:P176 wd:{manu_qid} .",
                    *_space_year_filter(y1, y2),
                ]
                return {
                    "template_id": "spacecraft_L3_country+manufacturer+launch_period",
                    "where_lines": where,
                    "query_text_ru": f"{_space_nlg_count_prefix(k)} страны происхождения «{country_ru}», произведённых организацией «{manu_ru}» и запущенных в период {y1}–{y2} годов.",
                    "constraints": {
                        "country_qid": country_qid,
                        "country_ru": country_ru,
                        "manufacturer_qid": manu_qid,
                        "manufacturer_ru": manu_ru,
                        "year_from": y1,
                        "year_to": y2,
                    },
                }
            patterns.append(p_country_manufacturer_year)

    elif complexity == "L4":
        if _space_pool_ok(spacecraft_seed_df, "seed_qid", "seedLabelRu"):
            def p_same_manufacturer_as_seed_year():
                seed_qid, seed_ru = _space_pick(spacecraft_seed_df, "seed_qid", "seedLabelRu", rng)
                y1, y2 = _space_year_range("narrow", rng)
                where = [
                    f"BIND(wd:{seed_qid} AS ?seed) .",
                    "?seed wdt:P176 ?manufacturer .",
                    "?item wdt:P176 ?manufacturer .",
                    "FILTER(?item != ?seed) .",
                    *_space_year_filter(y1, y2),
                ]
                return {
                    "template_id": "spacecraft_L4_same_manufacturer_as_seed+launch_period",
                    "where_lines": where,
                    "query_text_ru": f"{_space_nlg_count_prefix(k)}, произведённых той же организацией, что и «{seed_ru}», и запущенных в период {y1}–{y2} годов.",
                    "constraints": {
                        "seed_qid": seed_qid,
                        "seed_ru": seed_ru,
                        "bridge": "same_manufacturer",
                        "year_from": y1,
                        "year_to": y2,
                    },
                }
            patterns.append(p_same_manufacturer_as_seed_year)

            if _space_pool_ok(spacecraft_type_df, "type_qid", "typeLabelRu"):
                def p_same_operator_as_seed_type_year():
                    seed_qid, seed_ru = _space_pick(spacecraft_seed_df, "seed_qid", "seedLabelRu", rng)
                    type_qid, type_ru = _space_pick(spacecraft_type_df, "type_qid", "typeLabelRu", rng)
                    y1, y2 = _space_year_range("narrow", rng)
                    where = [
                        f"BIND(wd:{seed_qid} AS ?seed) .",
                        "?seed wdt:P137 ?operator .",
                        "?item wdt:P137 ?operator .",
                        f"?item wdt:P31/wdt:P279* wd:{type_qid} .",
                        "FILTER(?item != ?seed) .",
                        *_space_year_filter(y1, y2),
                    ]
                    return {
                        "template_id": "spacecraft_L4_same_operator_as_seed+type+launch_period",
                        "where_lines": where,
                        "query_text_ru": f"{_space_nlg_count_prefix(k)} типа «{type_ru}», оператор которых совпадает с оператором аппарата «{seed_ru}», и которые были запущены в период {y1}–{y2} годов.",
                        "constraints": {
                            "seed_qid": seed_qid,
                            "seed_ru": seed_ru,
                            "bridge": "same_operator",
                            "type_qid": type_qid,
                            "type_ru": type_ru,
                            "year_from": y1,
                            "year_to": y2,
                        },
                    }
                patterns.append(p_same_operator_as_seed_type_year)

        if (
            _space_pool_ok(spacecraft_launch_vehicle_df, "launch_vehicle_qid", "launch_vehicleLabelRu")
            and _space_pool_ok(spacecraft_operator_df, "operator_qid", "operatorLabelRu")
        ):
            def p_launch_vehicle_operator_year():
                lv_qid, lv_ru = _space_pick(spacecraft_launch_vehicle_df, "launch_vehicle_qid", "launch_vehicleLabelRu", rng)
                op_qid, op_ru = _space_pick(spacecraft_operator_df, "operator_qid", "operatorLabelRu", rng)
                y1, y2 = _space_year_range("narrow", rng)
                where = [
                    f"?item wdt:P375 wd:{lv_qid} .",
                    f"?item wdt:P137 wd:{op_qid} .",
                    *_space_year_filter(y1, y2),
                ]
                return {
                    "template_id": "spacecraft_L4_launch_vehicle+operator+launch_period",
                    "where_lines": where,
                    "query_text_ru": f"{_space_nlg_count_prefix(k)}, запущенных ракетой-носителем «{lv_ru}», оператором которых является «{op_ru}», в период {y1}–{y2} годов.",
                    "constraints": {
                        "launch_vehicle_qid": lv_qid,
                        "launch_vehicle_ru": lv_ru,
                        "operator_qid": op_qid,
                        "operator_ru": op_ru,
                        "year_from": y1,
                        "year_to": y2,
                    },
                }
            patterns.append(p_launch_vehicle_operator_year)

        if (
            _space_pool_ok(spacecraft_program_df, "program_qid", "programLabelRu")
            and _space_pool_ok(spacecraft_manu_df, "manufacturer_qid", "manufacturerLabelRu")
        ):
            def p_program_manufacturer_year():
                program_qid, program_ru = _space_pick(spacecraft_program_df, "program_qid", "programLabelRu", rng)
                manu_qid, manu_ru = _space_pick(spacecraft_manu_df, "manufacturer_qid", "manufacturerLabelRu", rng)
                y1, y2 = _space_year_range("narrow", rng)
                where = [
                    f"?item wdt:P361 wd:{program_qid} .",
                    f"?item wdt:P176 wd:{manu_qid} .",
                    *_space_year_filter(y1, y2),
                ]
                return {
                    "template_id": "spacecraft_L4_program+manufacturer+launch_period",
                    "where_lines": where,
                    "query_text_ru": f"{_space_nlg_count_prefix(k)}, относящихся к программе/серии «{program_ru}», произведённых организацией «{manu_ru}» и запущенных в период {y1}–{y2} годов.",
                    "constraints": {
                        "program_qid": program_qid,
                        "program_ru": program_ru,
                        "manufacturer_qid": manu_qid,
                        "manufacturer_ru": manu_ru,
                        "year_from": y1,
                        "year_to": y2,
                    },
                }
            patterns.append(p_program_manufacturer_year)

    elif complexity == "L5":
        if _space_pool_ok(spacecraft_seed_df, "seed_qid", "seedLabelRu"):
            def p_same_manufacturer_and_launch_vehicle_as_seed():
                seed_qid, seed_ru = _space_pick(spacecraft_seed_df, "seed_qid", "seedLabelRu", rng)
                y1, y2 = _space_year_range("narrow", rng)
                where = [
                    f"BIND(wd:{seed_qid} AS ?seed) .",
                    "?seed wdt:P176 ?manufacturer .",
                    "?seed wdt:P375 ?launch_vehicle .",
                    "?item wdt:P176 ?manufacturer .",
                    "?item wdt:P375 ?launch_vehicle .",
                    "FILTER(?item != ?seed) .",
                    *_space_year_filter(y1, y2),
                ]
                return {
                    "template_id": "spacecraft_L5_same_manufacturer+same_launch_vehicle_as_seed+launch_period",
                    "where_lines": where,
                    "query_text_ru": f"{_space_nlg_count_prefix(k)}, у которых совпадают производитель и ракета-носитель с аппаратом «{seed_ru}», и которые были запущены в период {y1}–{y2} годов.",
                    "constraints": {
                        "seed_qid": seed_qid,
                        "seed_ru": seed_ru,
                        "bridge": "same_manufacturer_and_launch_vehicle",
                        "year_from": y1,
                        "year_to": y2,
                    },
                }
            patterns.append(p_same_manufacturer_and_launch_vehicle_as_seed)

            def p_same_program_and_operator_as_seed():
                seed_qid, seed_ru = _space_pick(spacecraft_seed_df, "seed_qid", "seedLabelRu", rng)
                y1, y2 = _space_year_range("narrow", rng)
                where = [
                    f"BIND(wd:{seed_qid} AS ?seed) .",
                    "?seed wdt:P361 ?program .",
                    "?seed wdt:P137 ?operator .",
                    "?item wdt:P361 ?program .",
                    "?item wdt:P137 ?operator .",
                    "FILTER(?item != ?seed) .",
                    *_space_year_filter(y1, y2),
                ]
                return {
                    "template_id": "spacecraft_L5_same_program+same_operator_as_seed+launch_period",
                    "where_lines": where,
                    "query_text_ru": f"{_space_nlg_count_prefix(k)}, которые относятся к той же программе/серии и имеют того же оператора, что и аппарат «{seed_ru}», а также были запущены в период {y1}–{y2} годов.",
                    "constraints": {
                        "seed_qid": seed_qid,
                        "seed_ru": seed_ru,
                        "bridge": "same_program_and_operator",
                        "year_from": y1,
                        "year_to": y2,
                    },
                }
            patterns.append(p_same_program_and_operator_as_seed)

            if _space_pool_ok(spacecraft_type_df, "type_qid", "typeLabelRu"):
                def p_same_manufacturer_as_seed_type_year():
                    seed_qid, seed_ru = _space_pick(spacecraft_seed_df, "seed_qid", "seedLabelRu", rng)
                    type_qid, type_ru = _space_pick(spacecraft_type_df, "type_qid", "typeLabelRu", rng)
                    y1, y2 = _space_year_range("narrow", rng)
                    where = [
                        f"BIND(wd:{seed_qid} AS ?seed) .",
                        "?seed wdt:P176 ?manufacturer .",
                        "?item wdt:P176 ?manufacturer .",
                        f"?item wdt:P31/wdt:P279* wd:{type_qid} .",
                        "FILTER(?item != ?seed) .",
                        *_space_year_filter(y1, y2),
                    ]
                    return {
                        "template_id": "spacecraft_L5_same_manufacturer_as_seed+type+launch_period",
                        "where_lines": where,
                        "query_text_ru": f"{_space_nlg_count_prefix(k)} типа «{type_ru}», произведённых той же организацией, что и аппарат «{seed_ru}», и запущенных в период {y1}–{y2} годов.",
                        "constraints": {
                            "seed_qid": seed_qid,
                            "seed_ru": seed_ru,
                            "bridge": "same_manufacturer",
                            "type_qid": type_qid,
                            "type_ru": type_ru,
                            "year_from": y1,
                            "year_to": y2,
                        },
                    }
                patterns.append(p_same_manufacturer_as_seed_type_year)

        if (
            _space_pool_ok(spacecraft_type_df, "type_qid", "typeLabelRu")
            and _space_pool_ok(spacecraft_manu_df, "manufacturer_qid", "manufacturerLabelRu")
            and _space_pool_ok(spacecraft_operator_df, "operator_qid", "operatorLabelRu")
        ):
            def p_type_manufacturer_operator_year():
                type_qid, type_ru = _space_pick(spacecraft_type_df, "type_qid", "typeLabelRu", rng)
                manu_qid, manu_ru = _space_pick(spacecraft_manu_df, "manufacturer_qid", "manufacturerLabelRu", rng)
                op_qid, op_ru = _space_pick(spacecraft_operator_df, "operator_qid", "operatorLabelRu", rng)
                y1, y2 = _space_year_range("narrow", rng)
                where = [
                    f"?item wdt:P31/wdt:P279* wd:{type_qid} .",
                    f"?item wdt:P176 wd:{manu_qid} .",
                    f"?item wdt:P137 wd:{op_qid} .",
                    *_space_year_filter(y1, y2),
                ]
                return {
                    "template_id": "spacecraft_L5_type+manufacturer+operator+launch_period",
                    "where_lines": where,
                    "query_text_ru": f"{_space_nlg_count_prefix(k)} типа «{type_ru}», произведённых организацией «{manu_ru}», оператором которых является «{op_ru}», и запущенных в период {y1}–{y2} годов.",
                    "constraints": {
                        "type_qid": type_qid,
                        "type_ru": type_ru,
                        "manufacturer_qid": manu_qid,
                        "manufacturer_ru": manu_ru,
                        "operator_qid": op_qid,
                        "operator_ru": op_ru,
                        "year_from": y1,
                        "year_to": y2,
                    },
                }
            patterns.append(p_type_manufacturer_operator_year)

    else:
        raise ValueError(complexity)

    if not patterns:
        raise RuntimeError(f"No available Spacecraft patterns for {complexity}. Check pools.")

    return rng.choice(patterns)()


def generate_spacecraft_example(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 120,
) -> BenchmarkExample:
    """
    Генерирует только MAIN-примеры.
    Если за max_attempts не найден нормальный непустой запрос, бросает исключение:
    build_main_and_zero его поймает и попробует снова.
    """
    k = SPACECRAFT_REQUESTED_COUNT[complexity]
    last_debug = None

    for _ in range(max_attempts):
        cand = _space_make_candidate(complexity, rng)
        where = cand["where_lines"]

        try:
            sparql, items = _run_spacecraft_query(where, limit=SPACECRAFT_QUERY_LIMIT)
        except Exception as e:
            last_debug = f"SPARQL error for {cand.get('template_id')}: {type(e).__name__}: {e}"
            continue

        n = len(items)
        last_debug = f"{cand.get('template_id')}: n={n}"

        if not _space_accept_count(complexity, n, k):
            continue

        ask = build_ask_validator(Q_SPACECRAFT, where, item_var="item")

        return BenchmarkExample(
            id=f"spacecraft_{complexity.lower()}_{idx:04d}",
            domain="spacecraft",
            complexity=complexity,
            query_text_ru=cand["query_text_ru"],
            constraints=cand["constraints"],
            requested_count=k,
            gold_answer_qids=[q for q, _ in items],
            gold_answer_labels_ru=[l for _, l in items],
            sparql_query=sparql,
            created_at=utc_now_z(),
            is_advanced=(complexity in {"L4", "L5"}),
            template_id=cand["template_id"],
            template_family="spacecraft_v2",
            gold_truncated=(len(items) >= SPACECRAFT_QUERY_LIMIT),
            ask_validator_sparql=ask,
        )

    raise RuntimeError(
        f"Spacecraft {complexity}: failed to generate a valid MAIN example after "
        f"{max_attempts} attempts. Last: {last_debug}"
    )


print("✅ Spacecraft FINAL BLOCK loaded: diverse L1–L5 MAIN patterns, no intentional zero-gold generation.")

# Spacecraft refinement: viable co-occurrence templates for L3-L5.

import json
import random
import pandas as pd

if "_SPACECRAFT_BASE_GENERATOR" not in globals():
    _SPACECRAFT_BASE_GENERATOR = generate_spacecraft_example

SPACECRAFT_MAX_GOLD_BY_COMPLEXITY.update({
    "L3": 180,
    "L4": 140,
    "L5": 100,
})

SPACECRAFT_REQUESTED_COUNT.update({
    "L3": 4,
    "L4": 3,
    "L5": 2,
})

SPACECRAFT_V3_TEMPLATE_CACHE = {}


def _space_int(x, default=0):
    try:
        return int(float(x))
    except Exception:
        return default


def _space_values_year_ranges(kind: str) -> str:
    return "\n".join(
        f"        ({int(y1)} {int(y2)})"
        for y1, y2 in SPACECRAFT_YEAR_RANGES[kind]
    )


def _space_add_template_rows(records, template_id, sparql, make_record):
    try:
        rows = rows_from_select(wd.sparql_select(sparql))
    except Exception as e:
        print(f"[WARN] Spacecraft template pool failed: {template_id}: {type(e).__name__}: {e}")
        return

    for r in rows:
        try:
            rec = make_record(r)
            if not rec:
                continue

            rec["template_id"] = template_id
            rec["where_json"] = json.dumps(rec.pop("where_lines"), ensure_ascii=False)
            rec["constraints_json"] = json.dumps(rec.pop("constraints"), ensure_ascii=False)
            records.append(rec)
        except Exception as e:
            print(f"[WARN] bad template row for {template_id}: {type(e).__name__}: {e}")


def _build_spacecraft_l3_templates() -> pd.DataFrame:
    records = []
    k = SPACECRAFT_REQUESTED_COUNT["L3"]
    max_gold = SPACECRAFT_MAX_GOLD_BY_COMPLEXITY["L3"]
    years_medium = _space_values_year_ranges("medium")

    sparql = f"""
    SELECT ?operator ?operatorLabelRu ?manufacturer ?manufacturerLabelRu ?y1 ?y2
           (COUNT(DISTINCT ?item) AS ?cnt)
    WHERE {{
      VALUES (?y1 ?y2) {{
{years_medium}
      }}
      ?item wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ;
            wdt:P137 ?operator ;
            wdt:P176 ?manufacturer ;
            wdt:P619 ?launch_date .
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)

      ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") .
      ?operator rdfs:label ?operatorLabelRu FILTER(LANG(?operatorLabelRu) = "ru") .
      ?manufacturer rdfs:label ?manufacturerLabelRu FILTER(LANG(?manufacturerLabelRu) = "ru") .
    }}
    GROUP BY ?operator ?operatorLabelRu ?manufacturer ?manufacturerLabelRu ?y1 ?y2
    HAVING(COUNT(DISTINCT ?item) >= {k} && COUNT(DISTINCT ?item) <= {max_gold})
    LIMIT 300
    """

    def make_op_manu_year(r):
        op_qid = uri_to_qid(r.get("operator", ""))
        manu_qid = uri_to_qid(r.get("manufacturer", ""))
        op_ru = r.get("operatorLabelRu")
        manu_ru = r.get("manufacturerLabelRu")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        cnt = _space_int(r.get("cnt"))

        if not all([op_qid, manu_qid, op_ru, manu_ru, y1, y2]):
            return None

        return {
            "complexity": "L3",
            "expected_count": cnt,
            "query_text_ru": (
                f"{_space_nlg_count_prefix(k)}, оператором которых является «{op_ru}», "
                f"произведённых организацией «{manu_ru}» и запущенных в период {y1}–{y2} годов."
            ),
            "where_lines": [
                f"?item wdt:P137 wd:{op_qid} .",
                f"?item wdt:P176 wd:{manu_qid} .",
                *_space_year_filter(y1, y2),
            ],
            "constraints": {
                "operator_qid": op_qid,
                "operator_ru": op_ru,
                "manufacturer_qid": manu_qid,
                "manufacturer_ru": manu_ru,
                "year_from": y1,
                "year_to": y2,
                "expected_gold_count": cnt,
            },
        }

    _space_add_template_rows(
        records,
        "spacecraft_L3_operator+manufacturer+launch_period",
        sparql,
        make_op_manu_year,
    )

    sparql = f"""
    SELECT ?type ?typeLabelRu ?operator ?operatorLabelRu ?y1 ?y2
           (COUNT(DISTINCT ?item) AS ?cnt)
    WHERE {{
      VALUES (?y1 ?y2) {{
{years_medium}
      }}
      ?item wdt:P31 ?type ;
            wdt:P137 ?operator ;
            wdt:P619 ?launch_date .
      ?type wdt:P279* wd:{Q_SPACECRAFT} .
      FILTER(?type != wd:{Q_SPACECRAFT})

      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)

      ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") .
      ?type rdfs:label ?typeLabelRu FILTER(LANG(?typeLabelRu) = "ru") .
      ?operator rdfs:label ?operatorLabelRu FILTER(LANG(?operatorLabelRu) = "ru") .
    }}
    GROUP BY ?type ?typeLabelRu ?operator ?operatorLabelRu ?y1 ?y2
    HAVING(COUNT(DISTINCT ?item) >= {k} && COUNT(DISTINCT ?item) <= {max_gold})
    LIMIT 300
    """

    def make_type_op_year(r):
        type_qid = uri_to_qid(r.get("type", ""))
        op_qid = uri_to_qid(r.get("operator", ""))
        type_ru = r.get("typeLabelRu")
        op_ru = r.get("operatorLabelRu")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        cnt = _space_int(r.get("cnt"))

        if not all([type_qid, op_qid, type_ru, op_ru, y1, y2]):
            return None

        return {
            "complexity": "L3",
            "expected_count": cnt,
            "query_text_ru": (
                f"{_space_nlg_count_prefix(k)} типа «{type_ru}», оператором которых является «{op_ru}», "
                f"запущенных в период {y1}–{y2} годов."
            ),
            "where_lines": [
                f"?item wdt:P31/wdt:P279* wd:{type_qid} .",
                f"?item wdt:P137 wd:{op_qid} .",
                *_space_year_filter(y1, y2),
            ],
            "constraints": {
                "type_qid": type_qid,
                "type_ru": type_ru,
                "operator_qid": op_qid,
                "operator_ru": op_ru,
                "year_from": y1,
                "year_to": y2,
                "expected_gold_count": cnt,
            },
        }

    _space_add_template_rows(
        records,
        "spacecraft_L3_type+operator+launch_period",
        sparql,
        make_type_op_year,
    )

    sparql = f"""
    SELECT ?launch_vehicle ?launch_vehicleLabelRu ?operator ?operatorLabelRu ?y1 ?y2
           (COUNT(DISTINCT ?item) AS ?cnt)
    WHERE {{
      VALUES (?y1 ?y2) {{
{years_medium}
      }}
      ?item wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ;
            wdt:P375 ?launch_vehicle ;
            wdt:P137 ?operator ;
            wdt:P619 ?launch_date .
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)

      ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") .
      ?launch_vehicle rdfs:label ?launch_vehicleLabelRu FILTER(LANG(?launch_vehicleLabelRu) = "ru") .
      ?operator rdfs:label ?operatorLabelRu FILTER(LANG(?operatorLabelRu) = "ru") .
    }}
    GROUP BY ?launch_vehicle ?launch_vehicleLabelRu ?operator ?operatorLabelRu ?y1 ?y2
    HAVING(COUNT(DISTINCT ?item) >= {k} && COUNT(DISTINCT ?item) <= {max_gold})
    LIMIT 300
    """

    def make_lv_op_year(r):
        lv_qid = uri_to_qid(r.get("launch_vehicle", ""))
        op_qid = uri_to_qid(r.get("operator", ""))
        lv_ru = r.get("launch_vehicleLabelRu")
        op_ru = r.get("operatorLabelRu")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        cnt = _space_int(r.get("cnt"))

        if not all([lv_qid, op_qid, lv_ru, op_ru, y1, y2]):
            return None

        return {
            "complexity": "L3",
            "expected_count": cnt,
            "query_text_ru": (
                f"{_space_nlg_count_prefix(k)}, запущенных ракетой-носителем «{lv_ru}», "
                f"оператором которых является «{op_ru}», в период {y1}–{y2} годов."
            ),
            "where_lines": [
                f"?item wdt:P375 wd:{lv_qid} .",
                f"?item wdt:P137 wd:{op_qid} .",
                *_space_year_filter(y1, y2),
            ],
            "constraints": {
                "launch_vehicle_qid": lv_qid,
                "launch_vehicle_ru": lv_ru,
                "operator_qid": op_qid,
                "operator_ru": op_ru,
                "year_from": y1,
                "year_to": y2,
                "expected_gold_count": cnt,
            },
        }

    _space_add_template_rows(
        records,
        "spacecraft_L3_launch_vehicle+operator+launch_period",
        sparql,
        make_lv_op_year,
    )

    df = pd.DataFrame(records).drop_duplicates(subset=["query_text_ru"]).reset_index(drop=True)
    print(f"[Spacecraft] L3 viable templates:", len(df))
    return df


def _build_spacecraft_l4_templates() -> pd.DataFrame:
    records = []
    k = SPACECRAFT_REQUESTED_COUNT["L4"]
    max_gold = SPACECRAFT_MAX_GOLD_BY_COMPLEXITY["L4"]
    years_narrow = _space_values_year_ranges("narrow")

    sparql = f"""
    SELECT ?manufacturer ?manufacturerLabelRu ?operator ?operatorLabelRu
           ?launch_vehicle ?launch_vehicleLabelRu ?y1 ?y2
           (COUNT(DISTINCT ?item) AS ?cnt)
    WHERE {{
      VALUES (?y1 ?y2) {{
{years_narrow}
      }}
      ?item wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ;
            wdt:P176 ?manufacturer ;
            wdt:P137 ?operator ;
            wdt:P375 ?launch_vehicle ;
            wdt:P619 ?launch_date .
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)

      ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") .
      ?manufacturer rdfs:label ?manufacturerLabelRu FILTER(LANG(?manufacturerLabelRu) = "ru") .
      ?operator rdfs:label ?operatorLabelRu FILTER(LANG(?operatorLabelRu) = "ru") .
      ?launch_vehicle rdfs:label ?launch_vehicleLabelRu FILTER(LANG(?launch_vehicleLabelRu) = "ru") .
    }}
    GROUP BY ?manufacturer ?manufacturerLabelRu ?operator ?operatorLabelRu
             ?launch_vehicle ?launch_vehicleLabelRu ?y1 ?y2
    HAVING(COUNT(DISTINCT ?item) >= {k} && COUNT(DISTINCT ?item) <= {max_gold})
    LIMIT 300
    """

    def make_manu_op_lv_year(r):
        manu_qid = uri_to_qid(r.get("manufacturer", ""))
        op_qid = uri_to_qid(r.get("operator", ""))
        lv_qid = uri_to_qid(r.get("launch_vehicle", ""))
        manu_ru = r.get("manufacturerLabelRu")
        op_ru = r.get("operatorLabelRu")
        lv_ru = r.get("launch_vehicleLabelRu")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        cnt = _space_int(r.get("cnt"))

        if not all([manu_qid, op_qid, lv_qid, manu_ru, op_ru, lv_ru, y1, y2]):
            return None

        return {
            "complexity": "L4",
            "expected_count": cnt,
            "query_text_ru": (
                f"{_space_nlg_count_prefix(k)}, произведённых организацией «{manu_ru}», "
                f"оператором которых является «{op_ru}», запущенных ракетой-носителем «{lv_ru}» "
                f"в период {y1}–{y2} годов."
            ),
            "where_lines": [
                f"?item wdt:P176 wd:{manu_qid} .",
                f"?item wdt:P137 wd:{op_qid} .",
                f"?item wdt:P375 wd:{lv_qid} .",
                *_space_year_filter(y1, y2),
            ],
            "constraints": {
                "manufacturer_qid": manu_qid,
                "manufacturer_ru": manu_ru,
                "operator_qid": op_qid,
                "operator_ru": op_ru,
                "launch_vehicle_qid": lv_qid,
                "launch_vehicle_ru": lv_ru,
                "year_from": y1,
                "year_to": y2,
                "expected_gold_count": cnt,
            },
        }

    _space_add_template_rows(
        records,
        "spacecraft_L4_manufacturer+operator+launch_vehicle+launch_period",
        sparql,
        make_manu_op_lv_year,
    )

    sparql = f"""
    SELECT ?type ?typeLabelRu ?manufacturer ?manufacturerLabelRu ?operator ?operatorLabelRu ?y1 ?y2
           (COUNT(DISTINCT ?item) AS ?cnt)
    WHERE {{
      VALUES (?y1 ?y2) {{
{years_narrow}
      }}
      ?item wdt:P31 ?type ;
            wdt:P176 ?manufacturer ;
            wdt:P137 ?operator ;
            wdt:P619 ?launch_date .
      ?type wdt:P279* wd:{Q_SPACECRAFT} .
      FILTER(?type != wd:{Q_SPACECRAFT})

      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)

      ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") .
      ?type rdfs:label ?typeLabelRu FILTER(LANG(?typeLabelRu) = "ru") .
      ?manufacturer rdfs:label ?manufacturerLabelRu FILTER(LANG(?manufacturerLabelRu) = "ru") .
      ?operator rdfs:label ?operatorLabelRu FILTER(LANG(?operatorLabelRu) = "ru") .
    }}
    GROUP BY ?type ?typeLabelRu ?manufacturer ?manufacturerLabelRu ?operator ?operatorLabelRu ?y1 ?y2
    HAVING(COUNT(DISTINCT ?item) >= {k} && COUNT(DISTINCT ?item) <= {max_gold})
    LIMIT 300
    """

    def make_type_manu_op_year(r):
        type_qid = uri_to_qid(r.get("type", ""))
        manu_qid = uri_to_qid(r.get("manufacturer", ""))
        op_qid = uri_to_qid(r.get("operator", ""))
        type_ru = r.get("typeLabelRu")
        manu_ru = r.get("manufacturerLabelRu")
        op_ru = r.get("operatorLabelRu")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        cnt = _space_int(r.get("cnt"))

        if not all([type_qid, manu_qid, op_qid, type_ru, manu_ru, op_ru, y1, y2]):
            return None

        return {
            "complexity": "L4",
            "expected_count": cnt,
            "query_text_ru": (
                f"{_space_nlg_count_prefix(k)} типа «{type_ru}», произведённых организацией «{manu_ru}», "
                f"оператором которых является «{op_ru}», и запущенных в период {y1}–{y2} годов."
            ),
            "where_lines": [
                f"?item wdt:P31/wdt:P279* wd:{type_qid} .",
                f"?item wdt:P176 wd:{manu_qid} .",
                f"?item wdt:P137 wd:{op_qid} .",
                *_space_year_filter(y1, y2),
            ],
            "constraints": {
                "type_qid": type_qid,
                "type_ru": type_ru,
                "manufacturer_qid": manu_qid,
                "manufacturer_ru": manu_ru,
                "operator_qid": op_qid,
                "operator_ru": op_ru,
                "year_from": y1,
                "year_to": y2,
                "expected_gold_count": cnt,
            },
        }

    _space_add_template_rows(
        records,
        "spacecraft_L4_type+manufacturer+operator+launch_period",
        sparql,
        make_type_manu_op_year,
    )

    sparql = f"""
    SELECT ?program ?programLabelRu ?operator ?operatorLabelRu ?manufacturer ?manufacturerLabelRu ?y1 ?y2
           (COUNT(DISTINCT ?item) AS ?cnt)
    WHERE {{
      VALUES (?y1 ?y2) {{
{years_narrow}
      }}
      ?item wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ;
            wdt:P361 ?program ;
            wdt:P137 ?operator ;
            wdt:P176 ?manufacturer ;
            wdt:P619 ?launch_date .
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)

      ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") .
      ?program rdfs:label ?programLabelRu FILTER(LANG(?programLabelRu) = "ru") .
      ?operator rdfs:label ?operatorLabelRu FILTER(LANG(?operatorLabelRu) = "ru") .
      ?manufacturer rdfs:label ?manufacturerLabelRu FILTER(LANG(?manufacturerLabelRu) = "ru") .
    }}
    GROUP BY ?program ?programLabelRu ?operator ?operatorLabelRu ?manufacturer ?manufacturerLabelRu ?y1 ?y2
    HAVING(COUNT(DISTINCT ?item) >= {k} && COUNT(DISTINCT ?item) <= {max_gold})
    LIMIT 300
    """

    def make_program_op_manu_year(r):
        program_qid = uri_to_qid(r.get("program", ""))
        op_qid = uri_to_qid(r.get("operator", ""))
        manu_qid = uri_to_qid(r.get("manufacturer", ""))
        program_ru = r.get("programLabelRu")
        op_ru = r.get("operatorLabelRu")
        manu_ru = r.get("manufacturerLabelRu")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        cnt = _space_int(r.get("cnt"))

        if not all([program_qid, op_qid, manu_qid, program_ru, op_ru, manu_ru, y1, y2]):
            return None

        return {
            "complexity": "L4",
            "expected_count": cnt,
            "query_text_ru": (
                f"{_space_nlg_count_prefix(k)}, относящихся к программе/серии «{program_ru}», "
                f"оператором которых является «{op_ru}», произведённых организацией «{manu_ru}» "
                f"и запущенных в период {y1}–{y2} годов."
            ),
            "where_lines": [
                f"?item wdt:P361 wd:{program_qid} .",
                f"?item wdt:P137 wd:{op_qid} .",
                f"?item wdt:P176 wd:{manu_qid} .",
                *_space_year_filter(y1, y2),
            ],
            "constraints": {
                "program_qid": program_qid,
                "program_ru": program_ru,
                "operator_qid": op_qid,
                "operator_ru": op_ru,
                "manufacturer_qid": manu_qid,
                "manufacturer_ru": manu_ru,
                "year_from": y1,
                "year_to": y2,
                "expected_gold_count": cnt,
            },
        }

    _space_add_template_rows(
        records,
        "spacecraft_L4_program+operator+manufacturer+launch_period",
        sparql,
        make_program_op_manu_year,
    )

    df = pd.DataFrame(records).drop_duplicates(subset=["query_text_ru"]).reset_index(drop=True)
    print(f"[Spacecraft] L4 viable templates:", len(df))
    return df


def _build_spacecraft_l5_templates() -> pd.DataFrame:
    records = []
    k = SPACECRAFT_REQUESTED_COUNT["L5"]
    max_gold = SPACECRAFT_MAX_GOLD_BY_COMPLEXITY["L5"]
    years_narrow = _space_values_year_ranges("narrow")

    sparql = f"""
    SELECT ?seed ?seedLabelRu ?operator ?operatorLabelRu ?y1 ?y2
           (COUNT(DISTINCT ?item) AS ?cnt)
    WHERE {{
      VALUES (?y1 ?y2) {{
{years_narrow}
      }}
      ?seed wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ;
            wdt:P176 ?manufacturer .
      ?item wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ;
            wdt:P176 ?manufacturer ;
            wdt:P137 ?operator ;
            wdt:P619 ?launch_date .
      FILTER(?item != ?seed)

      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)

      ?seed rdfs:label ?seedLabelRu FILTER(LANG(?seedLabelRu) = "ru") .
      ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") .
      ?operator rdfs:label ?operatorLabelRu FILTER(LANG(?operatorLabelRu) = "ru") .
    }}
    GROUP BY ?seed ?seedLabelRu ?operator ?operatorLabelRu ?y1 ?y2
    HAVING(COUNT(DISTINCT ?item) >= {k} && COUNT(DISTINCT ?item) <= {max_gold})
    LIMIT 300
    """

    def make_same_manu_seed_op_year(r):
        seed_qid = uri_to_qid(r.get("seed", ""))
        op_qid = uri_to_qid(r.get("operator", ""))
        seed_ru = r.get("seedLabelRu")
        op_ru = r.get("operatorLabelRu")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        cnt = _space_int(r.get("cnt"))

        if not all([seed_qid, op_qid, seed_ru, op_ru, y1, y2]):
            return None

        return {
            "complexity": "L5",
            "expected_count": cnt,
            "query_text_ru": (
                f"{_space_nlg_count_prefix(k)}, произведённых той же организацией, что и аппарат «{seed_ru}», "
                f"оператором которых является «{op_ru}», и запущенных в период {y1}–{y2} годов."
            ),
            "where_lines": [
                f"BIND(wd:{seed_qid} AS ?seed) .",
                "?seed wdt:P176 ?manufacturer .",
                "?item wdt:P176 ?manufacturer .",
                f"?item wdt:P137 wd:{op_qid} .",
                "FILTER(?item != ?seed) .",
                *_space_year_filter(y1, y2),
            ],
            "constraints": {
                "seed_qid": seed_qid,
                "seed_ru": seed_ru,
                "bridge": "same_manufacturer_as_seed",
                "operator_qid": op_qid,
                "operator_ru": op_ru,
                "year_from": y1,
                "year_to": y2,
                "expected_gold_count": cnt,
            },
        }

    _space_add_template_rows(
        records,
        "spacecraft_L5_same_manufacturer_as_seed+operator+launch_period",
        sparql,
        make_same_manu_seed_op_year,
    )

    sparql = f"""
    SELECT ?seed ?seedLabelRu ?launch_vehicle ?launch_vehicleLabelRu ?y1 ?y2
           (COUNT(DISTINCT ?item) AS ?cnt)
    WHERE {{
      VALUES (?y1 ?y2) {{
{years_narrow}
      }}
      ?seed wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ;
            wdt:P361 ?program .
      ?item wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ;
            wdt:P361 ?program ;
            wdt:P375 ?launch_vehicle ;
            wdt:P619 ?launch_date .
      FILTER(?item != ?seed)

      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)

      ?seed rdfs:label ?seedLabelRu FILTER(LANG(?seedLabelRu) = "ru") .
      ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") .
      ?launch_vehicle rdfs:label ?launch_vehicleLabelRu FILTER(LANG(?launch_vehicleLabelRu) = "ru") .
    }}
    GROUP BY ?seed ?seedLabelRu ?launch_vehicle ?launch_vehicleLabelRu ?y1 ?y2
    HAVING(COUNT(DISTINCT ?item) >= {k} && COUNT(DISTINCT ?item) <= {max_gold})
    LIMIT 300
    """

    def make_same_program_seed_lv_year(r):
        seed_qid = uri_to_qid(r.get("seed", ""))
        lv_qid = uri_to_qid(r.get("launch_vehicle", ""))
        seed_ru = r.get("seedLabelRu")
        lv_ru = r.get("launch_vehicleLabelRu")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        cnt = _space_int(r.get("cnt"))

        if not all([seed_qid, lv_qid, seed_ru, lv_ru, y1, y2]):
            return None

        return {
            "complexity": "L5",
            "expected_count": cnt,
            "query_text_ru": (
                f"{_space_nlg_count_prefix(k)}, относящихся к той же программе/серии, что и аппарат «{seed_ru}», "
                f"запущенных ракетой-носителем «{lv_ru}» в период {y1}–{y2} годов."
            ),
            "where_lines": [
                f"BIND(wd:{seed_qid} AS ?seed) .",
                "?seed wdt:P361 ?program .",
                "?item wdt:P361 ?program .",
                f"?item wdt:P375 wd:{lv_qid} .",
                "FILTER(?item != ?seed) .",
                *_space_year_filter(y1, y2),
            ],
            "constraints": {
                "seed_qid": seed_qid,
                "seed_ru": seed_ru,
                "bridge": "same_program_as_seed",
                "launch_vehicle_qid": lv_qid,
                "launch_vehicle_ru": lv_ru,
                "year_from": y1,
                "year_to": y2,
                "expected_gold_count": cnt,
            },
        }

    _space_add_template_rows(
        records,
        "spacecraft_L5_same_program_as_seed+launch_vehicle+launch_period",
        sparql,
        make_same_program_seed_lv_year,
    )

    sparql = f"""
    SELECT ?seed ?seedLabelRu ?y1 ?y2
           (COUNT(DISTINCT ?item) AS ?cnt)
    WHERE {{
      VALUES (?y1 ?y2) {{
{years_narrow}
      }}
      ?seed wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ;
            wdt:P137 ?operator ;
            wdt:P176 ?manufacturer .
      ?item wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ;
            wdt:P137 ?operator ;
            wdt:P176 ?manufacturer ;
            wdt:P619 ?launch_date .
      FILTER(?item != ?seed)

      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)

      ?seed rdfs:label ?seedLabelRu FILTER(LANG(?seedLabelRu) = "ru") .
      ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") .
    }}
    GROUP BY ?seed ?seedLabelRu ?y1 ?y2
    HAVING(COUNT(DISTINCT ?item) >= {k} && COUNT(DISTINCT ?item) <= {max_gold})
    LIMIT 300
    """

    def make_same_op_manu_seed_year(r):
        seed_qid = uri_to_qid(r.get("seed", ""))
        seed_ru = r.get("seedLabelRu")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        cnt = _space_int(r.get("cnt"))

        if not all([seed_qid, seed_ru, y1, y2]):
            return None

        return {
            "complexity": "L5",
            "expected_count": cnt,
            "query_text_ru": (
                f"{_space_nlg_count_prefix(k)}, у которых совпадают оператор и производитель с аппаратом «{seed_ru}», "
                f"и которые были запущены в период {y1}–{y2} годов."
            ),
            "where_lines": [
                f"BIND(wd:{seed_qid} AS ?seed) .",
                "?seed wdt:P137 ?operator .",
                "?seed wdt:P176 ?manufacturer .",
                "?item wdt:P137 ?operator .",
                "?item wdt:P176 ?manufacturer .",
                "FILTER(?item != ?seed) .",
                *_space_year_filter(y1, y2),
            ],
            "constraints": {
                "seed_qid": seed_qid,
                "seed_ru": seed_ru,
                "bridge": "same_operator_and_manufacturer_as_seed",
                "year_from": y1,
                "year_to": y2,
                "expected_gold_count": cnt,
            },
        }

    _space_add_template_rows(
        records,
        "spacecraft_L5_same_operator+same_manufacturer_as_seed+launch_period",
        sparql,
        make_same_op_manu_seed_year,
    )

    df = pd.DataFrame(records).drop_duplicates(subset=["query_text_ru"]).reset_index(drop=True)
    print(f"[Spacecraft] L5 viable templates:", len(df))
    return df


def _get_spacecraft_template_pool(complexity: str) -> pd.DataFrame:
    if complexity in SPACECRAFT_V3_TEMPLATE_CACHE:
        return SPACECRAFT_V3_TEMPLATE_CACHE[complexity]

    if complexity == "L3":
        df = _build_spacecraft_l3_templates()
    elif complexity == "L4":
        df = _build_spacecraft_l4_templates()
    elif complexity == "L5":
        df = _build_spacecraft_l5_templates()
    else:
        raise ValueError(complexity)

    SPACECRAFT_V3_TEMPLATE_CACHE[complexity] = df
    return df


def _spacecraft_generate_from_viable_template(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 80,
) -> BenchmarkExample:
    k = SPACECRAFT_REQUESTED_COUNT[complexity]
    pool = _get_spacecraft_template_pool(complexity)

    if pool.empty:
        raise RuntimeError(f"Spacecraft {complexity}: no viable templates were found.")

    last_debug = None

    for _ in range(max_attempts):
        row = pool.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]

        where = json.loads(row["where_json"])
        constraints = json.loads(row["constraints_json"])
        template_id = row["template_id"]
        query_text_ru = row["query_text_ru"]

        try:
            sparql, items = _run_spacecraft_query(where, limit=SPACECRAFT_QUERY_LIMIT)
        except Exception as e:
            last_debug = f"{template_id}: SPARQL error {type(e).__name__}: {e}"
            continue

        n = len(items)
        last_debug = f"{template_id}: n={n}, expected={row.get('expected_count')}"

        if not _space_accept_count(complexity, n, k):
            continue

        ask = build_ask_validator(Q_SPACECRAFT, where, item_var="item")

        return BenchmarkExample(
            id=f"spacecraft_{complexity.lower()}_{idx:04d}",
            domain="spacecraft",
            complexity=complexity,
            query_text_ru=query_text_ru,
            constraints=constraints,
            requested_count=k,
            gold_answer_qids=[q for q, _ in items],
            gold_answer_labels_ru=[l for _, l in items],
            sparql_query=sparql,
            created_at=utc_now_z(),
            is_advanced=(complexity in {"L4", "L5"}),
            template_id=template_id,
            template_family="spacecraft_v3_viable_templates",
            gold_truncated=(len(items) >= SPACECRAFT_QUERY_LIMIT),
            ask_validator_sparql=ask,
        )

    raise RuntimeError(
        f"Spacecraft {complexity}: failed after {max_attempts} viable-template attempts. "
        f"Last: {last_debug}"
    )


def generate_spacecraft_example(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 120,
) -> BenchmarkExample:
    if complexity in {"L1", "L2"}:
        return _SPACECRAFT_BASE_GENERATOR(complexity, idx, rng, max_attempts=max_attempts)

    if complexity in {"L3", "L4", "L5"}:
        return _spacecraft_generate_from_viable_template(
            complexity=complexity,
            idx=idx,
            rng=rng,
            max_attempts=max_attempts,
        )

    raise ValueError(complexity)


print("✅ Spacecraft HOTFIX loaded: L3-L5 now use viable co-occurrence template pools.")

# Spacecraft L5 HOTFIX: robust seed-bridge L5 patterns
#

import random
import re
import pandas as pd

if "_SPACECRAFT_GENERATOR_BEFORE_L5_HOTFIX" not in globals():
    _SPACECRAFT_GENERATOR_BEFORE_L5_HOTFIX = generate_spacecraft_example

SPACECRAFT_REQUESTED_COUNT["L5"] = 1
SPACECRAFT_MAX_GOLD_BY_COMPLEXITY["L5"] = 220

SPACECRAFT_L5_QUERY_LIMIT = min(globals().get("SPACECRAFT_QUERY_LIMIT", 700), 700)


def _space_l5_qid(x):
    if not x:
        return None
    try:
        return uri_to_qid(str(x))
    except Exception:
        return None


def _space_l5_good_qid(x):
    return isinstance(x, str) and x.startswith("Q")


def _space_l5_good_label(x):
    if x is None:
        return False
    try:
        if pd.isna(x):
            return False
    except Exception:
        pass
    return isinstance(x, str) and len(x.strip()) > 0


def _space_l5_parse_year(x):
    if x is None:
        return None
    s = str(x)
    m = re.search(r"(-?\d{3,4})", s)
    if not m:
        return None
    y = int(m.group(1))
    if 1957 <= y <= 2026:
        return y
    return None


def _space_l5_year_window(center_year, half_width=14):
    if center_year is None:
        return None, None
    y1 = max(1957, int(center_year) - int(half_width))
    y2 = min(2026, int(center_year) + int(half_width))
    return y1, y2


def _build_spacecraft_l5_seed_pool(limit=1200) -> pd.DataFrame:
    """
    Лёгкий seed-pool для L5.
    Берём реальные космические аппараты с русскими label'ами и их свойства.
    Важно: здесь нет тяжёлого GROUP BY по всему Wikidata.
    """
    sparql = f"""
    SELECT DISTINCT
      ?seed ?seedLabelRu
      ?type ?typeLabelRu
      ?operator ?operatorLabelRu
      ?manufacturer ?manufacturerLabelRu
      ?launch_vehicle ?launch_vehicleLabelRu
      ?program ?programLabelRu
      ?launch_date
    WHERE {{
      ?seed wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} .
      ?seed rdfs:label ?seedLabelRu FILTER(LANG(?seedLabelRu) = "ru") .

      OPTIONAL {{
        ?seed wdt:P31 ?type .
        ?type wdt:P279* wd:{Q_SPACECRAFT} .
        FILTER(?type != wd:{Q_SPACECRAFT})
        ?type rdfs:label ?typeLabelRu FILTER(LANG(?typeLabelRu) = "ru") .
      }}

      OPTIONAL {{
        ?seed wdt:P137 ?operator .
        ?operator rdfs:label ?operatorLabelRu FILTER(LANG(?operatorLabelRu) = "ru") .
      }}

      OPTIONAL {{
        ?seed wdt:P176 ?manufacturer .
        ?manufacturer rdfs:label ?manufacturerLabelRu FILTER(LANG(?manufacturerLabelRu) = "ru") .
      }}

      OPTIONAL {{
        ?seed wdt:P375 ?launch_vehicle .
        ?launch_vehicle rdfs:label ?launch_vehicleLabelRu FILTER(LANG(?launch_vehicleLabelRu) = "ru") .
      }}

      OPTIONAL {{
        ?seed wdt:P361 ?program .
        ?program rdfs:label ?programLabelRu FILTER(LANG(?programLabelRu) = "ru") .
      }}

      OPTIONAL {{
        ?seed wdt:P619 ?launch_date .
      }}

      FILTER(
        BOUND(?manufacturer) ||
        BOUND(?operator) ||
        BOUND(?launch_vehicle) ||
        BOUND(?program)
      )
    }}
    LIMIT {int(limit)}
    """

    rows = rows_from_select(wd.sparql_select(sparql))
    data = []

    for r in rows:
        seed_qid = _space_l5_qid(r.get("seed"))
        seed_ru = r.get("seedLabelRu")

        if not (_space_l5_good_qid(seed_qid) and _space_l5_good_label(seed_ru)):
            continue

        data.append({
            "seed_qid": seed_qid,
            "seed_ru": seed_ru,

            "type_qid": _space_l5_qid(r.get("type")),
            "type_ru": r.get("typeLabelRu"),

            "operator_qid": _space_l5_qid(r.get("operator")),
            "operator_ru": r.get("operatorLabelRu"),

            "manufacturer_qid": _space_l5_qid(r.get("manufacturer")),
            "manufacturer_ru": r.get("manufacturerLabelRu"),

            "launch_vehicle_qid": _space_l5_qid(r.get("launch_vehicle")),
            "launch_vehicle_ru": r.get("launch_vehicleLabelRu"),

            "program_qid": _space_l5_qid(r.get("program")),
            "program_ru": r.get("programLabelRu"),

            "launch_year": _space_l5_parse_year(r.get("launch_date")),
        })

    df = pd.DataFrame(data).drop_duplicates().reset_index(drop=True)
    print("[Spacecraft L5] seed pool size:", len(df))
    return df


spacecraft_l5_seed_df = load_or_build_pool(
    "spacecraft_l5_seed_pool_robust_v1",
    lambda: _build_spacecraft_l5_seed_pool(limit=1200),
)


def _space_l5_row_has(row, fields):
    for qid_col, label_col in fields:
        if not _space_l5_good_qid(row.get(qid_col)):
            return False
        if not _space_l5_good_label(row.get(label_col)):
            return False
    return True


def _space_l5_sample_row(fields, rng, require_year=False, max_tries=600):
    if spacecraft_l5_seed_df.empty:
        raise RuntimeError("Spacecraft L5 seed pool is empty.")

    n = len(spacecraft_l5_seed_df)

    for _ in range(max_tries):
        row = spacecraft_l5_seed_df.iloc[rng.randrange(n)].to_dict()

        if not _space_l5_row_has(row, fields):
            continue

        if require_year and row.get("launch_year") is None:
            continue

        return row

    return None


def _space_l5_make_candidate(rng: random.Random):
    """
    L5-паттерны:
      1. тот же производитель, что у seed + период;
      2. тот же оператор, что у seed + тип;
      3. та же ракета-носитель, что у seed + период;
      4. та же программа/серия, что у seed;
      5. тот же производитель и оператор, что у seed.
    """
    k = SPACECRAFT_REQUESTED_COUNT["L5"]

    def p_same_manufacturer_period():
        row = _space_l5_sample_row(
            [("manufacturer_qid", "manufacturer_ru")],
            rng,
            require_year=True,
        )
        if not row:
            return None

        seed_qid, seed_ru = row["seed_qid"], row["seed_ru"]
        y1, y2 = _space_l5_year_window(row["launch_year"], half_width=18)

        return {
            "template_id": "spacecraft_L5_same_manufacturer_as_seed+launch_period",
            "query_text_ru": (
                f"{_space_nlg_count_prefix(k)}, произведённых той же организацией, что и аппарат «{seed_ru}», "
                f"и запущенных в период {y1}–{y2} годов. Не включай сам аппарат «{seed_ru}» в ответ."
            ),
            "where_lines": [
                f"BIND(wd:{seed_qid} AS ?seed) .",
                "?seed wdt:P176 ?manufacturer .",
                "?item wdt:P176 ?manufacturer .",
                "FILTER(?item != ?seed) .",
                *_space_year_filter(y1, y2),
            ],
            "constraints": {
                "seed_qid": seed_qid,
                "seed_ru": seed_ru,
                "bridge": "same_manufacturer_as_seed",
                "year_from": y1,
                "year_to": y2,
            },
        }

    def p_same_operator_type():
        row = _space_l5_sample_row(
            [("operator_qid", "operator_ru"), ("type_qid", "type_ru")],
            rng,
            require_year=False,
        )
        if not row:
            return None

        seed_qid, seed_ru = row["seed_qid"], row["seed_ru"]
        type_qid, type_ru = row["type_qid"], row["type_ru"]

        return {
            "template_id": "spacecraft_L5_same_operator_as_seed+type",
            "query_text_ru": (
                f"{_space_nlg_count_prefix(k)} типа «{type_ru}», оператор которых совпадает с оператором аппарата "
                f"«{seed_ru}». Не включай сам аппарат «{seed_ru}» в ответ."
            ),
            "where_lines": [
                f"BIND(wd:{seed_qid} AS ?seed) .",
                "?seed wdt:P137 ?operator .",
                "?item wdt:P137 ?operator .",
                f"?item wdt:P31/wdt:P279* wd:{type_qid} .",
                "FILTER(?item != ?seed) .",
            ],
            "constraints": {
                "seed_qid": seed_qid,
                "seed_ru": seed_ru,
                "bridge": "same_operator_as_seed",
                "type_qid": type_qid,
                "type_ru": type_ru,
            },
        }

    def p_same_launch_vehicle_period():
        row = _space_l5_sample_row(
            [("launch_vehicle_qid", "launch_vehicle_ru")],
            rng,
            require_year=True,
        )
        if not row:
            return None

        seed_qid, seed_ru = row["seed_qid"], row["seed_ru"]
        y1, y2 = _space_l5_year_window(row["launch_year"], half_width=20)

        return {
            "template_id": "spacecraft_L5_same_launch_vehicle_as_seed+launch_period",
            "query_text_ru": (
                f"{_space_nlg_count_prefix(k)}, запущенных той же ракетой-носителем, что и аппарат «{seed_ru}», "
                f"в период {y1}–{y2} годов. Не включай сам аппарат «{seed_ru}» в ответ."
            ),
            "where_lines": [
                f"BIND(wd:{seed_qid} AS ?seed) .",
                "?seed wdt:P375 ?launch_vehicle .",
                "?item wdt:P375 ?launch_vehicle .",
                "FILTER(?item != ?seed) .",
                *_space_year_filter(y1, y2),
            ],
            "constraints": {
                "seed_qid": seed_qid,
                "seed_ru": seed_ru,
                "bridge": "same_launch_vehicle_as_seed",
                "year_from": y1,
                "year_to": y2,
            },
        }

    def p_same_program():
        row = _space_l5_sample_row(
            [("program_qid", "program_ru")],
            rng,
            require_year=False,
        )
        if not row:
            return None

        seed_qid, seed_ru = row["seed_qid"], row["seed_ru"]

        return {
            "template_id": "spacecraft_L5_same_program_as_seed",
            "query_text_ru": (
                f"{_space_nlg_count_prefix(k)}, относящихся к той же программе/серии, что и аппарат «{seed_ru}». "
                f"Не включай сам аппарат «{seed_ru}» в ответ."
            ),
            "where_lines": [
                f"BIND(wd:{seed_qid} AS ?seed) .",
                "?seed wdt:P361 ?program .",
                "?item wdt:P361 ?program .",
                "FILTER(?item != ?seed) .",
            ],
            "constraints": {
                "seed_qid": seed_qid,
                "seed_ru": seed_ru,
                "bridge": "same_program_as_seed",
            },
        }

    def p_same_manufacturer_operator():
        row = _space_l5_sample_row(
            [("manufacturer_qid", "manufacturer_ru"), ("operator_qid", "operator_ru")],
            rng,
            require_year=False,
        )
        if not row:
            return None

        seed_qid, seed_ru = row["seed_qid"], row["seed_ru"]

        return {
            "template_id": "spacecraft_L5_same_manufacturer+same_operator_as_seed",
            "query_text_ru": (
                f"{_space_nlg_count_prefix(k)}, у которых совпадают производитель и оператор с аппаратом "
                f"«{seed_ru}». Не включай сам аппарат «{seed_ru}» в ответ."
            ),
            "where_lines": [
                f"BIND(wd:{seed_qid} AS ?seed) .",
                "?seed wdt:P176 ?manufacturer .",
                "?seed wdt:P137 ?operator .",
                "?item wdt:P176 ?manufacturer .",
                "?item wdt:P137 ?operator .",
                "FILTER(?item != ?seed) .",
            ],
            "constraints": {
                "seed_qid": seed_qid,
                "seed_ru": seed_ru,
                "bridge": "same_manufacturer_and_operator_as_seed",
            },
        }

    patterns = [
        p_same_manufacturer_period,
        p_same_operator_type,
        p_same_launch_vehicle_period,
        p_same_program,
        p_same_manufacturer_operator,
    ]

    for _ in range(80):
        cand = rng.choice(patterns)()
        if cand is not None:
            return cand

    raise RuntimeError("Spacecraft L5: could not build candidate from seed pool.")


def _spacecraft_generate_l5_robust(
    idx: int,
    rng: random.Random,
    max_attempts: int = 250,
) -> BenchmarkExample:
    complexity = "L5"
    k = SPACECRAFT_REQUESTED_COUNT["L5"]
    last_debug = None

    for _ in range(max_attempts):
        try:
            cand = _space_l5_make_candidate(rng)
        except Exception as e:
            last_debug = f"candidate error: {type(e).__name__}: {e}"
            continue

        try:
            sparql, items = _run_spacecraft_query(
                cand["where_lines"],
                limit=SPACECRAFT_L5_QUERY_LIMIT,
            )
        except Exception as e:
            last_debug = f"{cand.get('template_id')}: SPARQL error {type(e).__name__}: {e}"
            continue

        n = len(items)
        last_debug = f"{cand.get('template_id')}: n={n}"

        if n < k:
            continue

        if n > SPACECRAFT_MAX_GOLD_BY_COMPLEXITY["L5"]:
            continue

        ask = build_ask_validator(Q_SPACECRAFT, cand["where_lines"], item_var="item")

        return BenchmarkExample(
            id=f"spacecraft_l5_{idx:04d}",
            domain="spacecraft",
            complexity="L5",
            query_text_ru=cand["query_text_ru"],
            constraints=cand["constraints"],
            requested_count=k,
            gold_answer_qids=[q for q, _ in items],
            gold_answer_labels_ru=[l for _, l in items],
            sparql_query=sparql,
            created_at=utc_now_z(),
            is_advanced=True,
            template_id=cand["template_id"],
            template_family="spacecraft_l5_robust_seed_bridge",
            gold_truncated=(len(items) >= SPACECRAFT_L5_QUERY_LIMIT),
            ask_validator_sparql=ask,
        )

    raise RuntimeError(
        f"Spacecraft L5 robust: failed after {max_attempts} attempts. Last: {last_debug}"
    )


def generate_spacecraft_example(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 120,
) -> BenchmarkExample:
    if complexity == "L5":
        return _spacecraft_generate_l5_robust(
            idx=idx,
            rng=rng,
            max_attempts=max(max_attempts, 250),
        )

    return _SPACECRAFT_GENERATOR_BEFORE_L5_HOTFIX(
        complexity,
        idx,
        rng,
        max_attempts=max_attempts,
    )


print("✅ Spacecraft L5 HOTFIX loaded: robust seed-bridge L5 patterns.")

# Hard L5 spacecraft templates use multi-seed bridge constraints.

import random
import re
import pandas as pd

if "_SPACECRAFT_GENERATOR_BEFORE_L5_HARD_HOTFIX" not in globals():
    _SPACECRAFT_GENERATOR_BEFORE_L5_HARD_HOTFIX = generate_spacecraft_example

SPACECRAFT_REQUESTED_COUNT["L5"] = 2
SPACECRAFT_MAX_GOLD_BY_COMPLEXITY["L5"] = 120

SPACECRAFT_L5_HARD_QUERY_LIMIT = min(globals().get("SPACECRAFT_QUERY_LIMIT", 700), 700)

_SPACECRAFT_L5_REF_CACHE = {}


def _space_l5h_qid(x):
    if not x:
        return None
    try:
        return uri_to_qid(str(x))
    except Exception:
        return None


def _space_l5h_good_qid(x):
    return isinstance(x, str) and x.startswith("Q")


def _space_l5h_good_label(x):
    if x is None:
        return False
    try:
        if pd.isna(x):
            return False
    except Exception:
        pass
    return isinstance(x, str) and len(x.strip()) > 0


def _space_l5h_parse_year(x):
    if x is None:
        return None
    s = str(x)
    m = re.search(r"(-?\d{3,4})", s)
    if not m:
        return None
    y = int(m.group(1))
    if 1957 <= y <= 2026:
        return y
    return None


def _space_l5h_year_window(center_year, half_width=14):
    if center_year is None:
        return None, None
    y1 = max(1957, int(center_year) - int(half_width))
    y2 = min(2026, int(center_year) + int(half_width))
    return y1, y2


def _build_spacecraft_l5_hard_seed_pool(limit=1500) -> pd.DataFrame:
    """
    Seed-pool для сложных L5.
    Берём реальные аппараты с русскими названиями и свойствами.
    Без тяжёлого GROUP BY.
    """
    sparql = f"""
    SELECT DISTINCT
      ?seed ?seedLabelRu
      ?type ?typeLabelRu
      ?operator ?operatorLabelRu
      ?manufacturer ?manufacturerLabelRu
      ?launch_vehicle ?launch_vehicleLabelRu
      ?program ?programLabelRu
      ?launch_date
    WHERE {{
      ?seed wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} .
      ?seed rdfs:label ?seedLabelRu FILTER(LANG(?seedLabelRu) = "ru") .

      OPTIONAL {{
        ?seed wdt:P31 ?type .
        ?type wdt:P279* wd:{Q_SPACECRAFT} .
        FILTER(?type != wd:{Q_SPACECRAFT})
        ?type rdfs:label ?typeLabelRu FILTER(LANG(?typeLabelRu) = "ru") .
      }}

      OPTIONAL {{
        ?seed wdt:P137 ?operator .
        ?operator rdfs:label ?operatorLabelRu FILTER(LANG(?operatorLabelRu) = "ru") .
      }}

      OPTIONAL {{
        ?seed wdt:P176 ?manufacturer .
        ?manufacturer rdfs:label ?manufacturerLabelRu FILTER(LANG(?manufacturerLabelRu) = "ru") .
      }}

      OPTIONAL {{
        ?seed wdt:P375 ?launch_vehicle .
        ?launch_vehicle rdfs:label ?launch_vehicleLabelRu FILTER(LANG(?launch_vehicleLabelRu) = "ru") .
      }}

      OPTIONAL {{
        ?seed wdt:P361 ?program .
        ?program rdfs:label ?programLabelRu FILTER(LANG(?programLabelRu) = "ru") .
      }}

      OPTIONAL {{
        ?seed wdt:P619 ?launch_date .
      }}

      FILTER(
        BOUND(?manufacturer) ||
        BOUND(?operator) ||
        BOUND(?launch_vehicle) ||
        BOUND(?program) ||
        BOUND(?type)
      )
    }}
    LIMIT {int(limit)}
    """

    rows = rows_from_select(wd.sparql_select(sparql))
    data = []

    for r in rows:
        seed_qid = _space_l5h_qid(r.get("seed"))
        seed_ru = r.get("seedLabelRu")

        if not (_space_l5h_good_qid(seed_qid) and _space_l5h_good_label(seed_ru)):
            continue

        data.append({
            "seed_qid": seed_qid,
            "seed_ru": seed_ru,

            "type_qid": _space_l5h_qid(r.get("type")),
            "type_ru": r.get("typeLabelRu"),

            "operator_qid": _space_l5h_qid(r.get("operator")),
            "operator_ru": r.get("operatorLabelRu"),

            "manufacturer_qid": _space_l5h_qid(r.get("manufacturer")),
            "manufacturer_ru": r.get("manufacturerLabelRu"),

            "launch_vehicle_qid": _space_l5h_qid(r.get("launch_vehicle")),
            "launch_vehicle_ru": r.get("launch_vehicleLabelRu"),

            "program_qid": _space_l5h_qid(r.get("program")),
            "program_ru": r.get("programLabelRu"),

            "launch_year": _space_l5h_parse_year(r.get("launch_date")),
        })

    df = pd.DataFrame(data).drop_duplicates().reset_index(drop=True)

    interesting_tokens = [
        "Марс", "Луна", "Венера", "Юпитер", "Сатурн",
        "Вояджер", "Пионер", "Галилео", "Кассини",
        "Хаббл", "Спитцер", "Кеплер", "Союз", "Прогресс",
        "Аполлон", "Landsat", "Mars", "Voyager", "Galileo",
        "Cassini", "Hubble", "Спутник", "Интеркосмос",
        "Мир", "МКС", "Офек", "GPS", "ГЛОНАСС",
    ]

    def weight_row(row):
        text = " ".join(str(row.get(c, "")) for c in [
            "seed_ru", "type_ru", "operator_ru", "manufacturer_ru",
            "launch_vehicle_ru", "program_ru",
        ])
        w = 1.0
        if any(t in text for t in interesting_tokens):
            w += 3.0
        filled = sum(
            _space_l5h_good_qid(row.get(c))
            for c in ["type_qid", "operator_qid", "manufacturer_qid", "launch_vehicle_qid", "program_qid"]
        )
        w += 0.4 * filled
        if row.get("launch_year") is not None:
            w += 0.5
        return w

    if len(df):
        df["sample_weight"] = df.apply(weight_row, axis=1)

    print("[Spacecraft L5 hard] seed pool size:", len(df))
    return df


spacecraft_l5_hard_seed_df = load_or_build_pool(
    "spacecraft_l5_hard_seed_pool_v1",
    lambda: _build_spacecraft_l5_hard_seed_pool(limit=1500),
)


def _space_l5h_row_has(row, fields):
    for qid_col, label_col in fields:
        if not _space_l5h_good_qid(row.get(qid_col)):
            return False
        if not _space_l5h_good_label(row.get(label_col)):
            return False
    return True


def _space_l5h_sample_target(fields, rng, require_year=True, max_tries=800):
    """
    Берём target-строку, у которой есть нужные свойства.
    Из этих свойств потом строим seed-bridge запрос.
    """
    if spacecraft_l5_hard_seed_df.empty:
        raise RuntimeError("Spacecraft L5 hard seed pool is empty.")

    df = spacecraft_l5_hard_seed_df

    for _ in range(max_tries):
        if "sample_weight" in df.columns:
            row = df.sample(
                1,
                weights=df["sample_weight"],
                random_state=rng.randint(0, 10**9),
            ).iloc[0].to_dict()
        else:
            row = df.iloc[rng.randrange(len(df))].to_dict()

        if not _space_l5h_row_has(row, fields):
            continue

        if require_year and row.get("launch_year") is None:
            continue

        return row

    return None


def _space_l5h_find_reference_item(prop_pid, value_qid, exclude_qids, rng, limit=40):
    """
    Находит другой космический аппарат, у которого есть то же свойство.
    Например:
      prop_pid=P176, value_qid=Q..., значит ищем аппарат с тем же производителем.
    Это нужно, чтобы в запросе были разные seed A / seed B, а не просто явные константы.
    """
    exclude_qids = sorted({q for q in exclude_qids if _space_l5h_good_qid(q)})
    cache_key = (prop_pid, value_qid, tuple(exclude_qids))

    if cache_key in _SPACECRAFT_L5_REF_CACHE:
        candidates = _SPACECRAFT_L5_REF_CACHE[cache_key]
    else:
        filters = []
        for q in exclude_qids:
            filters.append(f"?x != wd:{q}")
        filter_line = ""
        if filters:
            filter_line = "FILTER(" + " && ".join(filters) + ") ."

        sparql = f"""
        SELECT DISTINCT ?x ?xLabelRu WHERE {{
          ?x wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ;
             wdt:{prop_pid} wd:{value_qid} .
          {filter_line}
          ?x rdfs:label ?xLabelRu FILTER(LANG(?xLabelRu) = "ru") .
        }}
        LIMIT {int(limit)}
        """

        try:
            rows = rows_from_select(wd.sparql_select(sparql))
        except Exception:
            rows = []

        candidates = []
        for r in rows:
            qid = _space_l5h_qid(r.get("x"))
            lbl = r.get("xLabelRu")
            if _space_l5h_good_qid(qid) and _space_l5h_good_label(lbl):
                candidates.append((qid, lbl))

        _SPACECRAFT_L5_REF_CACHE[cache_key] = candidates

    if not candidates:
        return None

    return rng.choice(candidates)


def _space_l5h_make_candidate(rng: random.Random):
    """
    Сложные L5-кандидаты:
      1. same manufacturer as A + same operator as B + launch period
      2. same program as A + same manufacturer as B + launch period
      3. same launch vehicle as A + same operator as B + launch period
      4. same type as A + same manufacturer as B + launch period
      5. same launch vehicle as A + same manufacturer as B + same operator as C
    """
    k = SPACECRAFT_REQUESTED_COUNT["L5"]

    def p_same_manufacturer_A_same_operator_B_period():
        row = _space_l5h_sample_target(
            [("manufacturer_qid", "manufacturer_ru"), ("operator_qid", "operator_ru")],
            rng,
            require_year=True,
        )
        if not row:
            return None

        target_qid = row["seed_qid"]
        y1, y2 = _space_l5h_year_window(row["launch_year"], half_width=16)

        seed_a = _space_l5h_find_reference_item(
            "P176",
            row["manufacturer_qid"],
            exclude_qids=[target_qid],
            rng=rng,
        )
        seed_b = _space_l5h_find_reference_item(
            "P137",
            row["operator_qid"],
            exclude_qids=[target_qid, seed_a[0] if seed_a else None],
            rng=rng,
        )

        if not seed_a or not seed_b:
            return None

        seed_a_qid, seed_a_ru = seed_a
        seed_b_qid, seed_b_ru = seed_b

        return {
            "template_id": "spacecraft_L5_same_manufacturer_A+same_operator_B+period",
            "query_text_ru": (
                f"Назови {k} космических аппарата, которые произведены той же организацией, что и «{seed_a_ru}», "
                f"оператор которых совпадает с оператором аппарата «{seed_b_ru}», "
                f"и которые были запущены в период {y1}–{y2} годов. "
                f"Не включай аппараты «{seed_a_ru}» и «{seed_b_ru}» в ответ."
            ),
            "where_lines": [
                f"BIND(wd:{seed_a_qid} AS ?seedA) .",
                f"BIND(wd:{seed_b_qid} AS ?seedB) .",
                "?seedA wdt:P176 ?manufacturer .",
                "?seedB wdt:P137 ?operator .",
                "?item wdt:P176 ?manufacturer .",
                "?item wdt:P137 ?operator .",
                f"FILTER(?item != wd:{seed_a_qid} && ?item != wd:{seed_b_qid}) .",
                *_space_year_filter(y1, y2),
            ],
            "constraints": {
                "seed_manufacturer_qid": seed_a_qid,
                "seed_manufacturer_ru": seed_a_ru,
                "seed_operator_qid": seed_b_qid,
                "seed_operator_ru": seed_b_ru,
                "bridge": "same_manufacturer_A_and_same_operator_B",
                "year_from": y1,
                "year_to": y2,
                "target_probe_qid": target_qid,
                "target_probe_ru": row["seed_ru"],
            },
        }

    def p_same_program_A_same_manufacturer_B_period():
        row = _space_l5h_sample_target(
            [("program_qid", "program_ru"), ("manufacturer_qid", "manufacturer_ru")],
            rng,
            require_year=True,
        )
        if not row:
            return None

        target_qid = row["seed_qid"]
        y1, y2 = _space_l5h_year_window(row["launch_year"], half_width=20)

        seed_a = _space_l5h_find_reference_item(
            "P361",
            row["program_qid"],
            exclude_qids=[target_qid],
            rng=rng,
        )
        seed_b = _space_l5h_find_reference_item(
            "P176",
            row["manufacturer_qid"],
            exclude_qids=[target_qid, seed_a[0] if seed_a else None],
            rng=rng,
        )

        if not seed_a or not seed_b:
            return None

        seed_a_qid, seed_a_ru = seed_a
        seed_b_qid, seed_b_ru = seed_b

        return {
            "template_id": "spacecraft_L5_same_program_A+same_manufacturer_B+period",
            "query_text_ru": (
                f"Назови {k} космических аппарата, относящихся к той же программе/серии, что и «{seed_a_ru}», "
                f"и произведённых той же организацией, что и «{seed_b_ru}», "
                f"если они были запущены в период {y1}–{y2} годов. "
                f"Не включай аппараты «{seed_a_ru}» и «{seed_b_ru}» в ответ."
            ),
            "where_lines": [
                f"BIND(wd:{seed_a_qid} AS ?seedA) .",
                f"BIND(wd:{seed_b_qid} AS ?seedB) .",
                "?seedA wdt:P361 ?program .",
                "?seedB wdt:P176 ?manufacturer .",
                "?item wdt:P361 ?program .",
                "?item wdt:P176 ?manufacturer .",
                f"FILTER(?item != wd:{seed_a_qid} && ?item != wd:{seed_b_qid}) .",
                *_space_year_filter(y1, y2),
            ],
            "constraints": {
                "seed_program_qid": seed_a_qid,
                "seed_program_ru": seed_a_ru,
                "seed_manufacturer_qid": seed_b_qid,
                "seed_manufacturer_ru": seed_b_ru,
                "bridge": "same_program_A_and_same_manufacturer_B",
                "year_from": y1,
                "year_to": y2,
                "target_probe_qid": target_qid,
                "target_probe_ru": row["seed_ru"],
            },
        }

    def p_same_launch_vehicle_A_same_operator_B_period():
        row = _space_l5h_sample_target(
            [("launch_vehicle_qid", "launch_vehicle_ru"), ("operator_qid", "operator_ru")],
            rng,
            require_year=True,
        )
        if not row:
            return None

        target_qid = row["seed_qid"]
        y1, y2 = _space_l5h_year_window(row["launch_year"], half_width=18)

        seed_a = _space_l5h_find_reference_item(
            "P375",
            row["launch_vehicle_qid"],
            exclude_qids=[target_qid],
            rng=rng,
        )
        seed_b = _space_l5h_find_reference_item(
            "P137",
            row["operator_qid"],
            exclude_qids=[target_qid, seed_a[0] if seed_a else None],
            rng=rng,
        )

        if not seed_a or not seed_b:
            return None

        seed_a_qid, seed_a_ru = seed_a
        seed_b_qid, seed_b_ru = seed_b

        return {
            "template_id": "spacecraft_L5_same_launch_vehicle_A+same_operator_B+period",
            "query_text_ru": (
                f"Назови {k} космических аппарата, запущенных той же ракетой-носителем, что и «{seed_a_ru}», "
                f"оператор которых совпадает с оператором аппарата «{seed_b_ru}», "
                f"и которые были запущены в период {y1}–{y2} годов. "
                f"Не включай аппараты «{seed_a_ru}» и «{seed_b_ru}» в ответ."
            ),
            "where_lines": [
                f"BIND(wd:{seed_a_qid} AS ?seedA) .",
                f"BIND(wd:{seed_b_qid} AS ?seedB) .",
                "?seedA wdt:P375 ?launch_vehicle .",
                "?seedB wdt:P137 ?operator .",
                "?item wdt:P375 ?launch_vehicle .",
                "?item wdt:P137 ?operator .",
                f"FILTER(?item != wd:{seed_a_qid} && ?item != wd:{seed_b_qid}) .",
                *_space_year_filter(y1, y2),
            ],
            "constraints": {
                "seed_launch_vehicle_qid": seed_a_qid,
                "seed_launch_vehicle_ru": seed_a_ru,
                "seed_operator_qid": seed_b_qid,
                "seed_operator_ru": seed_b_ru,
                "bridge": "same_launch_vehicle_A_and_same_operator_B",
                "year_from": y1,
                "year_to": y2,
                "target_probe_qid": target_qid,
                "target_probe_ru": row["seed_ru"],
            },
        }

    def p_same_type_A_same_manufacturer_B_period():
        row = _space_l5h_sample_target(
            [("type_qid", "type_ru"), ("manufacturer_qid", "manufacturer_ru")],
            rng,
            require_year=True,
        )
        if not row:
            return None

        target_qid = row["seed_qid"]
        y1, y2 = _space_l5h_year_window(row["launch_year"], half_width=18)

        seed_a = _space_l5h_find_reference_item(
            "P31",
            row["type_qid"],
            exclude_qids=[target_qid],
            rng=rng,
        )
        seed_b = _space_l5h_find_reference_item(
            "P176",
            row["manufacturer_qid"],
            exclude_qids=[target_qid, seed_a[0] if seed_a else None],
            rng=rng,
        )

        if not seed_a or not seed_b:
            return None

        seed_a_qid, seed_a_ru = seed_a
        seed_b_qid, seed_b_ru = seed_b

        return {
            "template_id": "spacecraft_L5_same_type_A+same_manufacturer_B+period",
            "query_text_ru": (
                f"Назови {k} космических аппарата того же типа, что и «{seed_a_ru}», "
                f"произведённых той же организацией, что и «{seed_b_ru}», "
                f"и запущенных в период {y1}–{y2} годов. "
                f"Не включай аппараты «{seed_a_ru}» и «{seed_b_ru}» в ответ."
            ),
            "where_lines": [
                f"BIND(wd:{seed_a_qid} AS ?seedA) .",
                f"BIND(wd:{seed_b_qid} AS ?seedB) .",
                "?seedA wdt:P31 ?type .",
                "?seedB wdt:P176 ?manufacturer .",
                "?item wdt:P31/wdt:P279* ?type .",
                "?item wdt:P176 ?manufacturer .",
                f"FILTER(?item != wd:{seed_a_qid} && ?item != wd:{seed_b_qid}) .",
                *_space_year_filter(y1, y2),
            ],
            "constraints": {
                "seed_type_qid": seed_a_qid,
                "seed_type_ru": seed_a_ru,
                "seed_manufacturer_qid": seed_b_qid,
                "seed_manufacturer_ru": seed_b_ru,
                "bridge": "same_type_A_and_same_manufacturer_B",
                "year_from": y1,
                "year_to": y2,
                "target_probe_qid": target_qid,
                "target_probe_ru": row["seed_ru"],
            },
        }

    patterns = [
        p_same_manufacturer_A_same_operator_B_period,
        p_same_program_A_same_manufacturer_B_period,
        p_same_launch_vehicle_A_same_operator_B_period,
        p_same_type_A_same_manufacturer_B_period,
    ]

    for _ in range(120):
        cand = rng.choice(patterns)()
        if cand is not None:
            return cand

    raise RuntimeError("Spacecraft L5 hard: could not build multi-seed candidate.")


def _spacecraft_generate_l5_hard(
    idx: int,
    rng: random.Random,
    max_attempts: int = 350,
) -> BenchmarkExample:
    complexity = "L5"
    k = SPACECRAFT_REQUESTED_COUNT["L5"]
    last_debug = None

    for _ in range(max_attempts):
        try:
            cand = _space_l5h_make_candidate(rng)
        except Exception as e:
            last_debug = f"candidate error: {type(e).__name__}: {e}"
            continue

        try:
            sparql, items = _run_spacecraft_query(
                cand["where_lines"],
                limit=SPACECRAFT_L5_HARD_QUERY_LIMIT,
            )
        except Exception as e:
            last_debug = f"{cand.get('template_id')}: SPARQL error {type(e).__name__}: {e}"
            continue

        n = len(items)
        last_debug = f"{cand.get('template_id')}: n={n}"

        if n < k:
            continue

        if n > SPACECRAFT_MAX_GOLD_BY_COMPLEXITY["L5"]:
            continue

        ask = build_ask_validator(Q_SPACECRAFT, cand["where_lines"], item_var="item")

        return BenchmarkExample(
            id=f"spacecraft_l5_{idx:04d}",
            domain="spacecraft",
            complexity="L5",
            query_text_ru=cand["query_text_ru"],
            constraints=cand["constraints"],
            requested_count=k,
            gold_answer_qids=[q for q, _ in items],
            gold_answer_labels_ru=[l for _, l in items],
            sparql_query=sparql,
            created_at=utc_now_z(),
            is_advanced=True,
            template_id=cand["template_id"],
            template_family="spacecraft_l5_hard_multi_seed_bridge",
            gold_truncated=(len(items) >= SPACECRAFT_L5_HARD_QUERY_LIMIT),
            ask_validator_sparql=ask,
        )

    raise RuntimeError(
        f"Spacecraft L5 hard: failed after {max_attempts} attempts. Last: {last_debug}"
    )


def generate_spacecraft_example(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 120,
) -> BenchmarkExample:
    if complexity == "L5":
        return _spacecraft_generate_l5_hard(
            idx=idx,
            rng=rng,
            max_attempts=max(max_attempts, 350),
        )

    return _SPACECRAFT_GENERATOR_BEFORE_L5_HARD_HOTFIX(
        complexity,
        idx,
        rng,
        max_attempts=max_attempts,
    )


print("✅ Spacecraft L5 HARD HOTFIX loaded: multi-seed bridge L5 patterns.")


✅ Spacecraft FINAL BLOCK loaded: diverse L1–L5 MAIN patterns, no intentional zero-gold generation.
✅ Spacecraft HOTFIX loaded: L3-L5 now use viable co-occurrence template pools.
✅ Spacecraft L5 HOTFIX loaded: robust seed-bridge L5 patterns.
✅ Spacecraft L5 HARD HOTFIX loaded: multi-seed bridge L5 patterns.


<a id="countries-domain"></a>

## 20. Countries domain

Country templates over geography, region, currency, language, population, and historical/current state filters.

In [20]:
import pandas as pd
import random
import json
import datetime as dt
from typing import Dict, List, Tuple, Any, Optional

Q_COUNTRY = ensure_qid("государство", fallback_qid="Q6256")  # country
Q_EU = ensure_qid("Европейский союз", fallback_qid="Q458")
Q_G7 = ensure_qid("Большая семёрка", fallback_qid="Q1762059")

COUNTRY_STATUS_CURRENT = "current"
COUNTRY_STATUS_HISTORICAL = "historical"

pool_countries_continents_ru = load_or_build_pool(
    "countries_continents_ru",
    lambda: build_value_pool_ru(Q_COUNTRY, "P30", "continent", limit=25),
)

pool_countries_languages_ru = load_or_build_pool(
    "countries_languages_ru",
    lambda: build_value_pool_ru(Q_COUNTRY, "P37", "language", limit=80),
)

pool_countries_currencies_ru = load_or_build_pool(
    "countries_currencies_ru",
    lambda: build_value_pool_ru(Q_COUNTRY, "P38", "currency", limit=80),
)

COUNTRY_CONTINENT_ANCHORS = [
    ("Q46", "Европа"),
    ("Q48", "Азия"),
    ("Q15", "Африка"),
    ("Q49", "Северная Америка"),
    ("Q18", "Южная Америка"),
    ("Q55643", "Океания"),
]

COUNTRY_LANG_ANCHORS = [
    ("Q1860", "английский язык"),
    ("Q1321", "испанский язык"),
    ("Q150", "французский язык"),
    ("Q7737", "русский язык"),
    ("Q13955", "арабский язык"),
    ("Q5146", "португальский язык"),
    ("Q188", "немецкий язык"),
]

COUNTRY_CURR_ANCHORS = [
    ("Q4916", "евро"),
    ("Q4917", "доллар США"),
    ("Q25224", "фунт стерлингов"),
    ("Q8146", "японская иена"),
    ("Q208526", "китайский юань"),
    ("Q132643", "российский рубль"),
]

def _norm_lbl(x: Any) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    s = str(x).strip()
    return "" if s.lower() == "nan" else s

def _pick_from_pool_or_anchors(
    pool_df: Optional[pd.DataFrame],
    anchors: List[Tuple[str,str]],
    qid_col: str,
    lbl_col: str,
    rng: random.Random
) -> Tuple[str, str]:
    """
    ВАЖНО: build_value_pool_ru(..., val_name="continent") -> continent_qid + continentLabelRu
           build_value_pool_ru(..., val_name="language")  -> language_qid + languageLabelRu
           build_value_pool_ru(..., val_name="currency")  -> currency_qid + currencyLabelRu
    """
    if isinstance(pool_df, pd.DataFrame) and len(pool_df) > 0 and qid_col in pool_df.columns:
        row = pool_df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
        qid = _norm_lbl(row.get(qid_col, ""))

        lbl = ""
        for c in [lbl_col, lbl_col.replace("_label_ru", "LabelRu"), "LabelRu", "label_ru", "label"]:
            if c and c in pool_df.columns:
                lbl = _norm_lbl(row.get(c, ""))
                if lbl:
                    break

        if qid and lbl:
            return qid, lbl

    qid, lbl = rng.choice(anchors)
    return _norm_lbl(qid), _norm_lbl(lbl)

def _normalize_country_status(country_status: Optional[str]) -> str:
    s = _norm_lbl(country_status).lower()
    aliases = {
        "": COUNTRY_STATUS_CURRENT,
        "current": COUNTRY_STATUS_CURRENT,
        "modern": COUNTRY_STATUS_CURRENT,
        "present": COUNTRY_STATUS_CURRENT,
        "active": COUNTRY_STATUS_CURRENT,
        "актуальные": COUNTRY_STATUS_CURRENT,
        "современные": COUNTRY_STATUS_CURRENT,
        "нынешние": COUNTRY_STATUS_CURRENT,
        "historical": COUNTRY_STATUS_HISTORICAL,
        "former": COUNTRY_STATUS_HISTORICAL,
        "past": COUNTRY_STATUS_HISTORICAL,
        "old": COUNTRY_STATUS_HISTORICAL,
        "исторические": COUNTRY_STATUS_HISTORICAL,
        "бывшие": COUNTRY_STATUS_HISTORICAL,
    }
    return aliases.get(s, COUNTRY_STATUS_CURRENT)

def _country_status_note_ru(country_status: Optional[str]) -> str:
    status = _normalize_country_status(country_status)
    if status == COUNTRY_STATUS_HISTORICAL:
        return " Учитывай только исторические государства, которых уже не существует."
    return " Учитывай только современные государства, которые существуют сейчас."

def _append_country_status_filters(where: List[str], country_var: str, country_status: Optional[str]) -> None:
    """
    Разделяем современные и исторические страны по наличию даты прекращения существования.
    В Wikidata для этого обычно используются:
      - dissolved, abolished or demolished date (P576)
      - end time (P582)
    """
    status = _normalize_country_status(country_status)

    if status == COUNTRY_STATUS_HISTORICAL:
        where.append(
            f"{{ {{ {country_var} wdt:P576 ?country_dissolved . }} "
            f"UNION "
            f"{{ {country_var} wdt:P582 ?country_end . }} }}"
        )
    else:
        where.append(f"FILTER NOT EXISTS {{ {country_var} wdt:P576 ?country_dissolved . }}")
        where.append(f"FILTER NOT EXISTS {{ {country_var} wdt:P582 ?country_end . }}")


def _current_membership_patterns(country_var: str, org_qid: str, stmt_prefix: str) -> List[str]:
    stmt_var = f"?{stmt_prefix}_stmt"
    end_var = f"?{stmt_prefix}_end"
    return [
        f"{country_var} p:P463 {stmt_var} .",
        f"{stmt_var} ps:P463 wd:{org_qid} .",
        f"MINUS {{ {stmt_var} pq:P582 {end_var} . }}",
    ]


def _not_current_member_filter(country_var: str, org_qid: str, stmt_prefix: str) -> str:
    stmt_var = f"?{stmt_prefix}_stmt"
    end_var = f"?{stmt_prefix}_end"
    return (
        f"FILTER NOT EXISTS {{ "
        f"{country_var} p:P463 {stmt_var} . "
        f"{stmt_var} ps:P463 wd:{org_qid} . "
        f"MINUS {{ {stmt_var} pq:P582 {end_var} . }} "
        f"}}"
    )


def build_countries_query_sparql(
    continent_qid: str,
    language_qid: Optional[str] = None,
    currency_qid: Optional[str] = None,
    pop_min: Optional[int] = None,
    pop_max: Optional[int] = None,
    not_member_of_qid: Optional[str] = None,
    country_status: str = COUNTRY_STATUS_CURRENT,
    limit: int = 400,
) -> str:
    where: List[str] = []
    where.append(f"?c wdt:P31/wdt:P279* wd:{Q_COUNTRY} .")
    where.append(f"?c wdt:P30 wd:{continent_qid} .")
    _append_country_status_filters(where, "?c", country_status)

    if language_qid:
        where.append(f"?c wdt:P37 wd:{language_qid} .")
    if currency_qid:
        where.append(f"?c wdt:P38 wd:{currency_qid} .")

    if pop_min is not None or pop_max is not None:
        where.append("?c wdt:P1082 ?pop .")
        if pop_min is not None:
            where.append(f"FILTER(?pop >= {int(pop_min)})")
        if pop_max is not None:
            where.append(f"FILTER(?pop <= {int(pop_max)})")

    if not_member_of_qid:
        where.append(_not_current_member_filter("?c", not_member_of_qid, "member"))

    where.append("?c wikibase:sitelinks ?sl . FILTER(?sl >= 1)")

    sparql = f"""
SELECT DISTINCT ?c ?cLabel WHERE {{
  {' '.join(where)}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
}}
LIMIT {int(limit)}
""".strip()
    return sparql


_COUNTRIES_QUERY_CACHE: Dict[Tuple[Any, ...], Tuple[str, List[Tuple[str, str]]]] = {}


def run_countries_query(
    continent_qid: str,
    language_qid: Optional[str] = None,
    currency_qid: Optional[str] = None,
    pop_min: Optional[int] = None,
    pop_max: Optional[int] = None,
    not_member_of_qid: Optional[str] = None,
    country_status: str = COUNTRY_STATUS_CURRENT,
    limit: int = 400
) -> Tuple[str, List[Tuple[str, str]]]:

    key = (
        _norm_lbl(continent_qid),
        _norm_lbl(language_qid),
        _norm_lbl(currency_qid),
        None if pop_min is None else int(pop_min),
        None if pop_max is None else int(pop_max),
        _norm_lbl(not_member_of_qid),
        _normalize_country_status(country_status),
        int(limit),
    )
    cached = _COUNTRIES_QUERY_CACHE.get(key)
    if cached is not None:
        sparql, items = cached
        return sparql, list(items)

    sparql = build_countries_query_sparql(
        continent_qid,
        language_qid=language_qid,
        currency_qid=currency_qid,
        pop_min=pop_min,
        pop_max=pop_max,
        not_member_of_qid=not_member_of_qid,
        country_status=country_status,
        limit=limit,
    )

    rows = rows_from_select(wd.sparql_select(sparql))
    out: List[Tuple[str, str]] = []
    seen: set = set()
    for r in rows:
        qid = uri_to_qid(r.get("c", ""))
        lbl = _norm_lbl(r.get("cLabel", ""))
        if qid and qid not in seen:
            seen.add(qid)
            out.append((qid, lbl))

    _COUNTRIES_QUERY_CACHE[key] = (sparql, list(out))
    return sparql, out


def build_countries_l2_combo_pool(
    country_status: str = COUNTRY_STATUS_CURRENT,
    min_gold: int = 3,
    limit: int = 25000,
) -> pd.DataFrame:
    """
    Предвычисляем только валидные тройки (continent, language, currency) для L2.
    Это резко снижает число пустых/малых запросов и убирает многоминутные подвисания
    на этапе генерации MAIN.
    """
    status = _normalize_country_status(country_status)
    where: List[str] = [
        f"?c wdt:P31/wdt:P279* wd:{Q_COUNTRY} .",
        "?c wdt:P30 ?continent .",
        "?c wdt:P37 ?language .",
        "?c wdt:P38 ?currency .",
        "?c wikibase:sitelinks ?sl . FILTER(?sl >= 1)",
    ]
    _append_country_status_filters(where, "?c", status)

    sparql = f"""
SELECT DISTINCT ?continent ?continentLabel ?language ?languageLabel ?currency ?currencyLabel ?c ?cLabel WHERE {{
  {' '.join(where)}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
}}
LIMIT {int(limit)}
""".strip()

    rows = rows_from_select(wd.sparql_select(sparql))
    grouped: Dict[Tuple[str, str, str], Dict[str, Any]] = {}

    for r in rows:
        cont_qid = uri_to_qid(r.get("continent", ""))
        lang_qid = uri_to_qid(r.get("language", ""))
        curr_qid = uri_to_qid(r.get("currency", ""))
        c_qid = uri_to_qid(r.get("c", ""))

        if not cont_qid or not lang_qid or not curr_qid or not c_qid:
            continue

        key = (cont_qid, lang_qid, curr_qid)
        rec = grouped.get(key)
        if rec is None:
            rec = {
                "continent_qid": cont_qid,
                "continentLabelRu": _norm_lbl(r.get("continentLabel", "")),
                "language_qid": lang_qid,
                "languageLabelRu": _norm_lbl(r.get("languageLabel", "")),
                "currency_qid": curr_qid,
                "currencyLabelRu": _norm_lbl(r.get("currencyLabel", "")),
                "gold_qids": [],
                "gold_labels_ru": [],
                "gold_count": 0,
                "_seen": set(),
            }
            grouped[key] = rec

        if c_qid in rec["_seen"]:
            continue
        rec["_seen"].add(c_qid)
        rec["gold_qids"].append(c_qid)
        rec["gold_labels_ru"].append(_norm_lbl(r.get("cLabel", "")))
        rec["gold_count"] += 1

    data: List[Dict[str, Any]] = []
    for rec in grouped.values():
        if int(rec.get("gold_count", 0)) >= int(min_gold):
            rec.pop("_seen", None)
            data.append(rec)

    df = pd.DataFrame(data)
    if len(df) > 0:
        df = df.sort_values(
            by=["gold_count", "continentLabelRu", "languageLabelRu", "currencyLabelRu"],
            ascending=[False, True, True, True],
        ).reset_index(drop=True)
    return df


pool_countries_l2_current_ru = load_or_build_pool(
    "countries_l2_current_ru",
    lambda: build_countries_l2_combo_pool(
        country_status=COUNTRY_STATUS_CURRENT,
        min_gold=3,
        limit=25000,
    ),
)


def _sample_countries_l2_current_row(rng: random.Random):
    df = globals().get("pool_countries_l2_current_ru", None)
    if not isinstance(df, pd.DataFrame) or len(df) == 0:
        return None
    return df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]


def _coerce_list_like(x: Any) -> List[Any]:
    if x is None:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, tuple):
        return list(x)
    try:
        if pd.isna(x):
            return []
    except Exception:
        pass
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        try:
            val = json.loads(s)
            if isinstance(val, list):
                return val
        except Exception:
            pass
    return []


def _items_from_countries_l2_pool_row(row: Any) -> List[Tuple[str, str]]:
    qids = [_norm_lbl(x) for x in _coerce_list_like(row.get("gold_qids", []))]
    lbls = [_norm_lbl(x) for x in _coerce_list_like(row.get("gold_labels_ru", []))]

    out: List[Tuple[str, str]] = []
    seen: set = set()
    for qid, lbl in zip(qids, lbls):
        if qid and qid not in seen:
            seen.add(qid)
            out.append((qid, lbl))
    return out

ADV_TEMPLATE_PROB_COUNTRIES = 0.25

def _generate_countries_example_default(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 80,
    country_status: str = COUNTRY_STATUS_CURRENT,
) -> BenchmarkExample:
    k = 3
    ex = None
    country_status = _normalize_country_status(country_status)
    status_note_ru = _country_status_note_ru(country_status)

    for _ in range(max_attempts):
        cont_qid, cont_lbl = _pick_from_pool_or_anchors(
            pool_countries_continents_ru, COUNTRY_CONTINENT_ANCHORS,
            "continent_qid", "continentLabelRu", rng
        )
        lang_qid, lang_lbl = _pick_from_pool_or_anchors(
            pool_countries_languages_ru, COUNTRY_LANG_ANCHORS,
            "language_qid", "languageLabelRu", rng
        )
        curr_qid, curr_lbl = _pick_from_pool_or_anchors(
            pool_countries_currencies_ru, COUNTRY_CURR_ANCHORS,
            "currency_qid", "currencyLabelRu", rng
        )

        if not cont_lbl or not lang_lbl:
            continue

        constraints: Dict[str, Any] = {
            "continent_qid": cont_qid,
            "continent_label_ru": cont_lbl,
            "country_status": country_status,
        }

        if complexity == "L1":
            constraints.update({"language_qid": lang_qid, "language_label_ru": lang_lbl})
            sparql, items = run_countries_query(
                cont_qid,
                language_qid=lang_qid,
                country_status=country_status,
                limit=500,
            )
            template_id = "countries_L1_continent+language"
            query_text_ru = (
                f"Назови {k} стран, которые находятся в регионе «{cont_lbl}» "
                f"и у которых один из официальных языков — «{lang_lbl}»."
                f"{status_note_ru}"
            )

        elif complexity == "L2":
            pool_row = _sample_countries_l2_current_row(rng) if country_status == COUNTRY_STATUS_CURRENT else None

            if pool_row is not None:
                cont_qid = _norm_lbl(pool_row.get("continent_qid", cont_qid))
                cont_lbl = _norm_lbl(pool_row.get("continentLabelRu", cont_lbl))
                lang_qid = _norm_lbl(pool_row.get("language_qid", lang_qid))
                lang_lbl = _norm_lbl(pool_row.get("languageLabelRu", lang_lbl))
                curr_qid = _norm_lbl(pool_row.get("currency_qid", curr_qid))
                curr_lbl = _norm_lbl(pool_row.get("currencyLabelRu", curr_lbl))
                items = _items_from_countries_l2_pool_row(pool_row)
                sparql = build_countries_query_sparql(
                    cont_qid,
                    language_qid=lang_qid,
                    currency_qid=curr_qid,
                    country_status=country_status,
                    limit=500,
                )
            else:
                if not curr_lbl:
                    continue
                sparql, items = run_countries_query(
                    cont_qid,
                    language_qid=lang_qid,
                    currency_qid=curr_qid,
                    country_status=country_status,
                    limit=300,
                )

            if not cont_lbl or not lang_lbl or not curr_lbl:
                continue

            constraints.update({
                "language_qid": lang_qid, "language_label_ru": lang_lbl,
                "currency_qid": curr_qid, "currency_label_ru": curr_lbl
            })
            template_id = "countries_L2_continent+language+currency"
            query_text_ru = (
                f"Подбери {k} стран в регионе «{cont_lbl}», где официальный язык — «{lang_lbl}», "
                f"а валюта — «{curr_lbl}»."
                f"{status_note_ru}"
            )

        elif complexity == "L3":
            pop_min = int(rng.choice([1_000_000, 5_000_000, 10_000_000, 30_000_000]))
            pop_max = int(pop_min * rng.choice([3, 5, 8]))
            constraints.update({
                "language_qid": lang_qid, "language_label_ru": lang_lbl,
                "pop_min": pop_min, "pop_max": pop_max
            })
            sparql, items = run_countries_query(
                cont_qid,
                language_qid=lang_qid,
                pop_min=pop_min,
                pop_max=pop_max,
                country_status=country_status,
                limit=600,
            )
            template_id = "countries_L3_continent+language+pop_range"
            query_text_ru = (
                f"Назови {k} стран в регионе «{cont_lbl}», где официальный язык — «{lang_lbl}», "
                f"и численность населения примерно в диапазоне {pop_min:,}–{pop_max:,} человек."
                f"{status_note_ru}"
            ).replace(",", " ")

        elif complexity == "L4":
            constraints.update({"country_status": COUNTRY_STATUS_CURRENT})
            template_id = "countries_L4_insufficient_EU_and_G7"
            query_text_ru = (
                "Перечисли 5 стран, которые сейчас одновременно состоят в Европейском союзе и в G7. "
                "Если таких стран меньше 5, перечисли все, сколько есть, и явно напиши, что их меньше 5."
            )
            query_gold = f"""
SELECT DISTINCT ?c ?cLabel WHERE {{
  ?c wdt:P31/wdt:P279* wd:{Q_COUNTRY} .
  FILTER NOT EXISTS {{ ?c wdt:P576 ?country_dissolved . }}
  FILTER NOT EXISTS {{ ?c wdt:P582 ?country_end . }}
  {' '.join(_current_membership_patterns("?c", Q_EU, "eu"))}
  {' '.join(_current_membership_patterns("?c", Q_G7, "g7"))}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
}}
LIMIT 50
""".strip()
            rows = rows_from_select(wd.sparql_select(query_gold))
            items = [(uri_to_qid(r.get("c","")), _norm_lbl(r.get("cLabel",""))) for r in rows if uri_to_qid(r.get("c",""))]
            sparql = query_gold

        else:  # L5
            cont_qid = ensure_qid("Антарктида", fallback_qid="Q51")
            cont_lbl = "Антарктида"
            constraints = {
                "continent_qid": cont_qid,
                "continent_label_ru": cont_lbl,
                "language_qid": lang_qid,
                "language_label_ru": lang_lbl,
                "country_status": country_status,
            }
            sparql, items = run_countries_query(
                cont_qid,
                language_qid=lang_qid,
                country_status=country_status,
                limit=50,
            )
            template_id = "countries_L5_impossible_antarctica_language"
            query_text_ru = (
                f"Назови {k} стран в регионе «{cont_lbl}», где официальный язык — «{lang_lbl}»."
                f"{status_note_ru}"
            )

        gold = items[:300]
        requested_count = 5 if complexity == "L4" else k
        ex = BenchmarkExample(
            id=f"countries_{complexity.lower()}_{idx:05d}",
            domain="countries",
            complexity=complexity,
            query_text_ru=query_text_ru,
            constraints=constraints,
            requested_count=requested_count,
            gold_answer_qids=[q for q, _ in gold if q],
            gold_answer_labels_ru=[l for _, l in gold],
            sparql_query=sparql,  # Keep the generation query for inspection
            created_at=dt.datetime.utcnow().isoformat(),
            is_advanced=False,
            template_id=template_id,
            template_family="default",
        )
        return ex

    return ex if ex is not None else BenchmarkExample(
        id=f"countries_{complexity.lower()}_{idx:05d}",
        domain="countries",
        complexity=complexity,
        query_text_ru="Назови 5 стран мира. Учитывай только современные государства, которые существуют сейчас.",
        constraints={"country_status": COUNTRY_STATUS_CURRENT},
        requested_count=5,
        gold_answer_qids=[],
        gold_answer_labels_ru=[],
        sparql_query="",
        created_at=dt.datetime.utcnow().isoformat(),
        is_advanced=False,
        template_id="countries_fallback",
        template_family="default",
    )

def generate_countries_example_advanced(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 80,
    country_status: str = COUNTRY_STATUS_CURRENT,
) -> BenchmarkExample:
    k = 5
    country_status = _normalize_country_status(country_status)
    status_note_ru = _country_status_note_ru(country_status)

    for _ in range(max_attempts):
        cont_qid, cont_lbl = _pick_from_pool_or_anchors(
            pool_countries_continents_ru, COUNTRY_CONTINENT_ANCHORS,
            "continent_qid", "continentLabelRu", rng
        )
        lang_qid, lang_lbl = _pick_from_pool_or_anchors(
            pool_countries_languages_ru, COUNTRY_LANG_ANCHORS,
            "language_qid", "languageLabelRu", rng
        )
        if not cont_lbl or not lang_lbl:
            continue

        not_org = Q_EU if rng.random() < 0.5 else None
        pop_min = int(rng.choice([5_000_000, 10_000_000, 20_000_000]))

        sparql, items = run_countries_query(
            cont_qid,
            language_qid=lang_qid,
            pop_min=pop_min,
            not_member_of_qid=not_org,
            country_status=country_status,
            limit=600,
        )

        query_text_ru = (
            f"Подбери ровно {k} стран, которые находятся в регионе «{cont_lbl}», "
            f"имеют официальный язык «{lang_lbl}», население больше {pop_min:,} человек"
        ).replace(",", " ")
        query_text_ru += " и при этом НЕ входят в Европейский союз." if not_org else "."
        query_text_ru += status_note_ru

        constraints = {
            "continent_qid": cont_qid, "continent_label_ru": cont_lbl,
            "language_qid": lang_qid, "language_label_ru": lang_lbl,
            "pop_min": pop_min,
            "not_member_of_qid": not_org,
            "country_status": country_status,
        }

        gold = items[:300]
        return BenchmarkExample(
            id=f"countries_{complexity.lower()}_{idx:05d}",
            domain="countries",
            complexity=complexity,
            query_text_ru=query_text_ru,
            constraints=constraints,
            requested_count=k,
            gold_answer_qids=[q for q, _ in gold if q],
            gold_answer_labels_ru=[l for _, l in gold],
            sparql_query=sparql,
            created_at=dt.datetime.utcnow().isoformat(),
            is_advanced=True,
            template_id="countries_adv_multihop_not_pop",
            template_family="advanced",
        )

    return _generate_countries_example_default(
        complexity,
        idx,
        rng,
        max_attempts=max_attempts,
        country_status=country_status,
    )

def generate_countries_example(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 80,
    country_status: str = COUNTRY_STATUS_CURRENT,
) -> BenchmarkExample:
    if complexity in ("L3","L4","L5") and rng.random() < ADV_TEMPLATE_PROB_COUNTRIES:
        try:
            return generate_countries_example_advanced(
                complexity,
                idx,
                rng,
                max_attempts=max_attempts,
                country_status=country_status,
            )
        except Exception:
            pass
    return _generate_countries_example_default(
        complexity,
        idx,
        rng,
        max_attempts=max_attempts,
        country_status=country_status,
    )


<a id="dishes-domain"></a>

## 21. Dishes domain

Dish templates with cuisine, ingredient inclusion/exclusion, country, and category constraints.

In [21]:
import pandas as pd
import random
import datetime as dt
from typing import Dict, List, Tuple, Any, Optional

Q_DISH = ensure_qid("блюдо", fallback_qid="Q746549")  # dish

pool_dishes_ingredients_ru = load_or_build_pool(
    "dishes_ingredients_ru",
    lambda: build_value_pool_ru(Q_DISH, "P186", "ing", limit=250),  # P186 = made from material / ingredient
)

pool_dishes_countries_ru = load_or_build_pool(
    "dishes_countries_ru",
    lambda: build_value_pool_ru(Q_DISH, "P495", "country", limit=150),  # P495 = country of origin
)

DISH_ING_ANCHORS = [
    ("Q1033", "курица"),
    ("Q83364", "говядина"),
    ("Q42555", "свинина"),
    ("Q152", "рыба"),          # Fish; improves pool matching
    ("Q54872", "рис"),
    ("Q10998", "картофель"),
    ("Q178", "томат"),
    ("Q10990", "сыр"),
    ("Q185144", "яйцо"),
]

DISH_COUNTRY_ANCHORS = [
    ("Q142", "Франция"),
    ("Q38", "Италия"),
    ("Q29", "Испания"),
    ("Q17", "Япония"),
    ("Q148", "Китай"),
    ("Q159", "Россия"),
    ("Q96", "Мексика"),
    ("Q408", "Австралия"),
    ("Q668", "Индия"),
]

def _norm_lbl(x: Any) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    s = str(x).strip()
    return "" if s.lower() == "nan" else s

def _pick_pool_or_anchor(
    pool_df: Optional[pd.DataFrame],
    anchors: List[Tuple[str, str]],
    qid_col: str,
    lbl_col: str,
    rng: random.Random
) -> Tuple[str, str]:
    """
    ВАЖНО: build_value_pool_ru кладёт лейбл в колонку {val_name}LabelRu
    например ingLabelRu / countryLabelRu.
    """
    if isinstance(pool_df, pd.DataFrame) and len(pool_df) > 0 and qid_col in pool_df.columns:
        row = pool_df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
        qid = _norm_lbl(row.get(qid_col, ""))
        lbl = _norm_lbl(row.get(lbl_col, ""))
        if qid and lbl:
            return qid, lbl
        if qid and not lbl:
            pass
    return rng.choice(anchors)

def run_dishes_query(
    ing_qid: str,
    country_qid: Optional[str] = None,
    not_ing_qid: Optional[str] = None,
    continent_qid: Optional[str] = None,
    limit: int = 500
) -> Tuple[str, List[Tuple[str, str]]]:
    where = []
    where.append(f"?dish wdt:P31/wdt:P279* wd:{Q_DISH} .")
    where.append(f"?dish wdt:P186 wd:{ing_qid} .")

    if not_ing_qid:
        where.append(f"FILTER NOT EXISTS {{ ?dish wdt:P186 wd:{not_ing_qid} . }}")

    if country_qid:
        where.append(f"?dish wdt:P495 wd:{country_qid} .")

    if continent_qid:
        where.append("?dish wdt:P495 ?c .")
        where.append(f"?c wdt:P30 wd:{continent_qid} .")

    where.append("?dish wikibase:sitelinks ?sl . FILTER(?sl >= 1)")

    query = f"""
SELECT DISTINCT ?dish ?dishLabel WHERE {{
  {' '.join(where)}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
}}
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(query))
    out: List[Tuple[str, str]] = []
    for r in rows:
        qid = uri_to_qid(r.get("dish", ""))
        lbl = _norm_lbl(r.get("dishLabel", ""))
        if qid:
            out.append((qid, lbl))
    return query.strip(), out

ADV_TEMPLATE_PROB_DISHES = 1.0

def _generate_dishes_example_default(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 80
) -> BenchmarkExample:
    k = 3

    for _ in range(max_attempts):
        ing_qid, ing_lbl = _pick_pool_or_anchor(
            pool_dishes_ingredients_ru, DISH_ING_ANCHORS,
            qid_col="ing_qid", lbl_col="ingLabelRu", rng=rng
        )
        c_qid, c_lbl = _pick_pool_or_anchor(
            pool_dishes_countries_ru, DISH_COUNTRY_ANCHORS,
            qid_col="country_qid", lbl_col="countryLabelRu", rng=rng
        )
        not_ing_qid, not_ing_lbl = rng.choice([a for a in DISH_ING_ANCHORS if a[0] != ing_qid])

        if not ing_lbl:
            continue

        constraints: Dict[str, Any] = {"ingredient_qid": ing_qid, "ingredient_label_ru": ing_lbl}

        if complexity == "L1":
            sparql, items = run_dishes_query(ing_qid, limit=600)
            query_text_ru = f"Назови {k} блюд, в составе которых есть ингредиент «{ing_lbl}»."
            template_id = "dishes_L1_ingredient"

        elif complexity == "L2":
            constraints.update({"country_qid": c_qid, "country_label_ru": c_lbl})
            sparql, items = run_dishes_query(ing_qid, country_qid=c_qid, limit=600)
            query_text_ru = f"Подбери {k} блюд с ингредиентом «{ing_lbl}», которые происходят из страны «{c_lbl}»."
            template_id = "dishes_L2_ingredient+country"

        elif complexity == "L3":
            constraints.update({"not_ingredient_qid": not_ing_qid, "not_ingredient_label_ru": not_ing_lbl})
            sparql, items = run_dishes_query(ing_qid, not_ing_qid=not_ing_qid, limit=800)
            query_text_ru = (
                f"Назови {k} блюд, в составе которых есть «{ing_lbl}», "
                f"но при этом НЕТ ингредиента «{not_ing_lbl}»."
            )
            template_id = "dishes_L3_ingredient_NOT_ingredient"

        elif complexity == "L4":
            constraints.update({
                "country_qid": c_qid, "country_label_ru": c_lbl,
                "not_ingredient_qid": not_ing_qid, "not_ingredient_label_ru": not_ing_lbl
            })
            sparql, items = run_dishes_query(ing_qid, country_qid=c_qid, not_ing_qid=not_ing_qid, limit=800)
            query_text_ru = (
                f"Подбери {k} блюд из страны «{c_lbl}», где есть «{ing_lbl}» и при этом нет «{not_ing_lbl}». "
                f"Если таких блюд меньше {k}, перечисли все, сколько нашлось."
            )
            template_id = "dishes_L4_insufficient_country_ing_not"

        else:  # L5
            ing2_qid, ing2_lbl = rng.choice([a for a in DISH_ING_ANCHORS if a[0] != ing_qid])
            constraints.update({"ingredient2_qid": ing2_qid, "ingredient2_label_ru": ing2_lbl})

            query_text_ru = f"Назови {k} блюд, в составе которых одновременно есть «{ing_lbl}» и «{ing2_lbl}»."

            query_gold = f"""
SELECT DISTINCT ?dish ?dishLabel WHERE {{
  ?dish wdt:P31/wdt:P279* wd:{Q_DISH} .
  ?dish wdt:P186 wd:{ing_qid} .
  ?dish wdt:P186 wd:{ing2_qid} .
  ?dish wikibase:sitelinks ?sl . FILTER(?sl >= 1)
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
}}
LIMIT 200
"""
            rows = rows_from_select(wd.sparql_select(query_gold))
            items = []
            for r in rows:
                qid = uri_to_qid(r.get("dish", ""))
                lbl = _norm_lbl(r.get("dishLabel", ""))
                if qid:
                    items.append((qid, lbl))
            sparql = query_gold.strip()
            template_id = "dishes_L5_two_ingredients"

        gold = items[:300]
        return BenchmarkExample(
            id=f"dishes_{complexity.lower()}_{idx:05d}",
            domain="dishes",
            complexity=complexity,
            query_text_ru=query_text_ru,
            constraints=constraints,
            requested_count=k,
            gold_answer_qids=[q for q, _ in gold if q],
            gold_answer_labels_ru=[l for _, l in gold],
            sparql_query=sparql,  # Keep the generation query for inspection
            created_at=dt.datetime.utcnow().isoformat(),
            is_advanced=False,
            template_id=template_id,
            template_family="default",
        )

    return BenchmarkExample(
        id=f"dishes_{complexity.lower()}_{idx:05d}",
        domain="dishes",
        complexity=complexity,
        query_text_ru="Назови 5 популярных блюд.",
        constraints={},
        requested_count=5,
        gold_answer_qids=[],
        gold_answer_labels_ru=[],
        sparql_query="",
        created_at=dt.datetime.utcnow().isoformat(),
        is_advanced=False,
        template_id="dishes_fallback",
        template_family="default",
    )

def generate_dishes_example_advanced(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 80
) -> BenchmarkExample:
    k = 5
    cont_qid, cont_lbl = rng.choice(COUNTRY_CONTINENT_ANCHORS)

    for _ in range(max_attempts):
        ing_qid, ing_lbl = _pick_pool_or_anchor(
            pool_dishes_ingredients_ru, DISH_ING_ANCHORS,
            qid_col="ing_qid", lbl_col="ingLabelRu", rng=rng
        )
        not_ing_qid, not_ing_lbl = rng.choice([a for a in DISH_ING_ANCHORS if a[0] != ing_qid])

        if not ing_lbl:
            continue

        sparql, items = run_dishes_query(ing_qid, not_ing_qid=not_ing_qid, continent_qid=cont_qid, limit=900)
        query_text_ru = (
            f"Подбери ровно {k} блюд, где есть «{ing_lbl}», "
            f"при этом нет «{not_ing_lbl}», а страна происхождения находится в регионе «{cont_lbl}»."
        )
        constraints = {
            "ingredient_qid": ing_qid, "ingredient_label_ru": ing_lbl,
            "not_ingredient_qid": not_ing_qid, "not_ingredient_label_ru": not_ing_lbl,
            "continent_qid": cont_qid, "continent_label_ru": cont_lbl,
        }
        gold = items[:300]
        return BenchmarkExample(
            id=f"dishes_{complexity.lower()}_{idx:05d}",
            domain="dishes",
            complexity=complexity,
            query_text_ru=query_text_ru,
            constraints=constraints,
            requested_count=k,
            gold_answer_qids=[q for q, _ in gold if q],
            gold_answer_labels_ru=[l for _, l in gold],
            sparql_query=sparql,
            created_at=dt.datetime.utcnow().isoformat(),
            is_advanced=True,
            template_id="dishes_adv_multihop_not",
            template_family="advanced",
        )

    return _generate_dishes_example_default(complexity, idx, rng, max_attempts=max_attempts)

def generate_dishes_example(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 80
) -> BenchmarkExample:
    if complexity in ("L3", "L4", "L5") and rng.random() < ADV_TEMPLATE_PROB_DISHES:
        try:
            return generate_dishes_example_advanced(complexity, idx, rng, max_attempts=max_attempts)
        except Exception:
            pass
    return _generate_dishes_example_default(complexity, idx, rng, max_attempts=max_attempts)


<a id="smartphones-domain"></a>

## 22. Smartphones domain

Smartphone templates with manufacturer, operating system, release year, screen, and CPU-style constraints.

In [22]:
import pandas as pd
import random
import datetime as dt
from collections import defaultdict
from typing import Optional, List, Tuple, Dict, Any

Q_SMARTPHONE = ensure_qid("смартфон", fallback_qid="Q22645")            # smartphone
Q_MOBILE_PHONE = ensure_qid("мобильный телефон", fallback_qid="Q17517")  # mobile phone
Q_ANDROID = ensure_qid("Android", fallback_qid="Q94")
Q_IOS = ensure_qid("iOS", fallback_qid="Q48493")

Q_SMARTPHONE_MODEL = resolve_qid("модель смартфона", "ru") or resolve_qid("smartphone model", "en")
Q_MOBILE_PHONE_MODEL = resolve_qid("модель мобильного телефона", "ru") or resolve_qid("mobile phone model", "en")

SMARTPHONE_TYPE_QIDS: List[str] = [q for q in [
    Q_SMARTPHONE,
    Q_MOBILE_PHONE,
    Q_SMARTPHONE_MODEL,
    Q_MOBILE_PHONE_MODEL,
] if q]

SMARTPHONE_MAKER_ANCHORS = [
    ("Q312", "Apple"),
    ("Q27414", "Samsung"),
    ("Q174187", "Xiaomi"),
    ("Q175751", "Huawei"),
    ("Q95", "Google"),
    ("Q318144", "Sony"),
    ("Q3884", "Nokia"),
    ("Q207194", "Motorola"),
    ("Q215380", "OnePlus"),
    ("Q1134006", "Honor"),
    ("Q1078460", "Oppo"),
    ("Q679694", "Vivo"),
    ("Q257998", "LG"),
    ("Q15148", "HTC"),
    ("Q192608", "Lenovo"),
]
SMARTPHONE_OS_ANCHORS = [
    (Q_ANDROID, "Android"),
    (Q_IOS, "iOS"),
]

_LABEL_CACHE: Dict[str, str] = {}
_SMARTPHONE_ENUM_CACHE: Dict[str, List[Any]] = {}
_SMARTPHONE_ENUM_CURSOR: Dict[str, int] = defaultdict(int)
_SMARTPHONE_CANDIDATE_CACHE: Dict[Tuple[Any, ...], List[Any]] = {}

def _norm_lbl(x: Any) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    s = str(x).strip()
    return "" if s.lower() == "nan" else s

def get_label_ru_en_cached(qid: str) -> str:
    qid = (qid or "").strip()
    if not qid or not qid.startswith("Q"):
        return ""
    if qid in _LABEL_CACHE:
        return _LABEL_CACHE[qid]

    query = f"""
SELECT ?ru ?en WHERE {{
  OPTIONAL {{ wd:{qid} rdfs:label ?ru FILTER(LANG(?ru)='ru') }}
  OPTIONAL {{ wd:{qid} rdfs:label ?en FILTER(LANG(?en)='en') }}
}}
LIMIT 1
"""
    lbl = ""
    try:
        rows = rows_from_select(wd.sparql_select(query))
        if rows:
            lbl = _norm_lbl(rows[0].get("ru")) or _norm_lbl(rows[0].get("en"))
    except Exception:
        lbl = ""
    _LABEL_CACHE[qid] = lbl
    return lbl

def _smartphone_type_block(item_var: str = "p") -> str:
    values = " ".join(f"wd:{q}" for q in SMARTPHONE_TYPE_QIDS if q)
    return f"""
  ?{item_var} wdt:P31/wdt:P279* ?phoneClass .
  VALUES ?phoneClass {{ {values} }}
  OPTIONAL {{ ?{item_var} wdt:P306 ?_anyOs . }}
  FILTER(
    ?phoneClass = wd:{Q_SMARTPHONE}
    {"|| ?phoneClass = wd:" + Q_SMARTPHONE_MODEL if Q_SMARTPHONE_MODEL else ""}
    || BOUND(?_anyOs)
  )
"""

def _smartphone_date_block(item_var: str = "p") -> List[str]:
    return [
        f"OPTIONAL {{ ?{item_var} wdt:P577 ?d1 . }}",
        f"OPTIONAL {{ ?{item_var} wdt:P571 ?d2 . }}",
        "BIND(COALESCE(?d1, ?d2) AS ?d)",
        "FILTER(BOUND(?d))",
    ]

def _smartphone_label_block(item_var: str = "p") -> str:
    return f"""
  OPTIONAL {{ ?{item_var} rdfs:label ?ru FILTER(LANG(?ru)='ru') }}
  OPTIONAL {{ ?{item_var} rdfs:label ?en FILTER(LANG(?en)='en') }}
  BIND(COALESCE(?ru, ?en) AS ?label)
"""

def _smartphone_gold_limit(k: int, complexity: str) -> int:
    if complexity in ("L4", "L5"):
        return max(50, k * 20)
    return max(80, k * 25)

def _smartphone_items_from_rows(rows: List[Dict[str, Any]], q_key: str = "p", lbl_key: str = "label") -> List[Tuple[str, str]]:
    out: List[Tuple[str, str]] = []
    seen = set()
    for r in rows:
        qid = uri_to_qid(r.get(q_key, ""))
        lbl = _norm_lbl(r.get(lbl_key))
        if not qid or qid in seen:
            continue
        seen.add(qid)
        if not lbl:
            lbl = get_label_ru_en_cached(qid)
        if qid:
            out.append((qid, lbl))
    return out

def run_smartphones_query(
    maker_qid: Optional[str] = None,
    os_qid: Optional[str] = None,
    year_from: Optional[int] = None,
    year_to: Optional[int] = None,
    not_maker_qid: Optional[str] = None,
    limit: int = 120,
) -> Tuple[str, List[Tuple[str, str]], List[str]]:

    where_lines: List[str] = [
        _smartphone_type_block("p"),
        "?p wdt:P176 ?maker .",
        "?p wikibase:sitelinks ?sl .",
        "FILTER(?sl >= 1)",
    ]

    if maker_qid:
        where_lines.append(f"?p wdt:P176 wd:{maker_qid} .")
    if os_qid:
        where_lines.append(f"?p wdt:P306 wd:{os_qid} .")
    if not_maker_qid:
        where_lines.append(f"FILTER NOT EXISTS {{ ?p wdt:P176 wd:{not_maker_qid} . }}")

    if year_from is not None or year_to is not None:
        where_lines.extend(_smartphone_date_block("p"))
        if year_from is not None:
            where_lines.append(f"FILTER(YEAR(?d) >= {int(year_from)})")
        if year_to is not None:
            where_lines.append(f"FILTER(YEAR(?d) <= {int(year_to)})")

    where = "\n  ".join(where_lines)
    sparql = f"""
SELECT DISTINCT ?p ?label WHERE {{
  {where}
  {_smartphone_label_block("p")}
}}
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))
    items = _smartphone_items_from_rows(rows, q_key="p", lbl_key="label")
    return sparql.strip(), items, where_lines

def _next_candidate(key: str, builder, rng: random.Random):
    if key not in _SMARTPHONE_ENUM_CACHE:
        items = list(builder() or [])
        rng.shuffle(items)
        _SMARTPHONE_ENUM_CACHE[key] = items
        _SMARTPHONE_ENUM_CURSOR[key] = 0

    items = _SMARTPHONE_ENUM_CACHE[key]
    if not items:
        return None

    cur = _SMARTPHONE_ENUM_CURSOR.get(key, 0)
    if cur >= len(items):
        if len(items) > 1:
            rng.shuffle(items)
        cur = 0
    _SMARTPHONE_ENUM_CURSOR[key] = cur + 1
    return items[cur]

def _fallback_maker_candidates() -> List[Tuple[str, str, int]]:
    out = []
    for qid, lbl in SMARTPHONE_MAKER_ANCHORS:
        out.append((qid, lbl or get_label_ru_en_cached(qid) or "", 999))
    return out

def _fallback_maker_os_candidates() -> List[Tuple[str, str, str, str, int]]:
    out = []
    for maker_qid, maker_lbl in SMARTPHONE_MAKER_ANCHORS:
        for os_qid, os_lbl in SMARTPHONE_OS_ANCHORS:
            out.append((maker_qid, maker_lbl or get_label_ru_en_cached(maker_qid) or "",
                        os_qid, os_lbl or get_label_ru_en_cached(os_qid) or "", 999))
    return out

def find_smartphone_maker_candidates(min_models: int = 5, limit: int = 80) -> List[Tuple[str, str, int]]:
    key = ("makers", int(min_models), int(limit))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    sparql = f"""
SELECT ?maker ?makerLabel (COUNT(DISTINCT ?p) AS ?cnt) WHERE {{
  {_smartphone_type_block("p")}
  ?p wdt:P176 ?maker .
  ?p wikibase:sitelinks ?sl .
  FILTER(?sl >= 1)
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
}}
GROUP BY ?maker ?makerLabel
HAVING(COUNT(DISTINCT ?p) >= {int(min_models)})
ORDER BY DESC(COUNT(DISTINCT ?p))
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))
    out: List[Tuple[str, str, int]] = []
    for r in rows:
        maker_qid = uri_to_qid(r.get("maker", ""))
        maker_lbl = _norm_lbl(r.get("makerLabel")) or (get_label_ru_en_cached(maker_qid) if maker_qid else "")
        try:
            cnt = int(float(r.get("cnt", "0")))
        except Exception:
            cnt = 0
        if maker_qid and maker_lbl and cnt >= int(min_models):
            out.append((maker_qid, maker_lbl, cnt))

    if not out:
        out = _fallback_maker_candidates()

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out

def find_smartphone_maker_year_candidates(
    min_models: int = 5,
    year_from: int = 2010,
    year_to: int = 2025,
    limit: int = 240,
) -> List[Tuple[str, str, int, int]]:
    key = ("maker_year", int(min_models), int(year_from), int(year_to), int(limit))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    sparql = f"""
SELECT ?maker ?makerLabel ?yy (COUNT(DISTINCT ?p) AS ?cnt) WHERE {{
  {_smartphone_type_block("p")}
  ?p wdt:P176 ?maker .
  {' '.join(_smartphone_date_block("p"))}
  BIND(YEAR(?d) AS ?yy)
  FILTER(?yy >= {int(year_from)} && ?yy <= {int(year_to)})
  ?p wikibase:sitelinks ?sl .
  FILTER(?sl >= 1)
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
}}
GROUP BY ?maker ?makerLabel ?yy
HAVING(COUNT(DISTINCT ?p) >= {int(min_models)})
ORDER BY DESC(COUNT(DISTINCT ?p))
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))
    out: List[Tuple[str, str, int, int]] = []
    for r in rows:
        maker_qid = uri_to_qid(r.get("maker", ""))
        maker_lbl = _norm_lbl(r.get("makerLabel")) or (get_label_ru_en_cached(maker_qid) if maker_qid else "")
        try:
            yy = int(float(r.get("yy", "0")))
            cnt = int(float(r.get("cnt", "0")))
        except Exception:
            yy, cnt = 0, 0
        if maker_qid and maker_lbl and yy and cnt >= int(min_models):
            out.append((maker_qid, maker_lbl, yy, cnt))

    if not out:
        years = list(range(2018, 2025))
        makers = find_smartphone_maker_candidates(min_models=max(2, min_models - 2), limit=20)
        for maker_qid, maker_lbl, _ in makers:
            for yy in years:
                out.append((maker_qid, maker_lbl, yy, max(min_models, 5)))

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out

def find_smartphone_maker_os_candidates(min_models: int = 5, limit: int = 160) -> List[Tuple[str, str, str, str, int]]:
    key = ("maker_os", int(min_models), int(limit))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    sparql = f"""
SELECT ?maker ?makerLabel ?os ?osLabel (COUNT(DISTINCT ?p) AS ?cnt) WHERE {{
  {_smartphone_type_block("p")}
  ?p wdt:P176 ?maker ;
     wdt:P306 ?os .
  ?p wikibase:sitelinks ?sl .
  FILTER(?sl >= 1)
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
}}
GROUP BY ?maker ?makerLabel ?os ?osLabel
HAVING(COUNT(DISTINCT ?p) >= {int(min_models)})
ORDER BY DESC(COUNT(DISTINCT ?p))
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))
    out: List[Tuple[str, str, str, str, int]] = []
    for r in rows:
        maker_qid = uri_to_qid(r.get("maker", ""))
        os_qid = uri_to_qid(r.get("os", ""))
        maker_lbl = _norm_lbl(r.get("makerLabel")) or (get_label_ru_en_cached(maker_qid) if maker_qid else "")
        os_lbl = _norm_lbl(r.get("osLabel")) or (get_label_ru_en_cached(os_qid) if os_qid else "")
        try:
            cnt = int(float(r.get("cnt", "0")))
        except Exception:
            cnt = 0
        if maker_qid and os_qid and maker_lbl and os_lbl and cnt >= int(min_models):
            out.append((maker_qid, maker_lbl, os_qid, os_lbl, cnt))

    if not out:
        out = _fallback_maker_os_candidates()

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out

def find_smartphone_maker_os_year_candidates(
    min_models: int = 1,
    year_from: int = 2010,
    year_to: int = 2025,
    limit: int = 320,
) -> List[Tuple[str, str, str, str, int, int]]:
    key = ("maker_os_year", int(min_models), int(year_from), int(year_to), int(limit))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    sparql = f"""
SELECT ?maker ?makerLabel ?os ?osLabel ?yy (COUNT(DISTINCT ?p) AS ?cnt) WHERE {{
  {_smartphone_type_block("p")}
  ?p wdt:P176 ?maker ;
     wdt:P306 ?os .
  {' '.join(_smartphone_date_block("p"))}
  BIND(YEAR(?d) AS ?yy)
  FILTER(?yy >= {int(year_from)} && ?yy <= {int(year_to)})
  ?p wikibase:sitelinks ?sl .
  FILTER(?sl >= 1)
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
}}
GROUP BY ?maker ?makerLabel ?os ?osLabel ?yy
HAVING(COUNT(DISTINCT ?p) >= {int(min_models)})
ORDER BY DESC(COUNT(DISTINCT ?p))
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))
    out: List[Tuple[str, str, str, str, int, int]] = []
    for r in rows:
        maker_qid = uri_to_qid(r.get("maker", ""))
        os_qid = uri_to_qid(r.get("os", ""))
        maker_lbl = _norm_lbl(r.get("makerLabel")) or (get_label_ru_en_cached(maker_qid) if maker_qid else "")
        os_lbl = _norm_lbl(r.get("osLabel")) or (get_label_ru_en_cached(os_qid) if os_qid else "")
        try:
            yy = int(float(r.get("yy", "0")))
            cnt = int(float(r.get("cnt", "0")))
        except Exception:
            yy, cnt = 0, 0
        if maker_qid and os_qid and maker_lbl and os_lbl and yy and cnt >= int(min_models):
            out.append((maker_qid, maker_lbl, os_qid, os_lbl, yy, cnt))

    if not out:
        base = find_smartphone_maker_os_candidates(min_models=max(2, min_models), limit=40)
        for maker_qid, maker_lbl, os_qid, os_lbl, _ in base:
            for yy in range(2018, 2025):
                out.append((maker_qid, maker_lbl, os_qid, os_lbl, yy, max(min_models, 1)))

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out

def build_smartphone_l5_candidates(
    min_models: int = 1,
    year_from: int = 2010,
    year_to: int = 2025,
    base_limit: int = 120,
    not_makers_per_base: int = 4,
) -> List[Tuple[str, str, str, str, int, str, str, int]]:
    key = ("l5", int(min_models), int(year_from), int(year_to), int(base_limit), int(not_makers_per_base))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    base = find_smartphone_maker_os_year_candidates(
        min_models=min_models,
        year_from=year_from,
        year_to=year_to,
        limit=base_limit,
    )
    makers = find_smartphone_maker_candidates(min_models=max(2, min_models), limit=40)

    out: List[Tuple[str, str, str, str, int, str, str, int]] = []
    for maker_qid, maker_lbl, os_qid, os_lbl, yy, cnt in base:
        others = [(q, l) for q, l, _ in makers if q != maker_qid][:max(1, int(not_makers_per_base))]
        for not_maker_qid, not_maker_lbl in others:
            out.append((maker_qid, maker_lbl, os_qid, os_lbl, yy, not_maker_qid, not_maker_lbl, cnt))

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out


_SMARTPHONE_MODELS_BY_MAKER_CACHE: Dict[Tuple[str, int, int, int], List[Dict[str, Any]]] = {}

def _smartphone_date_block_named(
    item_var: str = "p",
    date_var: str = "d",
    d1_var: str = "d1",
    d2_var: str = "d2",
) -> List[str]:
    return [
        f"OPTIONAL {{ ?{item_var} wdt:P577 ?{d1_var} . }}",
        f"OPTIONAL {{ ?{item_var} wdt:P571 ?{d2_var} . }}",
        f"BIND(COALESCE(?{d1_var}, ?{d2_var}) AS ?{date_var})",
        f"FILTER(BOUND(?{date_var}))",
    ]


def fetch_smartphone_models_for_maker(
    maker_qid: str,
    min_year: int = 2007,
    max_year: int = 2026,
    limit: int = 300,
) -> List[Dict[str, Any]]:
    key = (maker_qid, int(min_year), int(max_year), int(limit))
    if key in _SMARTPHONE_MODELS_BY_MAKER_CACHE:
        return _SMARTPHONE_MODELS_BY_MAKER_CACHE[key]

    sparql = f"""
SELECT DISTINCT ?p ?label ?d WHERE {{
  {_smartphone_type_block("p")}
  ?p wdt:P176 wd:{maker_qid} .
  ?p wikibase:sitelinks ?sl .
  FILTER(?sl >= 1)
  {' '.join(_smartphone_date_block_named("p", "d", "d1m", "d2m"))}
  FILTER(YEAR(?d) >= {int(min_year)} && YEAR(?d) <= {int(max_year)})
  {_smartphone_label_block("p")}
}}
ORDER BY ASC(?d) ?label
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))

    out: List[Dict[str, Any]] = []
    seen: set = set()
    for r in rows:
        qid = uri_to_qid(r.get("p", ""))
        if not qid or qid in seen:
            continue
        seen.add(qid)

        lbl = _norm_lbl(r.get("label")) or get_label_ru_en_cached(qid)
        d = _norm_lbl(r.get("d"))
        m = re.match(r"^(\d{4})", d)
        if not m:
            continue
        yy = int(m.group(1))
        if yy < int(min_year) or yy > int(max_year):
            continue
        out.append({
            "qid": qid,
            "label": lbl,
            "date": d,
            "year": yy,
        })

    _SMARTPHONE_MODELS_BY_MAKER_CACHE[key] = out
    return out


def find_smartphone_maker_year_window_candidates(
    min_models: int = 5,
    maker_limit: int = 24,
    min_span_years: int = 2,
    max_span_years: int = 4,
    max_candidates_per_maker: int = 12,
) -> List[Tuple[str, str, int, int, int]]:
    key = (
        "maker_year_window",
        int(min_models),
        int(maker_limit),
        int(min_span_years),
        int(max_span_years),
        int(max_candidates_per_maker),
    )
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    makers = find_smartphone_maker_candidates(min_models=max(min_models + 1, 6), limit=maker_limit)
    out: List[Tuple[str, str, int, int, int]] = []

    for maker_qid, maker_lbl, _ in makers:
        models = fetch_smartphone_models_for_maker(maker_qid)
        if len(models) < min_models:
            continue

        year_counts: Dict[int, int] = defaultdict(int)
        for m in models:
            year_counts[int(m["year"])] += 1

        years = sorted(year_counts)
        local: List[Tuple[str, str, int, int, int]] = []
        for i, y1 in enumerate(years):
            total = 0
            for j in range(i, len(years)):
                y2 = years[j]
                total += int(year_counts[y2])
                span = y2 - y1 + 1
                if span < int(min_span_years):
                    continue
                if span > int(max_span_years):
                    break
                if total >= int(min_models):
                    local.append((maker_qid, maker_lbl, y1, y2, total))

        local.sort(key=lambda x: (abs((x[3] - x[2] + 1) - 3), abs(x[4] - 7), x[2], x[3]))
        out.extend(local[:max_candidates_per_maker])

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out


def find_smartphone_maker_threshold_year_candidates(
    min_models: int = 5,
    maker_limit: int = 24,
    max_candidates_per_maker: int = 12,
) -> List[Tuple[str, str, str, int, int]]:
    key = ("maker_threshold_year", int(min_models), int(maker_limit), int(max_candidates_per_maker))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    makers = find_smartphone_maker_candidates(min_models=max(min_models + 1, 6), limit=maker_limit)
    out: List[Tuple[str, str, str, int, int]] = []

    for maker_qid, maker_lbl, _ in makers:
        models = fetch_smartphone_models_for_maker(maker_qid)
        years = sorted(int(m["year"]) for m in models)
        uniq = sorted(set(years))
        if len(uniq) < 3:
            continue

        local: List[Tuple[str, str, str, int, int]] = []
        for yy in uniq[1:-1]:
            cnt_ge = sum(1 for y in years if y >= yy)
            cnt_lt = sum(1 for y in years if y < yy)
            cnt_le = sum(1 for y in years if y <= yy)
            cnt_gt = sum(1 for y in years if y > yy)

            if cnt_ge >= int(min_models) and cnt_lt >= 1:
                local.append((maker_qid, maker_lbl, "from", yy, cnt_ge))
            if cnt_le >= int(min_models) and cnt_gt >= 1:
                local.append((maker_qid, maker_lbl, "to", yy, cnt_le))

        local.sort(key=lambda x: (abs(x[4] - 8), x[3], 0 if x[2] == "from" else 1))
        out.extend(local[:max_candidates_per_maker])

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out


def run_smartphones_query_relative_model(
    maker_qid: str,
    ref_model_qid: str,
    relation: str = "after",
    limit: int = 120,
) -> Tuple[str, List[Tuple[str, str]], List[str]]:
    if relation not in {"after", "before"}:
        raise ValueError(f"Unsupported relation: {relation}")

    where_lines: List[str] = [
        _smartphone_type_block("p"),
        f"?p wdt:P176 wd:{maker_qid} .",
        "?p wikibase:sitelinks ?sl .",
        "FILTER(?sl >= 1)",
        *_smartphone_date_block_named("p", "d", "d1p", "d2p"),
        f"VALUES ?refModel {{ wd:{ref_model_qid} }}",
        *_smartphone_date_block_named("refModel", "refDate", "d1ref", "d2ref"),
        "FILTER(?p != ?refModel)",
    ]
    if relation == "after":
        where_lines.append("FILTER(?d > ?refDate)")
    else:
        where_lines.append("FILTER(?d < ?refDate)")

    where = "\n  ".join(where_lines)
    sparql = f"""
SELECT DISTINCT ?p ?label WHERE {{
  {where}
  {_smartphone_label_block("p")}
}}
ORDER BY ASC(?label)
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))
    items = _smartphone_items_from_rows(rows, q_key="p", lbl_key="label")
    return sparql.strip(), items, where_lines


def run_smartphones_query_between_models(
    maker_qid: str,
    left_model_qid: str,
    right_model_qid: str,
    limit: int = 120,
) -> Tuple[str, List[Tuple[str, str]], List[str]]:
    where_lines: List[str] = [
        _smartphone_type_block("p"),
        f"?p wdt:P176 wd:{maker_qid} .",
        "?p wikibase:sitelinks ?sl .",
        "FILTER(?sl >= 1)",
        *_smartphone_date_block_named("p", "d", "d1p", "d2p"),
        f"VALUES ?leftRef {{ wd:{left_model_qid} }}",
        *_smartphone_date_block_named("leftRef", "leftDate", "d1l", "d2l"),
        f"VALUES ?rightRef {{ wd:{right_model_qid} }}",
        *_smartphone_date_block_named("rightRef", "rightDate", "d1r", "d2r"),
        "FILTER(?p != ?leftRef && ?p != ?rightRef)",
        "FILTER(?leftDate < ?rightDate)",
        "FILTER(?d > ?leftDate && ?d < ?rightDate)",
    ]

    where = "\n  ".join(where_lines)
    sparql = f"""
SELECT DISTINCT ?p ?label WHERE {{
  {where}
  {_smartphone_label_block("p")}
}}
ORDER BY ASC(?label)
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))
    items = _smartphone_items_from_rows(rows, q_key="p", lbl_key="label")
    return sparql.strip(), items, where_lines


def find_smartphone_reference_comparison_candidates(
    min_models: int = 5,
    maker_limit: int = 20,
    max_candidates_per_maker: int = 12,
) -> List[Tuple[str, str, str, str, int, str, int]]:
    key = ("ref_compare", int(min_models), int(maker_limit), int(max_candidates_per_maker))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    makers = find_smartphone_maker_candidates(min_models=max(min_models + 2, 7), limit=maker_limit)
    out: List[Tuple[str, str, str, str, int, str, int]] = []

    for maker_qid, maker_lbl, _ in makers:
        models = fetch_smartphone_models_for_maker(maker_qid)
        n = len(models)
        if n < min_models + 2:
            continue

        local: List[Tuple[str, str, str, str, int, str, int]] = []
        for i, m in enumerate(models):
            ref_qid = m["qid"]
            ref_lbl = _norm_lbl(m["label"])
            ref_year = int(m["year"])

            cnt_after = max(0, n - i - 1)
            cnt_before = max(0, i)

            if cnt_after >= int(min_models) and cnt_before >= 1 and ref_lbl:
                local.append((maker_qid, maker_lbl, ref_qid, ref_lbl, ref_year, "after", cnt_after))
            if cnt_before >= int(min_models) and cnt_after >= 1 and ref_lbl:
                local.append((maker_qid, maker_lbl, ref_qid, ref_lbl, ref_year, "before", cnt_before))

        local.sort(
            key=lambda x: (
                abs(x[6] - 8),
                abs(x[4] - 2018),
                0 if x[5] == "after" else 1,
            )
        )
        out.extend(local[:max_candidates_per_maker])

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out


def find_smartphone_between_model_candidates(
    min_models_between: int = 2,
    maker_limit: int = 18,
    max_candidates_per_maker: int = 10,
) -> List[Tuple[str, str, str, str, int, str, str, int, int]]:
    key = ("between_models", int(min_models_between), int(maker_limit), int(max_candidates_per_maker))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    makers = find_smartphone_maker_candidates(min_models=max(min_models_between + 5, 7), limit=maker_limit)
    out: List[Tuple[str, str, str, str, int, str, str, int, int]] = []

    for maker_qid, maker_lbl, _ in makers:
        models = fetch_smartphone_models_for_maker(maker_qid)
        n = len(models)
        if n < min_models_between + 3:
            continue

        local: List[Tuple[str, str, str, str, int, str, str, int, int]] = []
        for i in range(n):
            left = models[i]
            for j in range(i + min_models_between + 1, n):
                right = models[j]
                between = j - i - 1
                if between < int(min_models_between):
                    continue

                year_gap = int(right["year"]) - int(left["year"])
                if year_gap < 1:
                    continue

                local.append((
                    maker_qid,
                    maker_lbl,
                    left["qid"],
                    _norm_lbl(left["label"]),
                    int(left["year"]),
                    right["qid"],
                    _norm_lbl(right["label"]),
                    int(right["year"]),
                    between,
                ))

        local.sort(
            key=lambda x: (
                abs(x[8] - 4),
                abs((x[7] - x[4]) - 4),
                x[4],
                x[7],
            )
        )
        out.extend(local[:max_candidates_per_maker])

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out

def _smartphones_now_iso() -> str:
    try:
        return utc_now_z()
    except Exception:
        return dt.datetime.now(dt.timezone.utc).isoformat().replace("+00:00", "Z")

def _smartphone_make_example(
    idx: int,
    complexity: str,
    query_text_ru: str,
    constraints: Dict[str, Any],
    items: List[Tuple[str, str]],
    sparql_query: str,
    template_id: str,
) -> BenchmarkExample:
    return BenchmarkExample(
        id=f"smartphones_{complexity.lower()}_{idx:05d}",
        domain="smartphones",
        complexity=complexity,
        query_text_ru=query_text_ru,
        constraints=constraints,
        requested_count=5,
        gold_answer_qids=[q for q, _ in items],
        gold_answer_labels_ru=[l for _, l in items],
        sparql_query=sparql_query,
        created_at=_smartphones_now_iso(),
        is_advanced=False,
        template_id=template_id,
        template_family="default",
        gold_truncated=False,
        ask_validator_sparql=None,
    )


def _generate_smartphones_example_default(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 40,
) -> BenchmarkExample:
    k = 5

    if complexity == "L1":
        for _ in range(max_attempts):
            cand = _next_candidate(
                "smartphones_L1",
                lambda: find_smartphone_maker_candidates(min_models=k, limit=120),
                rng,
            )
            if not cand:
                break
            maker_qid, maker_lbl, _ = cand
            sparql, items, _ = run_smartphones_query(
                maker_qid=maker_qid,
                limit=_smartphone_gold_limit(k, complexity),
            )
            if len(items) < k:
                continue
            return _smartphone_make_example(
                idx=idx,
                complexity=complexity,
                query_text_ru=f"Назови {k} моделей смартфонов производителя «{maker_lbl}».",
                constraints={
                    "maker_qid": maker_qid,
                    "maker_label_ru": maker_lbl,
                },
                items=items,
                sparql_query=sparql,
                template_id="smartphones_maker_only",
            )
        raise RuntimeError("Smartphones L1: no valid maker candidates")

    if complexity == "L2":
        for _ in range(max_attempts):
            cand = _next_candidate(
                "smartphones_L2",
                lambda: find_smartphone_maker_year_window_candidates(
                    min_models=k,
                    maker_limit=24,
                    min_span_years=2,
                    max_span_years=4,
                    max_candidates_per_maker=12,
                ),
                rng,
            )
            if not cand:
                break
            maker_qid, maker_lbl, y1, y2, _ = cand
            sparql, items, _ = run_smartphones_query(
                maker_qid=maker_qid,
                year_from=y1,
                year_to=y2,
                limit=_smartphone_gold_limit(k, complexity),
            )
            if len(items) < k:
                continue
            return _smartphone_make_example(
                idx=idx,
                complexity=complexity,
                query_text_ru=f"Назови {k} смартфонов производителя «{maker_lbl}», выпущенных в период {y1}–{y2} годов.",
                constraints={
                    "maker_qid": maker_qid,
                    "maker_label_ru": maker_lbl,
                    "year_from": y1,
                    "year_to": y2,
                },
                items=items,
                sparql_query=sparql,
                template_id="smartphones_maker_year_window",
            )
        raise RuntimeError("Smartphones L2: no valid maker+year-window candidates")

    if complexity == "L3":
        for _ in range(max_attempts):
            cand = _next_candidate(
                "smartphones_L3",
                lambda: find_smartphone_maker_threshold_year_candidates(
                    min_models=k,
                    maker_limit=24,
                    max_candidates_per_maker=12,
                ),
                rng,
            )
            if not cand:
                break
            maker_qid, maker_lbl, mode, yy, _ = cand
            if mode == "from":
                sparql, items, _ = run_smartphones_query(
                    maker_qid=maker_qid,
                    year_from=yy,
                    limit=_smartphone_gold_limit(k, complexity),
                )
                query_text_ru = f"Назови {k} смартфонов производителя «{maker_lbl}», выпущенных не раньше {yy} года."
                constraints = {
                    "maker_qid": maker_qid,
                    "maker_label_ru": maker_lbl,
                    "year_from": yy,
                    "year_mode": "from",
                }
                template_id = "smartphones_maker_year_from"
            else:
                sparql, items, _ = run_smartphones_query(
                    maker_qid=maker_qid,
                    year_to=yy,
                    limit=_smartphone_gold_limit(k, complexity),
                )
                query_text_ru = f"Назови {k} смартфонов производителя «{maker_lbl}», выпущенных не позже {yy} года."
                constraints = {
                    "maker_qid": maker_qid,
                    "maker_label_ru": maker_lbl,
                    "year_to": yy,
                    "year_mode": "to",
                }
                template_id = "smartphones_maker_year_to"

            if len(items) < k:
                continue
            return _smartphone_make_example(
                idx=idx,
                complexity=complexity,
                query_text_ru=query_text_ru,
                constraints=constraints,
                items=items,
                sparql_query=sparql,
                template_id=template_id,
            )
        raise RuntimeError("Smartphones L3: no valid maker+threshold-year candidates")

    if complexity == "L4":
        for _ in range(max_attempts):
            cand = _next_candidate(
                "smartphones_L4",
                lambda: find_smartphone_reference_comparison_candidates(
                    min_models=k,
                    maker_limit=20,
                    max_candidates_per_maker=12,
                ),
                rng,
            )
            if not cand:
                break
            maker_qid, maker_lbl, ref_qid, ref_lbl, ref_year, relation, _ = cand
            sparql, items, _ = run_smartphones_query_relative_model(
                maker_qid=maker_qid,
                ref_model_qid=ref_qid,
                relation=relation,
                limit=_smartphone_gold_limit(k, complexity),
            )
            if len(items) < k:
                continue

            if relation == "after":
                query_text_ru = (
                    f"Назови {k} смартфонов производителя «{maker_lbl}», "
                    f"выпущенных позже модели «{ref_lbl}» ({ref_year})."
                )
                template_id = "smartphones_maker_after_model"
            else:
                query_text_ru = (
                    f"Назови {k} смартфонов производителя «{maker_lbl}», "
                    f"выпущенных раньше модели «{ref_lbl}» ({ref_year})."
                )
                template_id = "smartphones_maker_before_model"

            return _smartphone_make_example(
                idx=idx,
                complexity=complexity,
                query_text_ru=query_text_ru,
                constraints={
                    "maker_qid": maker_qid,
                    "maker_label_ru": maker_lbl,
                    "reference_model_qid": ref_qid,
                    "reference_model_label_ru": ref_lbl,
                    "reference_model_year": ref_year,
                    "relation": relation,
                },
                items=items,
                sparql_query=sparql,
                template_id=template_id,
            )
        raise RuntimeError("Smartphones L4: no valid maker+reference-model candidates")

    if complexity == "L5":
        for _ in range(max_attempts):
            cand = _next_candidate(
                "smartphones_L5",
                lambda: find_smartphone_between_model_candidates(
                    min_models_between=2,
                    maker_limit=18,
                    max_candidates_per_maker=10,
                ),
                rng,
            )
            if not cand:
                break
            maker_qid, maker_lbl, left_qid, left_lbl, left_year, right_qid, right_lbl, right_year, _ = cand
            sparql, items, _ = run_smartphones_query_between_models(
                maker_qid=maker_qid,
                left_model_qid=left_qid,
                right_model_qid=right_qid,
                limit=_smartphone_gold_limit(k, complexity),
            )
            if len(items) < 1:
                continue
            return _smartphone_make_example(
                idx=idx,
                complexity=complexity,
                query_text_ru=(
                    f"Подбери до {k} смартфонов производителя «{maker_lbl}», "
                    f"выпущенных позже модели «{left_lbl}» ({left_year}), "
                    f"но раньше модели «{right_lbl}» ({right_year})."
                ),
                constraints={
                    "maker_qid": maker_qid,
                    "maker_label_ru": maker_lbl,
                    "left_reference_model_qid": left_qid,
                    "left_reference_model_label_ru": left_lbl,
                    "left_reference_model_year": left_year,
                    "right_reference_model_qid": right_qid,
                    "right_reference_model_label_ru": right_lbl,
                    "right_reference_model_year": right_year,
                    "relation": "between_models",
                },
                items=items,
                sparql_query=sparql,
                template_id="smartphones_maker_between_models",
            )
        raise RuntimeError("Smartphones L5: no valid maker+between-models candidates")

    raise ValueError(f"Unknown complexity: {complexity}")


def generate_smartphones_example_advanced(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 40,
) -> BenchmarkExample:
    return _generate_smartphones_example_default(complexity, idx, rng, max_attempts=max_attempts)

def generate_smartphones_example(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 40,
) -> BenchmarkExample:
    return _generate_smartphones_example_default(complexity, idx, rng, max_attempts=max_attempts)


<a id="universities-airports-and-cars-domains"></a>

## 23. Universities, airports, and cars domains

Domain-specific templates for education, airport, and car-model entities.

In [23]:
import random
from typing import List, Tuple, Optional

def _gold_limit(k: int, complexity: str) -> int:
    """
    Сколько gold-элементов максимум запрашиваем у WDQS.
    Раньше код вызывал _gold_limit(...), но сам helper в ноутбуке отсутствовал,
    из-за чего генераторы (в т.ч. airports) молча падали в MAIN-цикле и только ретраились.
    """
    k = int(k)
    c = str(complexity)
    if c in ("L1", "L2"):
        return max(k + 5, 12)
    if c == "L3":
        return max(k + 6, 15)
    if c == "L4":
        return max(k + 8, 24)
    if c == "L5":
        return max(k + 5, 12)
    return max(k + 5, 12)


# Domain: Universities (RU)

# Domain: Universities (international, fast & stable)
Q_UNIVERSITY = "Q3918"  # university

UNI_COUNTRY_ROWS = [
    {"country_qid": "Q159", "country_ru": "Россия",          "macro_region_ru": "РФ"},
    {"country_qid": "Q30",  "country_ru": "США",             "macro_region_ru": "Северная Америка"},
    {"country_qid": "Q16",  "country_ru": "Канада",          "macro_region_ru": "Северная Америка"},
    {"country_qid": "Q145", "country_ru": "Великобритания",  "macro_region_ru": "Европа"},
    {"country_qid": "Q183", "country_ru": "Германия",        "macro_region_ru": "Европа"},
    {"country_qid": "Q142", "country_ru": "Франция",         "macro_region_ru": "Европа"},
    {"country_qid": "Q38",  "country_ru": "Италия",          "macro_region_ru": "Европа"},
    {"country_qid": "Q29",  "country_ru": "Испания",         "macro_region_ru": "Европа"},
    {"country_qid": "Q55",  "country_ru": "Нидерланды",      "macro_region_ru": "Европа"},
    {"country_qid": "Q39",  "country_ru": "Швейцария",       "macro_region_ru": "Европа"},
    {"country_qid": "Q36",  "country_ru": "Польша",          "macro_region_ru": "Европа"},
    {"country_qid": "Q40",  "country_ru": "Австрия",         "macro_region_ru": "Европа"},
    {"country_qid": "Q213", "country_ru": "Чехия",           "macro_region_ru": "Европа"},
    {"country_qid": "Q27",  "country_ru": "Ирландия",        "macro_region_ru": "Европа"},
    {"country_qid": "Q33",  "country_ru": "Финляндия",       "macro_region_ru": "Европа"},
]
universities_countries_df = pd.DataFrame(UNI_COUNTRY_ROWS)

def _build_universities_bucket_pool() -> pd.DataFrame:
    candidate_ranges = {
        "L1": [(None, None)],
        "L2": [(1800, 2025), (1850, 2025), (1900, 2025), (1950, 2025)],
        "L3": [(1800, 1999), (1850, 1999), (1900, 1999), (1950, 2025)],
        "L4": [(1600, 1899), (1700, 1899), (1800, 1949)],
        "L5": [(2500, 2500)],
    }
    need_k = {"L1": 5, "L2": 5, "L3": 4, "L4": 3, "L5": 3}
    rows = []

    for row in UNI_COUNTRY_ROWS:
        cqid = row["country_qid"]
        cru = row["country_ru"]
        macro = row["macro_region_ru"]

        for complexity, ranges in candidate_ranges.items():
            k = need_k[complexity]
            probe_limit = max(k, 6) if complexity != "L5" else 2

            for y1, y2 in ranges:
                where = [f"?uni wdt:P17 wd:{cqid} ."]
                if y1 is not None:
                    where += [
                        "?uni wdt:P571 ?inception .",
                        "BIND(YEAR(?inception) AS ?yy) .",
                        f"FILTER(?yy >= {int(y1)} && ?yy <= {int(y2)}) .",
                    ]
                try:
                    _, items = select_items_with_label_ru_en(
                        Q_UNIVERSITY,
                        where,
                        limit=probe_limit,
                        item_var="uni",
                        use_subclass_closure=True,
                    )
                    n = len(items)
                except Exception:
                    continue

                ok = (
                    (complexity in ("L1", "L2") and n >= 5) or
                    (complexity == "L3" and n >= 4) or
                    (complexity == "L4" and n >= 1) or
                    (complexity == "L5" and n == 0)
                )
                if ok:
                    rows.append({
                        "complexity": complexity,
                        "country_qid": cqid,
                        "country_ru": cru,
                        "macro_region_ru": macro,
                        "y1": y1 if y1 is not None else "",
                        "y2": y2 if y2 is not None else "",
                        "probe_count": n,
                    })

    return pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)

universities_bucket_pool_df = load_or_build_pool_safe(
    "universities_country_year_buckets_intl_strict_year_v2",
    _build_universities_bucket_pool,
)

def _pick_uni_bucket(complexity: str, rng: random.Random):
    df = universities_bucket_pool_df
    if isinstance(df, pd.DataFrame) and len(df) > 0:
        sub = df[df["complexity"] == complexity]
        if len(sub) > 0:
            row = sub.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
            y1 = row["y1"]
            y2 = row["y2"]
            y1 = None if y1 == "" or pd.isna(y1) else int(y1)
            y2 = None if y2 == "" or pd.isna(y2) else int(y2)
            return {
                "country_qid": str(row["country_qid"]),
                "country_ru": str(row["country_ru"]),
                "macro_region_ru": str(row["macro_region_ru"]),
                "y1": y1,
                "y2": y2,
            }

    base = rng.choice(UNI_COUNTRY_ROWS)
    if complexity == "L1":
        y1, y2 = None, None
    elif complexity == "L2":
        y1, y2 = rng.choice([(1800, 2025), (1850, 2025), (1900, 2025), (1950, 2025)])
    elif complexity == "L3":
        y1, y2 = rng.choice([(1800, 1999), (1850, 1999), (1900, 1999), (1950, 2025)])
    elif complexity == "L4":
        y1, y2 = rng.choice([(1600, 1899), (1700, 1899), (1800, 1949)])
    elif complexity == "L5":
        y1, y2 = 2500, 2500
    else:
        raise ValueError(f"Unknown complexity: {complexity}")
    return {**base, "y1": y1, "y2": y2}

def run_universities_country_query(
    country_qid: str,
    y1: Optional[int],
    y2: Optional[int],
    limit: int = 60,
) -> Tuple[str, List[Tuple[str, str]], List[str]]:
    where = [f"?uni wdt:P17 wd:{country_qid} ."]
    if y1 is not None:
        where += [
            "?uni wdt:P571 ?inception .",
                        "BIND(YEAR(?inception) AS ?yy) .",
                        f"FILTER(?yy >= {int(y1)} && ?yy <= {int(y2)}) .",
        ]

    sparql, items = select_items_with_label_ru_en(
        Q_UNIVERSITY,
        where,
        limit=limit,
        item_var="uni",
        use_subclass_closure=True,
    )
    return sparql, items, where

def _nlg_uni_country_ru(country_ru: str, macro_region_ru: str, y1: Optional[int], y2: Optional[int], k: int, complexity: str) -> str:
    if y1 is None:
        return f"Перечисли {k} университетов страны «{country_ru}»."
    if complexity == "L4":
        return (f"Перечисли {k} университетов страны «{country_ru}» ({macro_region_ru}), "
                f"которые были основаны в период {y1}–{y2}. Если таких меньше {k}, перечисли все, сколько найдётся.")
    if complexity == "L5":
        return f"Перечисли {k} университетов страны «{country_ru}», основанных в период {y1}–{y2}."
    return f"Перечисли {k} университетов страны «{country_ru}», которые были основаны в период {y1}–{y2}."

ADV_TEMPLATE_PROB_UNI = 0.0  # Disable advanced university templates for stable bucket filling

def generate_universities_example_default(complexity: str, idx: int, rng: random.Random, max_attempts: int = 40) -> BenchmarkExample:
    for _ in range(max_attempts):
        bucket = _pick_uni_bucket(complexity, rng)
        country_qid = bucket["country_qid"]
        country_ru = bucket["country_ru"]
        macro_region_ru = bucket["macro_region_ru"]
        y1 = bucket["y1"]
        y2 = bucket["y2"]

        if complexity == "L1":
            k = 5
        elif complexity == "L2":
            k = 5
        elif complexity == "L3":
            k = 4
        elif complexity == "L4":
            k = 3
        elif complexity == "L5":
            k = 3
        else:
            raise ValueError(f"Unknown complexity: {complexity}")

        sparql, items, where = run_universities_country_query(
            country_qid,
            y1,
            y2,
            limit=_gold_limit(k, complexity),
        )
        n = len(items)

        if complexity in ("L1", "L2") and n < k:
            continue
        if complexity == "L3" and n < k:
            continue
        if complexity == "L4" and n == 0:
            continue
        if complexity == "L5" and n != 0:
            continue

        ask = build_ask_validator(Q_UNIVERSITY, where, item_var="uni")
        return BenchmarkExample(
            id=f"universities_{complexity.lower()}_{idx:04d}",
            domain="universities",
            complexity=complexity,
            query_text_ru=_nlg_uni_country_ru(country_ru, macro_region_ru, y1, y2, k, complexity),
            constraints={
                "country_qid": country_qid,
                "country_ru": country_ru,
                "macro_region_ru": macro_region_ru,
                "y1": y1,
                "y2": y2,
            },
            requested_count=k,
            gold_answer_qids=[q for q, _ in items],
            gold_answer_labels_ru=[l for _, l in items],
            sparql_query=sparql,
            created_at=utc_now_z(),
            is_advanced=False,
            template_id="universities_country_year_fast",
            template_family="default",
            gold_truncated=len(items) >= _gold_limit(k, complexity),
            ask_validator_sparql=ask,
        )

    raise RuntimeError(f"Universities default:{complexity} failed after {max_attempts} attempts")

def generate_universities_example_advanced(complexity: str, idx: int, rng: random.Random, max_attempts: int = 20) -> BenchmarkExample:
    raise RuntimeError("Universities advanced disabled for speed/stability")

def generate_universities_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 40) -> BenchmarkExample:
    return generate_universities_example_default(complexity, idx, rng, max_attempts=max_attempts)

print("✅ Patched universities: strict-year buckets + universities label fallback ready")

# Domain: Airports (bucket-based, stable & fast)
Q_AIRPORT = ensure_qid("аэропорт", fallback_qid="Q1248784")
Q_CITY = "Q515"

AIRPORT_COUNTRY_ROWS = [
    {"country_qid": "Q159", "country_ru": "Россия"},
    {"country_qid": "Q30",  "country_ru": "США"},
    {"country_qid": "Q16",  "country_ru": "Канада"},
    {"country_qid": "Q145", "country_ru": "Великобритания"},
    {"country_qid": "Q183", "country_ru": "Германия"},
    {"country_qid": "Q142", "country_ru": "Франция"},
    {"country_qid": "Q38",  "country_ru": "Италия"},
    {"country_qid": "Q29",  "country_ru": "Испания"},
    {"country_qid": "Q55",  "country_ru": "Нидерланды"},
    {"country_qid": "Q17",  "country_ru": "Япония"},
    {"country_qid": "Q408", "country_ru": "Австралия"},
    {"country_qid": "Q155", "country_ru": "Бразилия"},
]

_AIRPORT_QUERY_CACHE: Dict[Tuple[str, int, bool, int], Tuple[str, List[Tuple[str, str]], List[str]]] = {}

def run_airports_query(country_qid: str, pop_min: int = 0, require_code: bool = False, limit: int = 60) -> Tuple[str, List[Tuple[str,str]], List[str]]:
    key = (str(country_qid), int(pop_min), bool(require_code), int(limit))
    if key in _AIRPORT_QUERY_CACHE:
        return _AIRPORT_QUERY_CACHE[key]

    where = [
        f"?item wdt:P17 wd:{country_qid} .",
    ]
    if require_code:
        where.append("{ ?item wdt:P238 ?code . } UNION { ?item wdt:P239 ?code . }")

    if int(pop_min) > 0:
        where.extend([
            "OPTIONAL {",
            "  { ?item wdt:P931 ?city . ?city wdt:P31/wdt:P279* wd:Q515 . }",
            "  UNION",
            "  { ?item wdt:P131 ?city . ?city wdt:P31/wdt:P279* wd:Q515 . }",
            "  UNION",
            "  { ?item wdt:P276 ?city . ?city wdt:P31/wdt:P279* wd:Q515 . }",
            "  ?city wdt:P1082 ?pop .",
            "}",
            "BIND(COALESCE(?pop, 0) AS ?p) .",
            f"FILTER(?p >= {int(pop_min)}) .",
        ])

    sparql, items = select_items_with_label_ru_en(
        Q_AIRPORT,
        where,
        limit=limit,
        item_var="item",
        use_subclass_closure=True,
    )
    res = (sparql, items, where)
    _AIRPORT_QUERY_CACHE[key] = res
    return res

def build_airports_bucket_pool() -> pd.DataFrame:
    rows = []

    def add_bucket(complexity: str, country_qid: str, country_ru: str, pop_min: int, require_code: bool, n_items: int):
        rows.append({
            "complexity": complexity,
            "country_qid": country_qid,
            "country_ru": country_ru,
            "pop_min": int(pop_min),
            "require_code": int(require_code),
            "n_items": int(n_items),
        })

    for crow in AIRPORT_COUNTRY_ROWS:
        c_qid = crow["country_qid"]
        c_ru = crow["country_ru"]

        _, items, _ = run_airports_query(c_qid, pop_min=0, require_code=False, limit=30)
        if len(items) >= 5:
            add_bucket("L1", c_qid, c_ru, 0, False, len(items))

        _, items, _ = run_airports_query(c_qid, pop_min=0, require_code=True, limit=30)
        if len(items) >= 5:
            add_bucket("L2", c_qid, c_ru, 0, True, len(items))

        for pop_min in (200_000, 500_000, 1_000_000):
            _, items, _ = run_airports_query(c_qid, pop_min=pop_min, require_code=False, limit=30)
            if len(items) >= 5:
                add_bucket("L3", c_qid, c_ru, pop_min, False, len(items))

        for pop_min in (2_000_000, 3_000_000, 5_000_000):
            _, items, _ = run_airports_query(c_qid, pop_min=pop_min, require_code=True, limit=20)
            if len(items) >= 1:
                add_bucket("L4", c_qid, c_ru, pop_min, True, len(items))

        _, items, _ = run_airports_query(c_qid, pop_min=50_000_000, require_code=True, limit=10)
        if len(items) == 0:
            add_bucket("L5", c_qid, c_ru, 50_000_000, True, 0)

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("airports bucket pool is empty")
    return df

airports_buckets_df = load_or_build_pool_safe(
    "airports_buckets_ru_v4",
    build_airports_bucket_pool,
)

def pick_airports_bucket(complexity: str, rng: random.Random) -> Dict[str, Any]:
    if not isinstance(airports_buckets_df, pd.DataFrame) or len(airports_buckets_df) == 0:
        raise RuntimeError("airports_buckets_df is empty")

    sub = airports_buckets_df[airports_buckets_df["complexity"] == complexity]
    if len(sub) == 0:
        raise RuntimeError(f"no airports buckets for {complexity}")

    row = sub.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
    return {
        "country_qid": str(row["country_qid"]),
        "country_ru": str(row["country_ru"]),
        "pop_min": int(row["pop_min"]),
        "require_code": bool(int(row["require_code"])),
        "n_items": int(row["n_items"]),
    }

def nlg_airports_ru(complexity: str, country_ru: str, pop_min: int, require_code: bool, k: int) -> str:
    if complexity == "L1":
        return f"Назови {k} аэропортов страны «{country_ru}»."
    if complexity == "L2":
        return f"Назови {k} аэропортов страны «{country_ru}», у которых указан код IATA или ICAO."
    if complexity == "L3":
        return f"Назови {k} аэропортов страны «{country_ru}», расположенных в городах с населением не меньше {pop_min} человек."
    if complexity == "L4":
        return f"Перечисли {k} аэропортов страны «{country_ru}», у которых указан код IATA или ICAO и которые расположены в городах с населением не меньше {pop_min} человек. Если таких меньше {k}, перечисли все, сколько есть."
    if complexity == "L5":
        return f"Назови {k} аэропортов страны «{country_ru}», у которых указан код IATA или ICAO и которые расположены в городах с населением не меньше {pop_min} человек. Если таких нет, так и напиши."
    raise ValueError(f"Unknown complexity: {complexity}")

ADV_TEMPLATE_PROB_AIRPORTS = 0.0

def generate_airports_example_default(complexity: str, idx: int, rng: random.Random, max_attempts: int = 25) -> BenchmarkExample:
    template_id = "airports_bucket_country_code_population"
    for _ in range(max_attempts):
        bucket = pick_airports_bucket(complexity, rng)

        if complexity in ("L1", "L2", "L3"):
            k = 5
        elif complexity == "L4":
            k = 12
        elif complexity == "L5":
            k = 5
        else:
            raise ValueError(f"Unknown complexity: {complexity}")

        sparql, items, where = run_airports_query(
            bucket["country_qid"],
            pop_min=bucket["pop_min"],
            require_code=bucket["require_code"],
            limit=_gold_limit(k, complexity),
        )
        n = len(items)

        if complexity in ("L1", "L2", "L3") and n < k:
            continue
        if complexity == "L4" and n == 0:
            continue
        if complexity == "L5" and n != 0:
            continue

        ask = build_ask_validator(Q_AIRPORT, where, item_var="item")
        return BenchmarkExample(
            id=f"airports_{complexity.lower()}_{idx:04d}",
            domain="airports",
            complexity=complexity,
            query_text_ru=nlg_airports_ru(complexity, bucket["country_ru"], bucket["pop_min"], bucket["require_code"], k),
            constraints={
                "country_qid": bucket["country_qid"],
                "country_ru": bucket["country_ru"],
                "pop_min": bucket["pop_min"],
                "require_code": bucket["require_code"],
            },
            requested_count=k,
            gold_answer_qids=[q for q, _ in items],
            gold_answer_labels_ru=[l for _, l in items],
            sparql_query=sparql,
            created_at=utc_now_z(),
            is_advanced=False,
            template_id=template_id,
            template_family="default",
            gold_truncated=len(items) >= _gold_limit(k, complexity),
            ask_validator_sparql=ask,
        )
    raise RuntimeError(f"Airports default:{complexity} failed after {max_attempts} attempts")

def generate_airports_example_advanced(complexity: str, idx: int, rng: random.Random, max_attempts: int = 1) -> BenchmarkExample:
    raise RuntimeError("airports advanced disabled")

def generate_airports_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 25) -> BenchmarkExample:
    return generate_airports_example_default(complexity, idx, rng, max_attempts=max_attempts)


Q_CAR_MODEL = ensure_qid("модель автомобиля", fallback_qid="Q3231690")  # automobile model

car_manufacturers_df = load_or_build_pool_safe(
    "cars_manufacturers_ru",
    lambda: build_value_pool_ru(Q_CAR_MODEL, "P176", "manu", limit=240),  # manufacturer
)

CAR_COUNTRY_ROWS = [
    {"country_qid": "Q17",  "country_ru": "Япония"},
    {"country_qid": "Q183", "country_ru": "Германия"},
    {"country_qid": "Q30",  "country_ru": "США"},
    {"country_qid": "Q142", "country_ru": "Франция"},
    {"country_qid": "Q38",  "country_ru": "Италия"},
    {"country_qid": "Q884", "country_ru": "Южная Корея"},
    {"country_qid": "Q148", "country_ru": "Китай"},
    {"country_qid": "Q145", "country_ru": "Великобритания"},
]

CAR_YEAR_RANGES_L2 = [(1960, 1979), (1980, 1999), (2000, 2020)]
CAR_YEAR_RANGES_L3 = [(1960, 1979), (1980, 1999), (2000, 2020)]
CAR_YEAR_RANGES_L4 = [(1960, 1999), (1980, 2020), (1950, 1985)]

def pick_car_country(rng: random.Random) -> Tuple[str, str]:
    row = rng.choice(CAR_COUNTRY_ROWS)
    return row["country_qid"], row["country_ru"]

def _car_country_where(country_qid: str) -> List[str]:
    return [
        "?item wdt:P176 ?manu .",
        "OPTIONAL { ?manu (wdt:P17|wdt:P495) ?manu_country . }",
        "OPTIONAL { ?item (wdt:P495|wdt:P17) ?item_country . }",
        f"FILTER((BOUND(?manu_country) && ?manu_country = wd:{country_qid}) || (BOUND(?item_country) && ?item_country = wd:{country_qid})) .",
    ]

def _car_country_or_where(country1_qid: str, country2_qid: str) -> List[str]:
    return [
        "?item wdt:P176 ?manu .",
        "OPTIONAL { ?manu (wdt:P17|wdt:P495) ?manu_country . }",
        "OPTIONAL { ?item (wdt:P495|wdt:P17) ?item_country . }",
        f"FILTER((BOUND(?manu_country) && (?manu_country = wd:{country1_qid} || ?manu_country = wd:{country2_qid})) || (BOUND(?item_country) && (?item_country = wd:{country1_qid} || ?item_country = wd:{country2_qid}))) .",
    ]

def _car_year_where_strict(y1: int, y2: int) -> List[str]:
    return [
        "?item wdt:P571 ?inception .",
        "BIND(YEAR(?inception) AS ?yy) .",
        f"FILTER(?yy >= {int(y1)} && ?yy <= {int(y2)}) .",
    ]

def _car_manufacturer_where(manu_qid: str) -> List[str]:
    return [
        f"?item wdt:P176 wd:{manu_qid} .",
    ]

CAR_BRAND_GENERIC_STOPWORDS = {
    "motor", "motors", "company", "corporation", "corp", "group", "auto",
    "automobile", "automobiles", "cars", "car", "co", "ltd", "inc", "ag", "gmbh",
    "spa", "s.p.a", "limited", "holding", "holdings"
}

CAR_MANUFACTURER_BRAND_HINTS = {
    "ford motor company": ["ford"],
    "toyota motor corporation": ["toyota"],
    "volkswagen ag": ["volkswagen"],
    "hyundai motor company": ["hyundai"],
    "honda motor company": ["honda"],
    "nissan motor": ["nissan"],
    "renault": ["renault"],
    "peugeot": ["peugeot"],
    "citroen": ["citroen", "citroën"],
    "fiat": ["fiat"],
    "bmw": ["bmw"],
    "bayerische motoren werke": ["bmw"],
    "mercedes benz group": ["mercedes-benz", "mercedes benz"],
    "mercedes benz": ["mercedes-benz", "mercedes benz"],
}

CAR_MANUFACTURER_BRAND_BLACKLIST = {
    "ford": ["lincoln", "mercury", "mercedes-benz", "mercedes benz"],
    "bmw": ["mini", "rolls-royce", "rolls royce"],
    "hyundai": ["kia", "genesis"],
    "toyota": ["lexus", "daihatsu", "hino"],
}

def _car_norm_brand_text(s: str) -> str:
    s = (s or "").strip().lower()
    for a, b in [
        ("ё", "е"), ("-", " "), ("/", " "), ("&", " and "),
        ("(", " "), (")", " "), (",", " "), (".", " "), ("'", " "), ('"', ' '),
    ]:
        s = s.replace(a, b)
    return " ".join(s.split())

def _car_brand_hints_for_manufacturer(manu_label: str) -> List[str]:
    raw = _car_norm_brand_text(manu_label)
    if not raw:
        return []

    hints: List[str] = []
    for key, vals in CAR_MANUFACTURER_BRAND_HINTS.items():
        if key in raw:
            hints.extend(vals)

    tokens = [t for t in raw.split() if t not in CAR_BRAND_GENERIC_STOPWORDS and len(t) >= 3]
    if tokens:
        hints.append(tokens[0])
        if len(tokens) >= 2:
            hints.append(" ".join(tokens[:2]))
    hints.append(raw)

    out: List[str] = []
    seen = set()
    for h in hints:
        h = _car_norm_brand_text(h)
        if h and h not in seen:
            seen.add(h)
            out.append(h)
    return out

def _car_brand_blacklist_for_manufacturer(manu_label: str) -> List[str]:
    hints = _car_brand_hints_for_manufacturer(manu_label)
    out: List[str] = []
    seen = set()
    for h in hints:
        for bad in CAR_MANUFACTURER_BRAND_BLACKLIST.get(h, []):
            bad = _car_norm_brand_text(bad)
            if bad and bad not in seen:
                seen.add(bad)
                out.append(bad)
    return out

def _car_brand_where_for_manufacturer(manu_label: str) -> List[str]:
    hints = _car_brand_hints_for_manufacturer(manu_label)
    blacklist = _car_brand_blacklist_for_manufacturer(manu_label)
    if not hints:
        return []

    contains_any_hint = " || ".join([f'CONTAINS(?brand_label_lc, "{h}")' for h in hints])
    not_blacklisted = " && ".join([f'!CONTAINS(?brand_label_lc, "{b}")' for b in blacklist]) or "true"

    return [
        "?item wdt:P1716 ?brand .",
        "OPTIONAL { ?brand rdfs:label ?brand_ru FILTER(LANG(?brand_ru)='ru') }",
        "OPTIONAL { ?brand rdfs:label ?brand_en FILTER(LANG(?brand_en)='en') }",
        "BIND(LCASE(COALESCE(STR(?brand_ru), STR(?brand_en), "")) AS ?brand_label_lc) .",
        f"FILTER(({contains_any_hint}) && ({not_blacklisted})) .",
    ]

def _run_car_models_where(where_lines: List[str], limit: int = 60) -> Tuple[str, List[Tuple[str, str]], List[str]]:
    sparql, items = select_items_with_label_ru_en(
        Q_CAR_MODEL,
        where_lines,
        limit=limit,
        item_var="item",
        use_subclass_closure=True,
    )
    return sparql, items, where_lines

def run_car_models_country_query(country_qid: str, limit: int = 60) -> Tuple[str, List[Tuple[str, str]], List[str]]:
    return _run_car_models_where([
        *_car_country_where(country_qid),
    ], limit=limit)

def run_car_models_country_year_query(country_qid: str, y1: int, y2: int, limit: int = 60) -> Tuple[str, List[Tuple[str, str]], List[str]]:
    return _run_car_models_where([
        *_car_country_where(country_qid),
        *_car_year_where_strict(y1, y2),
    ], limit=limit)

def run_car_models_country_year_manufacturer_query(country_qid: str, manu_qid: str, manu_label: str, y1: int, y2: int, limit: int = 60) -> Tuple[str, List[Tuple[str, str]], List[str]]:
    return _run_car_models_where([
        *_car_country_where(country_qid),
        *_car_manufacturer_where(manu_qid),
        *_car_brand_where_for_manufacturer(manu_label),
        *_car_year_where_strict(y1, y2),
    ], limit=limit)

def run_car_models_two_countries_year_query(country1_qid: str, country2_qid: str, y1: int, y2: int, limit: int = 60) -> Tuple[str, List[Tuple[str, str]], List[str]]:
    return _run_car_models_where([
        *_car_country_or_where(country1_qid, country2_qid),
        *_car_year_where_strict(y1, y2),
    ], limit=limit)

_CAR_MANUFACTURER_CANDIDATES_CACHE: Dict[Tuple[str, int, int, int, int], List[Tuple[str, str, int]]] = {}

def find_car_manufacturer_candidates(country_qid: str, y1: int, y2: int, min_models: int = 5, limit: int = 20) -> List[Tuple[str, str, int]]:
    key = (str(country_qid), int(y1), int(y2), int(min_models), int(limit))
    if key in _CAR_MANUFACTURER_CANDIDATES_CACHE:
        return _CAR_MANUFACTURER_CANDIDATES_CACHE[key]

    where = "\n      ".join([
        f"?item wdt:P31/wdt:P279* wd:{Q_CAR_MODEL} .",
        "?item wdt:P176 ?manu .",
        "OPTIONAL { ?manu (wdt:P17|wdt:P495) ?manu_country . }",
        "OPTIONAL { ?item (wdt:P495|wdt:P17) ?item_country . }",
        f"FILTER((BOUND(?manu_country) && ?manu_country = wd:{country_qid}) || (BOUND(?item_country) && ?item_country = wd:{country_qid})) .",
        "?item wdt:P571 ?inception .",
        "BIND(YEAR(?inception) AS ?yy) .",
        f"FILTER(?yy >= {int(y1)} && ?yy <= {int(y2)}) .",
    ])

    sparql = f"""
    SELECT ?manu ?manuLabel (COUNT(DISTINCT ?item) AS ?cnt) WHERE {{
      {where}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
    }}
    GROUP BY ?manu ?manuLabel
    HAVING(COUNT(DISTINCT ?item) >= {int(min_models)})
    ORDER BY DESC(COUNT(DISTINCT ?item))
    LIMIT {int(limit)}
    """

    rows = rows_from_select(wd.sparql_select(sparql))
    out: List[Tuple[str, str, int]] = []
    for r in rows:
        manu_qid = uri_to_qid(r.get("manu", ""))
        manu_label = r.get("manuLabel")
        cnt_raw = r.get("cnt", "0")
        try:
            cnt = int(float(cnt_raw))
        except Exception:
            cnt = 0
        if manu_qid and manu_label and cnt >= int(min_models):
            out.append((manu_qid, manu_label, cnt))

    _CAR_MANUFACTURER_CANDIDATES_CACHE[key] = out
    return out

def nlg_cars_l1_ru(country_ru: str, k: int) -> str:
    return f"Назови {k} моделей автомобилей производителей из страны {country_ru}."

def nlg_cars_l2_ru(country_ru: str, y1: int, y2: int, k: int) -> str:
    return f"Назови {k} моделей автомобилей производителей из страны {country_ru}, у которых указан год появления модели в диапазоне {y1}–{y2}."

def nlg_cars_l3_ru(country_ru: str, manu_ru: str, y1: int, y2: int, k: int) -> str:
    return f"Назови {k} моделей автомобилей производителя {manu_ru} из страны {country_ru}, у которых указан год появления модели в диапазоне {y1}–{y2}."

def nlg_cars_l4_ru(country1_ru: str, country2_ru: str, y1: int, y2: int, k: int) -> str:
    return f"Назови {k} моделей автомобилей производителей из страны {country1_ru} или {country2_ru}, у которых указан год появления модели в диапазоне {y1}–{y2}."

def nlg_cars_l5_ru(country_ru: str, manu_ru: str, y1: int, y2: int, k: int) -> str:
    return f"Назови {k} моделей автомобилей производителя {manu_ru} из страны {country_ru}, у которых указан год появления модели в диапазоне {y1}–{y2}."

ADV_TEMPLATE_PROB_CARS = 0.0

def generate_cars_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 80) -> BenchmarkExample:
    if complexity == "L1":
        template_id = "cars_country_only"
        k = 5
        for _ in range(max_attempts):
            c_qid, c_ru = pick_car_country(rng)
            sparql, items, where = run_car_models_country_query(c_qid, limit=_gold_limit(k, complexity))
            if len(items) < k:
                continue

            ask = build_ask_validator(Q_CAR_MODEL, where, item_var="item")
            return BenchmarkExample(
                id=f"cars_{complexity.lower()}_{idx:04d}",
                domain="cars",
                complexity=complexity,
                query_text_ru=nlg_cars_l1_ru(c_ru, k),
                constraints={"country_qid": c_qid, "country_ru": c_ru},
                requested_count=k,
                gold_answer_qids=[q for q, _ in items],
                gold_answer_labels_ru=[l for _, l in items],
                sparql_query=sparql,
                created_at=utc_now_z(),
                is_advanced=False,
                template_id=template_id,
                template_family="default",
                gold_truncated=len(items) >= _gold_limit(k, complexity),
                ask_validator_sparql=ask,
            )
        raise RuntimeError(f"Cars {complexity} failed after {max_attempts} attempts")

    if complexity == "L2":
        template_id = "cars_country_year"
        k = 5
        for _ in range(max_attempts):
            c_qid, c_ru = pick_car_country(rng)
            y1, y2 = rng.choice(CAR_YEAR_RANGES_L2)
            sparql, items, where = run_car_models_country_year_query(c_qid, y1, y2, limit=_gold_limit(k, complexity))
            if len(items) < k:
                continue

            ask = build_ask_validator(Q_CAR_MODEL, where, item_var="item")
            return BenchmarkExample(
                id=f"cars_{complexity.lower()}_{idx:04d}",
                domain="cars",
                complexity=complexity,
                query_text_ru=nlg_cars_l2_ru(c_ru, y1, y2, k),
                constraints={"country_qid": c_qid, "country_ru": c_ru, "y1": y1, "y2": y2},
                requested_count=k,
                gold_answer_qids=[q for q, _ in items],
                gold_answer_labels_ru=[l for _, l in items],
                sparql_query=sparql,
                created_at=utc_now_z(),
                is_advanced=False,
                template_id=template_id,
                template_family="default",
                gold_truncated=len(items) >= _gold_limit(k, complexity),
                ask_validator_sparql=ask,
            )
        raise RuntimeError(f"Cars {complexity} failed after {max_attempts} attempts")

    if complexity == "L3":
        template_id = "cars_country_year_manufacturer"
        k = 5
        for _ in range(max_attempts):
            c_qid, c_ru = pick_car_country(rng)
            y1, y2 = rng.choice(CAR_YEAR_RANGES_L3)
            candidates = find_car_manufacturer_candidates(c_qid, y1, y2, min_models=k, limit=20)
            if not candidates:
                continue

            top = candidates[:min(len(candidates), 8)]
            manu_qid, manu_ru, _ = rng.choice(top)
            sparql, items, where = run_car_models_country_year_manufacturer_query(
                c_qid, manu_qid, manu_ru, y1, y2, limit=_gold_limit(k, complexity)
            )
            if len(items) < k:
                continue

            ask = build_ask_validator(Q_CAR_MODEL, where, item_var="item")
            return BenchmarkExample(
                id=f"cars_{complexity.lower()}_{idx:04d}",
                domain="cars",
                complexity=complexity,
                query_text_ru=nlg_cars_l3_ru(c_ru, manu_ru, y1, y2, k),
                constraints={
                    "country_qid": c_qid,
                    "country_ru": c_ru,
                    "manufacturer_qid": manu_qid,
                    "manufacturer_ru": manu_ru,
                    "y1": y1,
                    "y2": y2,
                },
                requested_count=k,
                gold_answer_qids=[q for q, _ in items],
                gold_answer_labels_ru=[l for _, l in items],
                sparql_query=sparql,
                created_at=utc_now_z(),
                is_advanced=False,
                template_id=template_id,
                template_family="default",
                gold_truncated=len(items) >= _gold_limit(k, complexity),
                ask_validator_sparql=ask,
            )
        raise RuntimeError(f"Cars {complexity} failed after {max_attempts} attempts")

    if complexity == "L4":
        template_id = "cars_two_countries_year"
        k = 8
        for _ in range(max_attempts):
            c1_qid, c1_ru = pick_car_country(rng)
            c2_qid, c2_ru = pick_car_country(rng)
            if c1_qid == c2_qid:
                continue
            y1, y2 = rng.choice(CAR_YEAR_RANGES_L4)

            sparql, items, where = run_car_models_two_countries_year_query(
                c1_qid, c2_qid, y1, y2, limit=_gold_limit(k, complexity)
            )
            if len(items) < k:
                continue

            ask = build_ask_validator(Q_CAR_MODEL, where, item_var="item")
            return BenchmarkExample(
                id=f"cars_{complexity.lower()}_{idx:04d}",
                domain="cars",
                complexity=complexity,
                query_text_ru=nlg_cars_l4_ru(c1_ru, c2_ru, y1, y2, k),
                constraints={
                    "country1_qid": c1_qid,
                    "country1_ru": c1_ru,
                    "country2_qid": c2_qid,
                    "country2_ru": c2_ru,
                    "y1": y1,
                    "y2": y2,
                },
                requested_count=k,
                gold_answer_qids=[q for q, _ in items],
                gold_answer_labels_ru=[l for _, l in items],
                sparql_query=sparql,
                created_at=utc_now_z(),
                is_advanced=False,
                template_id=template_id,
                template_family="default",
                gold_truncated=len(items) >= _gold_limit(k, complexity),
                ask_validator_sparql=ask,
            )
        raise RuntimeError(f"Cars {complexity} failed after {max_attempts} attempts")

    if complexity == "L5":
        template_id = "cars_country_manufacturer_future_year_zero"
        k = 5
        for _ in range(max_attempts):
            c_qid, c_ru = pick_car_country(rng)
            seed_y1, seed_y2 = rng.choice(CAR_YEAR_RANGES_L3)
            candidates = find_car_manufacturer_candidates(c_qid, seed_y1, seed_y2, min_models=2, limit=20)
            if not candidates:
                continue

            manu_qid, manu_ru, _ = rng.choice(candidates[:min(len(candidates), 8)])
            y1, y2 = 2500, 2505
            sparql, items, where = run_car_models_country_year_manufacturer_query(
                c_qid, manu_qid, manu_ru, y1, y2, limit=_gold_limit(k, complexity)
            )
            if len(items) != 0:
                continue

            ask = build_ask_validator(Q_CAR_MODEL, where, item_var="item")
            return BenchmarkExample(
                id=f"cars_{complexity.lower()}_{idx:04d}",
                domain="cars",
                complexity=complexity,
                query_text_ru=nlg_cars_l5_ru(c_ru, manu_ru, y1, y2, k),
                constraints={
                    "country_qid": c_qid,
                    "country_ru": c_ru,
                    "manufacturer_qid": manu_qid,
                    "manufacturer_ru": manu_ru,
                    "y1": y1,
                    "y2": y2,
                },
                requested_count=k,
                gold_answer_qids=[],
                gold_answer_labels_ru=[],
                sparql_query=sparql,
                created_at=utc_now_z(),
                is_advanced=False,
                template_id=template_id,
                template_family="default",
                gold_truncated=False,
                ask_validator_sparql=ask,
            )
        raise RuntimeError(f"Cars {complexity} failed after {max_attempts} attempts")

    raise ValueError(f"Unknown complexity: {complexity}")

def generate_cars_example_default(complexity: str, idx: int, rng: random.Random, max_attempts: int = 80) -> BenchmarkExample:
    return generate_cars_example(complexity, idx, rng, max_attempts=max_attempts)

def generate_cars_example_advanced(complexity: str, idx: int, rng: random.Random, max_attempts: int = 80) -> BenchmarkExample:
    return generate_cars_example(complexity, idx, rng, max_attempts=max_attempts)


if 'DOMAIN_PAUSE_S' in globals() and DOMAIN_PAUSE_S and DOMAIN_PAUSE_S > 0:
    time.sleep(float(DOMAIN_PAUSE_S))


✅ Patched universities: strict-year buckets + universities label fallback ready


<a id="cinema-kinopoisk-extension"></a>

## 24. Cinema Kinopoisk extension

Additional L5 templates focused on Kinopoisk-related constraints.

In [24]:
import requests
import random
import time

KP_API_KEY = "G5B5M00-TT54Y2H-K0GZJYS-17G6727"

FAMOUS_ACTORS_KP = [
    "Леонардо ДиКаприо", "Брэд Питт", "Том Круз",
    "Киану Ривз", "Кристиан Бэйл", "Мэттью МакКонахи",
    "Джонни Депп", "Том Хэнкс", "Уилл Смит", "Скарлетт Йоханссон",
    "Марго Робби", "Джим Керри"
]

SAFE_CINEMA_GENRES_KP = [
    "драма", "боевик", "фантастика", "триллер", "криминал", "комедия"
]

LENGTH_CONDITIONS = [
    (1, 90, "не более 1.5 часов (90 минут)"),
    (1, 120, "не более 2 часов"),
    (120, 1000, "более 2 часов"),
    (1, 180, "не более 3 часов")
]

def generate_cinema_example_kinopoisk_l5_length(complexity: str, idx: int, rng: random.Random, max_attempts: int = 25) -> BenchmarkExample:
    if complexity != "L5":
        return BenchmarkExample(
            id=f"cinema_skip_{complexity.lower()}_{idx:04d}", domain="cinema", complexity=complexity, query_text_ru="(Пропущено)",
            constraints={"skipped": True}, requested_count=5, gold_answer_qids=[], gold_answer_labels_ru=[], sparql_query="",
            created_at=utc_now_z() if "utc_now_z" in globals() else ""
        )

    for attempt in range(max_attempts):
        g_ru = rng.choice(SAFE_CINEMA_GENRES_KP)
        actor_ru = rng.choice(FAMOUS_ACTORS_KP)
        
        min_len, max_len, len_text = rng.choice(LENGTH_CONDITIONS)
        
        score = rng.choice([6.5, 7.0, 7.5])
        src_name = rng.choice(["kp", "imdb"])
        
        url = "https://api.kinopoisk.dev/v1.4/movie"
        headers = {
            "X-API-KEY": KP_API_KEY,
            "accept": "application/json"
        }
        
        params = {
            "typeNumber": "1", 
            "genres.name": g_ru,
            "persons.name": actor_ru,
            "movieLength": f"{min_len}-{max_len}",
            f"rating.{src_name}": f"{score}-10",
            "votes.kp": "15000-5000000",
            "limit": 100,
            "selectFields": ["id", "name", "movieLength", "rating", "persons", "genres"]
        }
        
        print(f"[{idx:03d} | Поп. {attempt+1}/{max_attempts}] L5: {actor_ru}, {g_ru}, {min_len}-{max_len} мин, {src_name}>={score} ...", end=" ")
        
        try:
            resp = requests.get(url, headers=headers, params=params, timeout=10)
            if resp.status_code != 200:
                print(f"Ошибка API: {resp.status_code}")
                time.sleep(1)
                continue
            data = resp.json()
        except Exception as e:
            print(f"Сбой: {e}")
            time.sleep(1)
            continue
            
        docs = data.get("docs", [])
        
        valid_items = []
        for d in docs:
            name = d.get("name")
            if not name:
                continue
                
            is_real_actor = False
            for p in d.get("persons", []):
                if p.get("name") == actor_ru and p.get("enProfession") == "actor":
                    is_real_actor = True
                    break
                    
            if is_real_actor:
                valid_items.append((str(d.get("id")), name))
        
        n = len(valid_items)
        
        if n < 2:
            print(f"Мало настоящих фильмов ({n} шт). Ищем дальше.")
            continue 
            
        print(f"Найдено {n} шт! Успех.")
        
        k = rng.choice([2, 3, 4, 5])
        actual_k = min(k, n)
        
        src_ru = "IMDb" if src_name == "imdb" else "Кинопоиск"
        
        q_ru = f"Назови {actual_k} фильма в жанре «{g_ru}» с участием актера {actor_ru}, которые идут {len_text} и имеют рейтинг по {src_ru} от {score} и выше."
        if n > actual_k:
            q_ru += " Перечисли любые подходящие из них."
        else:
            q_ru += " Перечисли все подходящие."

        gold_qids = [f"kp:{_id}" for _id, _ in valid_items]
        gold_lbls = [name for _, name in valid_items]
        
        fake_sparql = f"# Kinopoisk L5 Length\\n# Genre: {g_ru}\\n# Actor: {actor_ru}\\n# Length: {min_len}-{max_len} min\\n# Rating {src_ru} >= {score}\\n# Votes >= 15000\\n# typeNumber: 1 (Movie)"
                
        return BenchmarkExample(
            id=f"cinema_l5_length_{idx:04d}", domain="cinema", complexity="L5",
            query_text_ru=q_ru,
            constraints={
                "type": "cinema_l5_length", 
                "genre_ru": g_ru, 
                "actor_ru": actor_ru, 
                "movie_length_range": (min_len, max_len),
                "rating": (src_name, ">=", float(score))
            },
            requested_count=actual_k, 
            gold_answer_qids=gold_qids, 
            gold_answer_labels_ru=gold_lbls,
            sparql_query=fake_sparql, 
            created_at=utc_now_z() if "utc_now_z" in globals() else "",
            is_advanced=True, 
            template_id="cinema_l5_length", 
            template_family="kinopoisk_api",
            gold_truncated=(n >= 100), 
            ask_validator_sparql=None
        )
        
    print(f"[{idx:03d}] Сдаемся.")
    return BenchmarkExample(
        id=f"cinema_l5_fail_{idx:04d}", domain="cinema", complexity="L5",
        query_text_ru="(Не удалось сгенерировать)", constraints={"failed": True},
        requested_count=3, gold_answer_qids=[], gold_answer_labels_ru=[], sparql_query="",
        created_at=utc_now_z() if "utc_now_z" in globals() else ""
    )

generate_cinema_example = generate_cinema_example_kinopoisk_l5_length
if "DOMAIN_GENERATORS" in globals():
    DOMAIN_GENERATORS["cinema"] = generate_cinema_example_kinopoisk_l5_length


<a id="final-dataset-assembly"></a>

## 25. Final dataset assembly

Append-only generation loop, strict Russian-label postprocessing, deduplication, progress reporting, and JSONL writing.

In [ ]:
import os, json, random, time
from dataclasses import asdict
from collections import Counter
from typing import Dict, List, Tuple, Any, Optional

from IPython.display import clear_output

COMPLEXITY_ORDER = ["L1", "L2", "L3", "L4", "L5"]

# Keep non-cinema golds only when a reliable Russian-facing label is available.

RU_GOLD_STRICT_NON_CINEMA = True
AIRPORTS_ALLOW_LABEL_FALLBACK = True
CARS_ALLOW_LABEL_FALLBACK = True
UNIVERSITIES_ALLOW_LABEL_FALLBACK = True
BOOKS_ALLOW_LABEL_FALLBACK = False
SMARTPHONES_ALLOW_LABEL_FALLBACK = True
VIDEOGAMES_ALLOW_LABEL_FALLBACK = True

def _keep_airport_fallback_label(label: str) -> Optional[str]:
    """
    Для аэропортов не режем gold слишком жёстко:
    у многих объектов нет популярного RU sitelink/label, но EN/local label вполне нормален.
    Иначе домен массово схлопывается в 0.
    """
    if label is None:
        return None
    s = str(label).strip()
    if not s:
        return None
    if _CINEMA_BAD_LABEL_RE.fullmatch(s):
        return None
    return s

def _keep_car_fallback_label(label: str) -> Optional[str]:
    """
    Для моделей автомобилей разрешаем латиницу/цифры:
    Toyota Camry, BMW E39, Peugeot 508 и т.п. — это нормальные gold labels,
    даже если у сущности нет сильного RU-анкера в Wikidata.
    """
    if label is None:
        return None
    s = str(label).strip()
    if not s:
        return None
    if _CINEMA_BAD_LABEL_RE.fullmatch(s):
        return None
    if re.fullmatch(r"Q\d+", s):
        return None
    if len(s) > 120:
        return None
    return s


def _keep_smartphone_fallback_label(label: str) -> Optional[str]:
    """
    Для смартфонов разрешаем валидные латинские названия моделей:
    iPhone 13 Pro, Galaxy S23, Pixel 8, Redmi Note 12 и т.п.
    Иначе домен схлопывается, потому что у многих моделей нет сильного RU-анкера.
    """
    if label is None:
        return None
    s = str(label).strip()
    if not s:
        return None
    if _CINEMA_BAD_LABEL_RE.fullmatch(s):
        return None
    if re.fullmatch(r"Q\d+", s):
        return None
    if len(s) < 2 or len(s) > 140:
        return None
    if not re.search(r"[A-Za-zА-Яа-я0-9]", s):
        return None
    return s


def _keep_videogame_fallback_label(label: str) -> Optional[str]:
    """
    Для видеоигр разрешаем хорошие латинские названия:
    Half-Life 2, Metal Gear Solid, Super Mario Odyssey, Baldur's Gate 3 и т.п.
    Иначе домен videogames после RU-нормализации легко схлопывается в 0.
    """
    if label is None:
        return None
    s = str(label).strip()
    if not s:
        return None
    if _CINEMA_BAD_LABEL_RE.fullmatch(s):
        return None
    if re.fullmatch(r"Q\d+", s):
        return None
    if len(s) < 2 or len(s) > 180:
        return None
    if not re.search(r"[A-Za-zА-Яа-я0-9]", s):
        return None
    return s


def _keep_university_fallback_label(label: str) -> Optional[str]:
    """
    Для университетов разрешаем fallback на исходный label:
    у многих международных университетов нет сильного RU-анкера в Wikidata,
    но EN/local label является нормальным gold answer.
    """
    if label is None:
        return None
    s = str(label).strip()
    if not s:
        return None
    if _CINEMA_BAD_LABEL_RE.fullmatch(s):
        return None
    if re.fullmatch(r"Q\d+", s):
        return None
    if len(s) < 3 or len(s) > 180:
        return None
    return s


def _keep_book_fallback_label(label: str) -> Optional[str]:
    """
    Для книг fallback отключён по смыслу датасета.
    Оставляем helper только как запасной кусок кода, но в books он не должен использоваться.
    """
    if label is None:
        return None
    s = str(label).strip()
    if not s:
        return None
    if _CINEMA_BAD_LABEL_RE.fullmatch(s):
        return None
    if re.fullmatch(r"Q\d+", s):
        return None
    if len(s) < 2 or len(s) > 220:
        return None
    return s

def enforce_ru_gold_labels_non_cinema(ex: BenchmarkExample) -> None:
    if not RU_GOLD_STRICT_NON_CINEMA:
        return
    if getattr(ex, "domain", None) == "cinema":
        return

    qids = list(ex.gold_answer_qids or [])
    lbls = list(ex.gold_answer_labels_ru or [])

    if not qids:
        return

    # align lengths (defensive)
    if len(lbls) < len(qids):
        lbls = lbls + [""] * (len(qids) - len(lbls))

    items = [(qids[i], lbls[i]) for i in range(len(qids))]

    # Fetch RU terms for ALL gold qids (cached on disk via wd.cache)
    uniq_qids = list(dict.fromkeys([q for q, _ in items if q]))
    ent_map = wd_get_entities_ru(uniq_qids) if uniq_qids else {}

    seen = set()
    new_items: List[Tuple[str, str]] = []
    domain_name = getattr(ex, "domain", None)
    is_airports = (domain_name == "airports")
    is_cars = (domain_name == "cars")
    is_universities = (domain_name == "universities")
    is_books = (domain_name == "books")
    is_smartphones = (domain_name == "smartphones")
    is_videogames = (domain_name == "videogames")

    for q, l in items:
        if not q or q in seen:
            continue
        seen.add(q)

        ent = ent_map.get(q)

        best = _best_popular_ru_name(ent, fallback_label="") if ent else None

        if best:
            new_items.append((q, best))
            continue

        if is_airports and AIRPORTS_ALLOW_LABEL_FALLBACK:
            keep = _keep_airport_fallback_label(l)
            if keep:
                new_items.append((q, keep))
                continue

        if is_cars and CARS_ALLOW_LABEL_FALLBACK:
            keep = _keep_car_fallback_label(l)
            if keep:
                new_items.append((q, keep))
                continue

        if is_universities and UNIVERSITIES_ALLOW_LABEL_FALLBACK:
            keep = _keep_university_fallback_label(l)
            if keep:
                new_items.append((q, keep))
                continue

        if is_books and BOOKS_ALLOW_LABEL_FALLBACK:
            keep = _keep_book_fallback_label(l)
            if keep:
                new_items.append((q, keep))
                continue

        if is_smartphones and SMARTPHONES_ALLOW_LABEL_FALLBACK:
            keep = _keep_smartphone_fallback_label(l)
            if keep:
                new_items.append((q, keep))
                continue

        if is_videogames and VIDEOGAMES_ALLOW_LABEL_FALLBACK:
            keep = _keep_videogame_fallback_label(l)
            if keep:
                new_items.append((q, keep))
                continue

        # If API failed for this qid, keep only if the original label already looks RU enough (has Cyrillic)
        try:
            if l and _cinema_has_cyrillic(str(l)):
                new_items.append((q, str(l).strip()))
        except Exception:
            # Drop labels that cannot be validated safely.
            pass

    ex.gold_answer_qids = [q for q, _ in new_items]
    ex.gold_answer_labels_ru = [l for _, l in new_items]

DOMAIN_GENERATORS = {
    "cinema": generate_cinema_example,
    "geo_ru": generate_geo_example,
    "geo": generate_geo_world_example,
    "books": generate_books_example,
    "videogames": generate_videogames_example,
    "music_albums": generate_music_albums_example,
    "software": generate_software_example,
    "people": generate_people_example_fast, 
    "scientists": generate_scientists_example,
    "physicists": generate_physicists_example,
    "mathematicians": generate_mathematicians_example,
    "paintings": generate_paintings_example,
    "museums": generate_museums_example,
    "spacecraft": generate_spacecraft_example,
    "universities": generate_universities_example,
    "airports": generate_airports_example,
    "cars": generate_cars_example,
    "countries": generate_countries_example,
    "dishes": generate_dishes_example,
    "smartphones": generate_smartphones_example,
}

TARGET_MAIN = {
    "dishes":         {"L1": 5,  "L2": 12, "L3": 15, "L4": 10, "L5": 8},
    "countries":      {"L1": 5,  "L2": 12, "L3": 10, "L4": 10, "L5": 2},
    "cinema":         {"L1": 0, "L2": 0, "L3": 5, "L4": 80, "L5": 40},
    "universities":   {"L1": 5,  "L2": 12, "L3": 15, "L4": 10, "L5": 8},
    "airports":       {"L1": 15,  "L2": 12, "L3": 15, "L4": 10, "L5": 8},
    "cars":           {"L1": 5,  "L2": 12, "L3": 15, "L4": 10, "L5": 8},
    "smartphones":    {"L1": 7,  "L2": 10, "L3": 15, "L4": 7, "L5": 5},
    "paintings":      {"L1": 0,  "L2": 0, "L3": 0, "L4": 0, "L5": 15},
    "books":          {"L1": 10,  "L2": 12, "L3": 15, "L4": 10, "L5": 8},
    "geo_ru":         {"L1": 10,  "L2": 15, "L3": 15, "L4": 15, "L5": 10},
    "geo":            {"L1": 0,  "L2": 0, "L3": 0, "L4": 15, "L5": 10},
    "spacecraft":     {"L1": 0,  "L2":  0, "L3": 0, "L4": 10, "L5": 20},
    "museums":        {"L1": 0,  "L2": 7, "L3": 10, "L4": 10, "L5": 10},
    "people":         {"L1": 5,  "L2": 12, "L3": 15, "L4": 10, "L5": 8},
    "scientists":     {"L1": 5,  "L2": 12, "L3": 15, "L4": 10, "L5": 8},
    "physicists":     {"L1": 5,  "L2": 12, "L3": 15, "L4": 10, "L5": 8},
    "mathematicians": {"L1": 5,  "L2": 12, "L3": 15, "L4": 10, "L5": 8},
    "software":       {"L1": 5,  "L2": 12, "L3": 15, "L4": 10, "L5": 8}, 
    "music_albums":   {"L1": 5,  "L2": 12, "L3": 15, "L4": 10, "L5": 8}, 
    "videogames":     {"L1": 5,  "L2": 12, "L3": 15, "L4": 10, "L5": 8}, 
}

def example_fingerprint(ex: BenchmarkExample) -> str:
    payload = {
        "domain": ex.domain,
        "complexity": ex.complexity,
        "constraints": ex.constraints,
        "requested_count": ex.requested_count,
    }
    return _sha1(json.dumps(payload, ensure_ascii=False, sort_keys=True))

def _safe_iter_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                break

def load_existing(path: str) -> Tuple[List[BenchmarkExample], set, Counter]:
    if not os.path.exists(path):
        return [], set(), Counter()
    exs: List[BenchmarkExample] = []
    seen = set()
    cnt = Counter()
    for obj in _safe_iter_jsonl(path):
        try:
            ex = BenchmarkExample(**obj)
        except Exception:
            continue
        fp = example_fingerprint(ex)
        seen.add(fp)
        cnt[(ex.domain, ex.complexity)] += 1
        exs.append(ex)
    return exs, seen, cnt

def append_jsonl(path: str, ex: BenchmarkExample, do_fsync: bool = False):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    line = json.dumps(asdict(ex), ensure_ascii=False) + "\n"
    with open(path, "a", encoding="utf-8") as f:
        f.write(line)
        f.flush()
        if do_fsync:
            os.fsync(f.fileno())

def min_gold_required(ex: BenchmarkExample) -> int:
    if ex.complexity in ("L1", "L2", "L3"):
        return int(ex.requested_count or 0)
    return 1

def zero_bucket_cap(domain: str, complexity: str) -> int:
    return 0

def build_main_and_zero(
    target_main: Dict[str, Dict[str, int]],
    out_main: str,
    out_zero: str,
    seed: int = 123,
    max_attempts_per_bucket: int = 2000,
    max_seconds_per_bucket: Optional[int] = 900,  
    fsync_each: bool = False,
    resume: bool = True,
    ui_every_s: float = 1.0,
):
    rng = random.Random(seed)

    if out_zero and os.path.exists(out_zero):
        os.remove(out_zero)

    if resume:
        main_ex, main_seen, main_cnt = load_existing(out_main)
    else:
        if os.path.exists(out_main):
            os.remove(out_main)
        main_ex, main_seen, main_cnt = [], set(), Counter()

    seen_all = set(main_seen)

    total_needed = sum(sum(plan.get(c, 0) for c in COMPLEXITY_ORDER) for plan in target_main.values())

    dropped_small = 0
    dropped_dup = 0
    dropped_zero = 0

    bucket_lines: Dict[str, str] = {}
    last_ui = 0.0
    global_start = time.time()

    def render(force: bool = False):
        nonlocal last_ui
        now = time.time()
        if not force and now - last_ui < ui_every_s:
            return
        last_ui = now

        done = sum(main_cnt.values())
        pct = (100.0 * done / total_needed) if total_needed else 0.0
        elapsed = int(now - global_start)

        lines = []
        lines.append(f"MAIN -> {out_main}")
        lines.append("ZERO -> disabled: empty gold lists are skipped")
        lines.append(f"MAIN total: {pct:5.1f}%  {done}/{total_needed}   elapsed={elapsed}s")
        lines.append(f"Stats: drop_zero={dropped_zero}  drop_small={dropped_small}  drop_dup={dropped_dup}")
        lines.append("")

        keys = list(bucket_lines.keys())
        tail = keys[-20:]
        for k in tail:
            lines.append(bucket_lines[k])

        clear_output(wait=True)
        print("\n".join(lines))

    render(force=True)

    for domain, plan in target_main.items():
        if domain not in DOMAIN_GENERATORS:
            continue
        gen = DOMAIN_GENERATORS[domain]

        for complexity in COMPLEXITY_ORDER:
            need_total = int(plan.get(complexity, 0))
            if need_total <= 0:
                continue

            have = main_cnt[(domain, complexity)]
            need = max(0, need_total - have)
            if need <= 0:
                continue

            bucket_key = f"{domain}:{complexity}"
            produced = 0
            attempts = 0
            start_ts = time.time()

            bucket_lines[bucket_key] = f"{bucket_key} 0/{need}  att=0  ok=0  drop_zero={dropped_zero}  drop_small={dropped_small}"
            render(force=True)

            while produced < need and attempts < max_attempts_per_bucket:
                if max_seconds_per_bucket is not None and (time.time() - start_ts) > max_seconds_per_bucket:
                    break

                attempts += 1
                bucket_lines[bucket_key] = (
                    f"{bucket_key} {produced}/{need}  att={attempts}  ok={produced}  "
                    f"drop_zero={dropped_zero}  drop_small={dropped_small}  drop_dup={dropped_dup}  "
                    f"t={int(time.time()-start_ts)}s"
                )
                render()

                idx = len(main_ex) + produced
                try:
                    ex = gen(complexity, idx, rng)
                    enforce_ru_gold_labels_non_cinema(ex)
                except Exception as e:
                    if globals().get("DEBUG_GENERATOR_ERRORS"):
                        print("[GENERATOR ERROR]", domain, complexity, repr(e))
                    continue

                fp = example_fingerprint(ex)
                if fp in seen_all:
                    dropped_dup += 1
                    continue

                g = len(ex.gold_answer_qids) if isinstance(ex.gold_answer_qids, list) else 0

                if g == 0:
                    dropped_zero += 1
                    continue

                if g < min_gold_required(ex):
                    dropped_small += 1
                    continue

                append_jsonl(out_main, ex, do_fsync=fsync_each)
                main_ex.append(ex)
                main_cnt[(domain, complexity)] += 1
                seen_all.add(fp)
                produced += 1

                bucket_lines[bucket_key] = (
                    f"{bucket_key} {produced}/{need}  att={attempts}  ok={produced}  "
                    f"drop_zero={dropped_zero}  drop_small={dropped_small}  drop_dup={dropped_dup}  "
                    f"t={int(time.time()-start_ts)}s"
                )
                render()

            if produced < need:
                extra = ""
                if max_seconds_per_bucket is not None and (time.time() - start_ts) > max_seconds_per_bucket:
                    extra = f", time_limit={max_seconds_per_bucket}s"
                bucket_lines[bucket_key] = f"[WARN] MAIN {bucket_key} need={need}, produced={produced} (attempts={attempts}{extra})"
                render(force=True)

    render(force=True)
    print("\nDone.")
    print("MAIN saved:", out_main, "count=", len(main_ex))
    print("ZERO disabled; skipped empty gold examples:", dropped_zero)
    return main_ex, []


main_examples, zero_examples = build_main_and_zero(
    TARGET_MAIN,
    out_main=DATASET_PATH_MAIN,
    out_zero=DATASET_PATH_ZERO,
    seed=123,
    max_attempts_per_bucket=25000,
    max_seconds_per_bucket=20000,  
    fsync_each=False,
    resume=True,
    ui_every_s=0.7,
)


MAIN -> out_wikidata_benchmark/dataset_main.jsonl
ZERO -> disabled: empty gold lists are skipped
MAIN total: 108.5%  1080/995   elapsed=0s
Stats: drop_zero=25000  drop_small=0  drop_dup=0

[WARN] MAIN cinema:L4 need=14, produced=0 (attempts=25000)
cinema:L5 0/6  att=0  ok=0  drop_zero=25000  drop_small=0
[1080 | Поп. 1/25] L5: Кристиан Бэйл, драма, 1-90 мин, imdb>=7.0 ... Ошибка API: 403
[1080 | Поп. 2/25] L5: Леонардо ДиКаприо, драма, 1-180 мин, imdb>=7.5 ... Ошибка API: 403
[1080 | Поп. 3/25] L5: Леонардо ДиКаприо, фантастика, 1-120 мин, imdb>=6.5 ... 

KeyboardInterrupt: 

<a id="summary"></a>

## 26. Summary

The notebook defines a reusable Wikidata-based pipeline for generating Russian multihop benchmark examples across multiple domains. It keeps the shared WDQS/cache utilities, domain-specific generators, Russian-label postprocessing, deduplication, and append-only JSONL assembly in separate sections. The expected output is a main dataset file with non-zero-gold examples that can be manually validated and then used for LLM evaluation.